In [11]:
from __future__ import annotations

import hashlib
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

VERIFY_LARGE_FIELD_HASH = True
EXPECTED_STRUCTURAL_TYPE = "W2"
EXPECTED_PERIOD_S = 0.40
EXPECTED_DESIGN_LEVELS = ["HighCode", "ModerateCode", "LowCode", "PreCode"]
EXPECTED_DAMAGE_STATES = ["Slight", "Moderate", "Extensive", "Complete"]


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "04_generate_ground_motion_fields.ipynb").exists():
            return candidate
        if (candidate / "data" / "metadata" / "notebook_4_final_handoff").exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run this notebook from the "
        "seismic-correlation-insurance-loss repository."
    )


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def normalize_structural_type(value: Any) -> str:
    return normalize_text(value).upper().replace(" ", "")


def normalize_occupancy(value: Any) -> str:
    text = normalize_text(value).upper().replace(" ", "")
    if not text:
        return ""
    if text.startswith("RES1-"):
        return "RES1"
    return text.split("-", 1)[0] if text.startswith("RES1") else text


def first_existing_column(columns: Iterable[str], aliases: Iterable[str]) -> str | None:
    lookup = {str(column).strip().casefold(): str(column) for column in columns}
    for alias in aliases:
        match = lookup.get(alias.casefold())
        if match is not None:
            return match
    return None


def resolve_one_file(
    preferred_folder: Path,
    aliases: list[str],
    label: str,
    fallback_root: Path | None = None,
) -> Path:
    normalized_aliases = {name.strip().casefold() for name in aliases}

    search_roots: list[Path] = []
    if preferred_folder.exists():
        search_roots.append(preferred_folder)
    if fallback_root is not None and fallback_root.exists():
        if fallback_root.resolve() not in {path.resolve() for path in search_roots}:
            search_roots.append(fallback_root)

    if not search_roots:
        raise FileNotFoundError(
            f"Neither the preferred folder nor fallback reference folder exists for "
            f"{label}: {preferred_folder}"
        )

    for root in search_roots:
        files = [path for path in root.rglob("*") if path.is_file()]
        matches = [
            path
            for path in files
            if path.name.strip().casefold() in normalized_aliases
        ]
        unique_matches = sorted({path.resolve() for path in matches})
        if len(unique_matches) == 1:
            resolved = Path(unique_matches[0])
            print(f"Located {label}: {resolved}")
            return resolved
        if len(unique_matches) > 1:
            raise RuntimeError(
                f"Multiple candidate files found for {label}: "
                f"{[str(path) for path in unique_matches]}"
            )

    available = sorted(
        str(path.relative_to(fallback_root))
        for path in fallback_root.rglob("*.xlsx")
        if path.is_file()
    ) if fallback_root is not None and fallback_root.exists() else []
    raise FileNotFoundError(
        f"Could not find {label}. Expected one of: {aliases}. "
        f"Excel files currently under {fallback_root}: {available}"
    )


def locate_handoff(project_root: Path) -> Path:
    candidates = [
        project_root
        / "data"
        / "metadata"
        / "notebook_4_final_handoff"
        / "notebook_5_input_handoff.json",
        project_root / "data" / "metadata" / "notebook_5_input_handoff.json",
    ]
    existing = [path for path in candidates if path.exists()]
    if len(existing) != 1:
        raise FileNotFoundError(
            "Expected exactly one Notebook 5 handoff file. Checked: "
            + ", ".join(str(path) for path in candidates)
        )
    return existing[0]


def resolve_handoff_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    recorded = recorded_path.replace("\\", "/")
    marker = "/data/"
    lower = recorded.lower()
    pos = lower.find(marker)
    if pos >= 0:
        relative = recorded[pos + 1 :]
        candidate = project_root / Path(relative)
        if candidate.exists():
            return candidate

    candidate = project_root / Path(recorded).name
    if candidate.exists():
        return candidate

    raise FileNotFoundError(f"Handoff path does not exist: {recorded_path}")


def clean_excel_table(path: Path, sheet_name: str | int = 0) -> pd.DataFrame:
    table = pd.read_excel(path, sheet_name=sheet_name)
    table = table.dropna(axis=1, how="all").dropna(axis=0, how="all")
    table.columns = [normalize_text(column) for column in table.columns]
    return table.reset_index(drop=True)


def inspect_fragility_workbook(
    path: Path,
    component: str,
    demand_type: str,
    median_unit: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    workbook = pd.ExcelFile(path)
    actual_sheets = list(workbook.sheet_names)
    if actual_sheets != EXPECTED_DESIGN_LEVELS:
        raise ValueError(
            f"{component} fragility sheets are {actual_sheets}; expected "
            f"{EXPECTED_DESIGN_LEVELS} in that order."
        )

    inventory_rows: list[dict[str, Any]] = []
    w2_rows: list[dict[str, Any]] = []

    for design_level in EXPECTED_DESIGN_LEVELS:
        table = clean_excel_table(path, design_level)
        slight_median_column = "sli_med" if "sli_med" in table.columns else "sli_mod"
        required = {
            "struct_typ",
            slight_median_column,
            "sli_beta",
            "mod_med",
            "mod_beta",
            "ext_med",
            "ext_beta",
            "com_med",
            "com_beta",
        }
        missing = sorted(required.difference(table.columns))
        if missing:
            raise ValueError(
                f"{component} fragility sheet {design_level} is missing columns: {missing}"
            )

        table["struct_typ_normalized"] = table["struct_typ"].map(normalize_structural_type)
        if table["struct_typ_normalized"].eq("").any():
            raise ValueError(f"Blank structural type found in {component} {design_level}.")
        if table["struct_typ_normalized"].duplicated().any():
            duplicates = sorted(
                table.loc[
                    table["struct_typ_normalized"].duplicated(keep=False),
                    "struct_typ_normalized",
                ].unique()
            )
            raise ValueError(
                f"Duplicate structural types in {component} {design_level}: {duplicates}"
            )

        median_columns = [slight_median_column, "mod_med", "ext_med", "com_med"]
        beta_columns = ["sli_beta", "mod_beta", "ext_beta", "com_beta"]
        numeric_columns = median_columns + beta_columns
        for column in numeric_columns:
            table[column] = pd.to_numeric(table[column], errors="coerce")
        if table[numeric_columns].isna().any().any():
            raise ValueError(f"Non-numeric fragility parameter in {component} {design_level}.")
        if (table[median_columns] <= 0).any().any():
            raise ValueError(f"Non-positive fragility median in {component} {design_level}.")
        if (table[beta_columns] <= 0).any().any():
            raise ValueError(f"Non-positive fragility beta in {component} {design_level}.")

        medians = table[median_columns].to_numpy(dtype=float)
        if not np.all(np.diff(medians, axis=1) > 0):
            raise ValueError(
                f"Fragility medians are not strictly ordered in {component} {design_level}."
            )

        w2 = table.loc[table["struct_typ_normalized"] == EXPECTED_STRUCTURAL_TYPE]
        if len(w2) != 1:
            raise ValueError(
                f"Expected exactly one W2 row in {component} {design_level}; found {len(w2)}."
            )
        w2 = w2.iloc[0]

        inventory_rows.append(
            {
                "component": component,
                "source_path": str(path),
                "sheet_name": design_level,
                "rows": int(len(table)),
                "structural_types": int(table["struct_typ_normalized"].nunique()),
                "slight_median_source_column": slight_median_column,
                "demand_type": demand_type,
                "median_unit": median_unit,
                "w2_present": True,
                "all_medians_positive": True,
                "all_betas_positive": True,
                "all_medians_strictly_ordered": True,
            }
        )

        source_medians = [
            float(w2[slight_median_column]),
            float(w2["mod_med"]),
            float(w2["ext_med"]),
            float(w2["com_med"]),
        ]
        source_betas = [
            float(w2["sli_beta"]),
            float(w2["mod_beta"]),
            float(w2["ext_beta"]),
            float(w2["com_beta"]),
        ]

        for state, median, beta in zip(
            EXPECTED_DAMAGE_STATES, source_medians, source_betas
        ):
            w2_rows.append(
                {
                    "component": component,
                    "structural_type": EXPECTED_STRUCTURAL_TYPE,
                    "design_level": design_level,
                    "damage_state": state,
                    "source_median": median,
                    "source_beta_ln": beta,
                    "source_demand_type": demand_type,
                    "source_median_unit": median_unit,
                    "source_path": str(path),
                    "source_sheet": design_level,
                    "transformed_for_production": False,
                }
            )

    return pd.DataFrame(inventory_rows), pd.DataFrame(w2_rows)


def expand_repair_ratio_table(path: Path, component: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = clean_excel_table(path)
    required = {"occ_type", "DS_0", "DS_1", "DS_2", "DS_3"}
    missing = sorted(required.difference(raw.columns))
    if missing:
        raise ValueError(f"{component} repair-cost table is missing columns: {missing}")

    for column in ["DS_0", "DS_1", "DS_2", "DS_3"]:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")
    if raw[["DS_0", "DS_1", "DS_2", "DS_3"]].isna().any().any():
        raise ValueError(f"Non-numeric value in {component} repair-cost ratios.")
    if (raw[["DS_0", "DS_1", "DS_2", "DS_3"]] < 0).any().any():
        raise ValueError(f"Negative value in {component} repair-cost ratios.")

    expanded_rows: list[dict[str, Any]] = []
    for _, row in raw.iterrows():
        key = normalize_occupancy(row["occ_type"])
        original_key = normalize_text(row["occ_type"]).upper().replace(" ", "")
        keys = [key]
        range_match = re.fullmatch(r"([A-Z]+\d+)([A-Z])-([A-Z])", original_key)
        if range_match:
            base, start, end = range_match.groups()
            keys = [base + chr(code) for code in range(ord(start), ord(end) + 1)]

        for occupancy in keys:
            expanded_rows.append(
                {
                    "component": component,
                    "occ_type": occupancy,
                    "source_occ_type": original_key,
                    "slight_percent": float(row["DS_0"]),
                    "moderate_percent": float(row["DS_1"]),
                    "extensive_percent": float(row["DS_2"]),
                    "complete_percent": float(row["DS_3"]),
                    "source_path": str(path),
                }
            )

    expanded = pd.DataFrame(expanded_rows)
    duplicate_groups = expanded.groupby("occ_type", sort=True)
    conflicts: list[str] = []
    keep_rows: list[pd.Series] = []
    ratio_columns = [
        "slight_percent",
        "moderate_percent",
        "extensive_percent",
        "complete_percent",
    ]
    for occupancy, group in duplicate_groups:
        unique_ratios = group[ratio_columns].drop_duplicates()
        if len(unique_ratios) > 1:
            conflicts.append(occupancy)
        keep_rows.append(group.iloc[-1])
    if conflicts:
        raise ValueError(
            f"Conflicting duplicate occupancy ratios in {component}: {sorted(conflicts)}"
        )

    expanded = pd.DataFrame(keep_rows).sort_values("occ_type").reset_index(drop=True)
    return raw, expanded


def inspect_replacement_costs(
    general_path: Path, res1_path: Path
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    general = clean_excel_table(general_path)
    if "occ_type" not in general.columns:
        raise ValueError("General replacement-cost table is missing occ_type.")

    cost_aliases = [
        "Structure Replacement Cost ($/ft²)",
        "Structure Replacement Cost ($/ft2)",
        "cost_per_sqft",
        "Cost per sqft",
        "Cost ($/ft²)",
        "Cost ($/ft2)",
    ]
    cost_column = first_existing_column(general.columns, cost_aliases)
    if cost_column is None:
        candidates = [column for column in general.columns if "cost" in column.casefold()]
        if len(candidates) == 1:
            cost_column = candidates[0]
        else:
            raise ValueError(
                f"Could not identify the general replacement-cost column. Found: {list(general.columns)}"
            )

    general["occ_type_normalized"] = general["occ_type"].map(normalize_occupancy)
    general["cost_per_sqft"] = pd.to_numeric(general[cost_column], errors="coerce")
    non_res1 = general["occ_type_normalized"] != "RES1"
    if general.loc[non_res1, "cost_per_sqft"].isna().any():
        missing = general.loc[non_res1 & general["cost_per_sqft"].isna(), "occ_type_normalized"]
        raise ValueError(
            f"Missing general replacement costs for occupancy classes: {sorted(missing.tolist())}"
        )
    if (general.loc[non_res1, "cost_per_sqft"] <= 0).any():
        raise ValueError("Non-positive value in the general replacement-cost table.")
    if general["occ_type_normalized"].duplicated().any():
        duplicates = sorted(
            general.loc[
                general["occ_type_normalized"].duplicated(keep=False),
                "occ_type_normalized",
            ].unique()
        )
        raise ValueError(f"Duplicate occupancy classes in replacement costs: {duplicates}")

    res1 = clean_excel_table(res1_path)
    required_res1 = {
        "Construction Class",
        "Height Class",
        "2022 Avg Cost per ft² (No Basement)",
        "2022 Avg Cost per ft² (Finished Basement)",
        "2022 Avg Cost per ft² (Unfinished Basement)",
    }
    missing_res1 = sorted(required_res1.difference(res1.columns))
    if missing_res1:
        raise ValueError(f"RES1 replacement-cost table is missing columns: {missing_res1}")

    res1_cost_columns = [column for column in res1.columns if "Cost per ft²" in column]
    for column in res1_cost_columns:
        res1[column] = pd.to_numeric(res1[column], errors="coerce")
    if res1[res1_cost_columns].isna().any().any():
        raise ValueError("Non-numeric value in the RES1 replacement-cost table.")
    if (res1[res1_cost_columns] <= 0).any().any():
        raise ValueError("Non-positive value in the RES1 replacement-cost table.")
    if res1.duplicated(["Construction Class", "Height Class"]).any():
        raise ValueError("Duplicate Construction Class and Height Class rows in RES1 costs.")

    return general, res1, cost_column


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


PROJECT_ROOT = find_project_root()
HANDOFF_PATH = locate_handoff(PROJECT_ROOT)
HANDOFF = load_json(HANDOFF_PATH)

REFERENCE_ROOT = PROJECT_ROOT / "data" / "reference" / "hazus"
FRAGILITY_DIR = REFERENCE_ROOT / "fragility"
REPAIR_DIR = REFERENCE_ROOT / "repair_cost_ratios"
REPLACEMENT_DIR = REFERENCE_ROOT / "replacement_costs"
README_PATH = REFERENCE_ROOT / "README.md"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

reference_paths = {
    "structural_fragility": resolve_one_file(
        FRAGILITY_DIR,
        ["HAZUS_W2_Structural_Fragility.xlsx", "HAZUS_Structural_Fragility_Source.xlsx", "FragilityCurves_EQ.xlsx"],
        "structural fragility workbook",
        fallback_root=REFERENCE_ROOT,
    ),
    "nsd_fragility": resolve_one_file(
        FRAGILITY_DIR,
        [
            "HAZUS_Nonstructural_Drift_Sensitive_Fragility.xlsx",
            "FragilityCurves_NS_EQ_.xlsx",
        ],
        "drift-sensitive nonstructural fragility workbook",
        fallback_root=REFERENCE_ROOT,
    ),
    "nsa_fragility": resolve_one_file(
        FRAGILITY_DIR,
        [
            "HAZUS_Nonstructural_Acceleration_Sensitive_Fragility.xlsx",
            "FragilityCurves_NS_EQ_accelerationsensitive.xlsx",
            "FragilityCurves_NS_EQ_accelerationsensitive .xlsx",
        ],
        "acceleration-sensitive nonstructural fragility workbook",
        fallback_root=REFERENCE_ROOT,
    ),
    "structural_repair_ratios": resolve_one_file(
        REPAIR_DIR,
        ["HAZUS_Structural_Repair_Cost_Ratios.xlsx"],
        "structural repair-cost-ratio workbook",
        fallback_root=REFERENCE_ROOT,
    ),
    "nsd_repair_ratios": resolve_one_file(
        REPAIR_DIR,
        ["HAZUS_Drift_Sensitive_Nonstructural_Repair_Costs.xlsx"],
        "drift-sensitive repair-cost-ratio workbook",
        fallback_root=REFERENCE_ROOT,
    ),
    "nsa_repair_ratios": resolve_one_file(
        REPAIR_DIR,
        ["HAZUS_Acceleration_Sensitive_Nonstructural_Repair_Costs.xlsx"],
        "acceleration-sensitive repair-cost-ratio workbook",
        fallback_root=REFERENCE_ROOT,
    ),
    "general_replacement_costs": resolve_one_file(
        REPLACEMENT_DIR,
        ["HAZUS_Structure_Replacement_Costs.xlsx"],
        "general replacement-cost workbook",
        fallback_root=REFERENCE_ROOT,
    ),
    "res1_replacement_costs": resolve_one_file(
        REPLACEMENT_DIR,
        ["HAZUS_RES1_Replacement_Costs.xlsx"],
        "RES1 replacement-cost workbook",
        fallback_root=REFERENCE_ROOT,
    ),
}

validation_rows: list[dict[str, Any]] = []
append_check(validation_rows, "handoff_exists", HANDOFF_PATH.exists(), str(HANDOFF_PATH))
append_check(
    validation_rows,
    "notebook4_complete",
    HANDOFF.get("notebook4_complete") is True,
    f"notebook4_complete={HANDOFF.get('notebook4_complete')}",
)
append_check(
    validation_rows,
    "notebook4_validation_passed",
    HANDOFF.get("validation", {}).get("all_checks_passed") is True,
    f"checks={HANDOFF.get('validation', {}).get('checks')}",
)
append_check(
    validation_rows,
    "catalog_duration_positive",
    int(HANDOFF.get("annual_catalog", {}).get("declared_duration_years", 0)) > 0,
    f"years={HANDOFF.get('annual_catalog', {}).get('declared_duration_years')}",
)
append_check(
    validation_rows,
    "reference_readme_exists",
    README_PATH.exists(),
    str(README_PATH),
)

primary_input = resolve_handoff_path(
    PROJECT_ROOT, HANDOFF["notebook5_primary_input"]["path"]
)
portfolio_path = resolve_handoff_path(PROJECT_ROOT, HANDOFF["portfolio"]["path"])

append_check(validation_rows, "primary_field_exists", primary_input.exists(), str(primary_input))
append_check(validation_rows, "portfolio_exists", portfolio_path.exists(), str(portfolio_path))

field_header = pd.read_csv(primary_input, nrows=5)
expected_field_columns = HANDOFF["notebook5_primary_input"]["available_columns"]
missing_field_columns = sorted(set(expected_field_columns).difference(field_header.columns))
append_check(
    validation_rows,
    "primary_field_schema_matches_handoff",
    not missing_field_columns,
    f"missing_columns={missing_field_columns}",
)

field_hash_verified = False
field_hash = None
if VERIFY_LARGE_FIELD_HASH:
    field_hash = sha256_file(primary_input)
    expected_field_hash = HANDOFF["notebook5_primary_input"]["sha256"]
    field_hash_verified = field_hash == expected_field_hash
    append_check(
        validation_rows,
        "primary_field_hash_matches_handoff",
        field_hash_verified,
        f"actual={field_hash}; expected={expected_field_hash}",
    )
else:
    append_check(
        validation_rows,
        "primary_field_hash_matches_handoff",
        True,
        "Not recalculated in Cell 1 because VERIFY_LARGE_FIELD_HASH=False; Cell 20 hash retained.",
        severity="informational",
    )

portfolio_hash = sha256_file(portfolio_path)
append_check(
    validation_rows,
    "portfolio_hash_matches_handoff",
    portfolio_hash == HANDOFF["portfolio"]["sha256"],
    f"actual={portfolio_hash}; expected={HANDOFF['portfolio']['sha256']}",
)

portfolio = pd.read_csv(portfolio_path)
site_column = HANDOFF["notebook5_primary_input"]["site_key"]
append_check(
    validation_rows,
    "portfolio_row_count_matches_handoff",
    len(portfolio) == int(HANDOFF["portfolio"]["sites"]),
    f"rows={len(portfolio)}; expected={HANDOFF['portfolio']['sites']}",
)
append_check(
    validation_rows,
    "portfolio_site_key_present",
    site_column in portfolio.columns,
    f"site_key={site_column}",
)
if site_column not in portfolio.columns:
    raise KeyError(f"Portfolio is missing the required site key: {site_column}")
append_check(
    validation_rows,
    "portfolio_site_key_complete_unique",
    not portfolio[site_column].isna().any() and portfolio[site_column].is_unique,
    f"nulls={int(portfolio[site_column].isna().sum())}; unique={portfolio[site_column].is_unique}",
)

column_aliases = {
    "structural_type": ["struct_typ", "structural_type", "structure_type", "bldg_type"],
    "occupancy": ["occtype", "occ_type", "occupancy", "occupancy_type"],
    "floor_area_sqft": ["sqft", "floor_area_sqft", "area_sqft", "building_sqft"],
    "year_built": ["med_yr_blt", "year_built", "built_year", "construction_year"],
    "period_s": ["period_s", "Period", "period", "fundamental_period_s"],
    "vs30_mps": ["vs30_mps", "vs30", "Vs30"],
}
resolved_columns = {
    role: first_existing_column(portfolio.columns, aliases)
    for role, aliases in column_aliases.items()
}

for role in ["structural_type", "occupancy", "floor_area_sqft", "year_built"]:
    append_check(
        validation_rows,
        f"portfolio_{role}_column_present",
        resolved_columns[role] is not None,
        f"resolved_column={resolved_columns[role]}; aliases={column_aliases[role]}",
    )

missing_required_roles = [
    role
    for role in ["structural_type", "occupancy", "floor_area_sqft", "year_built"]
    if resolved_columns[role] is None
]
if missing_required_roles:
    raise KeyError(
        "The GMM-ready portfolio lacks Notebook 5 attributes: "
        f"{missing_required_roles}. Available columns: {list(portfolio.columns)}"
    )

structural_column = resolved_columns["structural_type"]
occupancy_column = resolved_columns["occupancy"]
floor_area_column = resolved_columns["floor_area_sqft"]
year_built_column = resolved_columns["year_built"]

portfolio["structural_type_normalized"] = portfolio[structural_column].map(
    normalize_structural_type
)
portfolio["occupancy_normalized"] = portfolio[occupancy_column].map(normalize_occupancy)
portfolio["floor_area_sqft_numeric"] = pd.to_numeric(
    portfolio[floor_area_column], errors="coerce"
)
portfolio["year_built_numeric"] = pd.to_numeric(
    portfolio[year_built_column], errors="coerce"
)

append_check(
    validation_rows,
    "portfolio_all_w2",
    set(portfolio["structural_type_normalized"].unique()) == {EXPECTED_STRUCTURAL_TYPE},
    f"values={sorted(portfolio['structural_type_normalized'].unique().tolist())}",
)
append_check(
    validation_rows,
    "portfolio_occupancy_complete",
    not portfolio["occupancy_normalized"].eq("").any(),
    f"blank_count={int(portfolio['occupancy_normalized'].eq('').sum())}",
)
append_check(
    validation_rows,
    "portfolio_floor_area_positive",
    not portfolio["floor_area_sqft_numeric"].isna().any()
    and (portfolio["floor_area_sqft_numeric"] > 0).all(),
    f"missing={int(portfolio['floor_area_sqft_numeric'].isna().sum())}; "
    f"nonpositive={int((portfolio['floor_area_sqft_numeric'].fillna(0) <= 0).sum())}",
)
append_check(
    validation_rows,
    "portfolio_year_built_complete",
    not portfolio["year_built_numeric"].isna().any(),
    f"missing={int(portfolio['year_built_numeric'].isna().sum())}",
)
append_check(
    validation_rows,
    "portfolio_year_built_plausible",
    portfolio["year_built_numeric"].between(1700, datetime.now().year).all(),
    f"min={portfolio['year_built_numeric'].min()}; max={portfolio['year_built_numeric'].max()}",
)

if resolved_columns["period_s"] is not None:
    period_values = pd.to_numeric(portfolio[resolved_columns["period_s"]], errors="coerce")
    period_ok = (
        not period_values.isna().any()
        and np.allclose(period_values.to_numpy(), EXPECTED_PERIOD_S, rtol=0.0, atol=1e-12)
    )
    append_check(
        validation_rows,
        "portfolio_period_is_exact_sa0p4",
        period_ok,
        f"column={resolved_columns['period_s']}; unique={sorted(period_values.unique().tolist())}",
    )
else:
    append_check(
        validation_rows,
        "portfolio_period_is_exact_sa0p4",
        math.isclose(float(HANDOFF["portfolio"]["period_s"]), EXPECTED_PERIOD_S),
        "No period column in portfolio; accepted handoff period used.",
        severity="informational",
    )

reference_inventory_rows: list[dict[str, Any]] = []
for role, path in reference_paths.items():
    reference_inventory_rows.append(
        {
            "role": role,
            "path": str(path),
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "modified_at_utc": datetime.fromtimestamp(
                path.stat().st_mtime, tz=timezone.utc
            ).isoformat(),
        }
    )
append_check(
    validation_rows,
    "all_reference_files_resolved",
    len(reference_inventory_rows) == 8,
    f"resolved={len(reference_inventory_rows)}",
)

fragility_inventory_parts: list[pd.DataFrame] = []
w2_source_parts: list[pd.DataFrame] = []
fragility_specs = [
    (
        reference_paths["structural_fragility"],
        "structural",
        "spectral_displacement",
        "in",
    ),
    (
        reference_paths["nsd_fragility"],
        "nonstructural_drift_sensitive",
        "spectral_displacement",
        "in",
    ),
    (
        reference_paths["nsa_fragility"],
        "nonstructural_acceleration_sensitive",
        "spectral_acceleration",
        "g",
    ),
]
for path, component, demand_type, unit in fragility_specs:
    inventory, w2_source = inspect_fragility_workbook(
        path, component, demand_type, unit
    )
    fragility_inventory_parts.append(inventory)
    w2_source_parts.append(w2_source)

fragility_inventory = pd.concat(fragility_inventory_parts, ignore_index=True)
w2_source_fragility = pd.concat(w2_source_parts, ignore_index=True)
append_check(
    validation_rows,
    "w2_fragility_source_rows_complete",
    len(w2_source_fragility) == 3 * 4 * 4,
    f"rows={len(w2_source_fragility)}; expected=48",
)

structural_raw, structural_ratios = expand_repair_ratio_table(
    reference_paths["structural_repair_ratios"], "structural"
)
nsd_raw, nsd_ratios = expand_repair_ratio_table(
    reference_paths["nsd_repair_ratios"], "nonstructural_drift_sensitive"
)
nsa_raw, nsa_ratios = expand_repair_ratio_table(
    reference_paths["nsa_repair_ratios"], "nonstructural_acceleration_sensitive"
)
general_replacement, res1_replacement, general_cost_column = inspect_replacement_costs(
    reference_paths["general_replacement_costs"],
    reference_paths["res1_replacement_costs"],
)

portfolio_occupancies = sorted(portfolio["occupancy_normalized"].unique().tolist())
coverage_sets = {
    "structural_repair": set(structural_ratios["occ_type"]),
    "nsd_repair": set(nsd_ratios["occ_type"]),
    "nsa_repair": set(nsa_ratios["occ_type"]),
    "replacement": set(general_replacement["occ_type_normalized"]),
}
coverage_sets["replacement"].add("RES1")

coverage_rows: list[dict[str, Any]] = []
for occupancy in portfolio_occupancies:
    row = {"occ_type": occupancy}
    for table_name, values in coverage_sets.items():
        row[f"covered_by_{table_name}"] = occupancy in values
    row["all_required_tables_covered"] = all(
        row[f"covered_by_{name}"] for name in coverage_sets
    )
    coverage_rows.append(row)
occupancy_coverage = pd.DataFrame(coverage_rows)
append_check(
    validation_rows,
    "portfolio_occupancies_covered_by_all_hazus_tables",
    occupancy_coverage["all_required_tables_covered"].all(),
    "uncovered="
    + str(
        occupancy_coverage.loc[
            ~occupancy_coverage["all_required_tables_covered"], "occ_type"
        ].tolist()
    ),
)

complete_component_audit = (
    structural_ratios[["occ_type", "complete_percent"]]
    .rename(columns={"complete_percent": "structural_complete_percent"})
    .merge(
        nsd_ratios[["occ_type", "complete_percent"]].rename(
            columns={"complete_percent": "nsd_complete_percent"}
        ),
        on="occ_type",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        nsa_ratios[["occ_type", "complete_percent"]].rename(
            columns={"complete_percent": "nsa_complete_percent"}
        ),
        on="occ_type",
        how="outer",
        validate="one_to_one",
    )
)
complete_component_audit["complete_component_total_percent"] = complete_component_audit[
    [
        "structural_complete_percent",
        "nsd_complete_percent",
        "nsa_complete_percent",
    ]
].sum(axis=1, min_count=3)
complete_component_audit["difference_from_100_percentage_points"] = (
    complete_component_audit["complete_component_total_percent"] - 100.0
)
complete_component_audit["within_0p1_percentage_point"] = complete_component_audit[
    "difference_from_100_percentage_points"
].abs() <= 0.1
complete_component_audit["used_by_portfolio"] = complete_component_audit[
    "occ_type"
].isin(portfolio_occupancies)

component_total_warnings = complete_component_audit.loc[
    complete_component_audit["used_by_portfolio"]
    & ~complete_component_audit["within_0p1_percentage_point"],
    ["occ_type", "complete_component_total_percent"],
]
append_check(
    validation_rows,
    "portfolio_complete_component_totals_reconcile",
    component_total_warnings.empty,
    f"portfolio_warnings={component_total_warnings.to_dict(orient='records')}",
    severity="warning",
)

portfolio_schema_rows: list[dict[str, Any]] = []
for column in portfolio.columns:
    series = portfolio[column]
    sample_values = [normalize_text(value) for value in series.dropna().head(3).tolist()]
    portfolio_schema_rows.append(
        {
            "column": column,
            "dtype": str(series.dtype),
            "non_null_rows": int(series.notna().sum()),
            "null_rows": int(series.isna().sum()),
            "unique_non_null_values": int(series.nunique(dropna=True)),
            "sample_values": " | ".join(sample_values),
        }
    )
portfolio_schema = pd.DataFrame(portfolio_schema_rows)

occupancy_counts = (
    portfolio.groupby("occupancy_normalized", dropna=False)
    .agg(
        buildings=(site_column, "size"),
        total_floor_area_sqft=("floor_area_sqft_numeric", "sum"),
        minimum_year_built=("year_built_numeric", "min"),
        maximum_year_built=("year_built_numeric", "max"),
    )
    .reset_index()
    .rename(columns={"occupancy_normalized": "occ_type"})
    .sort_values("occ_type")
)

reference_inventory = pd.DataFrame(reference_inventory_rows).sort_values("role")
validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    (validation["severity"] == "critical") & ~validation["passed"]
]
warning_failures = validation.loc[
    (validation["severity"] == "warning") & ~validation["passed"]
]

paths = {
    "reference_inventory": METADATA_DIR / "notebook_5_cell_1_reference_inventory.csv",
    "fragility_inventory": METADATA_DIR / "notebook_5_cell_1_fragility_inventory.csv",
    "w2_source_fragility": OUTPUT_DIR / "notebook_5_w2_source_fragility_inventory.csv",
    "portfolio_schema": METADATA_DIR / "notebook_5_cell_1_portfolio_schema.csv",
    "portfolio_occupancy_counts": METADATA_DIR
    / "notebook_5_cell_1_portfolio_occupancy_counts.csv",
    "occupancy_coverage": METADATA_DIR
    / "notebook_5_cell_1_hazus_occupancy_coverage.csv",
    "component_audit": METADATA_DIR
    / "notebook_5_cell_1_complete_cost_component_audit.csv",
    "validation": METADATA_DIR / "notebook_5_cell_1_validation.csv",
    "summary": METADATA_DIR / "notebook_5_cell_1_summary.json",
}

reference_inventory.to_csv(paths["reference_inventory"], index=False)
fragility_inventory.to_csv(paths["fragility_inventory"], index=False)
w2_source_fragility.to_csv(paths["w2_source_fragility"], index=False)
portfolio_schema.to_csv(paths["portfolio_schema"], index=False)
occupancy_counts.to_csv(paths["portfolio_occupancy_counts"], index=False)
occupancy_coverage.to_csv(paths["occupancy_coverage"], index=False)
complete_component_audit.to_csv(paths["component_audit"], index=False)
validation.to_csv(paths["validation"], index=False)

summary = {
    "pipeline_version": "notebook5_cell1_input_validation_v1",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook4_handoff_path": str(HANDOFF_PATH),
    "notebook4_complete": HANDOFF.get("notebook4_complete") is True,
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int((validation["severity"] == "critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "annual_catalog": {
        "declared_duration_years": int(
            HANDOFF["annual_catalog"]["declared_duration_years"]
        ),
        "occurrences": int(HANDOFF["annual_catalog"]["occurrences"]),
        "occupied_years": int(HANDOFF["annual_catalog"]["occupied_years"]),
        "zero_event_years": int(HANDOFF["annual_catalog"]["zero_event_years"]),
    },
    "primary_ground_motion_fields": {
        "path": str(primary_input),
        "rows_from_handoff": int(HANDOFF["notebook5_primary_input"]["rows"]),
        "structural_damage_im_column": HANDOFF["notebook5_primary_input"][
            "structural_damage_im_column"
        ],
        "period_s": float(
            HANDOFF["notebook5_primary_input"]["structural_damage_im_period_s"]
        ),
        "sha256": field_hash
        if field_hash is not None
        else HANDOFF["notebook5_primary_input"]["sha256"],
        "hash_recalculated_in_cell1": bool(VERIFY_LARGE_FIELD_HASH),
        "hash_verified": bool(field_hash_verified)
        if VERIFY_LARGE_FIELD_HASH
        else None,
    },
    "portfolio": {
        "path": str(portfolio_path),
        "rows": int(len(portfolio)),
        "site_key": site_column,
        "resolved_columns": resolved_columns,
        "structural_types": sorted(
            portfolio["structural_type_normalized"].unique().tolist()
        ),
        "occupancy_types": portfolio_occupancies,
        "total_floor_area_sqft": float(portfolio["floor_area_sqft_numeric"].sum()),
        "minimum_year_built": float(portfolio["year_built_numeric"].min()),
        "maximum_year_built": float(portfolio["year_built_numeric"].max()),
        "sha256": portfolio_hash,
    },
    "hazus_reference_files": reference_inventory.to_dict(orient="records"),
    "fragility_interpretation": {
        "structural_source_demand": "spectral_displacement",
        "structural_source_unit": "in",
        "nsd_source_demand": "spectral_displacement",
        "nsd_source_unit": "in",
        "nsa_source_demand": "spectral_acceleration",
        "nsa_source_unit": "g",
        "direct_sa0p4_structural_table_generated": False,
        "next_required_transformation": (
            "Convert the accepted W2 structural spectral-displacement medians to "
            "equivalent direct SA0P4 medians at T=0.40 s, retain the source betas, "
            "and validate the transformation before sampling damage."
        ),
    },
    "replacement_costs": {
        "general_cost_column": general_cost_column,
        "general_rows": int(len(general_replacement)),
        "res1_rows": int(len(res1_replacement)),
        "dollar_year": 2022,
    },
    "outputs": {key: str(value) for key, value in paths.items()},
    "next_cell": (
        "Cell 2: assign a documented seismic design level to every W2 building "
        "and create the validated direct-SA0P4 structural fragility parameter table."
    ),
}
write_json(paths["summary"], summary)

print("=" * 78)
print("NOTEBOOK 5 CELL 1 INPUT AND HAZUS REFERENCE VALIDATION COMPLETE")
print("=" * 78)
print(f"Portfolio buildings:          {len(portfolio):,}")
print(f"Portfolio occupancy classes:  {len(portfolio_occupancies):,}")
print(f"HAZUS reference workbooks:    {len(reference_inventory):,}")
print(f"W2 source fragility rows:      {len(w2_source_fragility):,}")
print(f"Critical validation checks:   {int((validation['severity'] == 'critical').sum()):,}")
print(f"Critical failures:            {len(critical_failures):,}")
print(f"Warnings requiring review:    {len(warning_failures):,}")
print()
print("Resolved Notebook 5 portfolio columns:")
for role, column in resolved_columns.items():
    print(f"  {role:20s}: {column}")
print()
print("Source fragility demand measures:")
print("  Structural:                    spectral displacement, inches")
print("  Drift-sensitive nonstructural: spectral displacement, inches")
print("  Acceleration-sensitive:        spectral acceleration, g")
print()
print("Validation:")
print(f"  {paths['validation']}")
print("Summary:")
print(f"  {paths['summary']}")
print("W2 source fragility inventory:")
print(f"  {paths['w2_source_fragility']}")
print()
print(
    "Next: assign design levels and generate the validated direct-SA0P4 W2 "
    "structural fragility table."
)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 1 failed one or more critical checks. Review: "
        f"{paths['validation']}"
    )


Located structural fragility workbook: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\reference\hazus\fragility\HAZUS_W2_Structural_Fragility.xlsx
Located drift-sensitive nonstructural fragility workbook: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\reference\hazus\fragility\HAZUS_Nonstructural_Drift_Sensitive_Fragility.xlsx
Located acceleration-sensitive nonstructural fragility workbook: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\reference\hazus\fragility\HAZUS_Nonstructural_Acceleration_Sensitive_Fragility.xlsx
Located structural repair-cost-ratio workbook: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\reference\hazus\repair_cost_ratios\HAZUS_Structural_Repair_Cost_Ratios.xlsx
Located drift-sensitive repair-cost-ratio workbook: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\reference\hazus\repair_cost_ratios\HAZUS_Drift_Sensitive_Nonstructural_Repair_Costs.xl

In [12]:
from __future__ import annotations

import hashlib
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

EXPECTED_STRUCTURAL_TYPE = "W2"
EXPECTED_PERIOD_S = 0.40
STANDARD_GRAVITY_IN_S2 = 386.08858267716535
DESIGN_LEVEL_ORDER = ["PreCode", "LowCode", "ModerateCode", "HighCode"]
DAMAGE_STATE_ORDER = ["Slight", "Moderate", "Extensive", "Complete"]
DESIGN_LEVEL_RULES = [
    {"design_level": "PreCode", "minimum_year": None, "maximum_year_exclusive": 1979},
    {"design_level": "LowCode", "minimum_year": 1979, "maximum_year_exclusive": 1995},
    {"design_level": "ModerateCode", "minimum_year": 1995, "maximum_year_exclusive": 2003},
    {"design_level": "HighCode", "minimum_year": 2003, "maximum_year_exclusive": None},
]
DESIGN_LEVEL_RULE_ID = "oregon_hazus_year_mapping_v1"
DESIGN_LEVEL_SOURCE_NOTE = (
    "Project implementation of the Oregon Hazus seismic-design-level year mapping: "
    "PreCode before 1979, LowCode 1979-1994, ModerateCode 1995-2002, and "
    "HighCode from 2003 onward. Half-open intervals remove boundary overlap."
)
DESIGN_LEVEL_SOURCE_URL = (
    "https://www.fema.gov/sites/default/files/documents/"
    "fema_hazus-inventory-technical-manual-6.1.pdf"
)
CONVERSION_METHOD_ID = "fixed_period_pseudo_spectral_conversion_v1"
CONVERSION_NOTE = (
    "Equivalent direct-SA0P4 approximation. Source W2 structural fragility medians "
    "are spectral displacement in inches and are converted using "
    "Sa(g)=4*pi^2*Sd(in)/(g_in_s2*T^2) at T=0.40 s. This is not the full Hazus "
    "capacity-spectrum procedure."
)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_1_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = recorded_path.replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return " ".join(str(value).strip().split())


def normalize_structural_type(value: Any) -> str:
    return normalize_text(value).upper().replace(" ", "")


def normalize_occupancy(value: Any) -> str:
    text = normalize_text(value).upper().replace(" ", "")
    if text.startswith("RES1-"):
        return "RES1"
    return text


def assign_design_level(year: float) -> str:
    if year < 1979:
        return "PreCode"
    if year < 1995:
        return "LowCode"
    if year < 2003:
        return "ModerateCode"
    return "HighCode"


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CELL1_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_1_summary.json"
CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_1_validation.csv"
CELL1_W2_SOURCE_PATH = OUTPUT_DIR / "notebook_5_w2_source_fragility_inventory.csv"

validation_rows: list[dict[str, Any]] = []

append_check(validation_rows, "cell1_summary_exists", CELL1_SUMMARY_PATH.exists(), str(CELL1_SUMMARY_PATH))
append_check(validation_rows, "cell1_validation_exists", CELL1_VALIDATION_PATH.exists(), str(CELL1_VALIDATION_PATH))
append_check(validation_rows, "cell1_w2_source_exists", CELL1_W2_SOURCE_PATH.exists(), str(CELL1_W2_SOURCE_PATH))

if not all(path.exists() for path in [CELL1_SUMMARY_PATH, CELL1_VALIDATION_PATH, CELL1_W2_SOURCE_PATH]):
    raise FileNotFoundError("Notebook 5 Cell 1 outputs are incomplete. Rerun Cell 1 first.")

cell1_summary = load_json(CELL1_SUMMARY_PATH)
cell1_validation = pd.read_csv(CELL1_VALIDATION_PATH)

append_check(
    validation_rows,
    "cell1_critical_checks_passed",
    cell1_summary.get("all_critical_checks_passed") is True,
    f"all_critical_checks_passed={cell1_summary.get('all_critical_checks_passed')}",
)
cell1_unresolved = cell1_validation.loc[~parse_bool_series(cell1_validation["passed"])].copy()
append_check(
    validation_rows,
    "cell1_has_no_unresolved_checks",
    cell1_unresolved.empty,
    f"unresolved_rows={len(cell1_unresolved)}",
)

portfolio_info = cell1_summary["portfolio"]
portfolio_path = resolve_recorded_path(PROJECT_ROOT, portfolio_info["path"])
portfolio = pd.read_csv(portfolio_path)
resolved_columns = portfolio_info["resolved_columns"]
site_column = portfolio_info["site_key"]
structural_type_column = resolved_columns["structural_type"]
occupancy_column = resolved_columns["occupancy"]
year_built_column = resolved_columns["year_built"]

required_portfolio_columns = {
    site_column,
    structural_type_column,
    occupancy_column,
    year_built_column,
}
missing_portfolio_columns = sorted(required_portfolio_columns.difference(portfolio.columns))
append_check(
    validation_rows,
    "portfolio_required_columns_present",
    not missing_portfolio_columns,
    f"missing={missing_portfolio_columns}",
)
if missing_portfolio_columns:
    raise KeyError(f"Portfolio is missing columns: {missing_portfolio_columns}")

portfolio_hash = sha256_file(portfolio_path)
append_check(
    validation_rows,
    "portfolio_hash_matches_cell1",
    portfolio_hash == portfolio_info["sha256"],
    f"calculated={portfolio_hash}; cell1={portfolio_info['sha256']}",
)
append_check(
    validation_rows,
    "portfolio_row_count_matches_cell1",
    len(portfolio) == int(portfolio_info["rows"]),
    f"rows={len(portfolio)}; expected={portfolio_info['rows']}",
)

assignments = pd.DataFrame(
    {
        "site_id": portfolio[site_column].astype(str),
        "structural_type": portfolio[structural_type_column].map(normalize_structural_type),
        "occupancy_raw": portfolio[occupancy_column].map(normalize_text),
        "occ_type": portfolio[occupancy_column].map(normalize_occupancy),
        "year_built_source": portfolio[year_built_column],
    }
)
assignments["year_built"] = pd.to_numeric(assignments["year_built_source"], errors="coerce")

append_check(
    validation_rows,
    "site_ids_nonblank",
    assignments["site_id"].str.strip().ne("").all(),
    f"blank={int(assignments['site_id'].str.strip().eq('').sum())}",
)
append_check(
    validation_rows,
    "site_ids_unique",
    assignments["site_id"].is_unique,
    f"duplicates={int(assignments['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "all_buildings_are_w2",
    assignments["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(assignments['structural_type'].unique().tolist())}",
)
append_check(
    validation_rows,
    "year_built_numeric_complete",
    assignments["year_built"].notna().all(),
    f"missing={int(assignments['year_built'].isna().sum())}",
)
append_check(
    validation_rows,
    "year_built_plausible",
    assignments["year_built"].between(1800, datetime.now().year).all(),
    (
        f"min={assignments['year_built'].min()}; max={assignments['year_built'].max()}; "
        f"current_year={datetime.now().year}"
    ),
)

if assignments["year_built"].isna().any():
    bad_sites = assignments.loc[assignments["year_built"].isna(), "site_id"].head(20).tolist()
    raise ValueError(f"Missing or nonnumeric construction years for sites: {bad_sites}")

assignments["design_level"] = assignments["year_built"].map(assign_design_level)
assignments["design_level_rule_id"] = DESIGN_LEVEL_RULE_ID
assignments["design_level_source_note"] = DESIGN_LEVEL_SOURCE_NOTE

append_check(
    validation_rows,
    "all_design_levels_assigned",
    assignments["design_level"].isin(DESIGN_LEVEL_ORDER).all(),
    f"levels={sorted(assignments['design_level'].unique().tolist())}",
)

rules = pd.DataFrame(DESIGN_LEVEL_RULES)
rules["rule_id"] = DESIGN_LEVEL_RULE_ID
rules["source_note"] = DESIGN_LEVEL_SOURCE_NOTE
rules["source_url"] = DESIGN_LEVEL_SOURCE_URL
rules["interval_convention"] = "minimum inclusive; maximum exclusive"

source = pd.read_csv(CELL1_W2_SOURCE_PATH)
source_structural = source.loc[source["component"].eq("structural")].copy()
source_structural["design_level"] = source_structural["design_level"].astype(str)
source_structural["damage_state"] = source_structural["damage_state"].astype(str)
source_structural["source_median"] = pd.to_numeric(source_structural["source_median"], errors="coerce")
source_structural["source_beta_ln"] = pd.to_numeric(source_structural["source_beta_ln"], errors="coerce")

append_check(
    validation_rows,
    "structural_source_has_16_rows",
    len(source_structural) == 16,
    f"rows={len(source_structural)}",
)
append_check(
    validation_rows,
    "structural_source_design_levels_complete",
    set(source_structural["design_level"]) == set(DESIGN_LEVEL_ORDER),
    f"levels={sorted(source_structural['design_level'].unique().tolist())}",
)
append_check(
    validation_rows,
    "structural_source_damage_states_complete",
    set(source_structural["damage_state"]) == set(DAMAGE_STATE_ORDER),
    f"states={sorted(source_structural['damage_state'].unique().tolist())}",
)
append_check(
    validation_rows,
    "structural_source_is_w2",
    source_structural["structural_type"].astype(str).eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(source_structural['structural_type'].astype(str).unique().tolist())}",
)
append_check(
    validation_rows,
    "structural_source_is_spectral_displacement_inches",
    (
        source_structural["source_demand_type"].astype(str).eq("spectral_displacement").all()
        and source_structural["source_median_unit"].astype(str).eq("in").all()
    ),
    (
        f"demand={sorted(source_structural['source_demand_type'].astype(str).unique().tolist())}; "
        f"units={sorted(source_structural['source_median_unit'].astype(str).unique().tolist())}"
    ),
)
append_check(
    validation_rows,
    "structural_source_parameters_positive",
    (
        source_structural["source_median"].notna().all()
        and source_structural["source_beta_ln"].notna().all()
        and source_structural["source_median"].gt(0).all()
        and source_structural["source_beta_ln"].gt(0).all()
    ),
    "source medians and betas must be finite and positive",
)

period_s = float(cell1_summary["primary_ground_motion_fields"]["period_s"])
append_check(
    validation_rows,
    "period_is_exact_sa0p4",
    math.isclose(period_s, EXPECTED_PERIOD_S, rel_tol=0.0, abs_tol=1e-12),
    f"period_s={period_s}",
)

conversion_factor = 4.0 * math.pi**2 / (STANDARD_GRAVITY_IN_S2 * period_s**2)
fragility = source_structural.copy()
fragility["source_median_sd_in"] = fragility["source_median"].astype(float)
fragility["direct_sa0p4_median_g"] = fragility["source_median_sd_in"] * conversion_factor
fragility["direct_sa0p4_beta_ln"] = fragility["source_beta_ln"].astype(float)
fragility["period_s"] = period_s
fragility["gravity_in_s2"] = STANDARD_GRAVITY_IN_S2
fragility["sd_to_sa_factor_g_per_in"] = conversion_factor
fragility["conversion_method_id"] = CONVERSION_METHOD_ID
fragility["conversion_note"] = CONVERSION_NOTE
fragility["production_demand_type"] = "spectral_acceleration"
fragility["production_median_unit"] = "g"
fragility["transformed_for_production"] = True

state_rank = {state: rank for rank, state in enumerate(DAMAGE_STATE_ORDER)}
design_rank = {level: rank for rank, level in enumerate(DESIGN_LEVEL_ORDER)}
fragility["damage_state_rank"] = fragility["damage_state"].map(state_rank)
fragility["design_level_rank"] = fragility["design_level"].map(design_rank)
fragility = fragility.sort_values(["design_level_rank", "damage_state_rank"]).reset_index(drop=True)

ordered_check = (
    fragility.groupby("design_level", sort=False)["direct_sa0p4_median_g"]
    .apply(lambda values: bool(np.all(np.diff(values.to_numpy(dtype=float)) > 0)))
)
append_check(
    validation_rows,
    "direct_sa0p4_medians_strictly_ordered",
    ordered_check.all(),
    f"by_design_level={ordered_check.to_dict()}",
)

reconstructed_sd = (
    fragility["direct_sa0p4_median_g"].to_numpy(dtype=float)
    * STANDARD_GRAVITY_IN_S2
    * period_s**2
    / (4.0 * math.pi**2)
)
max_conversion_error = float(
    np.max(np.abs(reconstructed_sd - fragility["source_median_sd_in"].to_numpy(dtype=float)))
)
append_check(
    validation_rows,
    "sd_sa_conversion_reconstructs_source",
    max_conversion_error <= 1e-12,
    f"maximum_absolute_error_in={max_conversion_error:.3e}",
)
append_check(
    validation_rows,
    "beta_unchanged_by_constant_conversion",
    np.array_equal(
        fragility["direct_sa0p4_beta_ln"].to_numpy(dtype=float),
        fragility["source_beta_ln"].to_numpy(dtype=float),
    ),
    "direct beta equals source beta exactly",
)

wide_median = fragility.pivot(
    index="design_level", columns="damage_state", values="direct_sa0p4_median_g"
).rename(columns={state: f"{state.lower()}_median_sa0p4_g" for state in DAMAGE_STATE_ORDER})
wide_beta = fragility.pivot(
    index="design_level", columns="damage_state", values="direct_sa0p4_beta_ln"
).rename(columns={state: f"{state.lower()}_beta_ln" for state in DAMAGE_STATE_ORDER})
wide_source = fragility.pivot(
    index="design_level", columns="damage_state", values="source_median_sd_in"
).rename(columns={state: f"{state.lower()}_source_median_sd_in" for state in DAMAGE_STATE_ORDER})
fragility_wide = wide_source.join(wide_median).join(wide_beta).reset_index()

building_assignments = assignments.merge(
    fragility_wide,
    on="design_level",
    how="left",
    validate="many_to_one",
)
building_assignments["period_s"] = period_s
building_assignments["structural_damage_im_column"] = cell1_summary[
    "primary_ground_motion_fields"
]["structural_damage_im_column"]
building_assignments["conversion_method_id"] = CONVERSION_METHOD_ID
building_assignments["sd_to_sa_factor_g_per_in"] = conversion_factor
building_assignments["fragility_source_sha256"] = sha256_file(CELL1_W2_SOURCE_PATH)

parameter_columns = [
    f"{state.lower()}_median_sa0p4_g" for state in DAMAGE_STATE_ORDER
] + [f"{state.lower()}_beta_ln" for state in DAMAGE_STATE_ORDER]
append_check(
    validation_rows,
    "building_assignments_complete",
    building_assignments[parameter_columns].notna().all().all(),
    f"missing_parameter_cells={int(building_assignments[parameter_columns].isna().sum().sum())}",
)
append_check(
    validation_rows,
    "building_assignment_rows_match_portfolio",
    len(building_assignments) == len(portfolio),
    f"rows={len(building_assignments)}; portfolio={len(portfolio)}",
)
append_check(
    validation_rows,
    "building_assignment_site_ids_unique",
    building_assignments["site_id"].is_unique,
    f"duplicates={int(building_assignments['site_id'].duplicated().sum())}",
)

building_medians = building_assignments[
    [f"{state.lower()}_median_sa0p4_g" for state in DAMAGE_STATE_ORDER]
].to_numpy(dtype=float)
append_check(
    validation_rows,
    "building_medians_strictly_ordered",
    np.all(np.diff(building_medians, axis=1) > 0),
    "Slight < Moderate < Extensive < Complete for every building",
)

counts = (
    building_assignments.groupby("design_level", dropna=False)
    .agg(
        buildings=("site_id", "size"),
        minimum_year_built=("year_built", "min"),
        maximum_year_built=("year_built", "max"),
        total_floor_area_sqft=("site_id", "size"),
    )
    .reset_index()
)
if resolved_columns.get("floor_area_sqft") in portfolio.columns:
    floor_area_map = pd.DataFrame(
        {
            "site_id": portfolio[site_column].astype(str),
            "floor_area_sqft": pd.to_numeric(
                portfolio[resolved_columns["floor_area_sqft"]], errors="coerce"
            ),
        }
    )
    count_floor = (
        building_assignments[["site_id", "design_level"]]
        .merge(floor_area_map, on="site_id", how="left", validate="one_to_one")
        .groupby("design_level")["floor_area_sqft"]
        .sum()
    )
    counts = counts.drop(columns="total_floor_area_sqft").merge(
        count_floor.rename("total_floor_area_sqft"),
        on="design_level",
        how="left",
    )
counts["design_level_rank"] = counts["design_level"].map(design_rank)
counts = counts.sort_values("design_level_rank").drop(columns="design_level_rank").reset_index(drop=True)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    (validation["severity"] == "critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    (validation["severity"] == "warning") & ~validation["passed"]
].copy()

paths = {
    "design_level_rules": OUTPUT_DIR / "notebook_5_design_level_rules.csv",
    "design_level_assignments": OUTPUT_DIR / "seaside_w2_design_level_assignments.csv",
    "design_level_counts": METADATA_DIR / "notebook_5_cell_2_design_level_counts.csv",
    "direct_fragility_long": OUTPUT_DIR / "seaside_w2_direct_sa0p4_structural_fragility.csv",
    "direct_fragility_wide": OUTPUT_DIR / "seaside_w2_direct_sa0p4_structural_fragility_wide.csv",
    "building_assignments": OUTPUT_DIR / "seaside_w2_building_fragility_assignments.csv",
    "validation": METADATA_DIR / "notebook_5_cell_2_validation.csv",
    "summary": METADATA_DIR / "notebook_5_cell_2_summary.json",
}

rules.to_csv(paths["design_level_rules"], index=False)
assignments.to_csv(paths["design_level_assignments"], index=False)
counts.to_csv(paths["design_level_counts"], index=False)
fragility.drop(columns=["damage_state_rank", "design_level_rank"]).to_csv(
    paths["direct_fragility_long"], index=False
)
fragility_wide.to_csv(paths["direct_fragility_wide"], index=False)
building_assignments.to_csv(paths["building_assignments"], index=False)
validation.to_csv(paths["validation"], index=False)

summary = {
    "pipeline_version": "notebook5_cell2_design_levels_direct_sa0p4_v1",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int((validation["severity"] == "critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "portfolio": {
        "path": str(portfolio_path),
        "sha256": portfolio_hash,
        "buildings": int(len(building_assignments)),
        "structural_type": EXPECTED_STRUCTURAL_TYPE,
        "year_built_source_column": year_built_column,
        "design_level_counts": {
            str(row["design_level"]): int(row["buildings"])
            for _, row in counts.iterrows()
        },
    },
    "design_level_assignment": {
        "rule_id": DESIGN_LEVEL_RULE_ID,
        "rules": DESIGN_LEVEL_RULES,
        "interval_convention": "minimum inclusive; maximum exclusive",
        "source_note": DESIGN_LEVEL_SOURCE_NOTE,
        "source_url": DESIGN_LEVEL_SOURCE_URL,
        "important_limitation": (
            "Construction year is used as a portfolio-level proxy for seismic design level. "
            "The assignment does not verify permit date, adopted local code, enforcement, "
            "retrofit history, or building-specific design documents."
        ),
    },
    "fragility_transformation": {
        "structural_type": EXPECTED_STRUCTURAL_TYPE,
        "source_demand": "spectral displacement",
        "source_unit": "in",
        "production_demand": "SA0P4",
        "production_unit": "g",
        "period_s": period_s,
        "gravity_in_s2": STANDARD_GRAVITY_IN_S2,
        "conversion_factor_g_per_in": conversion_factor,
        "method_id": CONVERSION_METHOD_ID,
        "equation": "Sa_g = 4*pi^2*Sd_in/(g_in_s2*T_s^2)",
        "beta_rule": "unchanged under constant multiplicative transformation",
        "maximum_reconstruction_error_in": max_conversion_error,
        "method_note": CONVERSION_NOTE,
        "source_inventory_path": str(CELL1_W2_SOURCE_PATH),
        "source_inventory_sha256": sha256_file(CELL1_W2_SOURCE_PATH),
    },
    "outputs": {key: str(path) for key, path in paths.items()},
    "next_cell": (
        "Cell 3: calculate controlled structural damage-state exceedance and mutually "
        "exclusive probabilities from simulated SA0P4, then validate deterministic "
        "occurrence-building damage random streams before full production sampling."
    ),
}
write_json(paths["summary"], summary)

print("=" * 78)
print("NOTEBOOK 5 CELL 2 DESIGN LEVEL AND DIRECT-SA0P4 FRAGILITY COMPLETE")
print("=" * 78)
print(f"Portfolio buildings:              {len(building_assignments):,}")
print(f"Design levels represented:        {building_assignments['design_level'].nunique():,}")
for _, row in counts.iterrows():
    print(
        f"  {str(row['design_level']):14s}: {int(row['buildings']):4d} buildings "
        f"({row['minimum_year_built']:.0f}-{row['maximum_year_built']:.0f})"
    )
print(f"SA0P4 conversion factor:          {conversion_factor:.12f} g/in")
print(f"Maximum conversion error:         {max_conversion_error:.3e} in")
print(f"Critical validation checks:       {int((validation['severity'] == 'critical').sum()):,}")
print(f"Critical failures:                {len(critical_failures):,}")
print(f"Warnings requiring review:        {len(warning_failures):,}")
print()
print("Direct SA0P4 fragility:")
print(f"  {paths['direct_fragility_long']}")
print("Building assignments:")
print(f"  {paths['building_assignments']}")
print("Validation:")
print(f"  {paths['validation']}")
print("Summary:")
print(f"  {paths['summary']}")
print()
print("Next: controlled structural damage-probability and deterministic sampling validation.")

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 2 failed one or more critical checks. Review: "
        f"{paths['validation']}"
    )


NOTEBOOK 5 CELL 2 DESIGN LEVEL AND DIRECT-SA0P4 FRAGILITY COMPLETE
Portfolio buildings:              470
Design levels represented:        2
  PreCode       :  347 buildings (1951-1977)
  LowCode       :  123 buildings (1982-1992)
SA0P4 conversion factor:          0.639076422090 g/in
Maximum conversion error:         3.553e-15 in
Critical validation checks:       28
Critical failures:                0
Warnings requiring review:        0

Direct SA0P4 fragility:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_damage_loss_parameters\seaside_w2_direct_sa0p4_structural_fragility.csv
Building assignments:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_damage_loss_parameters\seaside_w2_building_fragility_assignments.csv
Validation:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_5_damage_loss\notebook_5_cell_2_validation.csv
Summary:
  c:\Users\USER\Document

In [13]:
from __future__ import annotations

import hashlib
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.special import ndtr
from scipy.stats import kstest

DAMAGE_STATES = ["None", "Slight", "Moderate", "Extensive", "Complete"]
EXCEEDANCE_STATES = ["Slight", "Moderate", "Extensive", "Complete"]
EXPECTED_PERIOD_S = 0.40
EXPECTED_STRUCTURAL_TYPE = "W2"
FIELD_CHUNK_ROWS = 250_000
SELECTION_MODULUS = 10_007
SELECTION_THRESHOLD = 50
MAX_SELECTION_ROWS_PER_GROUP = 5_000
CONTROL_QUANTILES = {"low": 0.10, "medium": 0.50, "high": 0.90}
DAMAGE_STREAM_NAMESPACE = "notebook5_structural_damage_v1"
CONTROL_SELECTION_NAMESPACE = "notebook5_cell3_control_selection_v1"
PROBABILITY_TOLERANCE = 1e-12
MONTE_CARLO_REPLICATES = 20_000
MONTE_CARLO_MAX_ABSOLUTE_ERROR = 0.015


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_2_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = recorded_path.replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def normalize_source_type(values: pd.Series) -> pd.Series:
    return values.astype(str).str.strip().str.upper()


def deterministic_digest(key: str, namespace: str) -> bytes:
    return hashlib.sha256(f"{namespace}|{key}".encode("utf-8")).digest()


def deterministic_uniforms(
    occurrence_ids: pd.Series,
    site_ids: pd.Series,
    namespace: str = DAMAGE_STREAM_NAMESPACE,
) -> tuple[np.ndarray, list[str]]:
    occurrence_values = occurrence_ids.astype(str).to_numpy()
    site_values = site_ids.astype(str).to_numpy()
    uniforms = np.empty(len(occurrence_values), dtype=np.float64)
    seed_hex: list[str] = []
    denominator = float(2**64)

    for index, (occurrence_id, site_id) in enumerate(
        zip(occurrence_values, site_values, strict=True)
    ):
        digest = deterministic_digest(f"{occurrence_id}|{site_id}", namespace)
        integer = int.from_bytes(digest[:8], byteorder="big", signed=False)
        uniforms[index] = (integer + 0.5) / denominator
        seed_hex.append(digest[:16].hex())

    return uniforms, seed_hex


def selection_priority(occurrence_id: str, site_id: str) -> str:
    return deterministic_digest(
        f"{occurrence_id}|{site_id}", CONTROL_SELECTION_NAMESPACE
    ).hex()


def parameter_columns() -> tuple[list[str], list[str]]:
    median_columns = [
        f"{state.lower()}_median_sa0p4_g" for state in EXCEEDANCE_STATES
    ]
    beta_columns = [f"{state.lower()}_beta_ln" for state in EXCEEDANCE_STATES]
    return median_columns, beta_columns


def calculate_damage_probabilities(frame: pd.DataFrame) -> pd.DataFrame:
    median_columns, beta_columns = parameter_columns()
    sa = pd.to_numeric(frame["sa0p4_simulated_g"], errors="raise").to_numpy(
        dtype=np.float64
    )
    medians = frame[median_columns].to_numpy(dtype=np.float64)
    betas = frame[beta_columns].to_numpy(dtype=np.float64)

    if not np.isfinite(sa).all() or np.any(sa <= 0.0):
        raise ValueError("All SA0P4 values must be finite and positive.")
    if not np.isfinite(medians).all() or np.any(medians <= 0.0):
        raise ValueError("All fragility medians must be finite and positive.")
    if not np.isfinite(betas).all() or np.any(betas <= 0.0):
        raise ValueError("All fragility betas must be finite and positive.")

    z = (np.log(sa)[:, None] - np.log(medians)) / betas
    exceedance = ndtr(z)

    raw_state_probabilities = np.column_stack(
        [
            1.0 - exceedance[:, 0],
            exceedance[:, 0] - exceedance[:, 1],
            exceedance[:, 1] - exceedance[:, 2],
            exceedance[:, 2] - exceedance[:, 3],
            exceedance[:, 3],
        ]
    )

    minimum_raw_probability = float(raw_state_probabilities.min())
    if minimum_raw_probability < -PROBABILITY_TOLERANCE:
        raise RuntimeError(
            "Fragility exceedance curves produced materially negative mutually "
            f"exclusive probabilities. Minimum={minimum_raw_probability:.6e}."
        )

    state_probabilities = np.clip(raw_state_probabilities, 0.0, 1.0)
    row_sums = state_probabilities.sum(axis=1)
    if np.any(row_sums <= 0.0):
        raise RuntimeError("At least one damage-probability row has zero total mass.")
    state_probabilities = state_probabilities / row_sums[:, None]

    output = pd.DataFrame(index=frame.index)
    for index, state in enumerate(EXCEEDANCE_STATES):
        output[f"p_exceed_{state.lower()}"] = exceedance[:, index]
    for index, state in enumerate(DAMAGE_STATES):
        output[f"p_state_{state.lower()}"] = state_probabilities[:, index]

    output["minimum_raw_state_probability"] = minimum_raw_probability
    output["state_probability_sum"] = state_probabilities.sum(axis=1)
    output["expected_damage_state"] = state_probabilities @ np.arange(5, dtype=float)
    return output


def sample_damage_states(
    probabilities: pd.DataFrame,
    uniforms: np.ndarray,
) -> np.ndarray:
    probability_columns = [f"p_state_{state.lower()}" for state in DAMAGE_STATES]
    cumulative = np.cumsum(
        probabilities[probability_columns].to_numpy(dtype=np.float64), axis=1
    )
    cumulative[:, -1] = 1.0
    states = (uniforms[:, None] > cumulative).sum(axis=1)
    return np.minimum(states, 4).astype(np.int8)


def calculate_control_outputs(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    probabilities = calculate_damage_probabilities(output)
    output = pd.concat([output.reset_index(drop=True), probabilities.reset_index(drop=True)], axis=1)
    uniforms, seed_hex = deterministic_uniforms(output["occurrence_id"], output["site_id"])
    output["structural_damage_seed_hex"] = seed_hex
    output["structural_damage_uniform"] = uniforms
    output["sampled_structural_damage_state"] = sample_damage_states(
        probabilities.reset_index(drop=True), uniforms
    )
    output["sampled_structural_damage_state_name"] = output[
        "sampled_structural_damage_state"
    ].map(dict(enumerate(DAMAGE_STATES)))
    return output


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
CONTROL_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_controlled_structural_damage"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

CELL1_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_1_summary.json"
CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_1_validation.csv"
CELL2_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_2_summary.json"
CELL2_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_2_validation.csv"
BUILDING_ASSIGNMENTS_PATH = (
    PARAMETER_DIR / "seaside_w2_building_fragility_assignments.csv"
)
DIRECT_FRAGILITY_WIDE_PATH = (
    PARAMETER_DIR / "seaside_w2_direct_sa0p4_structural_fragility_wide.csv"
)

required_paths = [
    CELL1_SUMMARY_PATH,
    CELL1_VALIDATION_PATH,
    CELL2_SUMMARY_PATH,
    CELL2_VALIDATION_PATH,
    BUILDING_ASSIGNMENTS_PATH,
    DIRECT_FRAGILITY_WIDE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required Cell 1 or Cell 2 outputs are missing: {missing_paths}")

validation_rows: list[dict[str, Any]] = []
cell1_summary = load_json(CELL1_SUMMARY_PATH)
cell2_summary = load_json(CELL2_SUMMARY_PATH)
cell1_validation = pd.read_csv(CELL1_VALIDATION_PATH)
cell2_validation = pd.read_csv(CELL2_VALIDATION_PATH)

append_check(
    validation_rows,
    "cell1_critical_checks_passed",
    cell1_summary.get("all_critical_checks_passed") is True,
    f"all_critical_checks_passed={cell1_summary.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell2_critical_checks_passed",
    cell2_summary.get("all_critical_checks_passed") is True,
    f"all_critical_checks_passed={cell2_summary.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell1_has_no_unresolved_checks",
    parse_bool_series(cell1_validation["passed"]).all(),
    f"unresolved={int((~parse_bool_series(cell1_validation['passed'])).sum())}",
)
append_check(
    validation_rows,
    "cell2_has_no_unresolved_checks",
    parse_bool_series(cell2_validation["passed"]).all(),
    f"unresolved={int((~parse_bool_series(cell2_validation['passed'])).sum())}",
)

field_info = cell1_summary["primary_ground_motion_fields"]
field_path = resolve_recorded_path(PROJECT_ROOT, field_info["path"])
expected_field_rows = int(field_info["rows_from_handoff"])
structural_im_column = str(field_info["structural_damage_im_column"])
period_s = float(field_info["period_s"])

append_check(validation_rows, "ground_motion_field_exists", field_path.exists(), str(field_path))
append_check(
    validation_rows,
    "structural_im_is_sa0p4",
    structural_im_column == "sa0p4_simulated_g",
    f"column={structural_im_column}",
)
append_check(
    validation_rows,
    "period_is_exact_sa0p4",
    math.isclose(period_s, EXPECTED_PERIOD_S, rel_tol=0.0, abs_tol=1e-12),
    f"period_s={period_s}",
)

building_assignments = pd.read_csv(BUILDING_ASSIGNMENTS_PATH)
fragility_wide = pd.read_csv(DIRECT_FRAGILITY_WIDE_PATH)
median_columns, beta_columns = parameter_columns()
required_assignment_columns = {
    "site_id",
    "structural_type",
    "design_level",
    *median_columns,
    *beta_columns,
}
missing_assignment_columns = sorted(
    required_assignment_columns.difference(building_assignments.columns)
)
append_check(
    validation_rows,
    "building_assignment_columns_present",
    not missing_assignment_columns,
    f"missing={missing_assignment_columns}",
)
if missing_assignment_columns:
    raise KeyError(f"Building assignments are missing columns: {missing_assignment_columns}")

building_assignments["site_id"] = building_assignments["site_id"].astype(str)
building_assignments["design_level"] = building_assignments["design_level"].astype(str)
building_assignments["structural_type"] = (
    building_assignments["structural_type"].astype(str).str.strip().str.upper()
)
represented_design_levels = building_assignments["design_level"].drop_duplicates().tolist()

append_check(
    validation_rows,
    "building_assignment_rows_match_portfolio",
    len(building_assignments) == int(cell2_summary["portfolio"]["buildings"]),
    f"rows={len(building_assignments)}; expected={cell2_summary['portfolio']['buildings']}",
)
append_check(
    validation_rows,
    "building_site_ids_unique",
    building_assignments["site_id"].is_unique,
    f"duplicates={int(building_assignments['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "all_buildings_are_w2",
    building_assignments["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(building_assignments['structural_type'].unique().tolist())}",
)
append_check(
    validation_rows,
    "fragility_parameters_complete",
    building_assignments[median_columns + beta_columns].notna().all().all(),
    f"missing={int(building_assignments[median_columns + beta_columns].isna().sum().sum())}",
)
append_check(
    validation_rows,
    "represented_design_levels_match_cell2",
    set(represented_design_levels)
    == set(cell2_summary["portfolio"]["design_level_counts"].keys()),
    f"levels={represented_design_levels}",
)

field_header = pd.read_csv(field_path, nrows=0)
required_field_columns = {
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    structural_im_column,
}
missing_field_columns = sorted(required_field_columns.difference(field_header.columns))
append_check(
    validation_rows,
    "ground_motion_columns_present",
    not missing_field_columns,
    f"missing={missing_field_columns}",
)
if missing_field_columns:
    raise KeyError(f"Ground-motion file is missing columns: {missing_field_columns}")

lookup_columns = ["site_id", "design_level", *median_columns, *beta_columns]
design_lookup = building_assignments[lookup_columns].copy()
field_usecols = [
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    structural_im_column,
]

candidate_frames: list[pd.DataFrame] = []
extreme_rows: dict[tuple[str, str, str], dict[str, Any]] = {}
selection_counts: dict[tuple[str, str], int] = {}
source_row_counts: dict[str, int] = {}
total_rows = 0
missing_site_rows = 0
nonpositive_sa_rows = 0

for chunk in pd.read_csv(field_path, usecols=field_usecols, chunksize=FIELD_CHUNK_ROWS):
    chunk = chunk.rename(columns={structural_im_column: "sa0p4_simulated_g"})
    chunk["site_id"] = chunk["site_id"].astype(str)
    chunk["source_type"] = normalize_source_type(chunk["source_type"])
    chunk["occurrence_ordinal"] = pd.to_numeric(
        chunk["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    chunk["site_ordinal"] = pd.to_numeric(chunk["site_ordinal"], errors="raise").astype(
        np.int64
    )
    chunk["sa0p4_simulated_g"] = pd.to_numeric(
        chunk["sa0p4_simulated_g"], errors="coerce"
    )
    chunk = chunk.merge(design_lookup, on="site_id", how="left", validate="many_to_one")

    total_rows += len(chunk)
    missing_site_rows += int(chunk["design_level"].isna().sum())
    nonpositive_sa_rows += int(
        (
            (~np.isfinite(chunk["sa0p4_simulated_g"]))
            | (chunk["sa0p4_simulated_g"] <= 0.0)
        ).sum()
    )

    for source_type, count in chunk["source_type"].value_counts().items():
        source_row_counts[str(source_type)] = source_row_counts.get(str(source_type), 0) + int(count)

    valid = chunk.loc[
        chunk["design_level"].notna()
        & np.isfinite(chunk["sa0p4_simulated_g"])
        & chunk["sa0p4_simulated_g"].gt(0.0)
    ].copy()

    for (design_level, source_type), group in valid.groupby(
        ["design_level", "source_type"], sort=False
    ):
        group_key = (str(design_level), str(source_type))
        selection_counts[group_key] = selection_counts.get(group_key, 0) + len(group)

        minimum_row = group.loc[group["sa0p4_simulated_g"].idxmin()].to_dict()
        maximum_row = group.loc[group["sa0p4_simulated_g"].idxmax()].to_dict()
        for label, row in [("minimum", minimum_row), ("maximum", maximum_row)]:
            key = (group_key[0], group_key[1], label)
            existing = extreme_rows.get(key)
            if existing is None:
                extreme_rows[key] = row
            elif label == "minimum" and row["sa0p4_simulated_g"] < existing["sa0p4_simulated_g"]:
                extreme_rows[key] = row
            elif label == "maximum" and row["sa0p4_simulated_g"] > existing["sa0p4_simulated_g"]:
                extreme_rows[key] = row

    selector = (
        (
            (valid["occurrence_ordinal"].to_numpy(dtype=np.int64) % SELECTION_MODULUS)
            * 37
            + (valid["site_ordinal"].to_numpy(dtype=np.int64) % SELECTION_MODULUS)
            * 101
        )
        % SELECTION_MODULUS
    )
    selected = valid.loc[selector < SELECTION_THRESHOLD].copy()
    if not selected.empty:
        candidate_frames.append(selected)

append_check(
    validation_rows,
    "ground_motion_row_count_matches_handoff",
    total_rows == expected_field_rows,
    f"rows={total_rows}; expected={expected_field_rows}",
)
append_check(
    validation_rows,
    "all_ground_motion_sites_mapped",
    missing_site_rows == 0,
    f"missing_site_rows={missing_site_rows}",
)
append_check(
    validation_rows,
    "all_sa0p4_values_positive_finite",
    nonpositive_sa_rows == 0,
    f"invalid_rows={nonpositive_sa_rows}",
)
append_check(
    validation_rows,
    "both_subduction_source_types_present",
    {"INTERFACE", "SLAB"}.issubset(source_row_counts),
    f"source_counts={source_row_counts}",
)

if not candidate_frames:
    raise RuntimeError("The deterministic control-row selection produced no candidates.")

candidate_pool = pd.concat(candidate_frames, ignore_index=True)
candidate_pool["control_selection_priority"] = [
    selection_priority(occurrence_id, site_id)
    for occurrence_id, site_id in zip(
        candidate_pool["occurrence_id"].astype(str),
        candidate_pool["site_id"].astype(str),
        strict=True,
    )
]
candidate_pool = candidate_pool.drop_duplicates(
    subset=["occurrence_id", "site_id"], keep="first"
)

control_rows: list[pd.Series] = []
selection_audit_rows: list[dict[str, Any]] = []
source_types = [source for source in ["INTERFACE", "SLAB"] if source in source_row_counts]

for design_level in represented_design_levels:
    for source_type in source_types:
        group = candidate_pool.loc[
            candidate_pool["design_level"].eq(design_level)
            & candidate_pool["source_type"].eq(source_type)
        ].copy()
        group = group.sort_values("control_selection_priority").head(
            MAX_SELECTION_ROWS_PER_GROUP
        )

        append_check(
            validation_rows,
            f"candidate_pool_{design_level}_{source_type}",
            len(group) >= 100,
            f"candidates={len(group)}",
        )
        if group.empty:
            continue

        used_indices: set[int] = set()
        quantile_values = group["sa0p4_simulated_g"].quantile(
            list(CONTROL_QUANTILES.values())
        )

        for intensity_level, quantile in CONTROL_QUANTILES.items():
            target_sa = float(quantile_values.loc[quantile])
            distances = (group["sa0p4_simulated_g"] - target_sa).abs()
            for index in distances.sort_values().index:
                if int(index) not in used_indices:
                    selected_row = group.loc[index].copy()
                    used_indices.add(int(index))
                    break
            selected_row["intensity_level"] = intensity_level
            selected_row["selection_quantile"] = quantile
            selected_row["selection_target_sa0p4_g"] = target_sa
            control_rows.append(selected_row)

        minimum_value = float(
            extreme_rows[(design_level, source_type, "minimum")]["sa0p4_simulated_g"]
        )
        maximum_value = float(
            extreme_rows[(design_level, source_type, "maximum")]["sa0p4_simulated_g"]
        )
        selection_audit_rows.append(
            {
                "design_level": design_level,
                "source_type": source_type,
                "full_group_rows": int(selection_counts[(design_level, source_type)]),
                "candidate_rows_before_cap": int(
                    len(
                        candidate_pool.loc[
                            candidate_pool["design_level"].eq(design_level)
                            & candidate_pool["source_type"].eq(source_type)
                        ]
                    )
                ),
                "candidate_rows_used": int(len(group)),
                "minimum_sa0p4_g": minimum_value,
                "sample_q10_sa0p4_g": float(quantile_values.loc[0.10]),
                "sample_q50_sa0p4_g": float(quantile_values.loc[0.50]),
                "sample_q90_sa0p4_g": float(quantile_values.loc[0.90]),
                "maximum_sa0p4_g": maximum_value,
            }
        )

controls = pd.DataFrame(control_rows).reset_index(drop=True)
expected_control_rows = len(represented_design_levels) * len(source_types) * len(
    CONTROL_QUANTILES
)
append_check(
    validation_rows,
    "controlled_row_count_complete",
    len(controls) == expected_control_rows,
    f"rows={len(controls)}; expected={expected_control_rows}",
)
append_check(
    validation_rows,
    "controlled_keys_unique",
    not controls.duplicated(["occurrence_id", "site_id"]).any(),
    f"duplicates={int(controls.duplicated(['occurrence_id', 'site_id']).sum())}",
)
append_check(
    validation_rows,
    "controlled_design_source_intensity_coverage",
    controls.groupby(["design_level", "source_type", "intensity_level"]).size().eq(1).all(),
    f"groups={len(controls.groupby(['design_level', 'source_type', 'intensity_level']))}",
)

controlled_output = calculate_control_outputs(controls)
exceedance_columns = [f"p_exceed_{state.lower()}" for state in EXCEEDANCE_STATES]
state_probability_columns = [f"p_state_{state.lower()}" for state in DAMAGE_STATES]

exceedance_values = controlled_output[exceedance_columns].to_numpy(dtype=float)
state_probability_values = controlled_output[state_probability_columns].to_numpy(dtype=float)
append_check(
    validation_rows,
    "controlled_exceedance_probabilities_ordered",
    np.all(np.diff(exceedance_values, axis=1) <= PROBABILITY_TOLERANCE),
    f"maximum_positive_difference={float(np.diff(exceedance_values, axis=1).max()):.3e}",
)
append_check(
    validation_rows,
    "controlled_state_probabilities_nonnegative",
    state_probability_values.min() >= -PROBABILITY_TOLERANCE,
    f"minimum={float(state_probability_values.min()):.3e}",
)
append_check(
    validation_rows,
    "controlled_state_probabilities_sum_to_one",
    np.allclose(state_probability_values.sum(axis=1), 1.0, rtol=0.0, atol=1e-14),
    f"maximum_error={float(np.abs(state_probability_values.sum(axis=1) - 1.0).max()):.3e}",
)
append_check(
    validation_rows,
    "controlled_uniforms_strictly_inside_unit_interval",
    controlled_output["structural_damage_uniform"].between(0.0, 1.0, inclusive="neither").all(),
    (
        f"min={controlled_output['structural_damage_uniform'].min():.12f}; "
        f"max={controlled_output['structural_damage_uniform'].max():.12f}"
    ),
)
append_check(
    validation_rows,
    "controlled_sampled_states_valid",
    controlled_output["sampled_structural_damage_state"].between(0, 4).all(),
    f"states={sorted(controlled_output['sampled_structural_damage_state'].unique().tolist())}",
)

repeated_output = calculate_control_outputs(controls.copy())
append_check(
    validation_rows,
    "deterministic_repeat_is_exact",
    (
        np.array_equal(
            controlled_output["structural_damage_uniform"].to_numpy(),
            repeated_output["structural_damage_uniform"].to_numpy(),
        )
        and np.array_equal(
            controlled_output["sampled_structural_damage_state"].to_numpy(),
            repeated_output["sampled_structural_damage_state"].to_numpy(),
        )
    ),
    "uniforms and states reproduce exactly",
)

shuffled = controls.sample(frac=1.0, random_state=20260802).reset_index(drop=True)
shuffled_output = calculate_control_outputs(shuffled)
comparison_columns = [
    "occurrence_id",
    "site_id",
    "structural_damage_uniform",
    "sampled_structural_damage_state",
]
ordered_a = controlled_output[comparison_columns].sort_values(
    ["occurrence_id", "site_id"]
).reset_index(drop=True)
ordered_b = shuffled_output[comparison_columns].sort_values(
    ["occurrence_id", "site_id"]
).reset_index(drop=True)
append_check(
    validation_rows,
    "damage_stream_is_row_order_invariant",
    ordered_a.equals(ordered_b),
    "shuffled rows reproduce identical uniforms and states",
)

chunked_frames: list[pd.DataFrame] = []
for start in range(0, len(controls), 5):
    chunked_frames.append(calculate_control_outputs(controls.iloc[start : start + 5].copy()))
chunked_output = pd.concat(chunked_frames, ignore_index=True)
ordered_chunked = chunked_output[comparison_columns].sort_values(
    ["occurrence_id", "site_id"]
).reset_index(drop=True)
append_check(
    validation_rows,
    "damage_stream_is_chunk_size_invariant",
    ordered_a.equals(ordered_chunked),
    "five-row chunks reproduce identical uniforms and states",
)

fragility_wide["design_level"] = fragility_wide["design_level"].astype(str)
fragility_wide = fragility_wide.loc[
    fragility_wide["design_level"].isin(represented_design_levels)
].copy()
minimum_median = float(fragility_wide[median_columns].min().min())
maximum_median = float(fragility_wide[median_columns].max().max())
minimum_field_sa = float(min(row["minimum_sa0p4_g"] for row in selection_audit_rows))
maximum_field_sa = float(max(row["maximum_sa0p4_g"] for row in selection_audit_rows))
grid_minimum = max(1e-6, min(minimum_field_sa, minimum_median / 20.0))
grid_maximum = max(maximum_field_sa, maximum_median * 4.0)
sa_grid = np.geomspace(grid_minimum, grid_maximum, 401)

grid_rows: list[pd.DataFrame] = []
for _, parameter_row in fragility_wide.iterrows():
    design_level = str(parameter_row["design_level"])
    frame = pd.DataFrame(
        {
            "design_level": design_level,
            "sa0p4_simulated_g": sa_grid,
        }
    )
    for column in median_columns + beta_columns:
        frame[column] = float(parameter_row[column])
    probabilities = calculate_damage_probabilities(frame)
    grid_rows.append(pd.concat([frame, probabilities], axis=1))

probability_grid = pd.concat(grid_rows, ignore_index=True)
monotonic_by_level: dict[str, bool] = {}
for design_level, group in probability_grid.groupby("design_level", sort=False):
    ordered = group.sort_values("sa0p4_simulated_g")
    monotonic_by_level[str(design_level)] = bool(
        np.all(np.diff(ordered["expected_damage_state"].to_numpy()) >= -1e-12)
    )
append_check(
    validation_rows,
    "expected_damage_state_monotonic_with_sa0p4",
    all(monotonic_by_level.values()),
    f"by_design_level={monotonic_by_level}",
)

if {"PreCode", "LowCode"}.issubset(set(represented_design_levels)):
    pre = probability_grid.loc[probability_grid["design_level"].eq("PreCode")].reset_index(
        drop=True
    )
    low = probability_grid.loc[probability_grid["design_level"].eq("LowCode")].reset_index(
        drop=True
    )
    pre_parameters = fragility_wide.loc[
        fragility_wide["design_level"].eq("PreCode"), median_columns
    ].iloc[0].to_numpy(dtype=float)
    low_parameters = fragility_wide.loc[
        fragility_wide["design_level"].eq("LowCode"), median_columns
    ].iloc[0].to_numpy(dtype=float)
    expected_state_difference = (
        pre["expected_damage_state"].to_numpy(dtype=float)
        - low["expected_damage_state"].to_numpy(dtype=float)
    )
    exceedance_difference = (
        pre[exceedance_columns].to_numpy(dtype=float)
        - low[exceedance_columns].to_numpy(dtype=float)
    )
    append_check(
        validation_rows,
        "precode_fragility_medians_not_greater_than_lowcode",
        np.all(pre_parameters <= low_parameters),
        f"precode={pre_parameters.tolist()}; lowcode={low_parameters.tolist()}",
    )
    append_check(
        validation_rows,
        "precode_expected_damage_not_less_than_lowcode",
        expected_state_difference.min() >= -1e-12,
        f"minimum_expected_state_difference={float(expected_state_difference.min()):.3e}",
    )
    append_check(
        validation_rows,
        "individual_precode_exceedance_curve_crossing_diagnostic",
        exceedance_difference.min() >= -1e-4,
        f"minimum_exceedance_difference={float(exceedance_difference.min()):.3e}",
        severity="warning",
    )

candidate_diagnostic = candidate_pool.sort_values("control_selection_priority").head(25_000)
candidate_uniforms, _ = deterministic_uniforms(
    candidate_diagnostic["occurrence_id"], candidate_diagnostic["site_id"]
)
uniform_mean = float(candidate_uniforms.mean())
uniform_variance = float(candidate_uniforms.var())
ks_result = kstest(candidate_uniforms, "uniform")
append_check(
    validation_rows,
    "damage_uniform_mean_reasonable",
    abs(uniform_mean - 0.5) <= 0.015,
    f"n={len(candidate_uniforms)}; mean={uniform_mean:.6f}",
)
append_check(
    validation_rows,
    "damage_uniform_variance_reasonable",
    abs(uniform_variance - 1.0 / 12.0) <= 0.006,
    f"variance={uniform_variance:.6f}; target={1.0/12.0:.6f}",
)
append_check(
    validation_rows,
    "damage_uniform_ks_diagnostic",
    float(ks_result.pvalue) >= 1e-4,
    f"statistic={float(ks_result.statistic):.6f}; pvalue={float(ks_result.pvalue):.6f}",
    severity="warning",
)

monte_carlo_rows: list[dict[str, Any]] = []
for _, parameter_row in fragility_wide.iterrows():
    design_level = str(parameter_row["design_level"])
    test_im_values = {
        "below_slight": 0.50 * float(parameter_row["slight_median_sa0p4_g"]),
        "at_moderate": float(parameter_row["moderate_median_sa0p4_g"]),
        "at_complete": float(parameter_row["complete_median_sa0p4_g"]),
    }

    for test_level, test_im in test_im_values.items():
        probability_input = pd.DataFrame({"sa0p4_simulated_g": [test_im]})
        for column in median_columns + beta_columns:
            probability_input[column] = float(parameter_row[column])
        analytical = calculate_damage_probabilities(probability_input)
        analytical_probabilities = analytical[state_probability_columns].iloc[0].to_numpy(
            dtype=float
        )

        occurrence_ids = pd.Series(
            [f"MC|{design_level}|{test_level}|{index:05d}" for index in range(MONTE_CARLO_REPLICATES)]
        )
        site_ids = pd.Series(["SYNTHETIC_W2"] * MONTE_CARLO_REPLICATES)
        uniforms, _ = deterministic_uniforms(occurrence_ids, site_ids)
        replicated_probabilities = pd.DataFrame(
            np.repeat(
                analytical_probabilities[None, :], MONTE_CARLO_REPLICATES, axis=0
            ),
            columns=state_probability_columns,
        )
        sampled_states = sample_damage_states(replicated_probabilities, uniforms)
        empirical_probabilities = np.bincount(sampled_states, minlength=5) / float(
            MONTE_CARLO_REPLICATES
        )
        maximum_probability_error = float(
            np.abs(empirical_probabilities - analytical_probabilities).max()
        )
        empirical_expected_state = float(sampled_states.mean())
        analytical_expected_state = float(
            analytical_probabilities @ np.arange(5, dtype=float)
        )

        row: dict[str, Any] = {
            "design_level": design_level,
            "test_level": test_level,
            "sa0p4_g": test_im,
            "replicates": MONTE_CARLO_REPLICATES,
            "maximum_state_probability_error": maximum_probability_error,
            "analytical_expected_damage_state": analytical_expected_state,
            "empirical_expected_damage_state": empirical_expected_state,
            "expected_state_error": empirical_expected_state - analytical_expected_state,
        }
        for index, state in enumerate(DAMAGE_STATES):
            row[f"analytical_p_{state.lower()}"] = float(analytical_probabilities[index])
            row[f"empirical_p_{state.lower()}"] = float(empirical_probabilities[index])
        monte_carlo_rows.append(row)

monte_carlo_validation = pd.DataFrame(monte_carlo_rows)
maximum_monte_carlo_error = float(
    monte_carlo_validation["maximum_state_probability_error"].max()
)
append_check(
    validation_rows,
    "categorical_sampling_matches_analytical_probabilities",
    maximum_monte_carlo_error <= MONTE_CARLO_MAX_ABSOLUTE_ERROR,
    (
        f"maximum_absolute_error={maximum_monte_carlo_error:.6f}; "
        f"tolerance={MONTE_CARLO_MAX_ABSOLUTE_ERROR:.6f}"
    ),
)

selection_audit = pd.DataFrame(selection_audit_rows)
validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()

paths = {
    "controlled_damage": CONTROL_DIR / "controlled_structural_damage_rows.csv",
    "selection_audit": METADATA_DIR / "notebook_5_cell_3_control_selection_audit.csv",
    "probability_grid": CONTROL_DIR / "controlled_structural_damage_probability_grid.csv.gz",
    "monte_carlo": CONTROL_DIR / "controlled_structural_damage_monte_carlo_validation.csv",
    "random_stream_specification": METADATA_DIR
    / "notebook_5_cell_3_structural_damage_random_stream.json",
    "validation": METADATA_DIR / "notebook_5_cell_3_validation.csv",
    "summary": METADATA_DIR / "notebook_5_cell_3_summary.json",
}

controlled_output.to_csv(paths["controlled_damage"], index=False)
selection_audit.to_csv(paths["selection_audit"], index=False)
probability_grid.to_csv(paths["probability_grid"], index=False, compression="gzip")
monte_carlo_validation.to_csv(paths["monte_carlo"], index=False)
validation.to_csv(paths["validation"], index=False)

random_stream_specification = {
    "namespace": DAMAGE_STREAM_NAMESPACE,
    "key_fields": ["occurrence_id", "site_id"],
    "key_format": "namespace|occurrence_id|site_id",
    "hash_function": "SHA-256",
    "uniform_construction": "(unsigned_big_endian_first_8_bytes + 0.5) / 2^64",
    "uniform_interval": "strictly between 0 and 1",
    "component": "structural_damage",
    "reuse_rule": (
        "Reuse the same occurrence-site structural damage uniform in future "
        "independent-versus-spatially-correlated ground-motion comparisons."
    ),
    "row_order_invariant": True,
    "chunk_size_invariant": True,
}
write_json(paths["random_stream_specification"], random_stream_specification)

summary = {
    "pipeline_version": "notebook5_cell3_controlled_structural_damage_v1",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "ground_motion_fields": {
        "path": str(field_path),
        "sha256_from_cell1": field_info["sha256"],
        "rows": int(total_rows),
        "source_row_counts": source_row_counts,
        "structural_im_column": structural_im_column,
        "period_s": period_s,
    },
    "building_fragility_assignments": {
        "path": str(BUILDING_ASSIGNMENTS_PATH),
        "sha256": sha256_file(BUILDING_ASSIGNMENTS_PATH),
        "buildings": int(len(building_assignments)),
        "design_levels": represented_design_levels,
        "structural_type": EXPECTED_STRUCTURAL_TYPE,
    },
    "controlled_selection": {
        "selection_method": (
            "Order-independent modular occurrence-site preselection followed by "
            "SHA-256 priority and nearest sample q10, q50, and q90 SA0P4 rows."
        ),
        "selection_modulus": SELECTION_MODULUS,
        "selection_threshold": SELECTION_THRESHOLD,
        "rows": int(len(controlled_output)),
        "intensity_levels": list(CONTROL_QUANTILES.keys()),
        "source_types": source_types,
    },
    "damage_model": {
        "equation": "P(DS>=k|Sa)=Phi((ln(Sa)-ln(theta_k))/beta_k)",
        "mutually_exclusive_states": DAMAGE_STATES,
        "probability_cleanup": (
            "Only numerical values within 1e-12 of zero are clipped before "
            "renormalization; materially negative probabilities stop the cell."
        ),
        "random_stream_namespace": DAMAGE_STREAM_NAMESPACE,
    },
    "uniform_diagnostics": {
        "sample_size": int(len(candidate_uniforms)),
        "mean": uniform_mean,
        "variance": uniform_variance,
        "ks_statistic": float(ks_result.statistic),
        "ks_pvalue": float(ks_result.pvalue),
    },
    "monte_carlo_validation": {
        "replicates_per_case": MONTE_CARLO_REPLICATES,
        "cases": int(len(monte_carlo_validation)),
        "maximum_state_probability_error": maximum_monte_carlo_error,
        "tolerance": MONTE_CARLO_MAX_ABSOLUTE_ERROR,
    },
    "outputs": {key: str(path) for key, path in paths.items()},
    "next_cell": (
        "Cell 4: apply the validated structural fragility, deterministic "
        "occurrence-site uniform, and categorical sampling algorithm to all "
        "4,996,100 annual-catalog occurrence-building rows."
    ),
}
write_json(paths["summary"], summary)

print("=" * 78)
print("NOTEBOOK 5 CELL 3 CONTROLLED STRUCTURAL DAMAGE VALIDATION COMPLETE")
print("=" * 78)
print(f"Ground-motion rows inspected:      {total_rows:,}")
print(f"Controlled occurrence-site rows:   {len(controlled_output):,}")
print(f"Design levels represented:         {len(represented_design_levels):,}")
print(f"Source types represented:          {len(source_types):,}")
print(f"Probability-grid rows:             {len(probability_grid):,}")
print(f"Monte Carlo validation cases:      {len(monte_carlo_validation):,}")
print(f"Maximum Monte Carlo error:         {maximum_monte_carlo_error:.6f}")
print(f"Damage-uniform mean:               {uniform_mean:.6f}")
print(f"Damage-uniform variance:           {uniform_variance:.6f}")
print(f"Critical validation checks:        {int(validation['severity'].eq('critical').sum()):,}")
print(f"Critical failures:                 {len(critical_failures):,}")
print(f"Warnings requiring review:         {len(warning_failures):,}")
print()
print("Controlled damage rows:")
print(f"  {paths['controlled_damage']}")
print("Validation:")
print(f"  {paths['validation']}")
print("Summary:")
print(f"  {paths['summary']}")
print()
print("Next: full production structural damage-state sampling.")

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 3 failed one or more critical checks. Review: "
        f"{paths['validation']}"
    )


NOTEBOOK 5 CELL 3 CONTROLLED STRUCTURAL DAMAGE VALIDATION COMPLETE
Ground-motion rows inspected:      4,996,100
Controlled occurrence-site rows:   12
Design levels represented:         2
Source types represented:          2
Probability-grid rows:             802
Monte Carlo validation cases:      6
Maximum Monte Carlo error:         0.007189
Damage-uniform mean:               0.501162
Damage-uniform variance:           0.083264
Critical validation checks:        39
Critical failures:                 0
Warnings requiring review:         0

Controlled damage rows:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_controlled_structural_damage\controlled_structural_damage_rows.csv
Validation:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_5_damage_loss\notebook_5_cell_3_validation.csv
Summary:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_5_damage_loss\notebo

In [14]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
import shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.special import ndtr

DAMAGE_STATES = ["None", "Slight", "Moderate", "Extensive", "Complete"]
EXCEEDANCE_STATES = ["Slight", "Moderate", "Extensive", "Complete"]
EXPECTED_STRUCTURAL_TYPE = "W2"
EXPECTED_PERIOD_S = 0.40
EXPECTED_SITES = 470
OCCURRENCES_PER_CHUNK = 250
DAMAGE_STREAM_NAMESPACE = "notebook5_structural_damage_v1"
PIPELINE_VERSION = "notebook5_cell4_full_structural_damage_v1"
PROBABILITY_TOLERANCE = 1e-12
CONTROL_TOLERANCE = 2e-12
SAMPLED_EXPECTED_PROPORTION_TOLERANCE = 0.002


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_3_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = recorded_path.replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def normalize_source_type(values: pd.Series) -> pd.Series:
    return values.astype(str).str.strip().str.upper()


def deterministic_uniforms(
    occurrence_ids: pd.Series,
    site_ids: pd.Series,
    namespace: str = DAMAGE_STREAM_NAMESPACE,
) -> np.ndarray:
    occurrence_values = occurrence_ids.astype(str).to_numpy()
    site_values = site_ids.astype(str).to_numpy()
    uniforms = np.empty(len(occurrence_values), dtype=np.float64)
    denominator = float(2**64)

    for index, (occurrence_id, site_id) in enumerate(
        zip(occurrence_values, site_values, strict=True)
    ):
        digest = hashlib.sha256(
            f"{namespace}|{occurrence_id}|{site_id}".encode("utf-8")
        ).digest()
        integer = int.from_bytes(digest[:8], byteorder="big", signed=False)
        uniforms[index] = (integer + 0.5) / denominator

    return uniforms


def parameter_columns() -> tuple[list[str], list[str]]:
    medians = [f"{state.lower()}_median_sa0p4_g" for state in EXCEEDANCE_STATES]
    betas = [f"{state.lower()}_beta_ln" for state in EXCEEDANCE_STATES]
    return medians, betas


def calculate_damage_probabilities(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, float]:
    median_columns, beta_columns = parameter_columns()
    sa = pd.to_numeric(frame["sa0p4_simulated_g"], errors="raise").to_numpy(
        dtype=np.float64
    )
    medians = frame[median_columns].to_numpy(dtype=np.float64)
    betas = frame[beta_columns].to_numpy(dtype=np.float64)

    if not np.isfinite(sa).all() or np.any(sa <= 0.0):
        raise ValueError("All SA0P4 values must be finite and positive.")
    if not np.isfinite(medians).all() or np.any(medians <= 0.0):
        raise ValueError("All fragility medians must be finite and positive.")
    if not np.isfinite(betas).all() or np.any(betas <= 0.0):
        raise ValueError("All fragility betas must be finite and positive.")

    exceedance = ndtr((np.log(sa)[:, None] - np.log(medians)) / betas)
    raw = np.column_stack(
        [
            1.0 - exceedance[:, 0],
            exceedance[:, 0] - exceedance[:, 1],
            exceedance[:, 1] - exceedance[:, 2],
            exceedance[:, 2] - exceedance[:, 3],
            exceedance[:, 3],
        ]
    )
    minimum_raw = float(raw.min())
    if minimum_raw < -PROBABILITY_TOLERANCE:
        raise RuntimeError(
            "Fragility curves produced materially negative mutually exclusive "
            f"probabilities. Minimum={minimum_raw:.6e}."
        )

    probabilities = np.clip(raw, 0.0, 1.0)
    totals = probabilities.sum(axis=1)
    if np.any(totals <= 0.0):
        raise RuntimeError("At least one probability row has zero total mass.")
    probabilities /= totals[:, None]

    output = pd.DataFrame(index=frame.index)
    for index, state in enumerate(DAMAGE_STATES):
        output[f"p_state_{state.lower()}"] = probabilities[:, index]
    output["expected_structural_damage_state"] = (
        probabilities @ np.arange(5, dtype=np.float64)
    )
    return output, minimum_raw


def sample_damage_states(probabilities: pd.DataFrame, uniforms: np.ndarray) -> np.ndarray:
    probability_columns = [f"p_state_{state.lower()}" for state in DAMAGE_STATES]
    cumulative = np.cumsum(
        probabilities[probability_columns].to_numpy(dtype=np.float64), axis=1
    )
    cumulative[:, -1] = 1.0
    states = (uniforms[:, None] > cumulative).sum(axis=1)
    return np.minimum(states, 4).astype(np.int8)


def stable_frame_hash(frame: pd.DataFrame, columns: list[str]) -> str:
    values = pd.util.hash_pandas_object(frame[columns], index=False).to_numpy(
        dtype=np.uint64
    )
    return hashlib.sha256(values.tobytes()).hexdigest()


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            filename="", mode="wb", fileobj=raw, compresslevel=6, mtime=0
        ) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8", newline="") as text:
                frame.to_csv(
                    text,
                    index=False,
                    float_format="%.17g",
                    lineterminator="\n",
                )
    temporary.replace(path)


def concatenate_gzip_csv_files(chunk_paths: list[Path], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    expected_header: bytes | None = None

    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            filename="", mode="wb", fileobj=raw, compresslevel=6, mtime=0
        ) as destination:
            for index, chunk_path in enumerate(chunk_paths):
                with gzip.open(chunk_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        destination.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            f"Chunk header mismatch while concatenating {chunk_path}."
                        )
                    shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)

    temporary.replace(output_path)


def summarize_occurrences(frame: pd.DataFrame) -> pd.DataFrame:
    group_columns = [
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
    ]
    probability_columns = [f"p_state_{state.lower()}" for state in DAMAGE_STATES]

    base = (
        frame.groupby(group_columns, sort=False, observed=True)
        .agg(
            buildings=("site_id", "size"),
            minimum_sa0p4_g=("sa0p4_simulated_g", "min"),
            mean_sa0p4_g=("sa0p4_simulated_g", "mean"),
            maximum_sa0p4_g=("sa0p4_simulated_g", "max"),
            mean_expected_damage_state=("expected_structural_damage_state", "mean"),
            mean_sampled_damage_state=("sampled_structural_damage_state", "mean"),
        )
        .reset_index()
    )

    sampled_counts = (
        frame.groupby(group_columns + ["sampled_structural_damage_state"], sort=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=range(5), fill_value=0)
        .rename(columns={index: f"sampled_{state.lower()}_buildings" for index, state in enumerate(DAMAGE_STATES)})
        .reset_index()
    )

    expected_counts = (
        frame.groupby(group_columns, sort=False, observed=True)[probability_columns]
        .sum()
        .rename(columns={f"p_state_{state.lower()}": f"expected_{state.lower()}_buildings" for state in DAMAGE_STATES})
        .reset_index()
    )

    output = base.merge(sampled_counts, on=group_columns, how="left", validate="one_to_one")
    output = output.merge(expected_counts, on=group_columns, how="left", validate="one_to_one")
    output["sampled_slight_or_worse_buildings"] = output[
        [f"sampled_{state.lower()}_buildings" for state in DAMAGE_STATES[1:]]
    ].sum(axis=1)
    output["expected_slight_or_worse_buildings"] = output[
        [f"expected_{state.lower()}_buildings" for state in DAMAGE_STATES[1:]]
    ].sum(axis=1)
    return output.sort_values("occurrence_ordinal").reset_index(drop=True)


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_structural_damage"
CHUNK_DIR = OUTPUT_DIR / "chunks"
WORK_DIR = METADATA_DIR / "notebook_5_cell_4_work"
MARKER_DIR = WORK_DIR / "markers"
CHUNK_VALIDATION_DIR = WORK_DIR / "chunk_validations"
for directory in [METADATA_DIR, OUTPUT_DIR, CHUNK_DIR, WORK_DIR, MARKER_DIR, CHUNK_VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CELL1_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_1_summary.json"
CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_1_validation.csv"
CELL2_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_2_summary.json"
CELL2_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_2_validation.csv"
CELL3_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_3_summary.json"
CELL3_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_3_validation.csv"
BUILDING_ASSIGNMENTS_PATH = PARAMETER_DIR / "seaside_w2_building_fragility_assignments.csv"
CONTROLLED_DAMAGE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_5_controlled_structural_damage"
    / "controlled_structural_damage_rows.csv"
)

required_paths = [
    CELL1_SUMMARY_PATH,
    CELL1_VALIDATION_PATH,
    CELL2_SUMMARY_PATH,
    CELL2_VALIDATION_PATH,
    CELL3_SUMMARY_PATH,
    CELL3_VALIDATION_PATH,
    BUILDING_ASSIGNMENTS_PATH,
    CONTROLLED_DAMAGE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required Cell 1 through Cell 3 outputs are missing: {missing_paths}")

validation_rows: list[dict[str, Any]] = []
cell1_summary = load_json(CELL1_SUMMARY_PATH)
cell2_summary = load_json(CELL2_SUMMARY_PATH)
cell3_summary = load_json(CELL3_SUMMARY_PATH)
cell1_validation = pd.read_csv(CELL1_VALIDATION_PATH)
cell2_validation = pd.read_csv(CELL2_VALIDATION_PATH)
cell3_validation = pd.read_csv(CELL3_VALIDATION_PATH)

for cell_number, summary, validation in [
    (1, cell1_summary, cell1_validation),
    (2, cell2_summary, cell2_validation),
    (3, cell3_summary, cell3_validation),
]:
    append_check(
        validation_rows,
        f"cell{cell_number}_critical_checks_passed",
        summary.get("all_critical_checks_passed") is True,
        f"all_critical_checks_passed={summary.get('all_critical_checks_passed')}",
    )
    parsed = parse_bool_series(validation["passed"])
    append_check(
        validation_rows,
        f"cell{cell_number}_has_no_unresolved_checks",
        parsed.all(),
        f"unresolved={int((~parsed).sum())}",
    )

field_info = cell3_summary["ground_motion_fields"]
field_path = resolve_recorded_path(PROJECT_ROOT, field_info["path"])
expected_rows = int(field_info["rows"])
structural_im_column = str(field_info["structural_im_column"])
period_s = float(field_info["period_s"])
expected_sites = int(cell2_summary["portfolio"]["buildings"])

append_check(validation_rows, "ground_motion_field_exists", field_path.exists(), str(field_path))
append_check(
    validation_rows,
    "structural_im_is_sa0p4",
    structural_im_column == "sa0p4_simulated_g",
    f"column={structural_im_column}",
)
append_check(
    validation_rows,
    "period_is_exact_sa0p4",
    math.isclose(period_s, EXPECTED_PERIOD_S, rel_tol=0.0, abs_tol=1e-12),
    f"period_s={period_s}",
)
append_check(
    validation_rows,
    "portfolio_has_470_sites",
    expected_sites == EXPECTED_SITES,
    f"sites={expected_sites}",
)
append_check(
    validation_rows,
    "field_rows_divisible_by_sites",
    expected_rows % expected_sites == 0,
    f"rows={expected_rows}; sites={expected_sites}",
)
expected_occurrences = expected_rows // expected_sites

actual_field_hash = sha256_file(field_path)
append_check(
    validation_rows,
    "ground_motion_hash_matches_cell1",
    actual_field_hash == str(field_info["sha256_from_cell1"]),
    f"calculated={actual_field_hash}; recorded={field_info['sha256_from_cell1']}",
)

building_assignments = pd.read_csv(BUILDING_ASSIGNMENTS_PATH)
controlled_damage = pd.read_csv(CONTROLLED_DAMAGE_PATH)
median_columns, beta_columns = parameter_columns()
probability_columns = [f"p_state_{state.lower()}" for state in DAMAGE_STATES]

required_assignment_columns = {
    "site_id",
    "structural_type",
    "design_level",
    "occ_type",
    "year_built",
    *median_columns,
    *beta_columns,
}
missing_assignment_columns = sorted(required_assignment_columns.difference(building_assignments.columns))
append_check(
    validation_rows,
    "building_assignment_columns_present",
    not missing_assignment_columns,
    f"missing={missing_assignment_columns}",
)
if missing_assignment_columns:
    raise KeyError(f"Building assignments are missing columns: {missing_assignment_columns}")

building_assignments["site_id"] = building_assignments["site_id"].astype(str)
building_assignments["structural_type"] = (
    building_assignments["structural_type"].astype(str).str.strip().str.upper()
)
building_assignments["design_level"] = building_assignments["design_level"].astype(str)
building_assignments["occ_type"] = building_assignments["occ_type"].astype(str)

append_check(
    validation_rows,
    "building_assignment_hash_matches_cell3",
    sha256_file(BUILDING_ASSIGNMENTS_PATH)
    == str(cell3_summary["building_fragility_assignments"]["sha256"]),
    "Cell 4 building assignments must match the file validated in Cell 3.",
)
append_check(
    validation_rows,
    "building_assignment_rows_match_sites",
    len(building_assignments) == expected_sites,
    f"rows={len(building_assignments)}; expected={expected_sites}",
)
append_check(
    validation_rows,
    "building_site_ids_unique",
    building_assignments["site_id"].is_unique,
    f"duplicates={int(building_assignments['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "all_buildings_are_w2",
    building_assignments["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(building_assignments['structural_type'].unique().tolist())}",
)

field_header = pd.read_csv(field_path, nrows=0)
required_field_columns = {
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    structural_im_column,
}
missing_field_columns = sorted(required_field_columns.difference(field_header.columns))
append_check(
    validation_rows,
    "ground_motion_columns_present",
    not missing_field_columns,
    f"missing={missing_field_columns}",
)
if missing_field_columns:
    raise KeyError(f"Ground-motion file is missing columns: {missing_field_columns}")

optional_field_columns = [
    column
    for column in ["catalog_event_id", "rupture_ordinal", "rupture_template_event_id"]
    if column in field_header.columns
]
field_usecols = [
    "catalog_year",
    *optional_field_columns,
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    structural_im_column,
]
lookup_columns = [
    "site_id",
    "structural_type",
    "design_level",
    "occ_type",
    "year_built",
    *median_columns,
    *beta_columns,
]
design_lookup = building_assignments[lookup_columns].copy()

basis_payload = {
    "pipeline_version": PIPELINE_VERSION,
    "ground_motion_sha256": actual_field_hash,
    "building_assignments_sha256": sha256_file(BUILDING_ASSIGNMENTS_PATH),
    "cell3_summary_sha256": sha256_file(CELL3_SUMMARY_PATH),
    "cell3_validation_sha256": sha256_file(CELL3_VALIDATION_PATH),
    "controlled_damage_sha256": sha256_file(CONTROLLED_DAMAGE_PATH),
    "damage_stream_namespace": DAMAGE_STREAM_NAMESPACE,
    "occurrences_per_chunk": OCCURRENCES_PER_CHUNK,
}
basis_hash = hashlib.sha256(
    json.dumps(basis_payload, sort_keys=True).encode("utf-8")
).hexdigest()

chunk_rows = expected_sites * OCCURRENCES_PER_CHUNK
expected_chunks = math.ceil(expected_rows / chunk_rows)
control_keys = set(
    zip(controlled_damage["occurrence_id"].astype(str), controlled_damage["site_id"].astype(str))
)
control_production_frames: list[pd.DataFrame] = []
chunk_manifest_rows: list[dict[str, Any]] = []
chunk_paths: list[Path] = []
event_summary_frames: list[pd.DataFrame] = []

seen_pairs = np.zeros(expected_rows, dtype=bool)
occurrence_counts = np.zeros(expected_occurrences, dtype=np.int64)
site_counts = np.zeros(expected_sites, dtype=np.int64)
seen_occurrences: set[str] = set()
source_row_counts: Counter[str] = Counter()
design_level_row_counts: Counter[str] = Counter()
sampled_state_counts = np.zeros(5, dtype=np.int64)
expected_state_counts = np.zeros(5, dtype=np.float64)
uniform_sum = 0.0
uniform_square_sum = 0.0
total_processed_rows = 0
global_minimum_raw_probability = math.inf
global_max_probability_sum_error = 0.0

print("=" * 78)
print("FULL ANNUAL-CATALOG STRUCTURAL DAMAGE-STATE SAMPLING")
print("=" * 78)
print(f"Ground-motion rows:       {expected_rows:,}")
print(f"Catalog occurrences:      {expected_occurrences:,}")
print(f"Portfolio buildings:      {expected_sites:,}")
print(f"Occurrences per chunk:    {OCCURRENCES_PER_CHUNK:,}")
print(f"Production chunks:        {expected_chunks:,}")
print(f"Structural IM:            exact SA0P4")
print(f"Damage random namespace:  {DAMAGE_STREAM_NAMESPACE}")

reader = pd.read_csv(field_path, usecols=field_usecols, chunksize=chunk_rows)
for chunk_index, raw_chunk in enumerate(reader):
    chunk_id = f"{chunk_index:04d}"
    print()
    print("-" * 78)
    print(f"STRUCTURAL DAMAGE CHUNK {chunk_index + 1} OF {expected_chunks} [ID {chunk_id}]")

    raw_chunk = raw_chunk.rename(columns={structural_im_column: "sa0p4_simulated_g"})
    raw_chunk["site_id"] = raw_chunk["site_id"].astype(str)
    raw_chunk["occurrence_id"] = raw_chunk["occurrence_id"].astype(str)
    raw_chunk["source_type"] = normalize_source_type(raw_chunk["source_type"])
    raw_chunk["occurrence_ordinal"] = pd.to_numeric(
        raw_chunk["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    raw_chunk["site_ordinal"] = pd.to_numeric(
        raw_chunk["site_ordinal"], errors="raise"
    ).astype(np.int64)
    raw_chunk["sa0p4_simulated_g"] = pd.to_numeric(
        raw_chunk["sa0p4_simulated_g"], errors="raise"
    )

    input_hash_columns = [
        "occurrence_ordinal",
        "occurrence_id",
        "site_ordinal",
        "site_id",
        "sa0p4_simulated_g",
    ]
    input_hash = stable_frame_hash(raw_chunk, input_hash_columns)
    chunk_output_path = CHUNK_DIR / f"structural_damage_chunk_{chunk_id}.csv.gz"
    marker_path = MARKER_DIR / f"structural_damage_chunk_{chunk_id}.json"
    chunk_validation_path = (
        CHUNK_VALIDATION_DIR / f"structural_damage_chunk_{chunk_id}_validation.csv"
    )

    marker_valid = False
    marker: dict[str, Any] = {}
    if marker_path.exists() and chunk_output_path.exists() and chunk_validation_path.exists():
        try:
            marker = load_json(marker_path)
            marker_valid = (
                marker.get("basis_hash") == basis_hash
                and marker.get("input_hash") == input_hash
                and int(marker.get("rows", -1)) == len(raw_chunk)
                and marker.get("output_sha256") == sha256_file(chunk_output_path)
            )
        except Exception:
            marker_valid = False

    if marker_valid:
        output = pd.read_csv(chunk_output_path)
        print(f"Reused validated chunk with {len(output):,} rows.")
    else:
        chunk_validation_rows: list[dict[str, Any]] = []
        occurrence_sizes = raw_chunk.groupby("occurrence_id", sort=False).size()
        append_check(
            chunk_validation_rows,
            "input_occurrences_complete",
            occurrence_sizes.eq(expected_sites).all(),
            (
                f"occurrences={len(occurrence_sizes)}; minimum={int(occurrence_sizes.min())}; "
                f"maximum={int(occurrence_sizes.max())}; expected={expected_sites}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "input_occurrence_ids_not_seen_in_prior_chunks",
            not set(occurrence_sizes.index.astype(str)).intersection(seen_occurrences),
            "Each occurrence must be fully contained in one production chunk.",
        )

        merged = raw_chunk.merge(
            design_lookup, on="site_id", how="left", validate="many_to_one"
        )
        missing_assignments = int(merged["design_level"].isna().sum())
        append_check(
            chunk_validation_rows,
            "all_sites_have_fragility_assignments",
            missing_assignments == 0,
            f"missing_rows={missing_assignments}",
        )
        if missing_assignments:
            raise RuntimeError(
                f"Chunk {chunk_id} has {missing_assignments} rows without assignments."
            )

        probabilities, minimum_raw = calculate_damage_probabilities(merged)
        uniforms = deterministic_uniforms(merged["occurrence_id"], merged["site_id"])
        sampled_states = sample_damage_states(probabilities, uniforms)

        output = merged[
            [
                "catalog_year",
                *optional_field_columns,
                "occurrence_ordinal",
                "occurrence_id",
                "rupture_id",
                "source_type",
                "magnitude",
                "site_ordinal",
                "site_id",
                "sa0p4_simulated_g",
                "structural_type",
                "design_level",
                "occ_type",
                "year_built",
            ]
        ].copy()
        output = pd.concat(
            [output.reset_index(drop=True), probabilities.reset_index(drop=True)], axis=1
        )
        output["structural_damage_uniform"] = uniforms
        output["sampled_structural_damage_state"] = sampled_states
        output["sampled_structural_damage_state_name"] = pd.Series(sampled_states).map(
            dict(enumerate(DAMAGE_STATES))
        )

        probability_matrix = output[probability_columns].to_numpy(dtype=np.float64)
        probability_sum_error = float(
            np.max(np.abs(probability_matrix.sum(axis=1) - 1.0))
        )
        append_check(
            chunk_validation_rows,
            "output_rows_match_input",
            len(output) == len(raw_chunk),
            f"output={len(output)}; input={len(raw_chunk)}",
        )
        append_check(
            chunk_validation_rows,
            "probabilities_finite",
            np.isfinite(probability_matrix).all(),
            "All mutually exclusive probabilities must be finite.",
        )
        append_check(
            chunk_validation_rows,
            "probabilities_nonnegative",
            probability_matrix.min() >= -PROBABILITY_TOLERANCE,
            f"minimum={float(probability_matrix.min()):.3e}",
        )
        append_check(
            chunk_validation_rows,
            "probabilities_sum_to_one",
            probability_sum_error <= 2e-15,
            f"maximum_error={probability_sum_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "expected_damage_state_in_range",
            output["expected_structural_damage_state"].between(0.0, 4.0).all(),
            (
                f"minimum={output['expected_structural_damage_state'].min():.6f}; "
                f"maximum={output['expected_structural_damage_state'].max():.6f}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "uniforms_strictly_inside_unit_interval",
            np.all((uniforms > 0.0) & (uniforms < 1.0)),
            f"minimum={uniforms.min():.6e}; maximum={uniforms.max():.6e}",
        )
        append_check(
            chunk_validation_rows,
            "sampled_states_in_range",
            np.all((sampled_states >= 0) & (sampled_states <= 4)),
            f"minimum={sampled_states.min()}; maximum={sampled_states.max()}",
        )
        repeated_uniforms = deterministic_uniforms(
            merged["occurrence_id"].iloc[: min(1000, len(merged))],
            merged["site_id"].iloc[: min(1000, len(merged))],
        )
        append_check(
            chunk_validation_rows,
            "deterministic_uniform_reproduction",
            np.array_equal(repeated_uniforms, uniforms[: len(repeated_uniforms)]),
            f"sample_rows={len(repeated_uniforms)}",
        )
        repeated_states = sample_damage_states(probabilities, uniforms)
        append_check(
            chunk_validation_rows,
            "categorical_state_reconstruction",
            np.array_equal(repeated_states, sampled_states),
            "Sampled states must be exactly reproducible from probabilities and uniforms.",
        )
        pair_duplicates = int(output.duplicated(["occurrence_id", "site_id"]).sum())
        append_check(
            chunk_validation_rows,
            "occurrence_site_pairs_unique",
            pair_duplicates == 0,
            f"duplicates={pair_duplicates}",
        )
        append_check(
            chunk_validation_rows,
            "source_types_valid",
            set(output["source_type"].unique()).issubset({"INTERFACE", "SLAB"}),
            f"types={sorted(output['source_type'].unique().tolist())}",
        )
        append_check(
            chunk_validation_rows,
            "structural_type_is_w2",
            output["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
            f"types={sorted(output['structural_type'].unique().tolist())}",
        )

        chunk_validation = pd.DataFrame(chunk_validation_rows)
        chunk_failures = chunk_validation.loc[
            chunk_validation["severity"].eq("critical") & ~chunk_validation["passed"]
        ]
        chunk_validation.to_csv(chunk_validation_path, index=False)
        if not chunk_failures.empty:
            raise RuntimeError(
                f"Structural damage chunk {chunk_id} failed validation. Review {chunk_validation_path}."
            )

        write_gzip_csv_deterministic(output, chunk_output_path)
        marker = {
            "pipeline_version": PIPELINE_VERSION,
            "basis_hash": basis_hash,
            "input_hash": input_hash,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_raw_probability": minimum_raw,
            "output_path": str(chunk_output_path),
            "output_sha256": sha256_file(chunk_output_path),
            "validation_path": str(chunk_validation_path),
            "validation_sha256": sha256_file(chunk_validation_path),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        write_json(marker_path, marker)
        print(
            f"Calculated {len(output):,} rows for "
            f"{output['occurrence_id'].nunique():,} occurrences."
        )

    required_output_columns = {
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
        "site_ordinal",
        "site_id",
        "sa0p4_simulated_g",
        "structural_type",
        "design_level",
        "occ_type",
        "year_built",
        *probability_columns,
        "expected_structural_damage_state",
        "structural_damage_uniform",
        "sampled_structural_damage_state",
        "sampled_structural_damage_state_name",
    }
    missing_output_columns = sorted(required_output_columns.difference(output.columns))
    if missing_output_columns:
        raise KeyError(f"Chunk {chunk_id} output is missing columns: {missing_output_columns}")

    output["occurrence_ordinal"] = pd.to_numeric(
        output["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    output["site_ordinal"] = pd.to_numeric(output["site_ordinal"], errors="raise").astype(
        np.int64
    )
    output["sampled_structural_damage_state"] = pd.to_numeric(
        output["sampled_structural_damage_state"], errors="raise"
    ).astype(np.int8)
    for column in probability_columns + [
        "expected_structural_damage_state",
        "structural_damage_uniform",
        "sa0p4_simulated_g",
    ]:
        output[column] = pd.to_numeric(output[column], errors="raise")

    occurrence_ordinals = output["occurrence_ordinal"].to_numpy(dtype=np.int64)
    site_ordinals = output["site_ordinal"].to_numpy(dtype=np.int64)
    if np.any(occurrence_ordinals < 0) or np.any(occurrence_ordinals >= expected_occurrences):
        raise RuntimeError(f"Chunk {chunk_id} has occurrence ordinals outside the expected range.")
    if np.any(site_ordinals < 0) or np.any(site_ordinals >= expected_sites):
        raise RuntimeError(f"Chunk {chunk_id} has site ordinals outside the expected range.")

    pair_indices = occurrence_ordinals * expected_sites + site_ordinals
    if len(np.unique(pair_indices)) != len(pair_indices):
        raise RuntimeError(f"Chunk {chunk_id} contains duplicate ordinal pair indices.")
    if seen_pairs[pair_indices].any():
        raise RuntimeError(f"Chunk {chunk_id} contains occurrence-site pairs seen previously.")
    seen_pairs[pair_indices] = True
    np.add.at(occurrence_counts, occurrence_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    occurrence_ids = set(output["occurrence_id"].astype(str).unique().tolist())
    repeated_occurrences = occurrence_ids.intersection(seen_occurrences)
    if repeated_occurrences:
        raise RuntimeError(
            f"Chunk {chunk_id} repeats occurrences from prior chunks: "
            f"{sorted(repeated_occurrences)[:5]}"
        )
    seen_occurrences.update(occurrence_ids)

    probability_matrix = output[probability_columns].to_numpy(dtype=np.float64)
    sampled_states = output["sampled_structural_damage_state"].to_numpy(dtype=np.int8)
    uniforms = output["structural_damage_uniform"].to_numpy(dtype=np.float64)
    sampled_state_counts += np.bincount(sampled_states, minlength=5)
    expected_state_counts += probability_matrix.sum(axis=0)
    uniform_sum += float(uniforms.sum())
    uniform_square_sum += float(np.square(uniforms).sum())
    total_processed_rows += len(output)
    source_row_counts.update(output["source_type"].astype(str).tolist())
    design_level_row_counts.update(output["design_level"].astype(str).tolist())
    global_minimum_raw_probability = min(
        global_minimum_raw_probability,
        float(marker.get("minimum_raw_probability", probability_matrix.min())),
    )
    global_max_probability_sum_error = max(
        global_max_probability_sum_error,
        float(np.max(np.abs(probability_matrix.sum(axis=1) - 1.0))),
    )

    key_mask = np.fromiter(
        (
            (occurrence_id, site_id) in control_keys
            for occurrence_id, site_id in zip(
                output["occurrence_id"].astype(str),
                output["site_id"].astype(str),
                strict=True,
            )
        ),
        dtype=bool,
        count=len(output),
    )
    if key_mask.any():
        control_production_frames.append(output.loc[key_mask].copy())

    event_summary_frames.append(summarize_occurrences(output))
    chunk_paths.append(chunk_output_path)
    chunk_manifest_rows.append(
        {
            "chunk_id": chunk_id,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_occurrence_ordinal": int(output["occurrence_ordinal"].min()),
            "maximum_occurrence_ordinal": int(output["occurrence_ordinal"].max()),
            "output_path": str(chunk_output_path),
            "output_size_bytes": int(chunk_output_path.stat().st_size),
            "output_sha256": sha256_file(chunk_output_path),
            "marker_path": str(marker_path),
            "validation_path": str(chunk_validation_path),
            "reused": bool(marker_valid),
        }
    )

if len(chunk_paths) != expected_chunks:
    raise RuntimeError(
        f"Processed {len(chunk_paths)} chunks but expected {expected_chunks}."
    )

final_damage_path = OUTPUT_DIR / "full_structural_damage_states.csv.gz"
final_event_summary_path = OUTPUT_DIR / "structural_damage_event_summary.csv.gz"
state_totals_path = METADATA_DIR / "notebook_5_cell_4_structural_damage_state_totals.csv"
chunk_manifest_path = METADATA_DIR / "notebook_5_cell_4_chunk_manifest.csv"
validation_path = METADATA_DIR / "notebook_5_cell_4_validation.csv"
summary_path = METADATA_DIR / "notebook_5_cell_4_summary.json"
random_specification_path = METADATA_DIR / "notebook_5_cell_4_random_stream_specification.json"

concatenate_gzip_csv_files(chunk_paths, final_damage_path)
event_summary = pd.concat(event_summary_frames, ignore_index=True)
event_summary = event_summary.sort_values("occurrence_ordinal").reset_index(drop=True)
write_gzip_csv_deterministic(event_summary, final_event_summary_path)

control_production = pd.concat(control_production_frames, ignore_index=True)
controlled_for_compare = controlled_damage.copy()
if (
    "expected_damage_state" in controlled_for_compare.columns
    and "expected_structural_damage_state" not in controlled_for_compare.columns
):
    controlled_for_compare = controlled_for_compare.rename(
        columns={"expected_damage_state": "expected_structural_damage_state"}
    )
control_compare = controlled_for_compare.merge(
    control_production,
    on=["occurrence_id", "site_id"],
    how="outer",
    suffixes=("_cell3", "_cell4"),
    indicator=True,
    validate="one_to_one",
)
append_check(
    validation_rows,
    "cell3_controls_reproduced_once",
    len(control_compare) == len(controlled_damage)
    and control_compare["_merge"].eq("both").all(),
    f"cell3={len(controlled_damage)}; compared={len(control_compare)}; merge={control_compare['_merge'].value_counts().to_dict()}",
)

control_column_pairs = [
    ("sa0p4_simulated_g", "sa0p4_simulated_g"),
    *[(column, column) for column in probability_columns],
    ("expected_structural_damage_state", "expected_structural_damage_state"),
    ("structural_damage_uniform", "structural_damage_uniform"),
]
controls_complete = (
    len(control_compare) == len(controlled_damage)
    and control_compare["_merge"].eq("both").all()
)
maximum_control_difference = 0.0 if controls_complete else math.inf
if controls_complete:
    for cell3_column, cell4_column in control_column_pairs:
        left = pd.to_numeric(
            control_compare[f"{cell3_column}_cell3"], errors="raise"
        ).to_numpy(dtype=np.float64)
        right = pd.to_numeric(
            control_compare[f"{cell4_column}_cell4"], errors="raise"
        ).to_numpy(dtype=np.float64)
        maximum_control_difference = max(
            maximum_control_difference, float(np.max(np.abs(left - right)))
        )
append_check(
    validation_rows,
    "cell3_control_numeric_values_reproduced",
    maximum_control_difference <= CONTROL_TOLERANCE,
    f"maximum_absolute_difference={maximum_control_difference:.3e}",
)
append_check(
    validation_rows,
    "cell3_control_sampled_states_reproduced",
    np.array_equal(
        pd.to_numeric(
            control_compare["sampled_structural_damage_state_cell3"], errors="raise"
        ).to_numpy(dtype=np.int8),
        pd.to_numeric(
            control_compare["sampled_structural_damage_state_cell4"], errors="raise"
        ).to_numpy(dtype=np.int8),
    ),
    "Cell 4 must exactly reproduce the 12 Cell 3 sampled states.",
)

uniform_mean = uniform_sum / total_processed_rows
uniform_variance = uniform_square_sum / total_processed_rows - uniform_mean**2
sampled_proportions = sampled_state_counts / float(total_processed_rows)
expected_proportions = expected_state_counts / float(total_processed_rows)
maximum_sampled_expected_difference = float(
    np.max(np.abs(sampled_proportions - expected_proportions))
)

append_check(
    validation_rows,
    "production_rows_complete",
    total_processed_rows == expected_rows,
    f"processed={total_processed_rows}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "all_occurrence_site_pairs_present_once",
    seen_pairs.all(),
    f"missing_pairs={int((~seen_pairs).sum())}",
)
append_check(
    validation_rows,
    "every_occurrence_has_all_sites",
    np.all(occurrence_counts == expected_sites),
    (
        f"minimum={int(occurrence_counts.min())}; maximum={int(occurrence_counts.max())}; "
        f"expected={expected_sites}"
    ),
)
append_check(
    validation_rows,
    "every_site_has_all_occurrences",
    np.all(site_counts == expected_occurrences),
    (
        f"minimum={int(site_counts.min())}; maximum={int(site_counts.max())}; "
        f"expected={expected_occurrences}"
    ),
)
append_check(
    validation_rows,
    "unique_occurrences_complete",
    len(seen_occurrences) == expected_occurrences,
    f"observed={len(seen_occurrences)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "event_summary_has_one_row_per_occurrence",
    len(event_summary) == expected_occurrences
    and event_summary["occurrence_id"].is_unique,
    f"rows={len(event_summary)}; unique={event_summary['occurrence_id'].nunique()}",
)
append_check(
    validation_rows,
    "event_summary_building_counts_are_470",
    event_summary["buildings"].eq(expected_sites).all(),
    f"minimum={event_summary['buildings'].min()}; maximum={event_summary['buildings'].max()}",
)
append_check(
    validation_rows,
    "sampled_state_counts_sum_to_rows",
    int(sampled_state_counts.sum()) == expected_rows,
    f"sum={int(sampled_state_counts.sum())}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "expected_state_counts_sum_to_rows",
    abs(float(expected_state_counts.sum()) - expected_rows) <= 1e-6,
    f"sum={float(expected_state_counts.sum()):.6f}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "production_uniform_mean_reasonable",
    abs(uniform_mean - 0.5) <= 0.001,
    f"mean={uniform_mean:.8f}",
)
append_check(
    validation_rows,
    "production_uniform_variance_reasonable",
    abs(uniform_variance - 1.0 / 12.0) <= 0.0005,
    f"variance={uniform_variance:.8f}; target={1.0/12.0:.8f}",
)
append_check(
    validation_rows,
    "sampled_state_proportions_match_expected_probabilities",
    maximum_sampled_expected_difference <= SAMPLED_EXPECTED_PROPORTION_TOLERANCE,
    (
        f"maximum_absolute_difference={maximum_sampled_expected_difference:.6f}; "
        f"tolerance={SAMPLED_EXPECTED_PROPORTION_TOLERANCE:.6f}"
    ),
)
append_check(
    validation_rows,
    "source_row_counts_match_cell3",
    dict(sorted(source_row_counts.items()))
    == {str(key): int(value) for key, value in sorted(field_info["source_row_counts"].items())},
    f"observed={dict(sorted(source_row_counts.items()))}; expected={field_info['source_row_counts']}",
)
append_check(
    validation_rows,
    "probability_sum_error_small",
    global_max_probability_sum_error <= 2e-15,
    f"maximum_error={global_max_probability_sum_error:.3e}",
)
append_check(
    validation_rows,
    "minimum_raw_probability_within_roundoff",
    global_minimum_raw_probability >= -PROBABILITY_TOLERANCE,
    f"minimum={global_minimum_raw_probability:.3e}",
)
append_check(
    validation_rows,
    "final_damage_file_exists",
    final_damage_path.exists() and final_damage_path.stat().st_size > 0,
    f"path={final_damage_path}; bytes={final_damage_path.stat().st_size if final_damage_path.exists() else 0}",
)
append_check(
    validation_rows,
    "final_event_summary_exists",
    final_event_summary_path.exists() and final_event_summary_path.stat().st_size > 0,
    f"path={final_event_summary_path}; bytes={final_event_summary_path.stat().st_size if final_event_summary_path.exists() else 0}",
)

state_totals = pd.DataFrame(
    {
        "damage_state": DAMAGE_STATES,
        "damage_state_index": np.arange(5, dtype=int),
        "sampled_building_event_rows": sampled_state_counts,
        "sampled_proportion": sampled_proportions,
        "analytical_expected_building_event_rows": expected_state_counts,
        "analytical_expected_proportion": expected_proportions,
        "sampled_minus_expected_proportion": sampled_proportions - expected_proportions,
    }
)
state_totals.to_csv(state_totals_path, index=False)
chunk_manifest = pd.DataFrame(chunk_manifest_rows)
chunk_manifest.to_csv(chunk_manifest_path, index=False)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()
validation.to_csv(validation_path, index=False)

random_specification = {
    "namespace": DAMAGE_STREAM_NAMESPACE,
    "key_fields": ["occurrence_id", "site_id"],
    "key_format": "namespace|occurrence_id|site_id",
    "hash_function": "SHA-256",
    "uniform_construction": "(unsigned_big_endian_first_8_bytes + 0.5) / 2^64",
    "uniform_interval": "strictly between 0 and 1",
    "row_order_invariant": True,
    "chunk_size_invariant": True,
    "reuse_rule": (
        "Reuse the same occurrence-site structural damage uniform for the future "
        "spatially correlated ground-motion case."
    ),
}
write_json(random_specification_path, random_specification)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "input_ground_motion_fields": {
        "path": str(field_path),
        "sha256": actual_field_hash,
        "rows": expected_rows,
        "occurrences": expected_occurrences,
        "sites": expected_sites,
        "structural_im_column": structural_im_column,
        "period_s": period_s,
    },
    "damage_model": {
        "structural_type": EXPECTED_STRUCTURAL_TYPE,
        "equation": "P(DS>=k|Sa)=Phi((ln(Sa)-ln(theta_k))/beta_k)",
        "states": DAMAGE_STATES,
        "random_stream_namespace": DAMAGE_STREAM_NAMESPACE,
        "sampled_state_rule": "inverse categorical CDF using one deterministic occurrence-site uniform",
        "minimum_raw_probability": global_minimum_raw_probability,
        "maximum_probability_sum_error": global_max_probability_sum_error,
    },
    "production": {
        "rows": total_processed_rows,
        "occurrences": len(seen_occurrences),
        "sites": expected_sites,
        "chunks": len(chunk_paths),
        "occurrences_per_chunk": OCCURRENCES_PER_CHUNK,
        "source_row_counts": dict(sorted(source_row_counts.items())),
        "design_level_row_counts": dict(sorted(design_level_row_counts.items())),
    },
    "uniform_diagnostics": {
        "mean": uniform_mean,
        "variance": uniform_variance,
        "target_mean": 0.5,
        "target_variance": 1.0 / 12.0,
    },
    "state_totals": {
        state: {
            "sampled_rows": int(sampled_state_counts[index]),
            "sampled_proportion": float(sampled_proportions[index]),
            "analytical_expected_rows": float(expected_state_counts[index]),
            "analytical_expected_proportion": float(expected_proportions[index]),
        }
        for index, state in enumerate(DAMAGE_STATES)
    },
    "control_reproduction": {
        "rows": int(len(control_compare)),
        "maximum_numeric_difference": maximum_control_difference,
        "sampled_states_exact": bool(
            np.array_equal(
                pd.to_numeric(
                    control_compare["sampled_structural_damage_state_cell3"],
                    errors="raise",
                ).to_numpy(dtype=np.int8),
                pd.to_numeric(
                    control_compare["sampled_structural_damage_state_cell4"],
                    errors="raise",
                ).to_numpy(dtype=np.int8),
            )
        ),
    },
    "outputs": {
        "full_structural_damage_states": {
            "path": str(final_damage_path),
            "sha256": sha256_file(final_damage_path),
            "size_bytes": int(final_damage_path.stat().st_size),
            "rows": expected_rows,
            "row_granularity": "one annual-catalog occurrence and one portfolio building",
        },
        "event_summary": {
            "path": str(final_event_summary_path),
            "sha256": sha256_file(final_event_summary_path),
            "size_bytes": int(final_event_summary_path.stat().st_size),
            "rows": int(len(event_summary)),
        },
        "state_totals": str(state_totals_path),
        "chunk_manifest": str(chunk_manifest_path),
        "validation": str(validation_path),
        "random_stream_specification": str(random_specification_path),
        "summary": str(summary_path),
    },
    "next_cell": (
        "Cell 5: assign HAZUS occupancy-specific replacement values and structural "
        "repair-cost ratios, then calculate sampled and analytical expected structural "
        "ground-up loss for every occurrence-building row."
    ),
}
write_json(summary_path, summary)

print()
print("=" * 78)
print("NOTEBOOK 5 CELL 4 FULL STRUCTURAL DAMAGE SAMPLING COMPLETE")
print("=" * 78)
print(f"Production damage rows:          {total_processed_rows:,}")
print(f"Catalog occurrences:             {len(seen_occurrences):,}")
print(f"Portfolio buildings:             {expected_sites:,}")
print(f"Production chunks:               {len(chunk_paths):,}")
print(f"Damage-uniform mean:              {uniform_mean:.6f}")
print(f"Damage-uniform variance:          {uniform_variance:.6f}")
print(f"Max sampled-expected difference:  {maximum_sampled_expected_difference:.6f}")
print(f"Max Cell 3 control difference:    {maximum_control_difference:.3e}")
print(f"Critical validation checks:       {int(validation['severity'].eq('critical').sum()):,}")
print(f"Critical failures:                {len(critical_failures):,}")
print(f"Warnings requiring review:        {len(warning_failures):,}")
print()
print("Full structural damage states:")
print(f"  {final_damage_path}")
print("Occurrence-level damage summary:")
print(f"  {final_event_summary_path}")
print("Validation:")
print(f"  {validation_path}")
print("Summary:")
print(f"  {summary_path}")
print()
print("Next: calculate sampled and analytical expected structural ground-up loss.")

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 4 failed one or more critical checks. Review: "
        f"{validation_path}"
    )


FULL ANNUAL-CATALOG STRUCTURAL DAMAGE-STATE SAMPLING
Ground-motion rows:       4,996,100
Catalog occurrences:      10,630
Portfolio buildings:      470
Occurrences per chunk:    250
Production chunks:        43
Structural IM:            exact SA0P4
Damage random namespace:  notebook5_structural_damage_v1

------------------------------------------------------------------------------
STRUCTURAL DAMAGE CHUNK 1 OF 43 [ID 0000]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
STRUCTURAL DAMAGE CHUNK 2 OF 43 [ID 0001]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
STRUCTURAL DAMAGE CHUNK 3 OF 43 [ID 0002]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
STRUCTURAL DAMAGE CHUNK 4 OF 43 [ID 0003]
Calculated 117,500 rows for 250 occurrences.

-------------------------

In [15]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
import re
import shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

DAMAGE_STATES = ["None", "Slight", "Moderate", "Extensive", "Complete"]
PROBABILITY_COLUMNS = [f"p_state_{state.lower()}" for state in DAMAGE_STATES]
RATIO_COLUMNS = [
    "structural_ratio_none",
    "structural_ratio_slight",
    "structural_ratio_moderate",
    "structural_ratio_extensive",
    "structural_ratio_complete",
]
EXPECTED_SITES = 470
EXPECTED_STRUCTURAL_TYPE = "W2"
DOLLAR_YEAR = 2022
PIPELINE_VERSION = "notebook5_cell5_structural_ground_up_loss_v1"
NUMERIC_TOLERANCE = 1e-10
LOSS_RECONCILIATION_WARNING_TOLERANCE = 0.05


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_4_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_occupancy(value: Any) -> str:
    text = normalize_text(value).upper().replace(" ", "")
    if not text:
        return ""
    if text.startswith("RES1-"):
        return "RES1"
    return text.split("-", 1)[0] if text.startswith("RES1") else text


def first_existing_column(columns: Iterable[str], aliases: Iterable[str]) -> str | None:
    lookup = {str(column).strip().casefold(): str(column) for column in columns}
    for alias in aliases:
        match = lookup.get(alias.casefold())
        if match is not None:
            return match
    return None


def clean_excel_table(path: Path, sheet_name: str | int = 0) -> pd.DataFrame:
    table = pd.read_excel(path, sheet_name=sheet_name)
    table = table.dropna(axis=1, how="all").dropna(axis=0, how="all")
    table.columns = [normalize_text(column) for column in table.columns]
    return table.reset_index(drop=True)


def expand_structural_repair_ratios(path: Path) -> pd.DataFrame:
    raw = clean_excel_table(path)
    required = {"occ_type", "DS_0", "DS_1", "DS_2", "DS_3"}
    missing = sorted(required.difference(raw.columns))
    if missing:
        raise ValueError(f"Structural repair-cost table is missing columns: {missing}")

    for column in ["DS_0", "DS_1", "DS_2", "DS_3"]:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")
    if raw[["DS_0", "DS_1", "DS_2", "DS_3"]].isna().any().any():
        raise ValueError("Structural repair-cost table contains nonnumeric ratios.")
    if (raw[["DS_0", "DS_1", "DS_2", "DS_3"]] < 0).any().any():
        raise ValueError("Structural repair-cost table contains negative ratios.")

    rows: list[dict[str, Any]] = []
    for _, row in raw.iterrows():
        original = normalize_text(row["occ_type"]).upper().replace(" ", "")
        normalized = normalize_occupancy(original)
        keys = [normalized]
        match = re.fullmatch(r"([A-Z]+\d+)([A-Z])-([A-Z])", original)
        if match:
            base, start, end = match.groups()
            keys = [base + chr(code) for code in range(ord(start), ord(end) + 1)]
        for occupancy in keys:
            rows.append(
                {
                    "occ_type": occupancy,
                    "source_occ_type": original,
                    "structural_ratio_none": 0.0,
                    "structural_ratio_slight": float(row["DS_0"]) / 100.0,
                    "structural_ratio_moderate": float(row["DS_1"]) / 100.0,
                    "structural_ratio_extensive": float(row["DS_2"]) / 100.0,
                    "structural_ratio_complete": float(row["DS_3"]) / 100.0,
                }
            )

    expanded = pd.DataFrame(rows)
    keep_rows: list[pd.Series] = []
    conflicts: list[str] = []
    for occupancy, group in expanded.groupby("occ_type", sort=True):
        if len(group[RATIO_COLUMNS].drop_duplicates()) > 1:
            conflicts.append(str(occupancy))
        keep_rows.append(group.iloc[-1])
    if conflicts:
        raise ValueError(
            "Conflicting structural repair ratios for occupancy classes: "
            f"{sorted(conflicts)}"
        )
    return pd.DataFrame(keep_rows).sort_values("occ_type").reset_index(drop=True)


def load_general_replacement_costs(path: Path) -> tuple[pd.DataFrame, str]:
    table = clean_excel_table(path)
    if "occ_type" not in table.columns:
        raise ValueError("General replacement-cost table is missing occ_type.")

    aliases = [
        "Structure Replacement Cost ($/ft²)",
        "Structure Replacement Cost ($/ft2)",
        "cost_per_sqft",
        "Cost per sqft",
        "Cost ($/ft²)",
        "Cost ($/ft2)",
    ]
    cost_column = first_existing_column(table.columns, aliases)
    if cost_column is None:
        candidates = [column for column in table.columns if "cost" in column.casefold()]
        if len(candidates) != 1:
            raise ValueError(
                "Could not identify the general replacement-cost column. "
                f"Available columns: {list(table.columns)}"
            )
        cost_column = candidates[0]

    table["occ_type"] = table["occ_type"].map(normalize_occupancy)
    table["replacement_cost_per_sqft_2022_usd"] = pd.to_numeric(
        table[cost_column], errors="coerce"
    )
    non_res1 = table["occ_type"] != "RES1"
    if table.loc[non_res1, "replacement_cost_per_sqft_2022_usd"].isna().any():
        missing = table.loc[
            non_res1 & table["replacement_cost_per_sqft_2022_usd"].isna(),
            "occ_type",
        ].tolist()
        raise ValueError(f"Missing replacement costs for occupancy classes: {missing}")
    if (table.loc[non_res1, "replacement_cost_per_sqft_2022_usd"] <= 0).any():
        raise ValueError("General replacement-cost table contains nonpositive costs.")
    if table["occ_type"].duplicated().any():
        duplicates = sorted(
            table.loc[table["occ_type"].duplicated(keep=False), "occ_type"].unique().tolist()
        )
        raise ValueError(f"Duplicate occupancy classes in replacement costs: {duplicates}")

    return (
        table[["occ_type", "replacement_cost_per_sqft_2022_usd"]].copy(),
        cost_column,
    )


def load_res1_replacement_costs(path: Path) -> pd.DataFrame:
    table = clean_excel_table(path)
    required = {
        "Construction Class",
        "Height Class",
        "2022 Avg Cost per ft² (No Basement)",
        "2022 Avg Cost per ft² (Finished Basement)",
        "2022 Avg Cost per ft² (Unfinished Basement)",
    }
    missing = sorted(required.difference(table.columns))
    if missing:
        raise ValueError(f"RES1 replacement-cost table is missing columns: {missing}")
    for column in [value for value in required if "Cost per ft²" in value]:
        table[column] = pd.to_numeric(table[column], errors="coerce")
    return table


def resolve_res1_cost_per_sqft(
    occupancy_raw: str,
    stories: float | int | None,
    res1_table: pd.DataFrame,
) -> tuple[float, str]:
    text = normalize_text(occupancy_raw).upper()
    suffix = text.split("-", 1)[1] if "-" in text else ""
    height_map = {1: "One-story", 2: "Two-story", 3: "Three-story"}

    height: str | None = None
    for number, label in height_map.items():
        if f"{number}S" in suffix:
            height = label
            break
    if height is None and stories is not None and not pd.isna(stories):
        story_number = int(min(max(round(float(stories)), 1), 3))
        height = height_map[story_number]
    if height is None:
        raise ValueError(
            f"Cannot resolve RES1 height for occupancy {occupancy_raw!r} and stories={stories}."
        )

    if suffix.endswith("WB"):
        basement = "Finished Basement"
        cost_column = "2022 Avg Cost per ft² (Finished Basement)"
    elif suffix.endswith("UB"):
        basement = "Unfinished Basement"
        cost_column = "2022 Avg Cost per ft² (Unfinished Basement)"
    else:
        basement = "No Basement"
        cost_column = "2022 Avg Cost per ft² (No Basement)"

    match = res1_table.loc[
        res1_table["Construction Class"].astype(str).str.strip().eq("Average")
        & res1_table["Height Class"].astype(str).str.strip().eq(height)
    ]
    if len(match) != 1:
        raise ValueError(
            "RES1 replacement-cost lookup did not return exactly one row for "
            f"Construction Class='Average', Height Class={height!r}."
        )
    value = float(match.iloc[0][cost_column])
    if not math.isfinite(value) or value <= 0:
        raise ValueError(f"Invalid RES1 replacement cost: {value}")
    return value, f"RES1 Average | {height} | {basement}"


def stable_frame_hash(frame: pd.DataFrame, columns: list[str]) -> str:
    digest = hashlib.sha256()
    subset = frame[columns].copy()
    for column in columns:
        values = subset[column]
        if pd.api.types.is_float_dtype(values):
            encoded = values.map(lambda value: format(float(value), ".17g"))
        else:
            encoded = values.astype(str)
        for value in encoded:
            digest.update(value.encode("utf-8"))
            digest.update(b"\x1f")
        digest.update(b"\x1e")
    return digest.hexdigest()


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    buffer = io.StringIO()
    frame.to_csv(buffer, index=False, lineterminator="\n")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as compressed:
            compressed.write(buffer.getvalue().encode("utf-8"))
    temporary.replace(path)


def concatenate_gzip_csv_files(chunk_paths: list[Path], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    expected_header: bytes | None = None
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(fileobj=raw_output, mode="wb", mtime=0) as compressed_output:
            for chunk_path in chunk_paths:
                with gzip.open(chunk_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        compressed_output.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            f"Chunk header mismatch while concatenating {chunk_path}."
                        )
                    shutil.copyfileobj(source, compressed_output, length=8 * 1024 * 1024)
    temporary.replace(output_path)


def reference_record(cell1_summary: dict[str, Any], role: str) -> dict[str, Any]:
    matches = [
        record
        for record in cell1_summary.get("hazus_reference_files", [])
        if str(record.get("role")) == role
    ]
    if len(matches) != 1:
        raise KeyError(f"Expected exactly one Cell 1 reference record for role={role!r}.")
    return matches[0]


def summarize_event_losses(frame: pd.DataFrame) -> pd.DataFrame:
    group_columns = [
        column
        for column in [
            "catalog_year",
            "catalog_event_id",
            "occurrence_ordinal",
            "occurrence_id",
            "rupture_ordinal",
            "rupture_template_event_id",
            "rupture_id",
            "source_type",
            "magnitude",
        ]
        if column in frame.columns
    ]

    base = (
        frame.groupby(group_columns, sort=False)
        .agg(
            buildings=("site_id", "size"),
            portfolio_replacement_value_2022_usd=(
                "building_replacement_value_2022_usd",
                "sum",
            ),
            sampled_structural_ground_up_loss_2022_usd=(
                "sampled_structural_ground_up_loss_2022_usd",
                "sum",
            ),
            analytical_expected_structural_ground_up_loss_2022_usd=(
                "analytical_expected_structural_ground_up_loss_2022_usd",
                "sum",
            ),
            sampled_structurally_damaged_buildings=(
                "sampled_structurally_damaged_indicator",
                "sum",
            ),
            analytical_expected_structurally_damaged_buildings=(
                "analytical_expected_structurally_damaged_indicator",
                "sum",
            ),
            maximum_building_sampled_structural_loss_2022_usd=(
                "sampled_structural_ground_up_loss_2022_usd",
                "max",
            ),
            mean_sa0p4_g=("sa0p4_simulated_g", "mean"),
            maximum_sa0p4_g=("sa0p4_simulated_g", "max"),
        )
        .reset_index()
    )

    counts = (
        frame.groupby(group_columns + ["sampled_structural_damage_state"], sort=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=range(5), fill_value=0)
        .rename(columns={index: f"sampled_{state.lower()}_buildings" for index, state in enumerate(DAMAGE_STATES)})
        .reset_index()
    )
    output = base.merge(counts, on=group_columns, how="left", validate="one_to_one")
    output["sampled_structural_loss_ratio"] = (
        output["sampled_structural_ground_up_loss_2022_usd"]
        / output["portfolio_replacement_value_2022_usd"]
    )
    output["analytical_expected_structural_loss_ratio"] = (
        output["analytical_expected_structural_ground_up_loss_2022_usd"]
        / output["portfolio_replacement_value_2022_usd"]
    )
    return output.sort_values("occurrence_ordinal").reset_index(drop=True)


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
DAMAGE_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_structural_damage"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_structural_loss"
CHUNK_DIR = OUTPUT_DIR / "chunks"
WORK_DIR = METADATA_DIR / "notebook_5_cell_5_work"
MARKER_DIR = WORK_DIR / "markers"
CHUNK_VALIDATION_DIR = WORK_DIR / "chunk_validations"
for directory in [
    METADATA_DIR,
    PARAMETER_DIR,
    OUTPUT_DIR,
    CHUNK_DIR,
    WORK_DIR,
    MARKER_DIR,
    CHUNK_VALIDATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CELL1_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_1_summary.json"
CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_1_validation.csv"
CELL4_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_4_summary.json"
CELL4_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_4_validation.csv"
CELL4_CHUNK_MANIFEST_PATH = METADATA_DIR / "notebook_5_cell_4_chunk_manifest.csv"
BUILDING_VALUE_PATH = PARAMETER_DIR / "seaside_w2_replacement_values_and_structural_ratios.csv"
REPLACEMENT_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_5_replacement_value_summary.csv"
LOSS_RECONCILIATION_PATH = METADATA_DIR / "notebook_5_cell_5_loss_reconciliation.csv"
CELL5_CHUNK_MANIFEST_PATH = METADATA_DIR / "notebook_5_cell_5_chunk_manifest.csv"
CELL5_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_5_validation.csv"
CELL5_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_5_summary.json"
FINAL_BUILDING_LOSS_PATH = OUTPUT_DIR / "full_structural_ground_up_loss.csv.gz"
FINAL_EVENT_LOSS_PATH = OUTPUT_DIR / "structural_ground_up_loss_event_summary.csv.gz"

required_paths = [
    CELL1_SUMMARY_PATH,
    CELL1_VALIDATION_PATH,
    CELL4_SUMMARY_PATH,
    CELL4_VALIDATION_PATH,
    CELL4_CHUNK_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required Cell 1 or Cell 4 outputs are missing: {missing_paths}")

validation_rows: list[dict[str, Any]] = []
cell1_summary = load_json(CELL1_SUMMARY_PATH)
cell4_summary = load_json(CELL4_SUMMARY_PATH)
cell1_validation = pd.read_csv(CELL1_VALIDATION_PATH)
cell4_validation = pd.read_csv(CELL4_VALIDATION_PATH)
cell4_manifest = pd.read_csv(CELL4_CHUNK_MANIFEST_PATH)

for cell_number, summary, validation in [
    (1, cell1_summary, cell1_validation),
    (4, cell4_summary, cell4_validation),
]:
    append_check(
        validation_rows,
        f"cell{cell_number}_critical_checks_passed",
        summary.get("all_critical_checks_passed") is True,
        f"all_critical_checks_passed={summary.get('all_critical_checks_passed')}",
    )
    parsed = parse_bool_series(validation["passed"])
    append_check(
        validation_rows,
        f"cell{cell_number}_has_no_unresolved_checks",
        parsed.all(),
        f"unresolved={int((~parsed).sum())}",
    )

expected_rows = int(cell4_summary["production"]["rows"])
expected_occurrences = int(cell4_summary["production"]["occurrences"])
expected_sites = int(cell4_summary["production"]["sites"])
declared_catalog_years = int(cell1_summary["annual_catalog"]["declared_duration_years"])

append_check(
    validation_rows,
    "portfolio_has_470_buildings",
    expected_sites == EXPECTED_SITES,
    f"sites={expected_sites}",
)
append_check(
    validation_rows,
    "catalog_duration_positive",
    declared_catalog_years > 0,
    f"years={declared_catalog_years}",
)
append_check(
    validation_rows,
    "cell4_rows_match_occurrences_times_sites",
    expected_rows == expected_occurrences * expected_sites,
    f"rows={expected_rows}; occurrences={expected_occurrences}; sites={expected_sites}",
)

cell4_damage_info = cell4_summary["outputs"]["full_structural_damage_states"]
cell4_damage_path = resolve_recorded_path(PROJECT_ROOT, cell4_damage_info["path"])
append_check(
    validation_rows,
    "cell4_final_damage_file_exists",
    cell4_damage_path.exists(),
    str(cell4_damage_path),
)
append_check(
    validation_rows,
    "cell4_final_damage_hash_matches_summary",
    sha256_file(cell4_damage_path) == str(cell4_damage_info["sha256"]),
    "Final Cell 4 structural damage file hash must match its accepted summary.",
)

portfolio_info = cell1_summary["portfolio"]
portfolio_path = resolve_recorded_path(PROJECT_ROOT, portfolio_info["path"])
portfolio = pd.read_csv(portfolio_path)
append_check(validation_rows, "portfolio_exists", portfolio_path.exists(), str(portfolio_path))
append_check(
    validation_rows,
    "portfolio_hash_matches_cell1",
    sha256_file(portfolio_path) == str(portfolio_info["sha256"]),
    "Portfolio hash must match Cell 1.",
)

resolved_columns = portfolio_info["resolved_columns"]
site_column = str(portfolio_info["site_key"])
occupancy_column = str(resolved_columns["occupancy"])
floor_area_column = str(resolved_columns["floor_area_sqft"])
structural_type_column = str(resolved_columns["structural_type"])
story_column = first_existing_column(
    portfolio.columns,
    ["num_story", "num_stories", "stories", "number_of_stories"],
)
required_portfolio_columns = {
    site_column,
    occupancy_column,
    floor_area_column,
    structural_type_column,
}
missing_portfolio_columns = sorted(required_portfolio_columns.difference(portfolio.columns))
append_check(
    validation_rows,
    "portfolio_columns_present",
    not missing_portfolio_columns,
    f"missing={missing_portfolio_columns}",
)
if missing_portfolio_columns:
    raise KeyError(f"Portfolio is missing columns: {missing_portfolio_columns}")

reference_roles = [
    "structural_repair_ratios",
    "general_replacement_costs",
    "res1_replacement_costs",
]
reference_paths: dict[str, Path] = {}
reference_hashes: dict[str, str] = {}
for role in reference_roles:
    record = reference_record(cell1_summary, role)
    path = resolve_recorded_path(PROJECT_ROOT, str(record["path"]))
    actual_hash = sha256_file(path)
    reference_paths[role] = path
    reference_hashes[role] = actual_hash
    append_check(
        validation_rows,
        f"{role}_hash_matches_cell1",
        actual_hash == str(record["sha256"]),
        f"calculated={actual_hash}; recorded={record['sha256']}",
    )

repair_ratios = expand_structural_repair_ratios(
    reference_paths["structural_repair_ratios"]
)
general_costs, general_cost_source_column = load_general_replacement_costs(
    reference_paths["general_replacement_costs"]
)
res1_costs = load_res1_replacement_costs(reference_paths["res1_replacement_costs"])

building_values = pd.DataFrame(
    {
        "site_id": portfolio[site_column].astype(str),
        "structural_type": portfolio[structural_type_column]
        .astype(str)
        .str.strip()
        .str.upper(),
        "occupancy_raw": portfolio[occupancy_column].map(normalize_text),
        "occ_type": portfolio[occupancy_column].map(normalize_occupancy),
        "floor_area_sqft": pd.to_numeric(portfolio[floor_area_column], errors="coerce"),
    }
)
if story_column is not None:
    building_values["number_of_stories"] = pd.to_numeric(
        portfolio[story_column], errors="coerce"
    )
else:
    building_values["number_of_stories"] = np.nan

append_check(
    validation_rows,
    "building_value_site_ids_unique",
    building_values["site_id"].is_unique,
    f"duplicates={int(building_values['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "all_buildings_are_w2",
    building_values["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(building_values['structural_type'].unique().tolist())}",
)
append_check(
    validation_rows,
    "floor_area_positive_complete",
    building_values["floor_area_sqft"].notna().all()
    and (building_values["floor_area_sqft"] > 0).all(),
    (
        f"missing={int(building_values['floor_area_sqft'].isna().sum())}; "
        f"nonpositive={int((building_values['floor_area_sqft'].fillna(0) <= 0).sum())}"
    ),
)

building_values = building_values.merge(
    general_costs,
    on="occ_type",
    how="left",
    validate="many_to_one",
)
building_values["replacement_cost_source"] = "HAZUS general occupancy table"

res1_mask = building_values["occ_type"].eq("RES1")
if res1_mask.any():
    for index in building_values.index[res1_mask]:
        cost, source = resolve_res1_cost_per_sqft(
            building_values.at[index, "occupancy_raw"],
            building_values.at[index, "number_of_stories"],
            res1_costs,
        )
        building_values.at[index, "replacement_cost_per_sqft_2022_usd"] = cost
        building_values.at[index, "replacement_cost_source"] = source

building_values = building_values.merge(
    repair_ratios,
    on="occ_type",
    how="left",
    validate="many_to_one",
)
building_values["building_replacement_value_2022_usd"] = (
    building_values["floor_area_sqft"]
    * building_values["replacement_cost_per_sqft_2022_usd"]
)
building_values["replacement_cost_dollar_year"] = DOLLAR_YEAR
building_values["general_replacement_cost_source_column"] = general_cost_source_column
building_values["general_replacement_cost_source_sha256"] = reference_hashes[
    "general_replacement_costs"
]
building_values["structural_repair_ratio_source_sha256"] = reference_hashes[
    "structural_repair_ratios"
]

missing_cost_sites = building_values.loc[
    building_values["replacement_cost_per_sqft_2022_usd"].isna(), "site_id"
].tolist()
missing_ratio_sites = building_values.loc[
    building_values[RATIO_COLUMNS].isna().any(axis=1), "site_id"
].tolist()
append_check(
    validation_rows,
    "replacement_cost_assigned_to_every_building",
    not missing_cost_sites,
    f"missing_sites={missing_cost_sites[:20]}",
)
append_check(
    validation_rows,
    "structural_repair_ratios_assigned_to_every_building",
    not missing_ratio_sites,
    f"missing_sites={missing_ratio_sites[:20]}",
)
append_check(
    validation_rows,
    "replacement_values_positive",
    np.isfinite(building_values["building_replacement_value_2022_usd"]).all()
    and (building_values["building_replacement_value_2022_usd"] > 0).all(),
    (
        f"minimum={building_values['building_replacement_value_2022_usd'].min():.2f}; "
        f"maximum={building_values['building_replacement_value_2022_usd'].max():.2f}"
    ),
)

ratio_matrix_by_building = building_values[RATIO_COLUMNS].to_numpy(dtype=np.float64)
append_check(
    validation_rows,
    "repair_ratios_finite_and_bounded",
    np.isfinite(ratio_matrix_by_building).all()
    and np.all((ratio_matrix_by_building >= 0.0) & (ratio_matrix_by_building <= 1.0)),
    (
        f"minimum={np.nanmin(ratio_matrix_by_building):.6f}; "
        f"maximum={np.nanmax(ratio_matrix_by_building):.6f}"
    ),
)
append_check(
    validation_rows,
    "repair_ratios_nondecreasing_by_state",
    np.all(np.diff(ratio_matrix_by_building, axis=1) >= -NUMERIC_TOLERANCE),
    "None <= Slight <= Moderate <= Extensive <= Complete for every building.",
)

building_values.to_csv(BUILDING_VALUE_PATH, index=False)
replacement_summary = (
    building_values.groupby("occ_type", sort=True)
    .agg(
        buildings=("site_id", "size"),
        total_floor_area_sqft=("floor_area_sqft", "sum"),
        replacement_cost_per_sqft_2022_usd=(
            "replacement_cost_per_sqft_2022_usd",
            "first",
        ),
        total_replacement_value_2022_usd=(
            "building_replacement_value_2022_usd",
            "sum",
        ),
        minimum_building_replacement_value_2022_usd=(
            "building_replacement_value_2022_usd",
            "min",
        ),
        maximum_building_replacement_value_2022_usd=(
            "building_replacement_value_2022_usd",
            "max",
        ),
    )
    .reset_index()
)
replacement_summary.to_csv(REPLACEMENT_SUMMARY_PATH, index=False)
portfolio_replacement_value = float(
    building_values["building_replacement_value_2022_usd"].sum()
)

required_manifest_columns = {
    "chunk_id",
    "rows",
    "occurrences",
    "output_path",
    "output_sha256",
}
missing_manifest_columns = sorted(required_manifest_columns.difference(cell4_manifest.columns))
append_check(
    validation_rows,
    "cell4_chunk_manifest_columns_present",
    not missing_manifest_columns,
    f"missing={missing_manifest_columns}",
)
if missing_manifest_columns:
    raise KeyError(f"Cell 4 chunk manifest is missing columns: {missing_manifest_columns}")

cell4_manifest["chunk_id"] = cell4_manifest["chunk_id"].astype(str).str.zfill(4)
cell4_manifest = cell4_manifest.sort_values("chunk_id").reset_index(drop=True)
append_check(
    validation_rows,
    "cell4_chunk_manifest_rows_match_summary",
    len(cell4_manifest) == int(cell4_summary["production"]["chunks"]),
    f"manifest={len(cell4_manifest)}; summary={cell4_summary['production']['chunks']}",
)

basis_payload = {
    "pipeline_version": PIPELINE_VERSION,
    "cell1_summary_sha256": sha256_file(CELL1_SUMMARY_PATH),
    "cell4_summary_sha256": sha256_file(CELL4_SUMMARY_PATH),
    "cell4_chunk_manifest_sha256": sha256_file(CELL4_CHUNK_MANIFEST_PATH),
    "portfolio_sha256": sha256_file(portfolio_path),
    "structural_repair_ratio_sha256": reference_hashes["structural_repair_ratios"],
    "general_replacement_cost_sha256": reference_hashes["general_replacement_costs"],
    "res1_replacement_cost_sha256": reference_hashes["res1_replacement_costs"],
    "building_value_sha256": sha256_file(BUILDING_VALUE_PATH),
    "dollar_year": DOLLAR_YEAR,
}
basis_hash = hashlib.sha256(
    json.dumps(basis_payload, sort_keys=True).encode("utf-8")
).hexdigest()

building_lookup_columns = [
    "site_id",
    "structural_type",
    "occupancy_raw",
    "occ_type",
    "floor_area_sqft",
    "replacement_cost_per_sqft_2022_usd",
    "building_replacement_value_2022_usd",
    *RATIO_COLUMNS,
]
building_lookup = building_values[building_lookup_columns].copy()

chunk_paths: list[Path] = []
chunk_manifest_rows: list[dict[str, Any]] = []
event_summary_frames: list[pd.DataFrame] = []
seen_pairs = np.zeros(expected_rows, dtype=bool)
occurrence_counts = np.zeros(expected_occurrences, dtype=np.int64)
site_counts = np.zeros(expected_sites, dtype=np.int64)
total_rows = 0
sampled_loss_sum = 0.0
expected_loss_sum = 0.0
maximum_sampled_equation_error = 0.0
maximum_expected_equation_error = 0.0
maximum_sampled_loss_over_replacement = -math.inf
maximum_expected_loss_over_replacement = -math.inf
source_loss_totals: Counter[str] = Counter()
occupancy_sampled_loss: Counter[str] = Counter()
occupancy_expected_loss: Counter[str] = Counter()

print("=" * 78)
print("FULL ANNUAL-CATALOG STRUCTURAL GROUND-UP LOSS")
print("=" * 78)
print(f"Structural damage rows:        {expected_rows:,}")
print(f"Catalog occurrences:          {expected_occurrences:,}")
print(f"Portfolio buildings:          {expected_sites:,}")
print(f"Cell 4 production chunks:     {len(cell4_manifest):,}")
print(f"Portfolio replacement value:  ${portfolio_replacement_value:,.0f} ({DOLLAR_YEAR} USD)")
print("Loss scope:                    structural repair component only")

for manifest_index, manifest_row in cell4_manifest.iterrows():
    chunk_id = str(manifest_row["chunk_id"]).zfill(4)
    print()
    print("-" * 78)
    print(
        f"STRUCTURAL LOSS CHUNK {manifest_index + 1} OF {len(cell4_manifest)} "
        f"[ID {chunk_id}]"
    )

    damage_chunk_path = resolve_recorded_path(
        PROJECT_ROOT, str(manifest_row["output_path"])
    )
    actual_input_hash = sha256_file(damage_chunk_path)
    expected_input_hash = str(manifest_row["output_sha256"])
    if actual_input_hash != expected_input_hash:
        raise RuntimeError(
            f"Cell 4 chunk {chunk_id} hash mismatch: "
            f"calculated={actual_input_hash}; expected={expected_input_hash}"
        )

    damage = pd.read_csv(damage_chunk_path)
    required_damage_columns = {
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
        "site_ordinal",
        "site_id",
        "sa0p4_simulated_g",
        "structural_type",
        "design_level",
        "occ_type",
        *PROBABILITY_COLUMNS,
        "sampled_structural_damage_state",
        "sampled_structural_damage_state_name",
    }
    missing_damage_columns = sorted(required_damage_columns.difference(damage.columns))
    if missing_damage_columns:
        raise KeyError(f"Cell 4 chunk {chunk_id} is missing columns: {missing_damage_columns}")

    damage["site_id"] = damage["site_id"].astype(str)
    damage["occurrence_id"] = damage["occurrence_id"].astype(str)
    damage["occ_type"] = damage["occ_type"].map(normalize_occupancy)
    damage["sampled_structural_damage_state"] = pd.to_numeric(
        damage["sampled_structural_damage_state"], errors="raise"
    ).astype(np.int8)
    for column in PROBABILITY_COLUMNS + ["sa0p4_simulated_g"]:
        damage[column] = pd.to_numeric(damage[column], errors="raise")

    input_hash = stable_frame_hash(
        damage,
        [
            "occurrence_ordinal",
            "occurrence_id",
            "site_ordinal",
            "site_id",
            "sampled_structural_damage_state",
            *PROBABILITY_COLUMNS,
        ],
    )
    chunk_output_path = CHUNK_DIR / f"structural_loss_chunk_{chunk_id}.csv.gz"
    marker_path = MARKER_DIR / f"structural_loss_chunk_{chunk_id}.json"
    chunk_validation_path = (
        CHUNK_VALIDATION_DIR / f"structural_loss_chunk_{chunk_id}_validation.csv"
    )

    marker_valid = False
    marker: dict[str, Any] = {}
    if marker_path.exists() and chunk_output_path.exists():
        marker = load_json(marker_path)
        marker_valid = (
            marker.get("pipeline_version") == PIPELINE_VERSION
            and marker.get("basis_hash") == basis_hash
            and marker.get("cell4_chunk_sha256") == actual_input_hash
            and marker.get("input_hash") == input_hash
            and int(marker.get("rows", -1)) == len(damage)
            and marker.get("output_sha256") == sha256_file(chunk_output_path)
        )

    if marker_valid:
        output = pd.read_csv(chunk_output_path)
        print(f"Reused validated loss chunk with {len(output):,} rows.")
    else:
        merged = damage.merge(
            building_lookup,
            on="site_id",
            how="left",
            suffixes=("_damage", "_portfolio"),
            validate="many_to_one",
            indicator=True,
        )
        chunk_validation_rows: list[dict[str, Any]] = []
        append_check(
            chunk_validation_rows,
            "all_damage_rows_joined_to_building_values",
            merged["_merge"].eq("both").all(),
            f"merge_counts={merged['_merge'].value_counts().to_dict()}",
        )
        if not merged["_merge"].eq("both").all():
            missing = merged.loc[merged["_merge"] != "both", "site_id"].head(20).tolist()
            raise RuntimeError(f"Chunk {chunk_id} has unmatched sites: {missing}")
        merged = merged.drop(columns="_merge")

        merged["occ_type_damage"] = merged["occ_type_damage"].map(normalize_occupancy)
        merged["occ_type_portfolio"] = merged["occ_type_portfolio"].map(normalize_occupancy)
        append_check(
            chunk_validation_rows,
            "damage_and_portfolio_occupancies_agree",
            merged["occ_type_damage"].eq(merged["occ_type_portfolio"]).all(),
            (
                f"disagreements={int((~merged['occ_type_damage'].eq(merged['occ_type_portfolio'])).sum())}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "damage_and_portfolio_structural_types_agree",
            merged["structural_type_damage"]
            .astype(str)
            .str.upper()
            .eq(merged["structural_type_portfolio"].astype(str).str.upper())
            .all(),
            "Structural type must remain W2 after the site join.",
        )

        probabilities = merged[PROBABILITY_COLUMNS].to_numpy(dtype=np.float64)
        ratios = merged[RATIO_COLUMNS].to_numpy(dtype=np.float64)
        sampled_states = merged["sampled_structural_damage_state"].to_numpy(dtype=np.int8)
        replacement_values = merged[
            "building_replacement_value_2022_usd"
        ].to_numpy(dtype=np.float64)

        sampled_ratios = ratios[np.arange(len(merged)), sampled_states]
        analytical_expected_ratios = np.sum(probabilities * ratios, axis=1)
        sampled_losses = replacement_values * sampled_ratios
        analytical_expected_losses = replacement_values * analytical_expected_ratios

        output = damage.copy()
        output["occupancy_raw"] = merged["occupancy_raw"]
        output["occ_type"] = merged["occ_type_portfolio"]
        output["floor_area_sqft"] = merged["floor_area_sqft"]
        output["replacement_cost_per_sqft_2022_usd"] = merged[
            "replacement_cost_per_sqft_2022_usd"
        ]
        output["building_replacement_value_2022_usd"] = replacement_values
        output["sampled_structural_repair_ratio"] = sampled_ratios
        output["analytical_expected_structural_repair_ratio"] = analytical_expected_ratios
        output["sampled_structural_ground_up_loss_2022_usd"] = sampled_losses
        output[
            "analytical_expected_structural_ground_up_loss_2022_usd"
        ] = analytical_expected_losses
        output["sampled_structurally_damaged_indicator"] = (
            sampled_states > 0
        ).astype(np.int8)
        output["analytical_expected_structurally_damaged_indicator"] = (
            1.0 - probabilities[:, 0]
        )

        sampled_reconstruction = replacement_values * output[
            "sampled_structural_repair_ratio"
        ].to_numpy(dtype=np.float64)
        expected_reconstruction = replacement_values * output[
            "analytical_expected_structural_repair_ratio"
        ].to_numpy(dtype=np.float64)
        sampled_equation_error = float(
            np.max(
                np.abs(
                    sampled_reconstruction
                    - output[
                        "sampled_structural_ground_up_loss_2022_usd"
                    ].to_numpy(dtype=np.float64)
                )
            )
        )
        expected_equation_error = float(
            np.max(
                np.abs(
                    expected_reconstruction
                    - output[
                        "analytical_expected_structural_ground_up_loss_2022_usd"
                    ].to_numpy(dtype=np.float64)
                )
            )
        )

        append_check(
            chunk_validation_rows,
            "output_rows_match_damage_rows",
            len(output) == len(damage),
            f"output={len(output)}; damage={len(damage)}",
        )
        append_check(
            chunk_validation_rows,
            "sampled_ratios_match_sampled_states",
            np.array_equal(
                sampled_ratios,
                ratios[np.arange(len(merged)), sampled_states],
            ),
            "Sampled ratio is gathered from the building-specific five-state ratio vector.",
        )
        append_check(
            chunk_validation_rows,
            "sampled_none_state_has_zero_loss",
            np.all(sampled_losses[sampled_states == 0] == 0.0),
            f"none_rows={int(np.sum(sampled_states == 0))}",
        )
        append_check(
            chunk_validation_rows,
            "losses_finite_and_nonnegative",
            np.isfinite(sampled_losses).all()
            and np.isfinite(analytical_expected_losses).all()
            and np.all(sampled_losses >= 0.0)
            and np.all(analytical_expected_losses >= 0.0),
            (
                f"sampled_min={sampled_losses.min():.6f}; "
                f"expected_min={analytical_expected_losses.min():.6f}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "structural_losses_do_not_exceed_replacement_value",
            np.all(sampled_losses <= replacement_values + NUMERIC_TOLERANCE)
            and np.all(analytical_expected_losses <= replacement_values + NUMERIC_TOLERANCE),
            (
                f"sampled_max_ratio={float(np.max(sampled_losses / replacement_values)):.6f}; "
                f"expected_max_ratio={float(np.max(analytical_expected_losses / replacement_values)):.6f}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "sampled_loss_equation_exact",
            sampled_equation_error <= NUMERIC_TOLERANCE,
            f"maximum_error={sampled_equation_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "analytical_expected_loss_equation_exact",
            expected_equation_error <= NUMERIC_TOLERANCE,
            f"maximum_error={expected_equation_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "occurrence_site_pairs_unique",
            not output.duplicated(["occurrence_id", "site_id"]).any(),
            f"duplicates={int(output.duplicated(['occurrence_id', 'site_id']).sum())}",
        )

        chunk_validation = pd.DataFrame(chunk_validation_rows)
        chunk_failures = chunk_validation.loc[
            chunk_validation["severity"].eq("critical") & ~chunk_validation["passed"]
        ]
        chunk_validation.to_csv(chunk_validation_path, index=False)
        if not chunk_failures.empty:
            raise RuntimeError(
                f"Structural loss chunk {chunk_id} failed validation. "
                f"Review {chunk_validation_path}."
            )

        write_gzip_csv_deterministic(output, chunk_output_path)
        marker = {
            "pipeline_version": PIPELINE_VERSION,
            "basis_hash": basis_hash,
            "cell4_chunk_sha256": actual_input_hash,
            "input_hash": input_hash,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "sampled_loss_sum_2022_usd": float(sampled_losses.sum()),
            "analytical_expected_loss_sum_2022_usd": float(
                analytical_expected_losses.sum()
            ),
            "maximum_sampled_equation_error": sampled_equation_error,
            "maximum_expected_equation_error": expected_equation_error,
            "output_path": str(chunk_output_path),
            "output_sha256": sha256_file(chunk_output_path),
            "validation_path": str(chunk_validation_path),
            "validation_sha256": sha256_file(chunk_validation_path),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        write_json(marker_path, marker)
        print(
            f"Calculated {len(output):,} rows for "
            f"{output['occurrence_id'].nunique():,} occurrences."
        )

    required_output_columns = {
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
        "site_ordinal",
        "site_id",
        "sa0p4_simulated_g",
        "design_level",
        "occ_type",
        *PROBABILITY_COLUMNS,
        "sampled_structural_damage_state",
        "sampled_structural_damage_state_name",
        "floor_area_sqft",
        "replacement_cost_per_sqft_2022_usd",
        "building_replacement_value_2022_usd",
        "sampled_structural_repair_ratio",
        "analytical_expected_structural_repair_ratio",
        "sampled_structural_ground_up_loss_2022_usd",
        "analytical_expected_structural_ground_up_loss_2022_usd",
        "sampled_structurally_damaged_indicator",
        "analytical_expected_structurally_damaged_indicator",
    }
    missing_output_columns = sorted(required_output_columns.difference(output.columns))
    if missing_output_columns:
        raise KeyError(f"Cell 5 chunk {chunk_id} is missing columns: {missing_output_columns}")

    output["occurrence_ordinal"] = pd.to_numeric(
        output["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    output["site_ordinal"] = pd.to_numeric(
        output["site_ordinal"], errors="raise"
    ).astype(np.int64)
    output["sampled_structural_damage_state"] = pd.to_numeric(
        output["sampled_structural_damage_state"], errors="raise"
    ).astype(np.int8)

    occurrence_ordinals = output["occurrence_ordinal"].to_numpy(dtype=np.int64)
    site_ordinals = output["site_ordinal"].to_numpy(dtype=np.int64)
    if np.any(occurrence_ordinals < 0) or np.any(occurrence_ordinals >= expected_occurrences):
        raise RuntimeError(f"Chunk {chunk_id} has occurrence ordinals outside range.")
    if np.any(site_ordinals < 0) or np.any(site_ordinals >= expected_sites):
        raise RuntimeError(f"Chunk {chunk_id} has site ordinals outside range.")

    pair_indices = occurrence_ordinals * expected_sites + site_ordinals
    if len(np.unique(pair_indices)) != len(pair_indices):
        raise RuntimeError(f"Chunk {chunk_id} contains duplicate ordinal pair indices.")
    if seen_pairs[pair_indices].any():
        raise RuntimeError(f"Chunk {chunk_id} repeats previously processed pairs.")
    seen_pairs[pair_indices] = True
    np.add.at(occurrence_counts, occurrence_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    sampled_losses = pd.to_numeric(
        output["sampled_structural_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    expected_losses = pd.to_numeric(
        output["analytical_expected_structural_ground_up_loss_2022_usd"],
        errors="raise",
    ).to_numpy(dtype=np.float64)
    replacement_values = pd.to_numeric(
        output["building_replacement_value_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    sampled_ratios = pd.to_numeric(
        output["sampled_structural_repair_ratio"], errors="raise"
    ).to_numpy(dtype=np.float64)
    expected_ratios = pd.to_numeric(
        output["analytical_expected_structural_repair_ratio"], errors="raise"
    ).to_numpy(dtype=np.float64)

    sampled_loss_sum += float(sampled_losses.sum())
    expected_loss_sum += float(expected_losses.sum())
    maximum_sampled_equation_error = max(
        maximum_sampled_equation_error,
        float(np.max(np.abs(sampled_losses - replacement_values * sampled_ratios))),
    )
    maximum_expected_equation_error = max(
        maximum_expected_equation_error,
        float(np.max(np.abs(expected_losses - replacement_values * expected_ratios))),
    )
    maximum_sampled_loss_over_replacement = max(
        maximum_sampled_loss_over_replacement,
        float(np.max(sampled_losses / replacement_values)),
    )
    maximum_expected_loss_over_replacement = max(
        maximum_expected_loss_over_replacement,
        float(np.max(expected_losses / replacement_values)),
    )
    total_rows += len(output)

    for source_type, value in output.groupby("source_type")[
        "sampled_structural_ground_up_loss_2022_usd"
    ].sum().items():
        source_loss_totals[str(source_type)] += float(value)
    for occupancy, value in output.groupby("occ_type")[
        "sampled_structural_ground_up_loss_2022_usd"
    ].sum().items():
        occupancy_sampled_loss[str(occupancy)] += float(value)
    for occupancy, value in output.groupby("occ_type")[
        "analytical_expected_structural_ground_up_loss_2022_usd"
    ].sum().items():
        occupancy_expected_loss[str(occupancy)] += float(value)

    event_summary_frames.append(summarize_event_losses(output))
    chunk_paths.append(chunk_output_path)
    chunk_manifest_rows.append(
        {
            "chunk_id": chunk_id,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_occurrence_ordinal": int(output["occurrence_ordinal"].min()),
            "maximum_occurrence_ordinal": int(output["occurrence_ordinal"].max()),
            "sampled_loss_sum_2022_usd": float(sampled_losses.sum()),
            "analytical_expected_loss_sum_2022_usd": float(expected_losses.sum()),
            "input_cell4_path": str(damage_chunk_path),
            "input_cell4_sha256": actual_input_hash,
            "output_path": str(chunk_output_path),
            "output_size_bytes": int(chunk_output_path.stat().st_size),
            "output_sha256": sha256_file(chunk_output_path),
            "marker_path": str(marker_path),
            "validation_path": str(chunk_validation_path),
            "reused": bool(marker_valid),
        }
    )

concatenate_gzip_csv_files(chunk_paths, FINAL_BUILDING_LOSS_PATH)
event_summary = pd.concat(event_summary_frames, ignore_index=True)
event_summary = event_summary.sort_values("occurrence_ordinal").reset_index(drop=True)
write_gzip_csv_deterministic(event_summary, FINAL_EVENT_LOSS_PATH)

sampled_event_sum = float(
    event_summary["sampled_structural_ground_up_loss_2022_usd"].sum()
)
expected_event_sum = float(
    event_summary[
        "analytical_expected_structural_ground_up_loss_2022_usd"
    ].sum()
)
sampled_aal = sampled_event_sum / declared_catalog_years
analytical_expected_aal = expected_event_sum / declared_catalog_years
relative_sampled_expected_difference = (
    abs(sampled_loss_sum - expected_loss_sum) / expected_loss_sum
    if expected_loss_sum > 0
    else 0.0
)

append_check(
    validation_rows,
    "all_production_rows_processed",
    total_rows == expected_rows and seen_pairs.all(),
    f"processed={total_rows}; expected={expected_rows}; unseen={int((~seen_pairs).sum())}",
)
append_check(
    validation_rows,
    "every_occurrence_has_470_buildings",
    np.all(occurrence_counts == expected_sites),
    f"minimum={occurrence_counts.min()}; maximum={occurrence_counts.max()}",
)
append_check(
    validation_rows,
    "every_building_has_all_occurrences",
    np.all(site_counts == expected_occurrences),
    f"minimum={site_counts.min()}; maximum={site_counts.max()}",
)
append_check(
    validation_rows,
    "event_summary_has_one_row_per_occurrence",
    len(event_summary) == expected_occurrences
    and event_summary["occurrence_id"].is_unique,
    f"rows={len(event_summary)}; unique={event_summary['occurrence_id'].nunique()}",
)
append_check(
    validation_rows,
    "event_summary_building_counts_are_470",
    event_summary["buildings"].eq(expected_sites).all(),
    f"minimum={event_summary['buildings'].min()}; maximum={event_summary['buildings'].max()}",
)
append_check(
    validation_rows,
    "portfolio_replacement_value_constant_by_occurrence",
    np.allclose(
        event_summary["portfolio_replacement_value_2022_usd"].to_numpy(dtype=float),
        portfolio_replacement_value,
        rtol=0.0,
        atol=max(1e-6, portfolio_replacement_value * 1e-12),
    ),
    (
        f"minimum={event_summary['portfolio_replacement_value_2022_usd'].min():.2f}; "
        f"maximum={event_summary['portfolio_replacement_value_2022_usd'].max():.2f}; "
        f"expected={portfolio_replacement_value:.2f}"
    ),
)
append_check(
    validation_rows,
    "building_and_event_sampled_loss_totals_agree",
    math.isclose(sampled_loss_sum, sampled_event_sum, rel_tol=1e-12, abs_tol=1e-4),
    f"building={sampled_loss_sum:.6f}; event={sampled_event_sum:.6f}",
)
append_check(
    validation_rows,
    "building_and_event_expected_loss_totals_agree",
    math.isclose(expected_loss_sum, expected_event_sum, rel_tol=1e-12, abs_tol=1e-4),
    f"building={expected_loss_sum:.6f}; event={expected_event_sum:.6f}",
)
append_check(
    validation_rows,
    "sampled_loss_equation_error_small",
    maximum_sampled_equation_error <= NUMERIC_TOLERANCE,
    f"maximum_error={maximum_sampled_equation_error:.3e}",
)
append_check(
    validation_rows,
    "analytical_expected_loss_equation_error_small",
    maximum_expected_equation_error <= NUMERIC_TOLERANCE,
    f"maximum_error={maximum_expected_equation_error:.3e}",
)
append_check(
    validation_rows,
    "structural_loss_ratios_bounded",
    maximum_sampled_loss_over_replacement <= 1.0 + NUMERIC_TOLERANCE
    and maximum_expected_loss_over_replacement <= 1.0 + NUMERIC_TOLERANCE,
    (
        f"sampled_max={maximum_sampled_loss_over_replacement:.6f}; "
        f"expected_max={maximum_expected_loss_over_replacement:.6f}"
    ),
)
append_check(
    validation_rows,
    "sampled_total_reasonably_close_to_analytical_expected_total",
    relative_sampled_expected_difference <= LOSS_RECONCILIATION_WARNING_TOLERANCE,
    (
        f"relative_difference={relative_sampled_expected_difference:.6%}; "
        f"warning_tolerance={LOSS_RECONCILIATION_WARNING_TOLERANCE:.2%}"
    ),
    severity="warning",
)
append_check(
    validation_rows,
    "preliminary_aal_values_nonnegative",
    sampled_aal >= 0.0 and analytical_expected_aal >= 0.0,
    f"sampled_aal={sampled_aal:.6f}; expected_aal={analytical_expected_aal:.6f}",
)
append_check(
    validation_rows,
    "final_building_loss_file_exists",
    FINAL_BUILDING_LOSS_PATH.exists() and FINAL_BUILDING_LOSS_PATH.stat().st_size > 0,
    f"path={FINAL_BUILDING_LOSS_PATH}",
)
append_check(
    validation_rows,
    "final_event_loss_file_exists",
    FINAL_EVENT_LOSS_PATH.exists() and FINAL_EVENT_LOSS_PATH.stat().st_size > 0,
    f"path={FINAL_EVENT_LOSS_PATH}",
)

loss_reconciliation = pd.DataFrame(
    {
        "occ_type": sorted(
            set(occupancy_sampled_loss).union(occupancy_expected_loss)
        )
    }
)
loss_reconciliation["sampled_structural_loss_2022_usd"] = loss_reconciliation[
    "occ_type"
].map(occupancy_sampled_loss).fillna(0.0)
loss_reconciliation[
    "analytical_expected_structural_loss_2022_usd"
] = loss_reconciliation["occ_type"].map(occupancy_expected_loss).fillna(0.0)
loss_reconciliation["sampled_minus_expected_2022_usd"] = (
    loss_reconciliation["sampled_structural_loss_2022_usd"]
    - loss_reconciliation["analytical_expected_structural_loss_2022_usd"]
)
loss_reconciliation["relative_difference"] = np.where(
    loss_reconciliation["analytical_expected_structural_loss_2022_usd"] > 0,
    loss_reconciliation["sampled_minus_expected_2022_usd"]
    / loss_reconciliation["analytical_expected_structural_loss_2022_usd"],
    0.0,
)
loss_reconciliation.to_csv(LOSS_RECONCILIATION_PATH, index=False)

chunk_manifest = pd.DataFrame(chunk_manifest_rows)
chunk_manifest.to_csv(CELL5_CHUNK_MANIFEST_PATH, index=False)
validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()
validation.to_csv(CELL5_VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "scope": {
        "loss_component": "structural repair only",
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
        "not_included": [
            "drift-sensitive nonstructural repair loss",
            "acceleration-sensitive nonstructural repair loss",
            "contents loss",
            "business interruption",
            "insurance policy terms",
            "reinsurance terms",
        ],
    },
    "annual_catalog": {
        "declared_duration_years": declared_catalog_years,
        "occurrences": expected_occurrences,
        "sites": expected_sites,
        "rows": expected_rows,
        "zero_event_years_rule": (
            "The preliminary AAL denominator uses all declared catalog years. "
            "Annual aggregation and AEP/OEP calculations are deferred to the next cell."
        ),
    },
    "portfolio_valuation": {
        "portfolio_path": str(portfolio_path),
        "portfolio_sha256": sha256_file(portfolio_path),
        "buildings": int(len(building_values)),
        "occupancy_types": sorted(building_values["occ_type"].unique().tolist()),
        "total_floor_area_sqft": float(building_values["floor_area_sqft"].sum()),
        "total_replacement_value_2022_usd": portfolio_replacement_value,
        "general_replacement_cost_column": general_cost_source_column,
        "general_replacement_cost_sha256": reference_hashes[
            "general_replacement_costs"
        ],
        "res1_replacement_cost_sha256": reference_hashes[
            "res1_replacement_costs"
        ],
    },
    "structural_repair_ratios": {
        "mapping": {
            "None": 0.0,
            "Slight": "DS_0 / 100",
            "Moderate": "DS_1 / 100",
            "Extensive": "DS_2 / 100",
            "Complete": "DS_3 / 100",
        },
        "source_path": str(reference_paths["structural_repair_ratios"]),
        "source_sha256": reference_hashes["structural_repair_ratios"],
    },
    "loss_equations": {
        "sampled": (
            "sampled structural loss = building replacement value * structural "
            "repair ratio for the sampled structural damage state"
        ),
        "analytical_expected": (
            "expected structural loss = building replacement value * "
            "sum_d[P(DS=d|SA0P4) * structural repair ratio_d]"
        ),
    },
    "production": {
        "rows": total_rows,
        "occurrences": expected_occurrences,
        "chunks": len(chunk_paths),
        "sampled_loss_total_2022_usd": sampled_loss_sum,
        "analytical_expected_loss_total_2022_usd": expected_loss_sum,
        "sampled_preliminary_aal_2022_usd": sampled_aal,
        "analytical_expected_preliminary_aal_2022_usd": analytical_expected_aal,
        "relative_sampled_expected_difference": relative_sampled_expected_difference,
        "maximum_sampled_loss_ratio": maximum_sampled_loss_over_replacement,
        "maximum_analytical_expected_loss_ratio": maximum_expected_loss_over_replacement,
        "source_sampled_loss_totals_2022_usd": dict(sorted(source_loss_totals.items())),
    },
    "outputs": {
        "building_value_and_ratio_table": {
            "path": str(BUILDING_VALUE_PATH),
            "sha256": sha256_file(BUILDING_VALUE_PATH),
            "rows": int(len(building_values)),
        },
        "full_structural_ground_up_loss": {
            "path": str(FINAL_BUILDING_LOSS_PATH),
            "sha256": sha256_file(FINAL_BUILDING_LOSS_PATH),
            "size_bytes": int(FINAL_BUILDING_LOSS_PATH.stat().st_size),
            "rows": total_rows,
        },
        "event_structural_ground_up_loss": {
            "path": str(FINAL_EVENT_LOSS_PATH),
            "sha256": sha256_file(FINAL_EVENT_LOSS_PATH),
            "size_bytes": int(FINAL_EVENT_LOSS_PATH.stat().st_size),
            "rows": int(len(event_summary)),
        },
        "replacement_summary": str(REPLACEMENT_SUMMARY_PATH),
        "loss_reconciliation": str(LOSS_RECONCILIATION_PATH),
        "chunk_manifest": str(CELL5_CHUNK_MANIFEST_PATH),
        "validation": str(CELL5_VALIDATION_PATH),
        "summary": str(CELL5_SUMMARY_PATH),
    },
    "next_cell": (
        "Cell 6: aggregate occurrence-level structural losses into the full "
        "2,000,000-year annual loss series, explicitly insert zero-event years, "
        "and calculate structural AAL, OEP, AEP, and PML diagnostics."
    ),
}
write_json(CELL5_SUMMARY_PATH, summary)

print()
print("=" * 78)
print("NOTEBOOK 5 CELL 5 STRUCTURAL GROUND-UP LOSS COMPLETE")
print("=" * 78)
print(f"Production loss rows:             {total_rows:,}")
print(f"Catalog occurrences:              {expected_occurrences:,}")
print(f"Portfolio replacement value:      ${portfolio_replacement_value:,.0f}")
print(f"Sampled structural loss total:    ${sampled_loss_sum:,.0f}")
print(f"Expected structural loss total:   ${expected_loss_sum:,.0f}")
print(f"Sampled preliminary AAL:          ${sampled_aal:,.2f}")
print(f"Expected preliminary AAL:         ${analytical_expected_aal:,.2f}")
print(f"Sampled-expected relative diff.:   {relative_sampled_expected_difference:.4%}")
print(f"Maximum sampled equation error:   {maximum_sampled_equation_error:.3e}")
print(f"Maximum expected equation error:  {maximum_expected_equation_error:.3e}")
print(f"Critical validation checks:       {int(validation['severity'].eq('critical').sum()):,}")
print(f"Critical failures:                {len(critical_failures):,}")
print(f"Warnings requiring review:        {len(warning_failures):,}")
print()
print("Full structural ground-up loss:")
print(f"  {FINAL_BUILDING_LOSS_PATH}")
print("Occurrence-level structural loss:")
print(f"  {FINAL_EVENT_LOSS_PATH}")
print("Building replacement values:")
print(f"  {BUILDING_VALUE_PATH}")
print("Validation:")
print(f"  {CELL5_VALIDATION_PATH}")
print("Summary:")
print(f"  {CELL5_SUMMARY_PATH}")
print()
print("Next: aggregate occurrence losses into annual loss and risk metrics.")

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 5 failed one or more critical checks. Review: "
        f"{CELL5_VALIDATION_PATH}"
    )


FULL ANNUAL-CATALOG STRUCTURAL GROUND-UP LOSS
Structural damage rows:        4,996,100
Catalog occurrences:          10,630
Portfolio buildings:          470
Cell 4 production chunks:     43
Portfolio replacement value:  $384,236,605 (2022 USD)
Loss scope:                    structural repair component only

------------------------------------------------------------------------------
STRUCTURAL LOSS CHUNK 1 OF 43 [ID 0000]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
STRUCTURAL LOSS CHUNK 2 OF 43 [ID 0001]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
STRUCTURAL LOSS CHUNK 3 OF 43 [ID 0002]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
STRUCTURAL LOSS CHUNK 4 OF 43 [ID 0003]
Calculated 117,500 rows for 250 occurrences.

------------------------------

In [16]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook5_cell6_structural_annual_risk_metrics_v1"
DOLLAR_YEAR = 2022
RETURN_PERIODS_YEARS = [
    50,
    100,
    200,
    250,
    500,
    1_000,
    2_000,
    5_000,
    10_000,
    20_000,
    50_000,
    100_000,
    200_000,
    500_000,
    1_000_000,
    2_000_000,
]
NUMERIC_ATOL_USD = 1e-4
TAIL_SUPPORT_WARNING_RANK = 20


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_5_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_output,
            compresslevel=6,
            mtime=0,
        ) as compressed_output:
            with io.TextIOWrapper(
                compressed_output,
                encoding="utf-8",
                newline="",
            ) as text_output:
                frame.to_csv(text_output, index=False, lineterminator="\n")
    temporary.replace(path)


def empirical_pml(losses: np.ndarray, return_period_years: int) -> tuple[float, int, float]:
    n_years = int(losses.size)
    if return_period_years <= 0 or return_period_years > n_years:
        raise ValueError(
            f"Return period must be in [1, {n_years}], received {return_period_years}."
        )
    descending_rank = max(1, int(math.ceil(n_years / return_period_years)))
    sorted_losses = np.sort(losses)[::-1]
    loss = float(sorted_losses[descending_rank - 1])
    empirical_aep = descending_rank / n_years
    return loss, descending_rank, empirical_aep


def build_ep_curve(
    losses: np.ndarray,
    curve_type: str,
    loss_basis: str,
    n_years: int,
) -> pd.DataFrame:
    positive = losses[losses > 0.0]
    if positive.size == 0:
        return pd.DataFrame(
            columns=[
                "curve_type",
                "loss_basis",
                "descending_rank",
                "annual_exceedance_probability",
                "return_period_years",
                "loss_2022_usd",
            ]
        )
    ordered = np.sort(positive)[::-1]
    ranks = np.arange(1, ordered.size + 1, dtype=np.int64)
    return pd.DataFrame(
        {
            "curve_type": curve_type,
            "loss_basis": loss_basis,
            "descending_rank": ranks,
            "annual_exceedance_probability": ranks / n_years,
            "return_period_years": n_years / ranks,
            "loss_2022_usd": ordered,
        }
    )


def annual_statistics(losses: np.ndarray) -> dict[str, float | int]:
    mean_loss = float(np.mean(losses))
    standard_deviation = float(np.std(losses, ddof=0))
    positive = losses[losses > 0.0]
    return {
        "years": int(losses.size),
        "positive_loss_years": int(positive.size),
        "zero_loss_years": int(losses.size - positive.size),
        "annual_probability_of_positive_loss": float(positive.size / losses.size),
        "mean_annual_loss_2022_usd": mean_loss,
        "annual_loss_standard_deviation_2022_usd": standard_deviation,
        "annual_loss_cov": float(standard_deviation / mean_loss) if mean_loss > 0 else 0.0,
        "mean_positive_year_loss_2022_usd": float(np.mean(positive)) if positive.size else 0.0,
        "median_positive_year_loss_2022_usd": float(np.median(positive)) if positive.size else 0.0,
        "maximum_annual_loss_2022_usd": float(np.max(losses)),
    }


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_structural_risk"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CELL5_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_5_summary.json"
CELL5_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_5_validation.csv"
HANDOFF_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_4_final_handoff"
    / "notebook_5_input_handoff.json"
)
ANNUAL_SERIES_PATH = OUTPUT_DIR / "structural_annual_loss_series.csv.gz"
EP_CURVE_PATH = OUTPUT_DIR / "structural_exceedance_probability_curve.csv.gz"
PML_TABLE_PATH = OUTPUT_DIR / "structural_pml_table.csv"
RISK_METRICS_PATH = OUTPUT_DIR / "structural_risk_metrics.csv"
SOURCE_AAL_PATH = OUTPUT_DIR / "structural_source_aal_summary.csv"
PLOT_PNG_PATH = PLOT_DIR / "structural_aep_oep_curve.png"
PLOT_PDF_PATH = PLOT_DIR / "structural_aep_oep_curve.pdf"
CELL6_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_6_validation.csv"
CELL6_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_6_summary.json"

validation_rows: list[dict[str, Any]] = []
append_check(validation_rows, "cell5_summary_exists", CELL5_SUMMARY_PATH.exists(), str(CELL5_SUMMARY_PATH))
append_check(validation_rows, "cell5_validation_exists", CELL5_VALIDATION_PATH.exists(), str(CELL5_VALIDATION_PATH))
append_check(validation_rows, "notebook4_handoff_exists", HANDOFF_PATH.exists(), str(HANDOFF_PATH))

cell5_summary = load_json(CELL5_SUMMARY_PATH)
handoff = load_json(HANDOFF_PATH)
cell5_validation = pd.read_csv(CELL5_VALIDATION_PATH)
cell5_validation["passed"] = parse_bool_series(cell5_validation["passed"])
cell5_critical_failures = cell5_validation.loc[
    cell5_validation["severity"].eq("critical") & ~cell5_validation["passed"]
]
append_check(
    validation_rows,
    "cell5_critical_checks_passed",
    cell5_critical_failures.empty and bool(cell5_summary.get("all_critical_checks_passed")),
    f"critical_failures={len(cell5_critical_failures)}",
)

annual_catalog = handoff["annual_catalog"]
declared_catalog_years = int(annual_catalog["declared_duration_years"])
expected_occurrences = int(annual_catalog["occurrences"])
expected_occupied_years = int(annual_catalog["occupied_years"])
expected_zero_event_years = int(annual_catalog["zero_event_years"])
expected_multiple_event_years = int(annual_catalog["multiple_event_years"])
expected_maximum_events = int(annual_catalog["maximum_events_in_one_year"])

append_check(
    validation_rows,
    "declared_catalog_duration_positive",
    declared_catalog_years > 0,
    f"years={declared_catalog_years}",
)
append_check(
    validation_rows,
    "return_periods_supported_by_catalog",
    max(RETURN_PERIODS_YEARS) <= declared_catalog_years,
    f"maximum_return_period={max(RETURN_PERIODS_YEARS)}; catalog_years={declared_catalog_years}",
)

record = cell5_summary["outputs"]["event_structural_ground_up_loss"]
event_loss_path = resolve_recorded_path(PROJECT_ROOT, str(record["path"]))
actual_event_hash = sha256_file(event_loss_path)
append_check(
    validation_rows,
    "cell5_event_loss_hash_matches",
    actual_event_hash == str(record["sha256"]),
    f"expected={record['sha256']}; actual={actual_event_hash}",
)

events = pd.read_csv(event_loss_path, compression="gzip")
required_event_columns = {
    "catalog_year",
    "occurrence_id",
    "source_type",
    "sampled_structural_ground_up_loss_2022_usd",
    "analytical_expected_structural_ground_up_loss_2022_usd",
}
missing_event_columns = sorted(required_event_columns.difference(events.columns))
append_check(
    validation_rows,
    "event_loss_schema_complete",
    not missing_event_columns,
    f"missing={missing_event_columns}",
)
if missing_event_columns:
    raise RuntimeError(f"Cell 5 event-loss file is missing columns: {missing_event_columns}")

for column in [
    "catalog_year",
    "sampled_structural_ground_up_loss_2022_usd",
    "analytical_expected_structural_ground_up_loss_2022_usd",
]:
    events[column] = pd.to_numeric(events[column], errors="raise")
events["catalog_year"] = events["catalog_year"].astype(np.int64)

append_check(
    validation_rows,
    "event_summary_occurrence_count_matches",
    len(events) == expected_occurrences and events["occurrence_id"].is_unique,
    f"rows={len(events)}; expected={expected_occurrences}; unique={events['occurrence_id'].nunique()}",
)
append_check(
    validation_rows,
    "catalog_year_labels_within_declared_duration",
    events["catalog_year"].between(1, declared_catalog_years).all(),
    f"minimum={events['catalog_year'].min()}; maximum={events['catalog_year'].max()}",
)
append_check(
    validation_rows,
    "event_losses_finite_and_nonnegative",
    np.isfinite(
        events[
            [
                "sampled_structural_ground_up_loss_2022_usd",
                "analytical_expected_structural_ground_up_loss_2022_usd",
            ]
        ].to_numpy(dtype=float)
    ).all()
    and (
        events[
            [
                "sampled_structural_ground_up_loss_2022_usd",
                "analytical_expected_structural_ground_up_loss_2022_usd",
            ]
        ]
        >= 0.0
    ).all().all(),
    "sampled and analytical expected occurrence losses checked",
)

annual_occupied = (
    events.groupby("catalog_year", sort=True)
    .agg(
        event_count=("occurrence_id", "size"),
        sampled_aep_structural_loss_2022_usd=(
            "sampled_structural_ground_up_loss_2022_usd",
            "sum",
        ),
        sampled_oep_structural_loss_2022_usd=(
            "sampled_structural_ground_up_loss_2022_usd",
            "max",
        ),
        analytical_expected_aep_structural_loss_2022_usd=(
            "analytical_expected_structural_ground_up_loss_2022_usd",
            "sum",
        ),
        analytical_expected_oep_structural_loss_2022_usd=(
            "analytical_expected_structural_ground_up_loss_2022_usd",
            "max",
        ),
    )
    .reset_index()
)

catalog_year = np.arange(1, declared_catalog_years + 1, dtype=np.int64)
event_count = np.zeros(declared_catalog_years, dtype=np.int16)
sampled_aep = np.zeros(declared_catalog_years, dtype=np.float64)
sampled_oep = np.zeros(declared_catalog_years, dtype=np.float64)
expected_aep = np.zeros(declared_catalog_years, dtype=np.float64)
expected_oep = np.zeros(declared_catalog_years, dtype=np.float64)

positions = annual_occupied["catalog_year"].to_numpy(dtype=np.int64) - 1
event_count[positions] = annual_occupied["event_count"].to_numpy(dtype=np.int16)
sampled_aep[positions] = annual_occupied[
    "sampled_aep_structural_loss_2022_usd"
].to_numpy(dtype=np.float64)
sampled_oep[positions] = annual_occupied[
    "sampled_oep_structural_loss_2022_usd"
].to_numpy(dtype=np.float64)
expected_aep[positions] = annual_occupied[
    "analytical_expected_aep_structural_loss_2022_usd"
].to_numpy(dtype=np.float64)
expected_oep[positions] = annual_occupied[
    "analytical_expected_oep_structural_loss_2022_usd"
].to_numpy(dtype=np.float64)

annual_series = pd.DataFrame(
    {
        "catalog_year": catalog_year,
        "event_count": event_count,
        "zero_event_year": event_count == 0,
        "sampled_aep_structural_loss_2022_usd": sampled_aep,
        "sampled_oep_structural_loss_2022_usd": sampled_oep,
        "analytical_expected_aep_structural_loss_2022_usd": expected_aep,
        "analytical_expected_oep_structural_loss_2022_usd": expected_oep,
    }
)
write_gzip_csv_deterministic(annual_series, ANNUAL_SERIES_PATH)

occupied_years = int(np.count_nonzero(event_count))
zero_event_years = int(np.count_nonzero(event_count == 0))
multiple_event_years = int(np.count_nonzero(event_count > 1))
maximum_events_in_year = int(event_count.max())

append_check(
    validation_rows,
    "annual_series_contains_all_catalog_years",
    len(annual_series) == declared_catalog_years
    and annual_series["catalog_year"].iloc[0] == 1
    and annual_series["catalog_year"].iloc[-1] == declared_catalog_years,
    f"rows={len(annual_series)}; first={annual_series['catalog_year'].iloc[0]}; last={annual_series['catalog_year'].iloc[-1]}",
)
append_check(
    validation_rows,
    "event_counts_reconcile",
    int(event_count.sum()) == expected_occurrences,
    f"annual_sum={int(event_count.sum())}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "occupied_year_count_matches_handoff",
    occupied_years == expected_occupied_years,
    f"actual={occupied_years}; expected={expected_occupied_years}",
)
append_check(
    validation_rows,
    "zero_event_year_count_matches_handoff",
    zero_event_years == expected_zero_event_years,
    f"actual={zero_event_years}; expected={expected_zero_event_years}",
)
append_check(
    validation_rows,
    "multiple_event_year_count_matches_handoff",
    multiple_event_years == expected_multiple_event_years,
    f"actual={multiple_event_years}; expected={expected_multiple_event_years}",
)
append_check(
    validation_rows,
    "maximum_events_per_year_matches_handoff",
    maximum_events_in_year == expected_maximum_events,
    f"actual={maximum_events_in_year}; expected={expected_maximum_events}",
)
append_check(
    validation_rows,
    "aep_not_less_than_oep",
    np.all(sampled_aep + NUMERIC_ATOL_USD >= sampled_oep)
    and np.all(expected_aep + NUMERIC_ATOL_USD >= expected_oep),
    "checked sampled and analytical expected annual losses",
)
append_check(
    validation_rows,
    "single_and_zero_event_years_have_equal_aep_oep",
    np.allclose(
        sampled_aep[event_count <= 1],
        sampled_oep[event_count <= 1],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    )
    and np.allclose(
        expected_aep[event_count <= 1],
        expected_oep[event_count <= 1],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    f"years_checked={int(np.count_nonzero(event_count <= 1))}",
)
append_check(
    validation_rows,
    "sampled_annual_total_matches_event_total",
    math.isclose(
        float(sampled_aep.sum()),
        float(events["sampled_structural_ground_up_loss_2022_usd"].sum()),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    f"annual={sampled_aep.sum():.6f}; event={events['sampled_structural_ground_up_loss_2022_usd'].sum():.6f}",
)
append_check(
    validation_rows,
    "expected_annual_total_matches_event_total",
    math.isclose(
        float(expected_aep.sum()),
        float(events["analytical_expected_structural_ground_up_loss_2022_usd"].sum()),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    f"annual={expected_aep.sum():.6f}; event={events['analytical_expected_structural_ground_up_loss_2022_usd'].sum():.6f}",
)

sampled_aal = float(sampled_aep.mean())
expected_aal = float(expected_aep.mean())
cell5_sampled_aal = float(
    cell5_summary["production"]["sampled_preliminary_aal_2022_usd"]
)
cell5_expected_aal = float(
    cell5_summary["production"]["analytical_expected_preliminary_aal_2022_usd"]
)
append_check(
    validation_rows,
    "sampled_aal_matches_cell5",
    math.isclose(sampled_aal, cell5_sampled_aal, rel_tol=1e-12, abs_tol=NUMERIC_ATOL_USD),
    f"cell6={sampled_aal:.9f}; cell5={cell5_sampled_aal:.9f}",
)
append_check(
    validation_rows,
    "expected_aal_matches_cell5",
    math.isclose(expected_aal, cell5_expected_aal, rel_tol=1e-12, abs_tol=NUMERIC_ATOL_USD),
    f"cell6={expected_aal:.9f}; cell5={cell5_expected_aal:.9f}",
)

curve_frames = [
    build_ep_curve(sampled_aep, "AEP", "sampled", declared_catalog_years),
    build_ep_curve(sampled_oep, "OEP", "sampled", declared_catalog_years),
    build_ep_curve(expected_aep, "AEP", "analytical_expected", declared_catalog_years),
    build_ep_curve(expected_oep, "OEP", "analytical_expected", declared_catalog_years),
]
ep_curve = pd.concat(curve_frames, ignore_index=True)
write_gzip_csv_deterministic(ep_curve, EP_CURVE_PATH)

pml_rows: list[dict[str, Any]] = []
loss_vectors = {
    ("sampled", "AEP"): sampled_aep,
    ("sampled", "OEP"): sampled_oep,
    ("analytical_expected", "AEP"): expected_aep,
    ("analytical_expected", "OEP"): expected_oep,
}
for return_period in RETURN_PERIODS_YEARS:
    row: dict[str, Any] = {
        "return_period_years": int(return_period),
        "target_annual_exceedance_probability": 1.0 / return_period,
    }
    supporting_ranks: list[int] = []
    for (loss_basis, curve_type), values in loss_vectors.items():
        loss, rank, empirical_aep = empirical_pml(values, return_period)
        prefix = f"{loss_basis}_{curve_type.lower()}"
        row[f"{prefix}_pml_2022_usd"] = loss
        row[f"{prefix}_supporting_descending_rank"] = rank
        row[f"{prefix}_empirical_aep"] = empirical_aep
        supporting_ranks.append(rank)
    row["minimum_supporting_descending_rank"] = min(supporting_ranks)
    row["tail_support_flag"] = (
        "thin_tail_support"
        if min(supporting_ranks) < TAIL_SUPPORT_WARNING_RANK
        else "adequate_order_statistic_support"
    )
    pml_rows.append(row)
pml_table = pd.DataFrame(pml_rows)
pml_table.to_csv(PML_TABLE_PATH, index=False)

for loss_basis in ["sampled", "analytical_expected"]:
    for curve_type in ["aep", "oep"]:
        column = f"{loss_basis}_{curve_type}_pml_2022_usd"
        append_check(
            validation_rows,
            f"{loss_basis}_{curve_type}_pml_nondecreasing_with_return_period",
            np.all(np.diff(pml_table[column].to_numpy(dtype=float)) >= -NUMERIC_ATOL_USD),
            f"minimum_difference={np.diff(pml_table[column].to_numpy(dtype=float)).min():.6f}",
        )
append_check(
    validation_rows,
    "sampled_aep_pml_not_less_than_sampled_oep_pml",
    np.all(
        pml_table["sampled_aep_pml_2022_usd"].to_numpy(dtype=float)
        + NUMERIC_ATOL_USD
        >= pml_table["sampled_oep_pml_2022_usd"].to_numpy(dtype=float)
    ),
    "checked all requested return periods",
)
append_check(
    validation_rows,
    "expected_aep_pml_not_less_than_expected_oep_pml",
    np.all(
        pml_table["analytical_expected_aep_pml_2022_usd"].to_numpy(dtype=float)
        + NUMERIC_ATOL_USD
        >= pml_table["analytical_expected_oep_pml_2022_usd"].to_numpy(dtype=float)
    ),
    "checked all requested return periods",
)

risk_rows: list[dict[str, Any]] = []
for loss_basis, curve_type, values in [
    ("sampled", "AEP", sampled_aep),
    ("sampled", "OEP", sampled_oep),
    ("analytical_expected", "AEP", expected_aep),
    ("analytical_expected", "OEP", expected_oep),
]:
    risk_rows.append(
        {
            "loss_basis": loss_basis,
            "curve_type": curve_type,
            **annual_statistics(values),
        }
    )
risk_metrics = pd.DataFrame(risk_rows)
risk_metrics.to_csv(RISK_METRICS_PATH, index=False)

source_aal = (
    events.groupby("source_type", sort=True)
    .agg(
        occurrences=("occurrence_id", "size"),
        sampled_structural_loss_total_2022_usd=(
            "sampled_structural_ground_up_loss_2022_usd",
            "sum",
        ),
        analytical_expected_structural_loss_total_2022_usd=(
            "analytical_expected_structural_ground_up_loss_2022_usd",
            "sum",
        ),
    )
    .reset_index()
)
source_aal["sampled_structural_aal_2022_usd"] = (
    source_aal["sampled_structural_loss_total_2022_usd"] / declared_catalog_years
)
source_aal["analytical_expected_structural_aal_2022_usd"] = (
    source_aal["analytical_expected_structural_loss_total_2022_usd"]
    / declared_catalog_years
)
source_aal["sampled_aal_share"] = np.where(
    sampled_aal > 0,
    source_aal["sampled_structural_aal_2022_usd"] / sampled_aal,
    0.0,
)
source_aal["analytical_expected_aal_share"] = np.where(
    expected_aal > 0,
    source_aal["analytical_expected_structural_aal_2022_usd"] / expected_aal,
    0.0,
)
source_aal.to_csv(SOURCE_AAL_PATH, index=False)

append_check(
    validation_rows,
    "source_sampled_aal_reconciles",
    math.isclose(
        float(source_aal["sampled_structural_aal_2022_usd"].sum()),
        sampled_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    f"source_sum={source_aal['sampled_structural_aal_2022_usd'].sum():.9f}; portfolio={sampled_aal:.9f}",
)
append_check(
    validation_rows,
    "source_expected_aal_reconciles",
    math.isclose(
        float(source_aal["analytical_expected_structural_aal_2022_usd"].sum()),
        expected_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    f"source_sum={source_aal['analytical_expected_structural_aal_2022_usd'].sum():.9f}; portfolio={expected_aal:.9f}",
)

plot_sampled = ep_curve.loc[ep_curve["loss_basis"].eq("sampled")].copy()
fig, ax = plt.subplots(figsize=(8.0, 5.0))
for curve_type in ["OEP", "AEP"]:
    subset = plot_sampled.loc[plot_sampled["curve_type"].eq(curve_type)]
    ax.plot(
        subset["return_period_years"],
        subset["loss_2022_usd"] / 1_000_000.0,
        linewidth=1.8,
        label=f"Sampled {curve_type}",
    )
ax.set_xscale("log")
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Structural ground-up loss (million 2022 USD)")
ax.set_title("Structural annual exceedance curves")
ax.grid(True, which="both", linewidth=0.4, alpha=0.35)
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_PNG_PATH, dpi=300)
fig.savefig(PLOT_PDF_PATH)
plt.close(fig)

for output_path, check_id in [
    (ANNUAL_SERIES_PATH, "annual_series_file_exists"),
    (EP_CURVE_PATH, "ep_curve_file_exists"),
    (PML_TABLE_PATH, "pml_table_file_exists"),
    (RISK_METRICS_PATH, "risk_metrics_file_exists"),
    (SOURCE_AAL_PATH, "source_aal_file_exists"),
    (PLOT_PNG_PATH, "ep_plot_png_exists"),
    (PLOT_PDF_PATH, "ep_plot_pdf_exists"),
]:
    append_check(
        validation_rows,
        check_id,
        output_path.exists() and output_path.stat().st_size > 0,
        str(output_path),
    )

thin_tail_return_periods = pml_table.loc[
    pml_table["tail_support_flag"].eq("thin_tail_support"),
    "return_period_years",
].astype(int).tolist()
append_check(
    validation_rows,
    "extreme_return_period_tail_support_documented",
    len(thin_tail_return_periods) == 0,
    f"thin_tail_return_periods={thin_tail_return_periods}; threshold_rank={TAIL_SUPPORT_WARNING_RANK}",
    severity="warning",
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()
validation.to_csv(CELL6_VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "scope": {
        "loss_component": "structural repair only",
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
        "official_ep_basis": "sampled structural damage states and sampled structural losses",
        "analytical_expected_curves_use": (
            "reconciliation diagnostic based on conditional expected structural loss; "
            "not a replacement for the sampled annual loss distribution"
        ),
    },
    "annual_catalog": {
        "declared_duration_years": declared_catalog_years,
        "occurrences": expected_occurrences,
        "occupied_years": occupied_years,
        "multiple_event_years": multiple_event_years,
        "zero_event_years": zero_event_years,
        "maximum_events_in_one_year": maximum_events_in_year,
    },
    "risk_metrics": {
        "sampled_structural_aal_2022_usd": sampled_aal,
        "analytical_expected_structural_aal_2022_usd": expected_aal,
        "sampled_positive_aep_years": int(np.count_nonzero(sampled_aep > 0.0)),
        "sampled_positive_oep_years": int(np.count_nonzero(sampled_oep > 0.0)),
        "maximum_sampled_annual_aggregate_loss_2022_usd": float(sampled_aep.max()),
        "maximum_sampled_annual_occurrence_loss_2022_usd": float(sampled_oep.max()),
    },
    "pml_method": {
        "definition": (
            "For return period R, sort all declared annual losses in descending order "
            "and select rank ceil(N/R), where N is the declared catalog duration."
        ),
        "return_periods_years": RETURN_PERIODS_YEARS,
        "tail_support_warning_rank": TAIL_SUPPORT_WARNING_RANK,
        "thin_tail_return_periods": thin_tail_return_periods,
    },
    "source_files": {
        "cell5_event_loss_path": str(event_loss_path),
        "cell5_event_loss_sha256": actual_event_hash,
        "cell5_summary_path": str(CELL5_SUMMARY_PATH),
        "cell5_summary_sha256": sha256_file(CELL5_SUMMARY_PATH),
        "notebook4_handoff_path": str(HANDOFF_PATH),
        "notebook4_handoff_sha256": sha256_file(HANDOFF_PATH),
    },
    "outputs": {
        "annual_loss_series": {
            "path": str(ANNUAL_SERIES_PATH),
            "sha256": sha256_file(ANNUAL_SERIES_PATH),
            "rows": declared_catalog_years,
        },
        "exceedance_probability_curve": {
            "path": str(EP_CURVE_PATH),
            "sha256": sha256_file(EP_CURVE_PATH),
            "rows": int(len(ep_curve)),
        },
        "pml_table": {
            "path": str(PML_TABLE_PATH),
            "sha256": sha256_file(PML_TABLE_PATH),
            "rows": int(len(pml_table)),
        },
        "risk_metrics": {
            "path": str(RISK_METRICS_PATH),
            "sha256": sha256_file(RISK_METRICS_PATH),
            "rows": int(len(risk_metrics)),
        },
        "source_aal_summary": {
            "path": str(SOURCE_AAL_PATH),
            "sha256": sha256_file(SOURCE_AAL_PATH),
            "rows": int(len(source_aal)),
        },
        "plot_png": str(PLOT_PNG_PATH),
        "plot_pdf": str(PLOT_PDF_PATH),
        "validation": str(CELL6_VALIDATION_PATH),
        "summary": str(CELL6_SUMMARY_PATH),
    },
    "next_cell": (
        "Cell 7: finalize the structural-only Notebook 5 baseline or begin the "
        "separate nonstructural damage and repair-loss extension before insurance terms."
    ),
}
write_json(CELL6_SUMMARY_PATH, summary)

print()
print("=" * 78)
print("NOTEBOOK 5 CELL 6 STRUCTURAL ANNUAL RISK METRICS COMPLETE")
print("=" * 78)
print(f"Declared catalog years:          {declared_catalog_years:,}")
print(f"Catalog occurrences:             {expected_occurrences:,}")
print(f"Occupied years:                  {occupied_years:,}")
print(f"Multiple-event years:            {multiple_event_years:,}")
print(f"Zero-event years:                {zero_event_years:,}")
print(f"Sampled structural AAL:          ${sampled_aal:,.2f}")
print(f"Analytical expected AAL:         ${expected_aal:,.2f}")
print(f"Maximum sampled AEP loss:        ${sampled_aep.max():,.0f}")
print(f"Maximum sampled OEP loss:        ${sampled_oep.max():,.0f}")
print(f"Critical validation checks:      {int(validation['severity'].eq('critical').sum())}")
print(f"Critical failures:               {len(critical_failures)}")
print(f"Warnings requiring review:       {len(warning_failures)}")
print()
print("Annual loss series:")
print(f"  {ANNUAL_SERIES_PATH}")
print("PML table:")
print(f"  {PML_TABLE_PATH}")
print("Exceedance curve:")
print(f"  {EP_CURVE_PATH}")
print("Validation:")
print(f"  {CELL6_VALIDATION_PATH}")
print("Summary:")
print(f"  {CELL6_SUMMARY_PATH}")
print()
print(
    "Next: finalize the structural-only Notebook 5 baseline or add the separate "
    "nonstructural damage and repair-loss extension."
)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 6 failed one or more critical checks. Review: "
        f"{CELL6_VALIDATION_PATH}"
    )



NOTEBOOK 5 CELL 6 STRUCTURAL ANNUAL RISK METRICS COMPLETE
Declared catalog years:          2,000,000
Catalog occurrences:             10,630
Occupied years:                  10,593
Multiple-event years:            36
Zero-event years:                1,989,407
Sampled structural AAL:          $17,125.97
Analytical expected AAL:         $17,138.05
Maximum sampled AEP loss:        $39,425,005
Maximum sampled OEP loss:        $39,425,005
Critical validation checks:      38
Critical failures:               0
Warnings requiring review:       1

Annual loss series:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_structural_risk\structural_annual_loss_series.csv.gz
PML table:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_structural_risk\structural_pml_table.csv
Exceedance curve:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_structural_risk\structural_ex

In [17]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path(
    r"."
)

validation_path = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_5_damage_loss"
    / "notebook_5_cell_6_validation.csv"
)

validation = pd.read_csv(validation_path)

warnings = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"].astype(bool)
]

display(
    warnings[
        ["check_id", "severity", "passed", "detail"]
    ]
)

,check_id,severity,passed,detail
38,extreme_return_period_tail_support_documented,warning,False,"thin_tail_return_periods=[200000, 500000, 1000000, 2000000]; threshold_rank=20"


In [18]:
from __future__ import annotations

import hashlib
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

EXPECTED_STRUCTURAL_TYPE = "W2"
EXPECTED_PERIOD_S = 0.40
STANDARD_GRAVITY_IN_S2 = 386.08858267716535
DAMAGE_STATES = ["Slight", "Moderate", "Extensive", "Complete"]
DESIGN_LEVELS = ["HighCode", "ModerateCode", "LowCode", "PreCode"]
DRIFT_METHOD_ID = "equivalent_direct_sa0p4_from_sd_v1"
ACCELERATION_METHOD_ID = "direct_sa0p4_identity_from_source_sa_v1"
PRODUCTION_IM_COLUMN = "sa0p4_simulated_g"


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (candidate / "data" / "metadata" / "notebook_5_damage_loss").exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run this cell from the "
        "seismic-correlation-insurance-loss repository."
    )


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def parse_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False)
        .astype(bool)
    )


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def normalize_component(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "_", normalize_text(value).lower()).strip("_")


def resolve_recorded_path(project_root: Path, recorded_path: str | Path) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    basename_matches = list((project_root / "data").rglob(Path(normalized).name))
    if len(basename_matches) == 1:
        return basename_matches[0]
    if len(basename_matches) > 1:
        raise RuntimeError(
            f"Multiple files match recorded path basename {Path(normalized).name}: "
            f"{[str(path) for path in basename_matches]}"
        )
    raise FileNotFoundError(f"Could not resolve recorded path: {recorded_path}")


def select_component_rows(source: pd.DataFrame, target: str) -> pd.DataFrame:
    component = source["component"].map(normalize_component)
    demand = source["source_demand_type"].map(normalize_component)

    structural_mask = component.eq("structural") | component.str.fullmatch(
        r"structural(_fragility)?", na=False
    )
    if target == "drift":
        mask = (
            component.str.contains("drift", na=False)
            | component.str.contains("nsd", na=False)
            | ((demand.str.contains("displacement", na=False)) & ~structural_mask)
        )
    elif target == "acceleration":
        mask = (
            component.str.contains("accel", na=False)
            | component.str.contains("nsa", na=False)
            | ((demand.str.contains("acceleration", na=False)) & ~structural_mask)
        )
    else:
        raise ValueError(f"Unsupported target component: {target}")

    selected = source.loc[mask].copy()
    if selected.empty:
        raise ValueError(
            f"Could not identify {target}-sensitive nonstructural rows. "
            f"Available component values: {sorted(source['component'].astype(str).unique())}"
        )
    return selected


def validate_source_component(
    frame: pd.DataFrame,
    component_label: str,
    expected_unit: str,
    checks: list[dict[str, Any]],
) -> pd.DataFrame:
    required = {
        "component",
        "structural_type",
        "design_level",
        "damage_state",
        "source_median",
        "source_beta_ln",
        "source_demand_type",
        "source_median_unit",
        "source_path",
        "source_sheet",
    }
    missing = sorted(required.difference(frame.columns))
    append_check(
        checks,
        f"{component_label}_source_columns_present",
        not missing,
        f"missing={missing}",
    )
    if missing:
        raise KeyError(f"{component_label} source rows are missing columns: {missing}")

    result = frame.copy()
    result["structural_type"] = result["structural_type"].astype(str).str.strip().str.upper()
    result["design_level"] = result["design_level"].astype(str).str.strip()
    result["damage_state"] = result["damage_state"].astype(str).str.strip()
    result["source_median"] = pd.to_numeric(result["source_median"], errors="coerce")
    result["source_beta_ln"] = pd.to_numeric(result["source_beta_ln"], errors="coerce")
    result["source_median_unit"] = result["source_median_unit"].astype(str).str.strip().str.lower()

    append_check(
        checks,
        f"{component_label}_source_has_16_rows",
        len(result) == 16,
        f"rows={len(result)}",
    )
    append_check(
        checks,
        f"{component_label}_source_is_w2",
        result["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
        f"types={sorted(result['structural_type'].unique().tolist())}",
    )
    append_check(
        checks,
        f"{component_label}_design_levels_complete",
        set(result["design_level"]) == set(DESIGN_LEVELS),
        f"levels={sorted(result['design_level'].unique().tolist())}",
    )
    append_check(
        checks,
        f"{component_label}_damage_states_complete",
        set(result["damage_state"]) == set(DAMAGE_STATES),
        f"states={sorted(result['damage_state'].unique().tolist())}",
    )
    append_check(
        checks,
        f"{component_label}_design_state_pairs_unique",
        not result.duplicated(["design_level", "damage_state"]).any(),
        f"duplicates={int(result.duplicated(['design_level', 'damage_state']).sum())}",
    )
    append_check(
        checks,
        f"{component_label}_source_unit_expected",
        set(result["source_median_unit"]) == {expected_unit.lower()},
        f"units={sorted(result['source_median_unit'].unique().tolist())}",
    )
    append_check(
        checks,
        f"{component_label}_source_medians_positive_complete",
        result["source_median"].notna().all() and (result["source_median"] > 0).all(),
        f"missing={int(result['source_median'].isna().sum())}; "
        f"nonpositive={int((result['source_median'].fillna(0) <= 0).sum())}",
    )
    append_check(
        checks,
        f"{component_label}_source_betas_positive_complete",
        result["source_beta_ln"].notna().all() and (result["source_beta_ln"] > 0).all(),
        f"missing={int(result['source_beta_ln'].isna().sum())}; "
        f"nonpositive={int((result['source_beta_ln'].fillna(0) <= 0).sum())}",
    )

    state_rank = {state: index for index, state in enumerate(DAMAGE_STATES)}
    result["damage_state_rank"] = result["damage_state"].map(state_rank)
    ordered = result.sort_values(["design_level", "damage_state_rank"])
    ordered_medians = ordered.groupby("design_level", sort=False)["source_median"].apply(
        lambda values: np.all(np.diff(values.to_numpy(dtype=float)) > 0)
    )
    append_check(
        checks,
        f"{component_label}_source_medians_strictly_ordered",
        bool(ordered_medians.all()),
        f"by_design_level={ordered_medians.to_dict()}",
    )

    return result


def build_long_parameter_table(
    source: pd.DataFrame,
    component: str,
    conversion_factor: float,
) -> pd.DataFrame:
    output = source.copy()
    output["component"] = component
    output = output.rename(
        columns={
            "source_median": "source_median_value",
            "source_beta_ln": "source_beta_ln",
        }
    )

    if component == "nonstructural_drift_sensitive":
        output["direct_sa0p4_median_g"] = (
            output["source_median_value"].to_numpy(dtype=float) * conversion_factor
        )
        output["conversion_method_id"] = DRIFT_METHOD_ID
        output["transformation_equation"] = (
            "Sa_g = 4*pi^2*Sd_in/(g_in_s2*T_s^2)"
        )
        output["transformation_note"] = (
            "Equivalent fixed-period direct-SA0P4 approximation to the source "
            "spectral-displacement fragility; this is not the full HAZUS "
            "capacity-spectrum procedure."
        )
    elif component == "nonstructural_acceleration_sensitive":
        output["direct_sa0p4_median_g"] = output["source_median_value"].to_numpy(dtype=float)
        output["conversion_method_id"] = ACCELERATION_METHOD_ID
        output["transformation_equation"] = "Sa0.4_g = source spectral-acceleration median_g"
        output["transformation_note"] = (
            "Identity mapping of source spectral-acceleration thresholds to SA0P4. "
            "This is a documented direct-IM approximation and not the full HAZUS "
            "capacity-spectrum procedure."
        )
    else:
        raise ValueError(component)

    output["direct_sa0p4_beta_ln"] = output["source_beta_ln"].to_numpy(dtype=float)
    output["production_im_column"] = PRODUCTION_IM_COLUMN
    output["production_period_s"] = EXPECTED_PERIOD_S
    output["production_unit"] = "g"
    output["source_file_sha256"] = ""
    return output


def attach_source_hashes(project_root: Path, table: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str]]:
    output = table.copy()
    hashes: dict[str, str] = {}
    resolved_paths: dict[str, str] = {}

    for recorded in sorted(output["source_path"].astype(str).unique()):
        resolved = resolve_recorded_path(project_root, recorded)
        file_hash = sha256_file(resolved)
        hashes[str(resolved)] = file_hash
        resolved_paths[recorded] = str(resolved)
        output.loc[output["source_path"].astype(str).eq(recorded), "source_path"] = str(resolved)
        output.loc[output["source_path"].astype(str).eq(str(resolved)), "source_file_sha256"] = file_hash

    return output, hashes


def to_wide(table: pd.DataFrame, prefix: str, source_suffix: str) -> pd.DataFrame:
    median_source = table.pivot(
        index="design_level", columns="damage_state", values="source_median_value"
    ).rename(
        columns={
            state: f"{prefix}_{state.lower()}_source_median_{source_suffix}"
            for state in DAMAGE_STATES
        }
    )
    median_direct = table.pivot(
        index="design_level", columns="damage_state", values="direct_sa0p4_median_g"
    ).rename(
        columns={
            state: f"{prefix}_{state.lower()}_median_sa0p4_g"
            for state in DAMAGE_STATES
        }
    )
    beta = table.pivot(
        index="design_level", columns="damage_state", values="direct_sa0p4_beta_ln"
    ).rename(
        columns={state: f"{prefix}_{state.lower()}_beta_ln" for state in DAMAGE_STATES}
    )
    return median_source.join(median_direct).join(beta).reset_index()


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
PARAMETER_DIR.mkdir(parents=True, exist_ok=True)

CELL1_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_1_summary.json"
CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_1_validation.csv"
CELL2_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_2_summary.json"
CELL2_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_2_validation.csv"
SOURCE_INVENTORY_PATH = PARAMETER_DIR / "notebook_5_w2_source_fragility_inventory.csv"
DESIGN_ASSIGNMENTS_PATH = PARAMETER_DIR / "seaside_w2_design_level_assignments.csv"
STRUCTURAL_ASSIGNMENTS_PATH = PARAMETER_DIR / "seaside_w2_building_fragility_assignments.csv"

required_paths = [
    CELL1_SUMMARY_PATH,
    CELL1_VALIDATION_PATH,
    CELL2_SUMMARY_PATH,
    CELL2_VALIDATION_PATH,
    SOURCE_INVENTORY_PATH,
    DESIGN_ASSIGNMENTS_PATH,
    STRUCTURAL_ASSIGNMENTS_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required Cell 1 or Cell 2 outputs are missing: {missing_paths}")

validation_rows: list[dict[str, Any]] = []
cell1_summary = load_json(CELL1_SUMMARY_PATH)
cell2_summary = load_json(CELL2_SUMMARY_PATH)
cell1_validation = pd.read_csv(CELL1_VALIDATION_PATH)
cell2_validation = pd.read_csv(CELL2_VALIDATION_PATH)

for cell_number, summary, validation in [
    (1, cell1_summary, cell1_validation),
    (2, cell2_summary, cell2_validation),
]:
    append_check(
        validation_rows,
        f"cell{cell_number}_critical_checks_passed",
        summary.get("all_critical_checks_passed") is True,
        f"all_critical_checks_passed={summary.get('all_critical_checks_passed')}",
    )
    passed = parse_bool_series(validation["passed"])
    unresolved_critical = validation.loc[
        validation["severity"].astype(str).str.lower().eq("critical") & ~passed
    ]
    append_check(
        validation_rows,
        f"cell{cell_number}_has_no_unresolved_critical_checks",
        unresolved_critical.empty,
        f"unresolved_critical={len(unresolved_critical)}",
    )

source_inventory = pd.read_csv(SOURCE_INVENTORY_PATH)
required_inventory_columns = {
    "component",
    "structural_type",
    "design_level",
    "damage_state",
    "source_median",
    "source_beta_ln",
    "source_demand_type",
    "source_median_unit",
    "source_path",
    "source_sheet",
}
missing_inventory_columns = sorted(required_inventory_columns.difference(source_inventory.columns))
append_check(
    validation_rows,
    "source_inventory_columns_present",
    not missing_inventory_columns,
    f"missing={missing_inventory_columns}",
)
if missing_inventory_columns:
    raise KeyError(f"Source fragility inventory is missing columns: {missing_inventory_columns}")

source_drift = validate_source_component(
    select_component_rows(source_inventory, "drift"),
    "nonstructural_drift",
    "in",
    validation_rows,
)
source_acceleration = validate_source_component(
    select_component_rows(source_inventory, "acceleration"),
    "nonstructural_acceleration",
    "g",
    validation_rows,
)

period_s = float(
    cell2_summary.get("fragility_transformation", {}).get("period_s", EXPECTED_PERIOD_S)
)
append_check(
    validation_rows,
    "production_period_is_0p4_seconds",
    math.isclose(period_s, EXPECTED_PERIOD_S, rel_tol=0.0, abs_tol=1e-12),
    f"period_s={period_s}",
)
if not math.isclose(period_s, EXPECTED_PERIOD_S, rel_tol=0.0, abs_tol=1e-12):
    raise ValueError(f"Cell 7 requires period 0.4 s; Cell 2 recorded {period_s} s.")

gravity_in_s2 = float(
    cell2_summary.get("fragility_transformation", {}).get(
        "gravity_in_s2", STANDARD_GRAVITY_IN_S2
    )
)
append_check(
    validation_rows,
    "gravity_constant_matches_cell2",
    math.isclose(gravity_in_s2, STANDARD_GRAVITY_IN_S2, rel_tol=0.0, abs_tol=1e-12),
    f"gravity_in_s2={gravity_in_s2}",
)
conversion_factor = 4.0 * math.pi**2 / (gravity_in_s2 * period_s**2)
drift = build_long_parameter_table(
    source_drift,
    "nonstructural_drift_sensitive",
    conversion_factor,
)
acceleration = build_long_parameter_table(
    source_acceleration,
    "nonstructural_acceleration_sensitive",
    conversion_factor,
)
drift, drift_hashes = attach_source_hashes(PROJECT_ROOT, drift)
acceleration, acceleration_hashes = attach_source_hashes(PROJECT_ROOT, acceleration)

reconstructed_sd = (
    drift["direct_sa0p4_median_g"].to_numpy(dtype=float)
    * gravity_in_s2
    * period_s**2
    / (4.0 * math.pi**2)
)
max_drift_reconstruction_error = float(
    np.max(np.abs(reconstructed_sd - drift["source_median_value"].to_numpy(dtype=float)))
)
max_acceleration_identity_error = float(
    np.max(
        np.abs(
            acceleration["direct_sa0p4_median_g"].to_numpy(dtype=float)
            - acceleration["source_median_value"].to_numpy(dtype=float)
        )
    )
)
append_check(
    validation_rows,
    "drift_sd_to_sa_conversion_reconstructs_source",
    max_drift_reconstruction_error <= 1e-12,
    f"maximum_absolute_error_in={max_drift_reconstruction_error:.3e}",
)
append_check(
    validation_rows,
    "acceleration_identity_mapping_exact",
    max_acceleration_identity_error == 0.0,
    f"maximum_absolute_error_g={max_acceleration_identity_error:.3e}",
)
append_check(
    validation_rows,
    "drift_beta_unchanged",
    np.array_equal(
        drift["direct_sa0p4_beta_ln"].to_numpy(dtype=float),
        drift["source_beta_ln"].to_numpy(dtype=float),
    ),
    "production beta equals source beta exactly",
)
append_check(
    validation_rows,
    "acceleration_beta_unchanged",
    np.array_equal(
        acceleration["direct_sa0p4_beta_ln"].to_numpy(dtype=float),
        acceleration["source_beta_ln"].to_numpy(dtype=float),
    ),
    "production beta equals source beta exactly",
)
append_check(
    validation_rows,
    "direct_im_approximation_documented",
    True,
    (
        "Both nonstructural components use documented direct-SA0P4 approximations. "
        "Neither table is presented as the full HAZUS capacity-spectrum method."
    ),
    severity="informational",
)

assignments = pd.read_csv(DESIGN_ASSIGNMENTS_PATH)
structural_assignments = pd.read_csv(STRUCTURAL_ASSIGNMENTS_PATH)
required_assignment_columns = {
    "site_id",
    "structural_type",
    "occ_type",
    "year_built",
    "design_level",
}
missing_assignment_columns = sorted(required_assignment_columns.difference(assignments.columns))
append_check(
    validation_rows,
    "design_assignment_columns_present",
    not missing_assignment_columns,
    f"missing={missing_assignment_columns}",
)
if missing_assignment_columns:
    raise KeyError(f"Design assignments are missing columns: {missing_assignment_columns}")

assignments["site_id"] = assignments["site_id"].astype(str)
assignments["structural_type"] = assignments["structural_type"].astype(str).str.strip().str.upper()
assignments["design_level"] = assignments["design_level"].astype(str).str.strip()
assignments["occ_type"] = assignments["occ_type"].astype(str).str.strip().str.upper()

append_check(
    validation_rows,
    "design_assignment_rows_match_cell2",
    len(assignments) == int(cell2_summary["portfolio"]["buildings"]),
    f"rows={len(assignments)}; expected={cell2_summary['portfolio']['buildings']}",
)
append_check(
    validation_rows,
    "design_assignment_site_ids_unique",
    assignments["site_id"].is_unique,
    f"duplicates={int(assignments['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "design_assignment_all_w2",
    assignments["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(assignments['structural_type'].unique().tolist())}",
)
append_check(
    validation_rows,
    "design_assignment_levels_covered_by_fragilities",
    set(assignments["design_level"]).issubset(set(DESIGN_LEVELS)),
    f"levels={sorted(assignments['design_level'].unique().tolist())}",
)
append_check(
    validation_rows,
    "design_assignments_match_structural_cell2_sites",
    set(assignments["site_id"]) == set(structural_assignments["site_id"].astype(str)),
    (
        f"design_sites={assignments['site_id'].nunique()}; "
        f"structural_sites={structural_assignments['site_id'].astype(str).nunique()}"
    ),
)

drift_wide = to_wide(drift, "nsd", "sd_in")
acceleration_wide = to_wide(acceleration, "nsa", "sa_g")
building_assignments = (
    assignments.merge(drift_wide, on="design_level", how="left", validate="many_to_one")
    .merge(acceleration_wide, on="design_level", how="left", validate="many_to_one")
)
building_assignments["period_s"] = period_s
building_assignments["nsd_damage_im_column"] = PRODUCTION_IM_COLUMN
building_assignments["nsa_damage_im_column"] = PRODUCTION_IM_COLUMN
building_assignments["nsd_conversion_method_id"] = DRIFT_METHOD_ID
building_assignments["nsa_conversion_method_id"] = ACCELERATION_METHOD_ID
building_assignments["sd_to_sa_factor_g_per_in"] = conversion_factor
building_assignments["source_fragility_inventory_sha256"] = sha256_file(SOURCE_INVENTORY_PATH)

nsd_median_columns = [f"nsd_{state.lower()}_median_sa0p4_g" for state in DAMAGE_STATES]
nsd_beta_columns = [f"nsd_{state.lower()}_beta_ln" for state in DAMAGE_STATES]
nsa_median_columns = [f"nsa_{state.lower()}_median_sa0p4_g" for state in DAMAGE_STATES]
nsa_beta_columns = [f"nsa_{state.lower()}_beta_ln" for state in DAMAGE_STATES]
parameter_columns = nsd_median_columns + nsd_beta_columns + nsa_median_columns + nsa_beta_columns

append_check(
    validation_rows,
    "building_nonstructural_assignments_complete",
    building_assignments[parameter_columns].notna().all().all(),
    f"missing_parameter_cells={int(building_assignments[parameter_columns].isna().sum().sum())}",
)
append_check(
    validation_rows,
    "building_nonstructural_assignment_rows_match_portfolio",
    len(building_assignments) == len(assignments),
    f"rows={len(building_assignments)}; portfolio={len(assignments)}",
)
append_check(
    validation_rows,
    "building_nonstructural_site_ids_unique",
    building_assignments["site_id"].is_unique,
    f"duplicates={int(building_assignments['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "building_nsd_medians_strictly_ordered",
    np.all(np.diff(building_assignments[nsd_median_columns].to_numpy(dtype=float), axis=1) > 0),
    "Slight < Moderate < Extensive < Complete for every building",
)
append_check(
    validation_rows,
    "building_nsa_medians_strictly_ordered",
    np.all(np.diff(building_assignments[nsa_median_columns].to_numpy(dtype=float), axis=1) > 0),
    "Slight < Moderate < Extensive < Complete for every building",
)
append_check(
    validation_rows,
    "building_nonstructural_betas_positive",
    (building_assignments[nsd_beta_columns + nsa_beta_columns] > 0).all().all(),
    "all drift-sensitive and acceleration-sensitive beta values are positive",
)
append_check(
    validation_rows,
    "building_nonstructural_medians_positive",
    (building_assignments[nsd_median_columns + nsa_median_columns] > 0).all().all(),
    "all direct-SA0P4 medians are positive",
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()

paths = {
    "drift_fragility": PARAMETER_DIR
    / "notebook_5_direct_sa0p4_nonstructural_drift_fragility.csv",
    "acceleration_fragility": PARAMETER_DIR
    / "notebook_5_direct_sa0p4_nonstructural_acceleration_fragility.csv",
    "building_assignments": PARAMETER_DIR
    / "seaside_w2_building_nonstructural_fragility_assignments.csv",
    "validation": METADATA_DIR / "notebook_5_cell_7_validation.csv",
    "summary": METADATA_DIR / "notebook_5_cell_7_summary.json",
}

output_drop_columns = ["damage_state_rank"]
drift.drop(columns=[column for column in output_drop_columns if column in drift.columns]).to_csv(
    paths["drift_fragility"], index=False
)
acceleration.drop(
    columns=[column for column in output_drop_columns if column in acceleration.columns]
).to_csv(paths["acceleration_fragility"], index=False)
building_assignments.to_csv(paths["building_assignments"], index=False)
validation.to_csv(paths["validation"], index=False)

summary = {
    "pipeline_version": "notebook5_cell7_nonstructural_fragility_parameters_v1",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "portfolio": {
        "buildings": int(len(building_assignments)),
        "structural_type": EXPECTED_STRUCTURAL_TYPE,
        "design_levels": sorted(building_assignments["design_level"].unique().tolist()),
        "occupancy_types": sorted(building_assignments["occ_type"].unique().tolist()),
        "design_level_counts": {
            str(key): int(value)
            for key, value in building_assignments["design_level"].value_counts().sort_index().items()
        },
    },
    "production_demand": {
        "im_column": PRODUCTION_IM_COLUMN,
        "period_s": period_s,
        "unit": "g",
    },
    "drift_sensitive": {
        "source_demand": "spectral displacement",
        "source_unit": "in",
        "production_demand": "SA0P4",
        "conversion_method_id": DRIFT_METHOD_ID,
        "gravity_in_s2": gravity_in_s2,
        "conversion_factor_g_per_in": conversion_factor,
        "maximum_reconstruction_error_in": max_drift_reconstruction_error,
        "source_files": drift_hashes,
        "limitation": (
            "Equivalent fixed-period direct-SA0P4 approximation; not the full HAZUS "
            "capacity-spectrum procedure."
        ),
    },
    "acceleration_sensitive": {
        "source_demand": "spectral acceleration",
        "source_unit": "g",
        "production_demand": "SA0P4",
        "conversion_method_id": ACCELERATION_METHOD_ID,
        "maximum_identity_error_g": max_acceleration_identity_error,
        "source_files": acceleration_hashes,
        "limitation": (
            "Source spectral-acceleration medians are mapped directly to SA0P4 as a "
            "documented direct-IM approximation; not the full HAZUS capacity-spectrum procedure."
        ),
    },
    "source_inventory": {
        "path": str(SOURCE_INVENTORY_PATH),
        "sha256": sha256_file(SOURCE_INVENTORY_PATH),
    },
    "outputs": {key: str(path) for key, path in paths.items()},
    "next_cell": (
        "Cell 8: calculate controlled drift-sensitive and acceleration-sensitive "
        "nonstructural damage probabilities, use separate deterministic random streams, "
        "and validate categorical sampling before full production processing."
    ),
}
write_json(paths["summary"], summary)

print("=" * 78)
print("NOTEBOOK 5 CELL 7 NONSTRUCTURAL FRAGILITY PARAMETERS COMPLETE")
print("=" * 78)
print(f"Portfolio buildings:                 {len(building_assignments):,}")
print(f"Design levels represented:           {building_assignments['design_level'].nunique():,}")
for level, count in building_assignments["design_level"].value_counts().sort_index().items():
    print(f"  {level:<14}: {count:>4,} buildings")
print(f"Drift conversion factor:             {conversion_factor:.12f} g/in")
print(f"Maximum drift conversion error:      {max_drift_reconstruction_error:.3e} in")
print(f"Maximum acceleration identity error: {max_acceleration_identity_error:.3e} g")
print(f"Critical validation checks:          {int(validation['severity'].eq('critical').sum()):,}")
print(f"Critical failures:                   {len(critical_failures):,}")
print(f"Warnings requiring review:           {len(warning_failures):,}")
print()
print("Drift-sensitive direct-SA0P4 fragility:")
print(f"  {paths['drift_fragility']}")
print("Acceleration-sensitive direct-SA0P4 fragility:")
print(f"  {paths['acceleration_fragility']}")
print("Building assignments:")
print(f"  {paths['building_assignments']}")
print("Validation:")
print(f"  {paths['validation']}")
print("Summary:")
print(f"  {paths['summary']}")
print()
print("Next: controlled nonstructural damage-probability and sampling validation.")

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 7 failed one or more critical checks. Review: "
        f"{paths['validation']}"
    )


NOTEBOOK 5 CELL 7 NONSTRUCTURAL FRAGILITY PARAMETERS COMPLETE
Portfolio buildings:                 470
Design levels represented:           2
  LowCode       :  123 buildings
  PreCode       :  347 buildings
Drift conversion factor:             0.639076422090 g/in
Maximum drift conversion error:      1.776e-15 in
Maximum acceleration identity error: 0.000e+00 g
Critical validation checks:          44
Critical failures:                   0
Warnings requiring review:           0

Drift-sensitive direct-SA0P4 fragility:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_damage_loss_parameters\notebook_5_direct_sa0p4_nonstructural_drift_fragility.csv
Acceleration-sensitive direct-SA0P4 fragility:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_damage_loss_parameters\notebook_5_direct_sa0p4_nonstructural_acceleration_fragility.csv
Building assignments:
  c:\Users\USER\Documents\GitHub\seismic-correlatio

In [19]:
from __future__ import annotations

import hashlib
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.special import ndtr
from scipy.stats import kstest

DAMAGE_STATES = ["None", "Slight", "Moderate", "Extensive", "Complete"]
EXCEEDANCE_STATES = ["Slight", "Moderate", "Extensive", "Complete"]
COMPONENTS = {
    "nsd": {
        "label": "nonstructural_drift_sensitive",
        "namespace": "notebook5_nonstructural_drift_damage_v1",
        "sample_column": "sampled_nsd_damage_state",
        "sample_name_column": "sampled_nsd_damage_state_name",
        "uniform_column": "nsd_damage_uniform",
        "seed_column": "nsd_damage_seed_hex",
    },
    "nsa": {
        "label": "nonstructural_acceleration_sensitive",
        "namespace": "notebook5_nonstructural_acceleration_damage_v1",
        "sample_column": "sampled_nsa_damage_state",
        "sample_name_column": "sampled_nsa_damage_state_name",
        "uniform_column": "nsa_damage_uniform",
        "seed_column": "nsa_damage_seed_hex",
    },
}
EXPECTED_PERIOD_S = 0.40
EXPECTED_STRUCTURAL_TYPE = "W2"
PROBABILITY_TOLERANCE = 1e-12
GRID_POINTS = 401
MONTE_CARLO_REPLICATES = 20_000
MONTE_CARLO_MAX_ABSOLUTE_ERROR = 0.015
STREAM_DIAGNOSTIC_SAMPLES = 25_000
STREAM_CORRELATION_TOLERANCE = 0.02
STRUCTURAL_STREAM_NAMESPACE = "notebook5_structural_damage_v1"


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_7_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def deterministic_digest(key: str, namespace: str) -> bytes:
    return hashlib.sha256(f"{namespace}|{key}".encode("utf-8")).digest()


def deterministic_uniforms(
    occurrence_ids: pd.Series,
    site_ids: pd.Series,
    namespace: str,
) -> tuple[np.ndarray, list[str]]:
    occurrence_values = occurrence_ids.astype(str).to_numpy()
    site_values = site_ids.astype(str).to_numpy()
    uniforms = np.empty(len(occurrence_values), dtype=np.float64)
    seed_hex: list[str] = []
    denominator = float(2**64)

    for index, (occurrence_id, site_id) in enumerate(
        zip(occurrence_values, site_values, strict=True)
    ):
        digest = deterministic_digest(f"{occurrence_id}|{site_id}", namespace)
        integer = int.from_bytes(digest[:8], byteorder="big", signed=False)
        uniforms[index] = (integer + 0.5) / denominator
        seed_hex.append(digest[:16].hex())

    return uniforms, seed_hex


def parameter_columns(prefix: str) -> tuple[list[str], list[str]]:
    medians = [
        f"{prefix}_{state.lower()}_median_sa0p4_g" for state in EXCEEDANCE_STATES
    ]
    betas = [f"{prefix}_{state.lower()}_beta_ln" for state in EXCEEDANCE_STATES]
    return medians, betas


def calculate_probability_arrays(
    sa0p4_g: np.ndarray,
    medians: np.ndarray,
    betas: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, float]:
    sa = np.asarray(sa0p4_g, dtype=np.float64)
    median_values = np.asarray(medians, dtype=np.float64)
    beta_values = np.asarray(betas, dtype=np.float64)

    if not np.isfinite(sa).all() or np.any(sa <= 0.0):
        raise ValueError("All SA0P4 values must be finite and positive.")
    if not np.isfinite(median_values).all() or np.any(median_values <= 0.0):
        raise ValueError("All fragility medians must be finite and positive.")
    if not np.isfinite(beta_values).all() or np.any(beta_values <= 0.0):
        raise ValueError("All fragility beta values must be finite and positive.")

    z = (np.log(sa)[:, None] - np.log(median_values)) / beta_values
    exceedance = ndtr(z)
    raw_states = np.column_stack(
        [
            1.0 - exceedance[:, 0],
            exceedance[:, 0] - exceedance[:, 1],
            exceedance[:, 1] - exceedance[:, 2],
            exceedance[:, 2] - exceedance[:, 3],
            exceedance[:, 3],
        ]
    )

    minimum_raw_probability = float(raw_states.min())
    if minimum_raw_probability < -PROBABILITY_TOLERANCE:
        raise RuntimeError(
            "Fragility exceedance curves produced materially negative mutually "
            f"exclusive probabilities. Minimum={minimum_raw_probability:.6e}."
        )

    states = np.clip(raw_states, 0.0, 1.0)
    totals = states.sum(axis=1)
    if np.any(totals <= 0.0):
        raise RuntimeError("At least one damage-probability row has zero total mass.")
    states = states / totals[:, None]
    return exceedance, states, minimum_raw_probability


def component_probabilities(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    median_columns, beta_columns = parameter_columns(prefix)
    exceedance, states, minimum_raw = calculate_probability_arrays(
        pd.to_numeric(frame["sa0p4_simulated_g"], errors="raise").to_numpy(dtype=float),
        frame[median_columns].to_numpy(dtype=float),
        frame[beta_columns].to_numpy(dtype=float),
    )

    output = pd.DataFrame(index=frame.index)
    for index, state in enumerate(EXCEEDANCE_STATES):
        output[f"{prefix}_p_exceed_{state.lower()}"] = exceedance[:, index]
    for index, state in enumerate(DAMAGE_STATES):
        output[f"{prefix}_p_state_{state.lower()}"] = states[:, index]
    output[f"{prefix}_minimum_raw_state_probability"] = minimum_raw
    output[f"{prefix}_state_probability_sum"] = states.sum(axis=1)
    output[f"{prefix}_expected_damage_state"] = states @ np.arange(5, dtype=float)
    return output


def sample_damage_states(state_probabilities: np.ndarray, uniforms: np.ndarray) -> np.ndarray:
    probabilities = np.asarray(state_probabilities, dtype=np.float64)
    cumulative = np.cumsum(probabilities, axis=1)
    cumulative[:, -1] = 1.0
    states = (uniforms[:, None] > cumulative).sum(axis=1)
    return np.minimum(states, 4).astype(np.int8)


def calculate_control_outputs(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy().reset_index(drop=True)
    for prefix, specification in COMPONENTS.items():
        probabilities = component_probabilities(output, prefix).reset_index(drop=True)
        output = pd.concat([output, probabilities], axis=1)
        uniforms, seed_hex = deterministic_uniforms(
            output["occurrence_id"],
            output["site_id"],
            specification["namespace"],
        )
        state_columns = [f"{prefix}_p_state_{state.lower()}" for state in DAMAGE_STATES]
        sampled = sample_damage_states(
            output[state_columns].to_numpy(dtype=float), uniforms
        )
        output[specification["seed_column"]] = seed_hex
        output[specification["uniform_column"]] = uniforms
        output[specification["sample_column"]] = sampled
        output[specification["sample_name_column"]] = pd.Series(sampled).map(
            dict(enumerate(DAMAGE_STATES))
        )
    return output


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
CONTROL_STRUCTURAL_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_controlled_structural_damage"
CONTROL_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_controlled_nonstructural_damage"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

CELL3_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_3_summary.json"
CELL3_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_3_validation.csv"
CELL3_CONTROLLED_PATH = CONTROL_STRUCTURAL_DIR / "controlled_structural_damage_rows.csv"
CELL3_STREAM_PATH = METADATA_DIR / "notebook_5_cell_3_structural_damage_random_stream.json"
CELL7_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_7_summary.json"
CELL7_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_7_validation.csv"
BUILDING_ASSIGNMENTS_PATH = (
    PARAMETER_DIR / "seaside_w2_building_nonstructural_fragility_assignments.csv"
)

required_paths = [
    CELL3_SUMMARY_PATH,
    CELL3_VALIDATION_PATH,
    CELL3_CONTROLLED_PATH,
    CELL3_STREAM_PATH,
    CELL7_SUMMARY_PATH,
    CELL7_VALIDATION_PATH,
    BUILDING_ASSIGNMENTS_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required Cell 3 or Cell 7 outputs are missing: {missing_paths}")

validation_rows: list[dict[str, Any]] = []
cell3_summary = load_json(CELL3_SUMMARY_PATH)
cell7_summary = load_json(CELL7_SUMMARY_PATH)
cell3_stream = load_json(CELL3_STREAM_PATH)
cell3_validation = pd.read_csv(CELL3_VALIDATION_PATH)
cell7_validation = pd.read_csv(CELL7_VALIDATION_PATH)

for cell_number, summary, validation in [
    (3, cell3_summary, cell3_validation),
    (7, cell7_summary, cell7_validation),
]:
    append_check(
        validation_rows,
        f"cell{cell_number}_critical_checks_passed",
        summary.get("all_critical_checks_passed") is True,
        f"all_critical_checks_passed={summary.get('all_critical_checks_passed')}",
    )
    passed = parse_bool_series(validation["passed"])
    unresolved = validation.loc[
        validation["severity"].astype(str).str.lower().eq("critical") & ~passed
    ]
    append_check(
        validation_rows,
        f"cell{cell_number}_has_no_unresolved_critical_checks",
        unresolved.empty,
        f"unresolved_critical={len(unresolved)}",
    )

append_check(
    validation_rows,
    "structural_stream_namespace_matches_cell3",
    cell3_stream.get("namespace") == STRUCTURAL_STREAM_NAMESPACE,
    f"namespace={cell3_stream.get('namespace')}",
)
append_check(
    validation_rows,
    "three_damage_stream_namespaces_are_distinct",
    len(
        {
            STRUCTURAL_STREAM_NAMESPACE,
            COMPONENTS["nsd"]["namespace"],
            COMPONENTS["nsa"]["namespace"],
        }
    )
    == 3,
    "structural, drift-sensitive, and acceleration-sensitive streams are separate",
)

controls_source = pd.read_csv(CELL3_CONTROLLED_PATH)
building_assignments = pd.read_csv(BUILDING_ASSIGNMENTS_PATH)

required_control_columns = {
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    "sa0p4_simulated_g",
    "design_level",
    "intensity_level",
    "selection_quantile",
    "selection_target_sa0p4_g",
}
missing_control_columns = sorted(required_control_columns.difference(controls_source.columns))
append_check(
    validation_rows,
    "cell3_control_columns_present",
    not missing_control_columns,
    f"missing={missing_control_columns}",
)
if missing_control_columns:
    raise KeyError(f"Cell 3 controlled rows are missing columns: {missing_control_columns}")

all_parameter_columns: list[str] = []
for prefix in COMPONENTS:
    medians, betas = parameter_columns(prefix)
    all_parameter_columns.extend(medians + betas)
required_assignment_columns = {
    "site_id",
    "structural_type",
    "occ_type",
    "design_level",
    "period_s",
    *all_parameter_columns,
}
missing_assignment_columns = sorted(
    required_assignment_columns.difference(building_assignments.columns)
)
append_check(
    validation_rows,
    "building_nonstructural_assignment_columns_present",
    not missing_assignment_columns,
    f"missing={missing_assignment_columns}",
)
if missing_assignment_columns:
    raise KeyError(
        "Cell 7 building assignments are missing columns: "
        f"{missing_assignment_columns}"
    )

controls_source["site_id"] = controls_source["site_id"].astype(str)
controls_source["occurrence_id"] = controls_source["occurrence_id"].astype(str)
controls_source["source_type"] = controls_source["source_type"].astype(str).str.strip().str.upper()
controls_source["design_level"] = controls_source["design_level"].astype(str).str.strip()
controls_source["sa0p4_simulated_g"] = pd.to_numeric(
    controls_source["sa0p4_simulated_g"], errors="raise"
)

building_assignments["site_id"] = building_assignments["site_id"].astype(str)
building_assignments["structural_type"] = (
    building_assignments["structural_type"].astype(str).str.strip().str.upper()
)
building_assignments["design_level"] = building_assignments["design_level"].astype(str).str.strip()
building_assignments["occ_type"] = building_assignments["occ_type"].astype(str).str.strip().str.upper()
building_assignments["period_s"] = pd.to_numeric(
    building_assignments["period_s"], errors="raise"
)

expected_buildings = int(cell7_summary["portfolio"]["buildings"])
append_check(
    validation_rows,
    "building_assignment_rows_match_cell7",
    len(building_assignments) == expected_buildings,
    f"rows={len(building_assignments)}; expected={expected_buildings}",
)
append_check(
    validation_rows,
    "building_assignment_site_ids_unique",
    building_assignments["site_id"].is_unique,
    f"duplicates={int(building_assignments['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "all_building_assignments_are_w2",
    building_assignments["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(building_assignments['structural_type'].unique().tolist())}",
)
append_check(
    validation_rows,
    "all_building_periods_are_0p4_seconds",
    np.allclose(
        building_assignments["period_s"].to_numpy(dtype=float),
        EXPECTED_PERIOD_S,
        rtol=0.0,
        atol=1e-12,
    ),
    f"periods={sorted(building_assignments['period_s'].unique().tolist())}",
)
append_check(
    validation_rows,
    "nonstructural_parameters_complete_positive",
    building_assignments[all_parameter_columns].notna().all().all()
    and (building_assignments[all_parameter_columns] > 0.0).all().all(),
    (
        f"missing={int(building_assignments[all_parameter_columns].isna().sum().sum())}; "
        f"nonpositive={int((building_assignments[all_parameter_columns].fillna(0) <= 0).sum().sum())}"
    ),
)

lookup_columns = [
    "site_id",
    "structural_type",
    "occ_type",
    "year_built",
    "design_level",
    "period_s",
    *all_parameter_columns,
]
lookup_columns = [column for column in lookup_columns if column in building_assignments.columns]
controls_base_columns = [
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    "sa0p4_simulated_g",
    "intensity_level",
    "selection_quantile",
    "selection_target_sa0p4_g",
]
controls = controls_source[controls_base_columns].merge(
    building_assignments[lookup_columns],
    on="site_id",
    how="left",
    validate="many_to_one",
)

append_check(
    validation_rows,
    "controlled_rows_match_cell3_summary",
    len(controls) == int(cell3_summary["controlled_selection"]["rows"]),
    f"rows={len(controls)}; expected={cell3_summary['controlled_selection']['rows']}",
)
append_check(
    validation_rows,
    "controlled_occurrence_site_keys_unique",
    not controls.duplicated(["occurrence_id", "site_id"]).any(),
    f"duplicates={int(controls.duplicated(['occurrence_id', 'site_id']).sum())}",
)
append_check(
    validation_rows,
    "controlled_sites_all_mapped",
    controls["design_level"].notna().all(),
    f"missing={int(controls['design_level'].isna().sum())}",
)
append_check(
    validation_rows,
    "controlled_sa0p4_positive_finite",
    np.isfinite(controls["sa0p4_simulated_g"]).all()
    and controls["sa0p4_simulated_g"].gt(0.0).all(),
    (
        f"minimum={controls['sa0p4_simulated_g'].min():.6g}; "
        f"maximum={controls['sa0p4_simulated_g'].max():.6g}"
    ),
)
append_check(
    validation_rows,
    "controlled_design_levels_match_cell7",
    set(controls["design_level"])
    == set(cell7_summary["portfolio"]["design_levels"]),
    f"levels={sorted(controls['design_level'].unique().tolist())}",
)
append_check(
    validation_rows,
    "controlled_source_types_are_interface_and_slab",
    set(controls["source_type"]) == {"INTERFACE", "SLAB"},
    f"source_types={sorted(controls['source_type'].unique().tolist())}",
)
append_check(
    validation_rows,
    "controlled_intensity_levels_complete",
    set(controls["intensity_level"]) == {"low", "medium", "high"},
    f"levels={sorted(controls['intensity_level'].unique().tolist())}",
)

controlled_output = calculate_control_outputs(controls)

for prefix, specification in COMPONENTS.items():
    exceedance_columns = [
        f"{prefix}_p_exceed_{state.lower()}" for state in EXCEEDANCE_STATES
    ]
    state_columns = [f"{prefix}_p_state_{state.lower()}" for state in DAMAGE_STATES]
    exceedance = controlled_output[exceedance_columns].to_numpy(dtype=float)
    states = controlled_output[state_columns].to_numpy(dtype=float)

    append_check(
        validation_rows,
        f"{prefix}_controlled_exceedance_probabilities_ordered",
        np.all(np.diff(exceedance, axis=1) <= PROBABILITY_TOLERANCE),
        f"maximum_positive_difference={float(np.diff(exceedance, axis=1).max()):.3e}",
    )
    append_check(
        validation_rows,
        f"{prefix}_controlled_state_probabilities_nonnegative",
        states.min() >= -PROBABILITY_TOLERANCE,
        f"minimum={float(states.min()):.3e}",
    )
    append_check(
        validation_rows,
        f"{prefix}_controlled_state_probabilities_sum_to_one",
        np.allclose(states.sum(axis=1), 1.0, rtol=0.0, atol=1e-14),
        f"maximum_error={float(np.abs(states.sum(axis=1) - 1.0).max()):.3e}",
    )
    append_check(
        validation_rows,
        f"{prefix}_controlled_uniforms_inside_unit_interval",
        controlled_output[specification["uniform_column"]]
        .between(0.0, 1.0, inclusive="neither")
        .all(),
        (
            f"min={controlled_output[specification['uniform_column']].min():.12f}; "
            f"max={controlled_output[specification['uniform_column']].max():.12f}"
        ),
    )
    append_check(
        validation_rows,
        f"{prefix}_controlled_sampled_states_valid",
        controlled_output[specification["sample_column"]].between(0, 4).all(),
        (
            f"states={sorted(controlled_output[specification['sample_column']].unique().tolist())}"
        ),
    )

repeated_output = calculate_control_outputs(controls.copy())
repeat_exact = True
for specification in COMPONENTS.values():
    repeat_exact = repeat_exact and np.array_equal(
        controlled_output[specification["uniform_column"]].to_numpy(),
        repeated_output[specification["uniform_column"]].to_numpy(),
    )
    repeat_exact = repeat_exact and np.array_equal(
        controlled_output[specification["sample_column"]].to_numpy(),
        repeated_output[specification["sample_column"]].to_numpy(),
    )
append_check(
    validation_rows,
    "nonstructural_deterministic_repeat_is_exact",
    repeat_exact,
    "both component uniforms and sampled states reproduce exactly",
)

comparison_columns = ["occurrence_id", "site_id"]
for specification in COMPONENTS.values():
    comparison_columns.extend(
        [specification["uniform_column"], specification["sample_column"]]
    )

shuffled = controls.sample(frac=1.0, random_state=20260802).reset_index(drop=True)
shuffled_output = calculate_control_outputs(shuffled)
ordered_reference = controlled_output[comparison_columns].sort_values(
    ["occurrence_id", "site_id"]
).reset_index(drop=True)
ordered_shuffled = shuffled_output[comparison_columns].sort_values(
    ["occurrence_id", "site_id"]
).reset_index(drop=True)
append_check(
    validation_rows,
    "nonstructural_streams_are_row_order_invariant",
    ordered_reference.equals(ordered_shuffled),
    "shuffled rows reproduce identical component uniforms and states",
)

chunked_frames: list[pd.DataFrame] = []
for start in range(0, len(controls), 5):
    chunked_frames.append(calculate_control_outputs(controls.iloc[start : start + 5].copy()))
chunked_output = pd.concat(chunked_frames, ignore_index=True)
ordered_chunked = chunked_output[comparison_columns].sort_values(
    ["occurrence_id", "site_id"]
).reset_index(drop=True)
append_check(
    validation_rows,
    "nonstructural_streams_are_chunk_size_invariant",
    ordered_reference.equals(ordered_chunked),
    "five-row chunks reproduce identical component uniforms and states",
)

parameter_rows: list[pd.DataFrame] = []
represented_design_levels = sorted(controls["design_level"].unique().tolist())
for design_level in represented_design_levels:
    design_group = building_assignments.loc[
        building_assignments["design_level"].eq(design_level)
    ]
    row: dict[str, Any] = {"design_level": design_level}
    for column in all_parameter_columns:
        unique_values = np.unique(design_group[column].to_numpy(dtype=float))
        append_check(
            validation_rows,
            f"{design_level}_{column}_unique_within_design_level",
            len(unique_values) == 1,
            f"unique_values={unique_values.tolist()}",
        )
        row[column] = float(unique_values[0])
    parameter_rows.append(pd.DataFrame([row]))
parameters_by_design = pd.concat(parameter_rows, ignore_index=True)

minimum_control_sa = float(controls["sa0p4_simulated_g"].min())
maximum_control_sa = float(controls["sa0p4_simulated_g"].max())
minimum_median = float(parameters_by_design[all_parameter_columns].filter(like="_median_").min().min())
maximum_median = float(parameters_by_design[all_parameter_columns].filter(like="_median_").max().max())
grid_minimum = max(1e-6, min(minimum_control_sa, minimum_median / 20.0))
grid_maximum = max(maximum_control_sa, maximum_median * 4.0)
sa_grid = np.geomspace(grid_minimum, grid_maximum, GRID_POINTS)

grid_rows: list[pd.DataFrame] = []
for prefix, specification in COMPONENTS.items():
    median_columns, beta_columns = parameter_columns(prefix)
    for _, parameter_row in parameters_by_design.iterrows():
        frame = pd.DataFrame(
            {
                "component": specification["label"],
                "component_prefix": prefix,
                "design_level": str(parameter_row["design_level"]),
                "sa0p4_simulated_g": sa_grid,
            }
        )
        for column in median_columns + beta_columns:
            frame[column] = float(parameter_row[column])
        probabilities = component_probabilities(frame, prefix)
        rename_map = {
            f"{prefix}_p_exceed_{state.lower()}": f"p_exceed_{state.lower()}"
            for state in EXCEEDANCE_STATES
        }
        rename_map.update(
            {
                f"{prefix}_p_state_{state.lower()}": f"p_state_{state.lower()}"
                for state in DAMAGE_STATES
            }
        )
        rename_map.update(
            {
                f"{prefix}_minimum_raw_state_probability": "minimum_raw_state_probability",
                f"{prefix}_state_probability_sum": "state_probability_sum",
                f"{prefix}_expected_damage_state": "expected_damage_state",
            }
        )
        grid_rows.append(
            pd.concat([frame, probabilities.rename(columns=rename_map)], axis=1)
        )

probability_grid = pd.concat(grid_rows, ignore_index=True)
monotonic_diagnostics: dict[str, bool] = {}
for (component, design_level), group in probability_grid.groupby(
    ["component", "design_level"], sort=False
):
    ordered = group.sort_values("sa0p4_simulated_g")
    key = f"{component}|{design_level}"
    monotonic_diagnostics[key] = bool(
        np.all(np.diff(ordered["expected_damage_state"].to_numpy(dtype=float)) >= -1e-12)
    )
append_check(
    validation_rows,
    "nonstructural_expected_damage_monotonic_with_sa0p4",
    all(monotonic_diagnostics.values()),
    f"by_component_design={monotonic_diagnostics}",
)

stream_occurrence_ids = pd.Series(
    [f"STREAM_DIAGNOSTIC_{index:06d}" for index in range(STREAM_DIAGNOSTIC_SAMPLES)]
)
stream_site_ids = pd.Series(
    [f"SITE_{index % max(expected_buildings, 1):04d}" for index in range(STREAM_DIAGNOSTIC_SAMPLES)]
)
stream_uniforms: dict[str, np.ndarray] = {}
stream_diagnostic_rows: list[dict[str, Any]] = []
all_streams = {
    "structural": STRUCTURAL_STREAM_NAMESPACE,
    "nsd": COMPONENTS["nsd"]["namespace"],
    "nsa": COMPONENTS["nsa"]["namespace"],
}
for stream_name, namespace in all_streams.items():
    uniforms, _ = deterministic_uniforms(stream_occurrence_ids, stream_site_ids, namespace)
    stream_uniforms[stream_name] = uniforms
    ks_result = kstest(uniforms, "uniform")
    mean = float(uniforms.mean())
    variance = float(uniforms.var())
    stream_diagnostic_rows.append(
        {
            "stream": stream_name,
            "namespace": namespace,
            "samples": STREAM_DIAGNOSTIC_SAMPLES,
            "mean": mean,
            "variance": variance,
            "ks_statistic": float(ks_result.statistic),
            "ks_pvalue": float(ks_result.pvalue),
        }
    )
    append_check(
        validation_rows,
        f"{stream_name}_stream_mean_reasonable",
        abs(mean - 0.5) <= 0.015,
        f"mean={mean:.6f}",
    )
    append_check(
        validation_rows,
        f"{stream_name}_stream_variance_reasonable",
        abs(variance - 1.0 / 12.0) <= 0.006,
        f"variance={variance:.6f}; target={1.0/12.0:.6f}",
    )
    append_check(
        validation_rows,
        f"{stream_name}_stream_ks_diagnostic",
        float(ks_result.pvalue) >= 1e-4,
        f"statistic={float(ks_result.statistic):.6f}; pvalue={float(ks_result.pvalue):.6f}",
        severity="warning",
    )

correlation_rows: list[dict[str, Any]] = []
for left, right in [("structural", "nsd"), ("structural", "nsa"), ("nsd", "nsa")]:
    correlation = float(np.corrcoef(stream_uniforms[left], stream_uniforms[right])[0, 1])
    identical = int(np.equal(stream_uniforms[left], stream_uniforms[right]).sum())
    correlation_rows.append(
        {
            "stream_a": left,
            "stream_b": right,
            "correlation": correlation,
            "identical_uniform_count": identical,
        }
    )
    append_check(
        validation_rows,
        f"{left}_{right}_streams_are_distinct",
        identical == 0,
        f"identical_uniform_count={identical}",
    )
    append_check(
        validation_rows,
        f"{left}_{right}_stream_correlation_small",
        abs(correlation) <= STREAM_CORRELATION_TOLERANCE,
        (
            f"correlation={correlation:.6f}; "
            f"tolerance={STREAM_CORRELATION_TOLERANCE:.6f}"
        ),
    )

monte_carlo_rows: list[dict[str, Any]] = []
for prefix, specification in COMPONENTS.items():
    median_columns, beta_columns = parameter_columns(prefix)
    for _, parameter_row in parameters_by_design.iterrows():
        design_level = str(parameter_row["design_level"])
        test_im_values = {
            "below_slight": 0.50 * float(parameter_row[median_columns[0]]),
            "at_moderate": float(parameter_row[median_columns[1]]),
            "at_complete": float(parameter_row[median_columns[3]]),
        }

        for test_level, test_im in test_im_values.items():
            medians = np.repeat(
                parameter_row[median_columns].to_numpy(dtype=float)[None, :],
                1,
                axis=0,
            )
            betas = np.repeat(
                parameter_row[beta_columns].to_numpy(dtype=float)[None, :],
                1,
                axis=0,
            )
            _, analytical_states, _ = calculate_probability_arrays(
                np.array([test_im], dtype=float), medians, betas
            )
            analytical_probabilities = analytical_states[0]

            occurrence_ids = pd.Series(
                [
                    f"MC|{prefix}|{design_level}|{test_level}|{index:05d}"
                    for index in range(MONTE_CARLO_REPLICATES)
                ]
            )
            site_ids = pd.Series(["SYNTHETIC_W2"] * MONTE_CARLO_REPLICATES)
            uniforms, _ = deterministic_uniforms(
                occurrence_ids, site_ids, specification["namespace"]
            )
            replicated_probabilities = np.repeat(
                analytical_probabilities[None, :],
                MONTE_CARLO_REPLICATES,
                axis=0,
            )
            sampled_states = sample_damage_states(replicated_probabilities, uniforms)
            empirical_probabilities = np.bincount(sampled_states, minlength=5) / float(
                MONTE_CARLO_REPLICATES
            )
            maximum_probability_error = float(
                np.abs(empirical_probabilities - analytical_probabilities).max()
            )
            analytical_expected_state = float(
                analytical_probabilities @ np.arange(5, dtype=float)
            )
            empirical_expected_state = float(sampled_states.mean())

            row: dict[str, Any] = {
                "component": specification["label"],
                "component_prefix": prefix,
                "design_level": design_level,
                "test_level": test_level,
                "sa0p4_g": test_im,
                "replicates": MONTE_CARLO_REPLICATES,
                "maximum_state_probability_error": maximum_probability_error,
                "analytical_expected_damage_state": analytical_expected_state,
                "empirical_expected_damage_state": empirical_expected_state,
                "expected_state_error": empirical_expected_state
                - analytical_expected_state,
            }
            for index, state in enumerate(DAMAGE_STATES):
                row[f"analytical_p_{state.lower()}"] = float(
                    analytical_probabilities[index]
                )
                row[f"empirical_p_{state.lower()}"] = float(
                    empirical_probabilities[index]
                )
            monte_carlo_rows.append(row)

monte_carlo_validation = pd.DataFrame(monte_carlo_rows)
maximum_monte_carlo_error = float(
    monte_carlo_validation["maximum_state_probability_error"].max()
)
append_check(
    validation_rows,
    "nonstructural_categorical_sampling_matches_analytical_probabilities",
    maximum_monte_carlo_error <= MONTE_CARLO_MAX_ABSOLUTE_ERROR,
    (
        f"maximum_absolute_error={maximum_monte_carlo_error:.6f}; "
        f"tolerance={MONTE_CARLO_MAX_ABSOLUTE_ERROR:.6f}"
    ),
)

stream_diagnostics = pd.DataFrame(stream_diagnostic_rows)
stream_correlations = pd.DataFrame(correlation_rows)
validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()

paths = {
    "controlled_damage": CONTROL_DIR / "controlled_nonstructural_damage_rows.csv",
    "probability_grid": CONTROL_DIR
    / "controlled_nonstructural_damage_probability_grid.csv.gz",
    "monte_carlo": CONTROL_DIR
    / "controlled_nonstructural_damage_monte_carlo_validation.csv",
    "stream_diagnostics": CONTROL_DIR
    / "controlled_nonstructural_damage_stream_diagnostics.csv",
    "stream_correlations": CONTROL_DIR
    / "controlled_nonstructural_damage_stream_correlations.csv",
    "random_stream_specification": METADATA_DIR
    / "notebook_5_cell_8_nonstructural_damage_random_streams.json",
    "validation": METADATA_DIR / "notebook_5_cell_8_validation.csv",
    "summary": METADATA_DIR / "notebook_5_cell_8_summary.json",
}

controlled_output.to_csv(paths["controlled_damage"], index=False)
probability_grid.to_csv(paths["probability_grid"], index=False, compression="gzip")
monte_carlo_validation.to_csv(paths["monte_carlo"], index=False)
stream_diagnostics.to_csv(paths["stream_diagnostics"], index=False)
stream_correlations.to_csv(paths["stream_correlations"], index=False)
validation.to_csv(paths["validation"], index=False)

random_stream_specification = {
    "hash_function": "SHA-256",
    "key_fields": ["occurrence_id", "site_id"],
    "key_format": "namespace|occurrence_id|site_id",
    "uniform_construction": "(unsigned_big_endian_first_8_bytes + 0.5) / 2^64",
    "uniform_interval": "strictly between 0 and 1",
    "row_order_invariant": True,
    "chunk_size_invariant": True,
    "conditional_dependence_assumption": (
        "Structural, drift-sensitive nonstructural, and acceleration-sensitive "
        "nonstructural damage-state uniforms are modeled as separate deterministic "
        "streams conditional on the simulated ground-motion intensity."
    ),
    "streams": {
        "structural_damage": {
            "namespace": STRUCTURAL_STREAM_NAMESPACE,
            "defined_by": "Notebook 5 Cell 3",
        },
        "nonstructural_drift_sensitive_damage": {
            "namespace": COMPONENTS["nsd"]["namespace"],
            "defined_by": "Notebook 5 Cell 8",
        },
        "nonstructural_acceleration_sensitive_damage": {
            "namespace": COMPONENTS["nsa"]["namespace"],
            "defined_by": "Notebook 5 Cell 8",
        },
    },
    "reuse_rule": (
        "Reuse the same occurrence-site component uniforms when comparing the "
        "baseline independent ground-motion case with future spatially correlated cases."
    ),
}
write_json(paths["random_stream_specification"], random_stream_specification)

summary = {
    "pipeline_version": "notebook5_cell8_controlled_nonstructural_damage_v1",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "controlled_rows": {
        "source": str(CELL3_CONTROLLED_PATH),
        "source_sha256": sha256_file(CELL3_CONTROLLED_PATH),
        "rows": int(len(controlled_output)),
        "design_levels": represented_design_levels,
        "source_types": sorted(controlled_output["source_type"].unique().tolist()),
        "intensity_levels": sorted(controlled_output["intensity_level"].unique().tolist()),
    },
    "building_assignments": {
        "path": str(BUILDING_ASSIGNMENTS_PATH),
        "sha256": sha256_file(BUILDING_ASSIGNMENTS_PATH),
        "buildings": int(len(building_assignments)),
    },
    "damage_model": {
        "equation": "P(DS>=k|Sa)=Phi((ln(Sa)-ln(theta_k))/beta_k)",
        "mutually_exclusive_states": DAMAGE_STATES,
        "production_im": "sa0p4_simulated_g",
        "period_s": EXPECTED_PERIOD_S,
        "components": [specification["label"] for specification in COMPONENTS.values()],
        "conditional_component_dependence": (
            "Separate deterministic damage streams conditional on SA0P4."
        ),
    },
    "probability_grid": {
        "rows": int(len(probability_grid)),
        "points_per_component_design_level": GRID_POINTS,
        "minimum_sa0p4_g": grid_minimum,
        "maximum_sa0p4_g": grid_maximum,
    },
    "monte_carlo_validation": {
        "cases": int(len(monte_carlo_validation)),
        "replicates_per_case": MONTE_CARLO_REPLICATES,
        "maximum_state_probability_error": maximum_monte_carlo_error,
        "tolerance": MONTE_CARLO_MAX_ABSOLUTE_ERROR,
    },
    "stream_diagnostics": {
        "samples": STREAM_DIAGNOSTIC_SAMPLES,
        "maximum_absolute_pairwise_correlation": float(
            stream_correlations["correlation"].abs().max()
        ),
        "correlation_tolerance": STREAM_CORRELATION_TOLERANCE,
    },
    "outputs": {key: str(path) for key, path in paths.items()},
    "next_cell": (
        "Cell 9: apply the validated drift-sensitive and acceleration-sensitive "
        "damage equations and deterministic component streams to all 4,996,100 "
        "occurrence-building rows using restartable production chunks."
    ),
}
write_json(paths["summary"], summary)

print("=" * 78)
print("NOTEBOOK 5 CELL 8 CONTROLLED NONSTRUCTURAL DAMAGE VALIDATION COMPLETE")
print("=" * 78)
print(f"Controlled occurrence-site rows:     {len(controlled_output):,}")
print(f"Components validated:                {len(COMPONENTS):,}")
print(f"Design levels represented:           {len(represented_design_levels):,}")
print(f"Source types represented:            {controlled_output['source_type'].nunique():,}")
print(f"Probability-grid rows:               {len(probability_grid):,}")
print(f"Monte Carlo validation cases:        {len(monte_carlo_validation):,}")
print(f"Maximum Monte Carlo error:           {maximum_monte_carlo_error:.6f}")
print(
    "Maximum cross-stream correlation: "
    f"{stream_correlations['correlation'].abs().max():.6f}"
)
print(f"Critical validation checks:          {int(validation['severity'].eq('critical').sum()):,}")
print(f"Critical failures:                   {len(critical_failures):,}")
print(f"Warnings requiring review:           {len(warning_failures):,}")
print()
print("Controlled nonstructural damage rows:")
print(f"  {paths['controlled_damage']}")
print("Validation:")
print(f"  {paths['validation']}")
print("Summary:")
print(f"  {paths['summary']}")
print()
print("Next: full production nonstructural damage-state sampling.")

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 8 failed one or more critical validation checks. "
        "Inspect notebook_5_cell_8_validation.csv before continuing."
    )


NOTEBOOK 5 CELL 8 CONTROLLED NONSTRUCTURAL DAMAGE VALIDATION COMPLETE
Controlled occurrence-site rows:     12
Components validated:                2
Design levels represented:           2
Source types represented:            2
Probability-grid rows:               1,604
Monte Carlo validation cases:        12
Maximum Monte Carlo error:           0.008116
Maximum cross-stream correlation: 0.007044
Critical validation checks:          79
Critical failures:                   0
Warnings requiring review:           0

Controlled nonstructural damage rows:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_controlled_nonstructural_damage\controlled_nonstructural_damage_rows.csv
Validation:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_5_damage_loss\notebook_5_cell_8_validation.csv
Summary:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_5_damage_loss\notebook_5_ce

In [20]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
import shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.special import ndtr

DAMAGE_STATES = ["None", "Slight", "Moderate", "Extensive", "Complete"]
EXCEEDANCE_STATES = ["Slight", "Moderate", "Extensive", "Complete"]
COMPONENTS = {
    "nsd": {
        "label": "nonstructural_drift_sensitive",
        "namespace": "notebook5_nonstructural_drift_damage_v1",
        "uniform_column": "nsd_damage_uniform",
        "sample_column": "sampled_nsd_damage_state",
        "sample_name_column": "sampled_nsd_damage_state_name",
        "expected_column": "nsd_expected_damage_state",
    },
    "nsa": {
        "label": "nonstructural_acceleration_sensitive",
        "namespace": "notebook5_nonstructural_acceleration_damage_v1",
        "uniform_column": "nsa_damage_uniform",
        "sample_column": "sampled_nsa_damage_state",
        "sample_name_column": "sampled_nsa_damage_state_name",
        "expected_column": "nsa_expected_damage_state",
    },
}
EXPECTED_STRUCTURAL_TYPE = "W2"
EXPECTED_PERIOD_S = 0.40
EXPECTED_SITES = 470
OCCURRENCES_PER_CHUNK = 250
PIPELINE_VERSION = "notebook5_cell9_full_nonstructural_damage_v1"
PROBABILITY_TOLERANCE = 1e-12
CONTROL_TOLERANCE = 2e-12
BASE_SAMPLED_EXPECTED_PROPORTION_TOLERANCE = 0.002
BASE_UNIFORM_MEAN_TOLERANCE = 0.001
BASE_UNIFORM_VARIANCE_TOLERANCE = 0.0005
BASE_PRODUCTION_STREAM_CORRELATION_TOLERANCE = 0.002


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_8_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = recorded_path.replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def normalize_source_type(values: pd.Series) -> pd.Series:
    return values.astype(str).str.strip().str.upper()


def deterministic_uniforms(
    occurrence_ids: pd.Series,
    site_ids: pd.Series,
    namespace: str,
) -> np.ndarray:
    occurrence_values = occurrence_ids.astype(str).to_numpy()
    site_values = site_ids.astype(str).to_numpy()
    uniforms = np.empty(len(occurrence_values), dtype=np.float64)
    denominator = float(2**64)

    for index, (occurrence_id, site_id) in enumerate(
        zip(occurrence_values, site_values, strict=True)
    ):
        digest = hashlib.sha256(
            f"{namespace}|{occurrence_id}|{site_id}".encode("utf-8")
        ).digest()
        integer = int.from_bytes(digest[:8], byteorder="big", signed=False)
        uniforms[index] = (integer + 0.5) / denominator

    return uniforms


def parameter_columns(prefix: str) -> tuple[list[str], list[str]]:
    medians = [
        f"{prefix}_{state.lower()}_median_sa0p4_g" for state in EXCEEDANCE_STATES
    ]
    betas = [f"{prefix}_{state.lower()}_beta_ln" for state in EXCEEDANCE_STATES]
    return medians, betas


def probability_columns(prefix: str) -> list[str]:
    return [f"{prefix}_p_state_{state.lower()}" for state in DAMAGE_STATES]


def calculate_component_probabilities(
    frame: pd.DataFrame,
    prefix: str,
) -> tuple[pd.DataFrame, float]:
    median_columns, beta_columns = parameter_columns(prefix)
    sa = pd.to_numeric(frame["sa0p4_simulated_g"], errors="raise").to_numpy(
        dtype=np.float64
    )
    medians = frame[median_columns].to_numpy(dtype=np.float64)
    betas = frame[beta_columns].to_numpy(dtype=np.float64)

    if not np.isfinite(sa).all() or np.any(sa <= 0.0):
        raise ValueError("All SA0P4 values must be finite and positive.")
    if not np.isfinite(medians).all() or np.any(medians <= 0.0):
        raise ValueError(f"All {prefix} fragility medians must be finite and positive.")
    if not np.isfinite(betas).all() or np.any(betas <= 0.0):
        raise ValueError(f"All {prefix} fragility betas must be finite and positive.")

    exceedance = ndtr((np.log(sa)[:, None] - np.log(medians)) / betas)
    raw = np.column_stack(
        [
            1.0 - exceedance[:, 0],
            exceedance[:, 0] - exceedance[:, 1],
            exceedance[:, 1] - exceedance[:, 2],
            exceedance[:, 2] - exceedance[:, 3],
            exceedance[:, 3],
        ]
    )
    minimum_raw = float(raw.min())
    if minimum_raw < -PROBABILITY_TOLERANCE:
        raise RuntimeError(
            f"{prefix} fragility curves produced materially negative mutually "
            f"exclusive probabilities. Minimum={minimum_raw:.6e}."
        )

    probabilities = np.clip(raw, 0.0, 1.0)
    totals = probabilities.sum(axis=1)
    if np.any(totals <= 0.0):
        raise RuntimeError(f"At least one {prefix} probability row has zero total mass.")
    probabilities /= totals[:, None]

    output = pd.DataFrame(index=frame.index)
    for index, state in enumerate(DAMAGE_STATES):
        output[f"{prefix}_p_state_{state.lower()}"] = probabilities[:, index]
    output[f"{prefix}_expected_damage_state"] = (
        probabilities @ np.arange(5, dtype=np.float64)
    )
    return output, minimum_raw


def sample_damage_states(probabilities: pd.DataFrame, uniforms: np.ndarray, prefix: str) -> np.ndarray:
    columns = probability_columns(prefix)
    cumulative = np.cumsum(
        probabilities[columns].to_numpy(dtype=np.float64), axis=1
    )
    cumulative[:, -1] = 1.0
    states = (uniforms[:, None] > cumulative).sum(axis=1)
    return np.minimum(states, 4).astype(np.int8)


def stable_frame_hash(frame: pd.DataFrame, columns: list[str]) -> str:
    values = pd.util.hash_pandas_object(frame[columns], index=False).to_numpy(
        dtype=np.uint64
    )
    return hashlib.sha256(values.tobytes()).hexdigest()


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            filename="", mode="wb", fileobj=raw, compresslevel=6, mtime=0
        ) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8", newline="") as text:
                frame.to_csv(
                    text,
                    index=False,
                    float_format="%.17g",
                    lineterminator="\n",
                )
    temporary.replace(path)


def concatenate_gzip_csv_files(chunk_paths: list[Path], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    expected_header: bytes | None = None

    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            filename="", mode="wb", fileobj=raw, compresslevel=6, mtime=0
        ) as destination:
            for chunk_path in chunk_paths:
                with gzip.open(chunk_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        destination.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            f"Chunk header mismatch while concatenating {chunk_path}."
                        )
                    shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)

    temporary.replace(output_path)


def summarize_occurrences(frame: pd.DataFrame) -> pd.DataFrame:
    group_columns = [
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
    ]

    base_aggregations: dict[str, tuple[str, str]] = {
        "buildings": ("site_id", "size"),
        "minimum_sa0p4_g": ("sa0p4_simulated_g", "min"),
        "mean_sa0p4_g": ("sa0p4_simulated_g", "mean"),
        "maximum_sa0p4_g": ("sa0p4_simulated_g", "max"),
    }
    for prefix, specification in COMPONENTS.items():
        base_aggregations[f"{prefix}_mean_expected_damage_state"] = (
            specification["expected_column"],
            "mean",
        )
        base_aggregations[f"{prefix}_mean_sampled_damage_state"] = (
            specification["sample_column"],
            "mean",
        )

    output = (
        frame.groupby(group_columns, sort=False, observed=True)
        .agg(**base_aggregations)
        .reset_index()
    )

    for prefix, specification in COMPONENTS.items():
        sampled_counts = (
            frame.groupby(
                group_columns + [specification["sample_column"]], sort=False
            )
            .size()
            .unstack(fill_value=0)
            .reindex(columns=range(5), fill_value=0)
            .rename(
                columns={
                    index: f"{prefix}_sampled_{state.lower()}_buildings"
                    for index, state in enumerate(DAMAGE_STATES)
                }
            )
            .reset_index()
        )
        expected_counts = (
            frame.groupby(group_columns, sort=False, observed=True)[
                probability_columns(prefix)
            ]
            .sum()
            .rename(
                columns={
                    f"{prefix}_p_state_{state.lower()}": (
                        f"{prefix}_expected_{state.lower()}_buildings"
                    )
                    for state in DAMAGE_STATES
                }
            )
            .reset_index()
        )
        output = output.merge(
            sampled_counts, on=group_columns, how="left", validate="one_to_one"
        )
        output = output.merge(
            expected_counts, on=group_columns, how="left", validate="one_to_one"
        )
        output[f"{prefix}_sampled_slight_or_worse_buildings"] = output[
            [
                f"{prefix}_sampled_{state.lower()}_buildings"
                for state in DAMAGE_STATES[1:]
            ]
        ].sum(axis=1)
        output[f"{prefix}_expected_slight_or_worse_buildings"] = output[
            [
                f"{prefix}_expected_{state.lower()}_buildings"
                for state in DAMAGE_STATES[1:]
            ]
        ].sum(axis=1)

    return output.sort_values("occurrence_ordinal").reset_index(drop=True)


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
CONTROL_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_controlled_nonstructural_damage"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_nonstructural_damage"
CHUNK_DIR = OUTPUT_DIR / "chunks"
WORK_DIR = METADATA_DIR / "notebook_5_cell_9_work"
MARKER_DIR = WORK_DIR / "markers"
CHUNK_VALIDATION_DIR = WORK_DIR / "chunk_validations"
for directory in [
    METADATA_DIR,
    OUTPUT_DIR,
    CHUNK_DIR,
    WORK_DIR,
    MARKER_DIR,
    CHUNK_VALIDATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CELL3_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_3_summary.json"
CELL3_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_3_validation.csv"
CELL7_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_7_summary.json"
CELL7_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_7_validation.csv"
CELL8_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_8_summary.json"
CELL8_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_8_validation.csv"
CELL8_RANDOM_STREAM_PATH = (
    METADATA_DIR / "notebook_5_cell_8_nonstructural_damage_random_streams.json"
)
BUILDING_ASSIGNMENTS_PATH = (
    PARAMETER_DIR / "seaside_w2_building_nonstructural_fragility_assignments.csv"
)
CONTROLLED_DAMAGE_PATH = CONTROL_DIR / "controlled_nonstructural_damage_rows.csv"

required_paths = [
    CELL3_SUMMARY_PATH,
    CELL3_VALIDATION_PATH,
    CELL7_SUMMARY_PATH,
    CELL7_VALIDATION_PATH,
    CELL8_SUMMARY_PATH,
    CELL8_VALIDATION_PATH,
    CELL8_RANDOM_STREAM_PATH,
    BUILDING_ASSIGNMENTS_PATH,
    CONTROLLED_DAMAGE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        f"Required Cell 3, Cell 7, or Cell 8 outputs are missing: {missing_paths}"
    )

validation_rows: list[dict[str, Any]] = []
cell3_summary = load_json(CELL3_SUMMARY_PATH)
cell7_summary = load_json(CELL7_SUMMARY_PATH)
cell8_summary = load_json(CELL8_SUMMARY_PATH)
cell8_streams = load_json(CELL8_RANDOM_STREAM_PATH)
cell3_validation = pd.read_csv(CELL3_VALIDATION_PATH)
cell7_validation = pd.read_csv(CELL7_VALIDATION_PATH)
cell8_validation = pd.read_csv(CELL8_VALIDATION_PATH)

for cell_number, summary, validation in [
    (3, cell3_summary, cell3_validation),
    (7, cell7_summary, cell7_validation),
    (8, cell8_summary, cell8_validation),
]:
    append_check(
        validation_rows,
        f"cell{cell_number}_critical_checks_passed",
        summary.get("all_critical_checks_passed") is True,
        f"all_critical_checks_passed={summary.get('all_critical_checks_passed')}",
    )
    passed = parse_bool_series(validation["passed"])
    unresolved = validation.loc[
        validation["severity"].astype(str).str.lower().eq("critical") & ~passed
    ]
    append_check(
        validation_rows,
        f"cell{cell_number}_has_no_unresolved_critical_checks",
        unresolved.empty,
        f"unresolved_critical={len(unresolved)}",
    )

stream_entries = cell8_streams.get("streams", {})
for prefix, specification in COMPONENTS.items():
    stream_key = (
        "nonstructural_drift_sensitive_damage"
        if prefix == "nsd"
        else "nonstructural_acceleration_sensitive_damage"
    )
    recorded_namespace = stream_entries.get(stream_key, {}).get("namespace")
    append_check(
        validation_rows,
        f"{prefix}_stream_namespace_matches_cell8",
        recorded_namespace == specification["namespace"],
        f"recorded={recorded_namespace}; expected={specification['namespace']}",
    )
append_check(
    validation_rows,
    "nonstructural_stream_namespaces_are_distinct",
    COMPONENTS["nsd"]["namespace"] != COMPONENTS["nsa"]["namespace"],
    "Drift-sensitive and acceleration-sensitive damage streams must be separate.",
)

field_info = cell3_summary["ground_motion_fields"]
field_path = resolve_recorded_path(PROJECT_ROOT, str(field_info["path"]))
expected_rows = int(field_info["rows"])
structural_im_column = str(field_info["structural_im_column"])
period_s = float(field_info["period_s"])
expected_sites = int(cell7_summary["portfolio"]["buildings"])

append_check(validation_rows, "ground_motion_field_exists", field_path.exists(), str(field_path))
append_check(
    validation_rows,
    "production_im_is_sa0p4",
    structural_im_column == "sa0p4_simulated_g",
    f"column={structural_im_column}",
)
append_check(
    validation_rows,
    "period_is_exact_sa0p4",
    math.isclose(period_s, EXPECTED_PERIOD_S, rel_tol=0.0, abs_tol=1e-12),
    f"period_s={period_s}",
)
append_check(
    validation_rows,
    "portfolio_has_470_sites",
    expected_sites == EXPECTED_SITES,
    f"sites={expected_sites}",
)
append_check(
    validation_rows,
    "field_rows_divisible_by_sites",
    expected_rows % expected_sites == 0,
    f"rows={expected_rows}; sites={expected_sites}",
)
expected_occurrences = expected_rows // expected_sites

actual_field_hash = sha256_file(field_path)
append_check(
    validation_rows,
    "ground_motion_hash_matches_cell3",
    actual_field_hash == str(field_info["sha256_from_cell1"]),
    f"calculated={actual_field_hash}; recorded={field_info['sha256_from_cell1']}",
)

building_assignments = pd.read_csv(BUILDING_ASSIGNMENTS_PATH)
controlled_damage = pd.read_csv(CONTROLLED_DAMAGE_PATH)
all_parameter_columns: list[str] = []
for prefix in COMPONENTS:
    medians, betas = parameter_columns(prefix)
    all_parameter_columns.extend(medians + betas)

required_assignment_columns = {
    "site_id",
    "structural_type",
    "design_level",
    "occ_type",
    "year_built",
    "period_s",
    *all_parameter_columns,
}
missing_assignment_columns = sorted(
    required_assignment_columns.difference(building_assignments.columns)
)
append_check(
    validation_rows,
    "building_assignment_columns_present",
    not missing_assignment_columns,
    f"missing={missing_assignment_columns}",
)
if missing_assignment_columns:
    raise KeyError(
        f"Nonstructural building assignments are missing columns: {missing_assignment_columns}"
    )

building_assignments["site_id"] = building_assignments["site_id"].astype(str)
building_assignments["structural_type"] = (
    building_assignments["structural_type"].astype(str).str.strip().str.upper()
)
building_assignments["design_level"] = (
    building_assignments["design_level"].astype(str).str.strip()
)
building_assignments["occ_type"] = (
    building_assignments["occ_type"].astype(str).str.strip().str.upper()
)
building_assignments["period_s"] = pd.to_numeric(
    building_assignments["period_s"], errors="raise"
)

append_check(
    validation_rows,
    "building_assignment_hash_matches_cell8",
    sha256_file(BUILDING_ASSIGNMENTS_PATH)
    == str(cell8_summary["building_assignments"]["sha256"]),
    "Cell 9 assignments must match the file validated in Cell 8.",
)
append_check(
    validation_rows,
    "building_assignment_rows_match_sites",
    len(building_assignments) == expected_sites,
    f"rows={len(building_assignments)}; expected={expected_sites}",
)
append_check(
    validation_rows,
    "building_site_ids_unique",
    building_assignments["site_id"].is_unique,
    f"duplicates={int(building_assignments['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "all_buildings_are_w2",
    building_assignments["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
    f"types={sorted(building_assignments['structural_type'].unique().tolist())}",
)
append_check(
    validation_rows,
    "all_building_periods_are_0p4_seconds",
    np.allclose(
        building_assignments["period_s"].to_numpy(dtype=float),
        EXPECTED_PERIOD_S,
        rtol=0.0,
        atol=1e-12,
    ),
    f"periods={sorted(building_assignments['period_s'].unique().tolist())}",
)
append_check(
    validation_rows,
    "nonstructural_parameters_complete_positive",
    building_assignments[all_parameter_columns].notna().all().all()
    and (building_assignments[all_parameter_columns] > 0.0).all().all(),
    (
        f"missing={int(building_assignments[all_parameter_columns].isna().sum().sum())}; "
        f"nonpositive={int((building_assignments[all_parameter_columns].fillna(0) <= 0).sum().sum())}"
    ),
)

field_header = pd.read_csv(field_path, nrows=0)
required_field_columns = {
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    structural_im_column,
}
missing_field_columns = sorted(required_field_columns.difference(field_header.columns))
append_check(
    validation_rows,
    "ground_motion_columns_present",
    not missing_field_columns,
    f"missing={missing_field_columns}",
)
if missing_field_columns:
    raise KeyError(f"Ground-motion file is missing columns: {missing_field_columns}")

optional_field_columns = [
    column
    for column in ["catalog_event_id", "rupture_ordinal", "rupture_template_event_id"]
    if column in field_header.columns
]
field_usecols = [
    "catalog_year",
    *optional_field_columns,
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    structural_im_column,
]
lookup_columns = [
    "site_id",
    "structural_type",
    "design_level",
    "occ_type",
    "year_built",
    "period_s",
    *all_parameter_columns,
]
design_lookup = building_assignments[lookup_columns].copy()

basis_payload = {
    "pipeline_version": PIPELINE_VERSION,
    "ground_motion_sha256": actual_field_hash,
    "building_assignments_sha256": sha256_file(BUILDING_ASSIGNMENTS_PATH),
    "cell8_summary_sha256": sha256_file(CELL8_SUMMARY_PATH),
    "cell8_validation_sha256": sha256_file(CELL8_VALIDATION_PATH),
    "controlled_damage_sha256": sha256_file(CONTROLLED_DAMAGE_PATH),
    "random_stream_specification_sha256": sha256_file(CELL8_RANDOM_STREAM_PATH),
    "stream_namespaces": {
        prefix: specification["namespace"] for prefix, specification in COMPONENTS.items()
    },
    "occurrences_per_chunk": OCCURRENCES_PER_CHUNK,
}
basis_hash = hashlib.sha256(
    json.dumps(basis_payload, sort_keys=True).encode("utf-8")
).hexdigest()

chunk_rows = expected_sites * OCCURRENCES_PER_CHUNK
expected_chunks = math.ceil(expected_rows / chunk_rows)
controlled_damage["occurrence_id"] = controlled_damage["occurrence_id"].astype(str)
controlled_damage["site_id"] = controlled_damage["site_id"].astype(str)
control_keys = set(zip(controlled_damage["occurrence_id"], controlled_damage["site_id"]))
control_production_frames: list[pd.DataFrame] = []
chunk_manifest_rows: list[dict[str, Any]] = []
chunk_paths: list[Path] = []
event_summary_frames: list[pd.DataFrame] = []

seen_pairs = np.zeros(expected_rows, dtype=bool)
occurrence_counts = np.zeros(expected_occurrences, dtype=np.int64)
site_counts = np.zeros(expected_sites, dtype=np.int64)
seen_occurrences: set[str] = set()
source_row_counts: Counter[str] = Counter()
design_level_row_counts: Counter[str] = Counter()
sampled_state_counts = {
    prefix: np.zeros(5, dtype=np.int64) for prefix in COMPONENTS
}
expected_state_counts = {
    prefix: np.zeros(5, dtype=np.float64) for prefix in COMPONENTS
}
uniform_sum = {prefix: 0.0 for prefix in COMPONENTS}
uniform_square_sum = {prefix: 0.0 for prefix in COMPONENTS}
uniform_cross_sum = 0.0
total_processed_rows = 0
global_minimum_raw_probability = {prefix: math.inf for prefix in COMPONENTS}
global_max_probability_sum_error = {prefix: 0.0 for prefix in COMPONENTS}

print("=" * 78)
print("FULL ANNUAL-CATALOG NONSTRUCTURAL DAMAGE-STATE SAMPLING")
print("=" * 78)
print(f"Ground-motion rows:       {expected_rows:,}")
print(f"Catalog occurrences:      {expected_occurrences:,}")
print(f"Portfolio buildings:      {expected_sites:,}")
print(f"Occurrences per chunk:    {OCCURRENCES_PER_CHUNK:,}")
print(f"Production chunks:        {expected_chunks:,}")
print("Damage IM:                exact SA0P4")
print(f"Drift damage namespace:   {COMPONENTS['nsd']['namespace']}")
print(f"Acceleration namespace:   {COMPONENTS['nsa']['namespace']}")

reader = pd.read_csv(field_path, usecols=field_usecols, chunksize=chunk_rows)
for chunk_index, raw_chunk in enumerate(reader):
    chunk_id = f"{chunk_index:04d}"
    print()
    print("-" * 78)
    print(
        f"NONSTRUCTURAL DAMAGE CHUNK {chunk_index + 1} OF {expected_chunks} "
        f"[ID {chunk_id}]"
    )

    raw_chunk = raw_chunk.rename(columns={structural_im_column: "sa0p4_simulated_g"})
    raw_chunk["site_id"] = raw_chunk["site_id"].astype(str)
    raw_chunk["occurrence_id"] = raw_chunk["occurrence_id"].astype(str)
    raw_chunk["source_type"] = normalize_source_type(raw_chunk["source_type"])
    raw_chunk["occurrence_ordinal"] = pd.to_numeric(
        raw_chunk["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    raw_chunk["site_ordinal"] = pd.to_numeric(
        raw_chunk["site_ordinal"], errors="raise"
    ).astype(np.int64)
    raw_chunk["sa0p4_simulated_g"] = pd.to_numeric(
        raw_chunk["sa0p4_simulated_g"], errors="raise"
    )

    input_hash_columns = [
        "occurrence_ordinal",
        "occurrence_id",
        "site_ordinal",
        "site_id",
        "sa0p4_simulated_g",
    ]
    input_hash = stable_frame_hash(raw_chunk, input_hash_columns)
    chunk_output_path = CHUNK_DIR / f"nonstructural_damage_chunk_{chunk_id}.csv.gz"
    marker_path = MARKER_DIR / f"nonstructural_damage_chunk_{chunk_id}.json"
    chunk_validation_path = (
        CHUNK_VALIDATION_DIR / f"nonstructural_damage_chunk_{chunk_id}_validation.csv"
    )

    marker_valid = False
    marker: dict[str, Any] = {}
    if marker_path.exists() and chunk_output_path.exists() and chunk_validation_path.exists():
        try:
            marker = load_json(marker_path)
            marker_valid = (
                marker.get("basis_hash") == basis_hash
                and marker.get("input_hash") == input_hash
                and int(marker.get("rows", -1)) == len(raw_chunk)
                and marker.get("output_sha256") == sha256_file(chunk_output_path)
            )
        except Exception:
            marker_valid = False

    if marker_valid:
        output = pd.read_csv(chunk_output_path)
        print(f"Reused validated chunk with {len(output):,} rows.")
    else:
        chunk_validation_rows: list[dict[str, Any]] = []
        occurrence_sizes = raw_chunk.groupby("occurrence_id", sort=False).size()
        append_check(
            chunk_validation_rows,
            "input_occurrences_complete",
            occurrence_sizes.eq(expected_sites).all(),
            (
                f"occurrences={len(occurrence_sizes)}; minimum={int(occurrence_sizes.min())}; "
                f"maximum={int(occurrence_sizes.max())}; expected={expected_sites}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "input_occurrence_ids_not_seen_in_prior_chunks",
            not set(occurrence_sizes.index.astype(str)).intersection(seen_occurrences),
            "Each occurrence must be fully contained in one production chunk.",
        )

        merged = raw_chunk.merge(
            design_lookup, on="site_id", how="left", validate="many_to_one"
        )
        missing_assignments = int(merged["design_level"].isna().sum())
        append_check(
            chunk_validation_rows,
            "all_sites_have_nonstructural_fragility_assignments",
            missing_assignments == 0,
            f"missing_rows={missing_assignments}",
        )
        if missing_assignments:
            raise RuntimeError(
                f"Chunk {chunk_id} has {missing_assignments} rows without assignments."
            )

        component_probability_frames: dict[str, pd.DataFrame] = {}
        component_uniforms: dict[str, np.ndarray] = {}
        component_states: dict[str, np.ndarray] = {}
        component_minimum_raw: dict[str, float] = {}

        for prefix, specification in COMPONENTS.items():
            probabilities, minimum_raw = calculate_component_probabilities(
                merged, prefix
            )
            uniforms = deterministic_uniforms(
                merged["occurrence_id"],
                merged["site_id"],
                specification["namespace"],
            )
            sampled_states = sample_damage_states(probabilities, uniforms, prefix)
            component_probability_frames[prefix] = probabilities
            component_uniforms[prefix] = uniforms
            component_states[prefix] = sampled_states
            component_minimum_raw[prefix] = minimum_raw

        output = merged[
            [
                "catalog_year",
                *optional_field_columns,
                "occurrence_ordinal",
                "occurrence_id",
                "rupture_id",
                "source_type",
                "magnitude",
                "site_ordinal",
                "site_id",
                "sa0p4_simulated_g",
                "structural_type",
                "design_level",
                "occ_type",
                "year_built",
            ]
        ].copy()
        output = output.reset_index(drop=True)

        for prefix, specification in COMPONENTS.items():
            output = pd.concat(
                [
                    output,
                    component_probability_frames[prefix].reset_index(drop=True),
                ],
                axis=1,
            )
            output[specification["uniform_column"]] = component_uniforms[prefix]
            output[specification["sample_column"]] = component_states[prefix]
            output[specification["sample_name_column"]] = pd.Series(
                component_states[prefix]
            ).map(dict(enumerate(DAMAGE_STATES)))

        append_check(
            chunk_validation_rows,
            "output_rows_match_input",
            len(output) == len(raw_chunk),
            f"output={len(output)}; input={len(raw_chunk)}",
        )

        for prefix, specification in COMPONENTS.items():
            matrix = output[probability_columns(prefix)].to_numpy(dtype=np.float64)
            probability_sum_error = float(
                np.max(np.abs(matrix.sum(axis=1) - 1.0))
            )
            sampled_states = component_states[prefix]
            uniforms = component_uniforms[prefix]
            append_check(
                chunk_validation_rows,
                f"{prefix}_probabilities_finite",
                np.isfinite(matrix).all(),
                "All mutually exclusive probabilities must be finite.",
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_probabilities_nonnegative",
                matrix.min() >= -PROBABILITY_TOLERANCE,
                f"minimum={float(matrix.min()):.3e}",
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_probabilities_sum_to_one",
                probability_sum_error <= 2e-15,
                f"maximum_error={probability_sum_error:.3e}",
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_expected_damage_state_in_range",
                output[specification["expected_column"]].between(0.0, 4.0).all(),
                (
                    f"minimum={output[specification['expected_column']].min():.6f}; "
                    f"maximum={output[specification['expected_column']].max():.6f}"
                ),
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_uniforms_strictly_inside_unit_interval",
                np.all((uniforms > 0.0) & (uniforms < 1.0)),
                f"minimum={uniforms.min():.6e}; maximum={uniforms.max():.6e}",
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_sampled_states_in_range",
                np.all((sampled_states >= 0) & (sampled_states <= 4)),
                f"minimum={sampled_states.min()}; maximum={sampled_states.max()}",
            )
            sample_rows = min(1000, len(merged))
            repeated_uniforms = deterministic_uniforms(
                merged["occurrence_id"].iloc[:sample_rows],
                merged["site_id"].iloc[:sample_rows],
                specification["namespace"],
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_deterministic_uniform_reproduction",
                np.array_equal(repeated_uniforms, uniforms[:sample_rows]),
                f"sample_rows={sample_rows}",
            )
            repeated_states = sample_damage_states(
                component_probability_frames[prefix], uniforms, prefix
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_categorical_state_reconstruction",
                np.array_equal(repeated_states, sampled_states),
                "Sampled states must reproduce exactly from probabilities and uniforms.",
            )

        append_check(
            chunk_validation_rows,
            "component_uniform_streams_are_distinct",
            not np.array_equal(
                component_uniforms["nsd"], component_uniforms["nsa"]
            ),
            "Drift-sensitive and acceleration-sensitive streams must differ.",
        )
        pair_duplicates = int(output.duplicated(["occurrence_id", "site_id"]).sum())
        append_check(
            chunk_validation_rows,
            "occurrence_site_pairs_unique",
            pair_duplicates == 0,
            f"duplicates={pair_duplicates}",
        )
        append_check(
            chunk_validation_rows,
            "source_types_valid",
            set(output["source_type"].unique()).issubset({"INTERFACE", "SLAB"}),
            f"types={sorted(output['source_type'].unique().tolist())}",
        )
        append_check(
            chunk_validation_rows,
            "structural_type_is_w2",
            output["structural_type"].eq(EXPECTED_STRUCTURAL_TYPE).all(),
            f"types={sorted(output['structural_type'].unique().tolist())}",
        )

        chunk_validation = pd.DataFrame(chunk_validation_rows)
        chunk_failures = chunk_validation.loc[
            chunk_validation["severity"].eq("critical") & ~chunk_validation["passed"]
        ]
        chunk_validation.to_csv(chunk_validation_path, index=False)
        if not chunk_failures.empty:
            raise RuntimeError(
                f"Nonstructural damage chunk {chunk_id} failed validation. "
                f"Review {chunk_validation_path}."
            )

        write_gzip_csv_deterministic(output, chunk_output_path)
        marker = {
            "pipeline_version": PIPELINE_VERSION,
            "basis_hash": basis_hash,
            "input_hash": input_hash,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_raw_probability": component_minimum_raw,
            "output_path": str(chunk_output_path),
            "output_sha256": sha256_file(chunk_output_path),
            "validation_path": str(chunk_validation_path),
            "validation_sha256": sha256_file(chunk_validation_path),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        write_json(marker_path, marker)
        print(
            f"Calculated {len(output):,} rows for "
            f"{output['occurrence_id'].nunique():,} occurrences."
        )

    required_output_columns = {
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
        "site_ordinal",
        "site_id",
        "sa0p4_simulated_g",
        "structural_type",
        "design_level",
        "occ_type",
        "year_built",
    }
    for prefix, specification in COMPONENTS.items():
        required_output_columns.update(probability_columns(prefix))
        required_output_columns.update(
            {
                specification["expected_column"],
                specification["uniform_column"],
                specification["sample_column"],
                specification["sample_name_column"],
            }
        )
    missing_output_columns = sorted(required_output_columns.difference(output.columns))
    if missing_output_columns:
        raise KeyError(f"Chunk {chunk_id} output is missing columns: {missing_output_columns}")

    output["occurrence_ordinal"] = pd.to_numeric(
        output["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    output["site_ordinal"] = pd.to_numeric(
        output["site_ordinal"], errors="raise"
    ).astype(np.int64)
    output["sa0p4_simulated_g"] = pd.to_numeric(
        output["sa0p4_simulated_g"], errors="raise"
    )
    for prefix, specification in COMPONENTS.items():
        output[specification["sample_column"]] = pd.to_numeric(
            output[specification["sample_column"]], errors="raise"
        ).astype(np.int8)
        for column in probability_columns(prefix) + [
            specification["expected_column"],
            specification["uniform_column"],
        ]:
            output[column] = pd.to_numeric(output[column], errors="raise")

    occurrence_ordinals = output["occurrence_ordinal"].to_numpy(dtype=np.int64)
    site_ordinals = output["site_ordinal"].to_numpy(dtype=np.int64)
    if np.any(occurrence_ordinals < 0) or np.any(
        occurrence_ordinals >= expected_occurrences
    ):
        raise RuntimeError(
            f"Chunk {chunk_id} has occurrence ordinals outside the expected range."
        )
    if np.any(site_ordinals < 0) or np.any(site_ordinals >= expected_sites):
        raise RuntimeError(
            f"Chunk {chunk_id} has site ordinals outside the expected range."
        )

    pair_indices = occurrence_ordinals * expected_sites + site_ordinals
    if len(np.unique(pair_indices)) != len(pair_indices):
        raise RuntimeError(f"Chunk {chunk_id} contains duplicate ordinal pair indices.")
    if seen_pairs[pair_indices].any():
        raise RuntimeError(
            f"Chunk {chunk_id} contains occurrence-site pairs seen previously."
        )
    seen_pairs[pair_indices] = True
    np.add.at(occurrence_counts, occurrence_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    occurrence_ids = set(output["occurrence_id"].astype(str).unique().tolist())
    repeated_occurrences = occurrence_ids.intersection(seen_occurrences)
    if repeated_occurrences:
        raise RuntimeError(
            f"Chunk {chunk_id} repeats occurrences from prior chunks: "
            f"{sorted(repeated_occurrences)[:5]}"
        )
    seen_occurrences.update(occurrence_ids)

    for prefix, specification in COMPONENTS.items():
        matrix = output[probability_columns(prefix)].to_numpy(dtype=np.float64)
        sampled_states = output[specification["sample_column"]].to_numpy(dtype=np.int8)
        uniforms = output[specification["uniform_column"]].to_numpy(dtype=np.float64)
        sampled_state_counts[prefix] += np.bincount(sampled_states, minlength=5)
        expected_state_counts[prefix] += matrix.sum(axis=0)
        uniform_sum[prefix] += float(uniforms.sum())
        uniform_square_sum[prefix] += float(np.square(uniforms).sum())
        marker_minimum_raw = marker.get("minimum_raw_probability", {})
        if isinstance(marker_minimum_raw, dict):
            minimum_value = float(
                marker_minimum_raw.get(prefix, float(matrix.min()))
            )
        else:
            minimum_value = float(matrix.min())
        global_minimum_raw_probability[prefix] = min(
            global_minimum_raw_probability[prefix], minimum_value
        )
        global_max_probability_sum_error[prefix] = max(
            global_max_probability_sum_error[prefix],
            float(np.max(np.abs(matrix.sum(axis=1) - 1.0))),
        )

    nsd_uniforms = output[COMPONENTS["nsd"]["uniform_column"]].to_numpy(
        dtype=np.float64
    )
    nsa_uniforms = output[COMPONENTS["nsa"]["uniform_column"]].to_numpy(
        dtype=np.float64
    )
    uniform_cross_sum += float(np.dot(nsd_uniforms, nsa_uniforms))
    total_processed_rows += len(output)
    source_row_counts.update(output["source_type"].astype(str).tolist())
    design_level_row_counts.update(output["design_level"].astype(str).tolist())

    key_mask = np.fromiter(
        (
            (occurrence_id, site_id) in control_keys
            for occurrence_id, site_id in zip(
                output["occurrence_id"].astype(str),
                output["site_id"].astype(str),
                strict=True,
            )
        ),
        dtype=bool,
        count=len(output),
    )
    if key_mask.any():
        control_production_frames.append(output.loc[key_mask].copy())

    event_summary_frames.append(summarize_occurrences(output))
    chunk_paths.append(chunk_output_path)
    chunk_manifest_rows.append(
        {
            "chunk_id": chunk_id,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_occurrence_ordinal": int(output["occurrence_ordinal"].min()),
            "maximum_occurrence_ordinal": int(output["occurrence_ordinal"].max()),
            "output_path": str(chunk_output_path),
            "output_size_bytes": int(chunk_output_path.stat().st_size),
            "output_sha256": sha256_file(chunk_output_path),
            "marker_path": str(marker_path),
            "validation_path": str(chunk_validation_path),
            "reused": bool(marker_valid),
        }
    )

if len(chunk_paths) != expected_chunks:
    raise RuntimeError(f"Processed {len(chunk_paths)} chunks but expected {expected_chunks}.")

final_damage_path = OUTPUT_DIR / "full_nonstructural_damage_states.csv.gz"
final_event_summary_path = OUTPUT_DIR / "nonstructural_damage_event_summary.csv.gz"
state_totals_path = METADATA_DIR / "notebook_5_cell_9_nonstructural_damage_state_totals.csv"
chunk_manifest_path = METADATA_DIR / "notebook_5_cell_9_chunk_manifest.csv"
validation_path = METADATA_DIR / "notebook_5_cell_9_validation.csv"
summary_path = METADATA_DIR / "notebook_5_cell_9_summary.json"
random_specification_path = (
    METADATA_DIR / "notebook_5_cell_9_random_stream_specification.json"
)

concatenate_gzip_csv_files(chunk_paths, final_damage_path)
event_summary = pd.concat(event_summary_frames, ignore_index=True)
event_summary = event_summary.sort_values("occurrence_ordinal").reset_index(drop=True)
write_gzip_csv_deterministic(event_summary, final_event_summary_path)

if not control_production_frames:
    raise RuntimeError("No Cell 8 controlled occurrence-site rows were found in production.")
control_production = pd.concat(control_production_frames, ignore_index=True)
control_compare = controlled_damage.merge(
    control_production,
    on=["occurrence_id", "site_id"],
    how="outer",
    suffixes=("_cell8", "_cell9"),
    indicator=True,
    validate="one_to_one",
)
append_check(
    validation_rows,
    "cell8_controls_reproduced_once",
    len(control_compare) == len(controlled_damage)
    and control_compare["_merge"].eq("both").all(),
    (
        f"cell8={len(controlled_damage)}; compared={len(control_compare)}; "
        f"merge={control_compare['_merge'].value_counts().to_dict()}"
    ),
)

controls_complete = (
    len(control_compare) == len(controlled_damage)
    and control_compare["_merge"].eq("both").all()
)
maximum_control_difference = 0.0 if controls_complete else math.inf
if controls_complete:
    numeric_control_columns = ["sa0p4_simulated_g"]
    for prefix, specification in COMPONENTS.items():
        numeric_control_columns.extend(probability_columns(prefix))
        numeric_control_columns.extend(
            [specification["expected_column"], specification["uniform_column"]]
        )
    for column in numeric_control_columns:
        left = pd.to_numeric(
            control_compare[f"{column}_cell8"], errors="raise"
        ).to_numpy(dtype=np.float64)
        right = pd.to_numeric(
            control_compare[f"{column}_cell9"], errors="raise"
        ).to_numpy(dtype=np.float64)
        maximum_control_difference = max(
            maximum_control_difference, float(np.max(np.abs(left - right)))
        )
append_check(
    validation_rows,
    "cell8_control_numeric_values_reproduced",
    maximum_control_difference <= CONTROL_TOLERANCE,
    f"maximum_absolute_difference={maximum_control_difference:.3e}",
)
for prefix, specification in COMPONENTS.items():
    append_check(
        validation_rows,
        f"cell8_{prefix}_sampled_states_reproduced",
        controls_complete
        and np.array_equal(
            pd.to_numeric(
                control_compare[f"{specification['sample_column']}_cell8"],
                errors="raise",
            ).to_numpy(dtype=np.int8),
            pd.to_numeric(
                control_compare[f"{specification['sample_column']}_cell9"],
                errors="raise",
            ).to_numpy(dtype=np.int8),
        ),
        f"Cell 9 must exactly reproduce the Cell 8 {prefix} sampled states.",
    )

uniform_means: dict[str, float] = {}
uniform_variances: dict[str, float] = {}
sampled_proportions: dict[str, np.ndarray] = {}
expected_proportions: dict[str, np.ndarray] = {}
maximum_sampled_expected_difference: dict[str, float] = {}
for prefix in COMPONENTS:
    uniform_means[prefix] = uniform_sum[prefix] / total_processed_rows
    uniform_variances[prefix] = (
        uniform_square_sum[prefix] / total_processed_rows
        - uniform_means[prefix] ** 2
    )
    sampled_proportions[prefix] = (
        sampled_state_counts[prefix] / float(total_processed_rows)
    )
    expected_proportions[prefix] = (
        expected_state_counts[prefix] / float(total_processed_rows)
    )
    maximum_sampled_expected_difference[prefix] = float(
        np.max(
            np.abs(
                sampled_proportions[prefix] - expected_proportions[prefix]
            )
        )
    )

uniform_covariance = (
    uniform_cross_sum / total_processed_rows
    - uniform_means["nsd"] * uniform_means["nsa"]
)
uniform_correlation = uniform_covariance / math.sqrt(
    uniform_variances["nsd"] * uniform_variances["nsa"]
)
uniform_mean_tolerance = max(
    BASE_UNIFORM_MEAN_TOLERANCE,
    5.0 / math.sqrt(12.0 * total_processed_rows),
)
uniform_variance_tolerance = max(
    BASE_UNIFORM_VARIANCE_TOLERANCE,
    5.0 * math.sqrt((1.0 / 180.0) / total_processed_rows),
)
sampled_expected_proportion_tolerance = max(
    BASE_SAMPLED_EXPECTED_PROPORTION_TOLERANCE,
    2.5 / math.sqrt(total_processed_rows),
)
production_stream_correlation_tolerance = max(
    BASE_PRODUCTION_STREAM_CORRELATION_TOLERANCE,
    5.0 / math.sqrt(total_processed_rows),
)

append_check(
    validation_rows,
    "production_rows_complete",
    total_processed_rows == expected_rows,
    f"processed={total_processed_rows}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "all_occurrence_site_pairs_present_once",
    seen_pairs.all(),
    f"missing_pairs={int((~seen_pairs).sum())}",
)
append_check(
    validation_rows,
    "every_occurrence_has_all_sites",
    np.all(occurrence_counts == expected_sites),
    (
        f"minimum={int(occurrence_counts.min())}; maximum={int(occurrence_counts.max())}; "
        f"expected={expected_sites}"
    ),
)
append_check(
    validation_rows,
    "every_site_has_all_occurrences",
    np.all(site_counts == expected_occurrences),
    (
        f"minimum={int(site_counts.min())}; maximum={int(site_counts.max())}; "
        f"expected={expected_occurrences}"
    ),
)
append_check(
    validation_rows,
    "unique_occurrences_complete",
    len(seen_occurrences) == expected_occurrences,
    f"observed={len(seen_occurrences)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "event_summary_has_one_row_per_occurrence",
    len(event_summary) == expected_occurrences
    and event_summary["occurrence_id"].is_unique,
    f"rows={len(event_summary)}; unique={event_summary['occurrence_id'].nunique()}",
)
append_check(
    validation_rows,
    "event_summary_building_counts_are_470",
    event_summary["buildings"].eq(expected_sites).all(),
    (
        f"minimum={event_summary['buildings'].min()}; "
        f"maximum={event_summary['buildings'].max()}"
    ),
)
for prefix in COMPONENTS:
    append_check(
        validation_rows,
        f"{prefix}_sampled_state_counts_sum_to_rows",
        int(sampled_state_counts[prefix].sum()) == expected_rows,
        f"sum={int(sampled_state_counts[prefix].sum())}; expected={expected_rows}",
    )
    append_check(
        validation_rows,
        f"{prefix}_expected_state_counts_sum_to_rows",
        abs(float(expected_state_counts[prefix].sum()) - expected_rows) <= 1e-6,
        (
            f"sum={float(expected_state_counts[prefix].sum()):.6f}; "
            f"expected={expected_rows}"
        ),
    )
    append_check(
        validation_rows,
        f"{prefix}_production_uniform_mean_reasonable",
        abs(uniform_means[prefix] - 0.5) <= uniform_mean_tolerance,
        (
            f"mean={uniform_means[prefix]:.8f}; "
            f"tolerance={uniform_mean_tolerance:.8f}"
        ),
    )
    append_check(
        validation_rows,
        f"{prefix}_production_uniform_variance_reasonable",
        abs(uniform_variances[prefix] - 1.0 / 12.0) <= uniform_variance_tolerance,
        (
            f"variance={uniform_variances[prefix]:.8f}; "
            f"target={1.0/12.0:.8f}; "
            f"tolerance={uniform_variance_tolerance:.8f}"
        ),
    )
    append_check(
        validation_rows,
        f"{prefix}_sampled_state_proportions_match_expected_probabilities",
        maximum_sampled_expected_difference[prefix]
        <= sampled_expected_proportion_tolerance,
        (
            f"maximum_absolute_difference="
            f"{maximum_sampled_expected_difference[prefix]:.6f}; "
            f"tolerance={sampled_expected_proportion_tolerance:.6f}"
        ),
    )
    append_check(
        validation_rows,
        f"{prefix}_probability_sum_error_small",
        global_max_probability_sum_error[prefix] <= 2e-15,
        f"maximum_error={global_max_probability_sum_error[prefix]:.3e}",
    )
    append_check(
        validation_rows,
        f"{prefix}_minimum_raw_probability_within_roundoff",
        global_minimum_raw_probability[prefix] >= -PROBABILITY_TOLERANCE,
        f"minimum={global_minimum_raw_probability[prefix]:.3e}",
    )
append_check(
    validation_rows,
    "production_component_stream_correlation_small",
    abs(uniform_correlation) <= production_stream_correlation_tolerance,
    (
        f"correlation={uniform_correlation:.6f}; "
        f"tolerance={production_stream_correlation_tolerance:.6f}"
    ),
)
append_check(
    validation_rows,
    "source_row_counts_match_cell3",
    dict(sorted(source_row_counts.items()))
    == {
        str(key): int(value)
        for key, value in sorted(field_info["source_row_counts"].items())
    },
    (
        f"observed={dict(sorted(source_row_counts.items()))}; "
        f"expected={field_info['source_row_counts']}"
    ),
)
append_check(
    validation_rows,
    "design_level_row_counts_match_cell7",
    dict(sorted(design_level_row_counts.items()))
    == {
        str(key): int(value) * expected_occurrences
        for key, value in sorted(
            cell7_summary["portfolio"]["design_level_counts"].items()
        )
    },
    (
        f"observed={dict(sorted(design_level_row_counts.items()))}; "
        f"expected_per_occurrence={cell7_summary['portfolio']['design_level_counts']}"
    ),
)
append_check(
    validation_rows,
    "final_damage_file_exists",
    final_damage_path.exists() and final_damage_path.stat().st_size > 0,
    (
        f"path={final_damage_path}; "
        f"bytes={final_damage_path.stat().st_size if final_damage_path.exists() else 0}"
    ),
)
append_check(
    validation_rows,
    "final_event_summary_exists",
    final_event_summary_path.exists() and final_event_summary_path.stat().st_size > 0,
    (
        f"path={final_event_summary_path}; "
        f"bytes={final_event_summary_path.stat().st_size if final_event_summary_path.exists() else 0}"
    ),
)

state_total_rows: list[dict[str, Any]] = []
for prefix, specification in COMPONENTS.items():
    for index, state in enumerate(DAMAGE_STATES):
        state_total_rows.append(
            {
                "component": specification["label"],
                "component_prefix": prefix,
                "damage_state": state,
                "damage_state_index": index,
                "sampled_building_event_rows": int(
                    sampled_state_counts[prefix][index]
                ),
                "sampled_proportion": float(sampled_proportions[prefix][index]),
                "analytical_expected_building_event_rows": float(
                    expected_state_counts[prefix][index]
                ),
                "analytical_expected_proportion": float(
                    expected_proportions[prefix][index]
                ),
                "sampled_minus_expected_proportion": float(
                    sampled_proportions[prefix][index]
                    - expected_proportions[prefix][index]
                ),
            }
        )
state_totals = pd.DataFrame(state_total_rows)
state_totals.to_csv(state_totals_path, index=False)
chunk_manifest = pd.DataFrame(chunk_manifest_rows)
chunk_manifest.to_csv(chunk_manifest_path, index=False)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()
validation.to_csv(validation_path, index=False)

random_specification = {
    "hash_function": "SHA-256",
    "key_fields": ["occurrence_id", "site_id"],
    "key_format": "namespace|occurrence_id|site_id",
    "uniform_construction": "(unsigned_big_endian_first_8_bytes + 0.5) / 2^64",
    "uniform_interval": "strictly between 0 and 1",
    "row_order_invariant": True,
    "chunk_size_invariant": True,
    "streams": {
        specification["label"]: specification["namespace"]
        for specification in COMPONENTS.values()
    },
    "conditional_component_dependence": (
        "Separate deterministic component damage streams conditional on the same "
        "simulated SA0P4 demand."
    ),
    "production_nsd_nsa_uniform_correlation": uniform_correlation,
    "reuse_rule": (
        "Reuse the same occurrence-site component uniforms when comparing the "
        "baseline independent ground-motion case with future spatially correlated cases."
    ),
}
write_json(random_specification_path, random_specification)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "input_ground_motion_fields": {
        "path": str(field_path),
        "sha256": actual_field_hash,
        "rows": expected_rows,
        "occurrences": expected_occurrences,
        "sites": expected_sites,
        "im_column": structural_im_column,
        "period_s": period_s,
    },
    "damage_model": {
        "equation": "P(DS>=k|Sa)=Phi((ln(Sa)-ln(theta_k))/beta_k)",
        "states": DAMAGE_STATES,
        "components": {
            prefix: {
                "label": specification["label"],
                "random_stream_namespace": specification["namespace"],
                "sampled_state_rule": (
                    "inverse categorical CDF using one deterministic occurrence-site uniform"
                ),
                "minimum_raw_probability": global_minimum_raw_probability[prefix],
                "maximum_probability_sum_error": global_max_probability_sum_error[prefix],
            }
            for prefix, specification in COMPONENTS.items()
        },
        "production_im": "sa0p4_simulated_g",
        "period_s": EXPECTED_PERIOD_S,
        "conditional_component_dependence": (
            "Separate deterministic damage streams conditional on SA0P4."
        ),
    },
    "production": {
        "rows": total_processed_rows,
        "occurrences": len(seen_occurrences),
        "sites": expected_sites,
        "chunks": len(chunk_paths),
        "occurrences_per_chunk": OCCURRENCES_PER_CHUNK,
        "source_row_counts": dict(sorted(source_row_counts.items())),
        "design_level_row_counts": dict(sorted(design_level_row_counts.items())),
    },
    "uniform_diagnostics": {
        prefix: {
            "mean": uniform_means[prefix],
            "variance": uniform_variances[prefix],
            "target_mean": 0.5,
            "target_variance": 1.0 / 12.0,
        }
        for prefix in COMPONENTS
    }
    | {
        "nsd_nsa_correlation": uniform_correlation,
        "correlation_tolerance": production_stream_correlation_tolerance,
        "uniform_mean_tolerance": uniform_mean_tolerance,
        "uniform_variance_tolerance": uniform_variance_tolerance,
        "sampled_expected_proportion_tolerance": sampled_expected_proportion_tolerance,
    },
    "state_totals": {
        prefix: {
            state: {
                "sampled_rows": int(sampled_state_counts[prefix][index]),
                "sampled_proportion": float(sampled_proportions[prefix][index]),
                "analytical_expected_rows": float(
                    expected_state_counts[prefix][index]
                ),
                "analytical_expected_proportion": float(
                    expected_proportions[prefix][index]
                ),
            }
            for index, state in enumerate(DAMAGE_STATES)
        }
        for prefix in COMPONENTS
    },
    "control_reproduction": {
        "rows": int(len(control_compare)),
        "maximum_numeric_difference": maximum_control_difference,
        "sampled_states_exact": {
            prefix: bool(
                np.array_equal(
                    pd.to_numeric(
                        control_compare[
                            f"{specification['sample_column']}_cell8"
                        ],
                        errors="raise",
                    ).to_numpy(dtype=np.int8),
                    pd.to_numeric(
                        control_compare[
                            f"{specification['sample_column']}_cell9"
                        ],
                        errors="raise",
                    ).to_numpy(dtype=np.int8),
                )
            )
            for prefix, specification in COMPONENTS.items()
        },
    },
    "outputs": {
        "full_nonstructural_damage_states": {
            "path": str(final_damage_path),
            "sha256": sha256_file(final_damage_path),
            "size_bytes": int(final_damage_path.stat().st_size),
            "rows": expected_rows,
            "row_granularity": (
                "one annual-catalog occurrence and one portfolio building"
            ),
        },
        "event_summary": {
            "path": str(final_event_summary_path),
            "sha256": sha256_file(final_event_summary_path),
            "size_bytes": int(final_event_summary_path.stat().st_size),
            "rows": int(len(event_summary)),
        },
        "state_totals": str(state_totals_path),
        "chunk_manifest": str(chunk_manifest_path),
        "validation": str(validation_path),
        "random_stream_specification": str(random_specification_path),
        "summary": str(summary_path),
    },
    "next_cell": (
        "Cell 10: assign HAZUS occupancy-specific drift-sensitive and acceleration-sensitive "
        "repair-cost ratios, then calculate sampled and analytical expected nonstructural "
        "ground-up losses for every occurrence-building row."
    ),
}
write_json(summary_path, summary)

print()
print("=" * 78)
print("NOTEBOOK 5 CELL 9 FULL NONSTRUCTURAL DAMAGE SAMPLING COMPLETE")
print("=" * 78)
print(f"Production damage rows:            {total_processed_rows:,}")
print(f"Catalog occurrences:               {len(seen_occurrences):,}")
print(f"Portfolio buildings:               {expected_sites:,}")
print(f"Production chunks:                 {len(chunk_paths):,}")
print(f"NSD damage-uniform mean:           {uniform_means['nsd']:.6f}")
print(f"NSD damage-uniform variance:       {uniform_variances['nsd']:.6f}")
print(f"NSA damage-uniform mean:           {uniform_means['nsa']:.6f}")
print(f"NSA damage-uniform variance:       {uniform_variances['nsa']:.6f}")
print(f"NSD-NSA stream correlation:        {uniform_correlation:.6f}")
print(
    "Max NSD sampled-expected diff.:  "
    f"{maximum_sampled_expected_difference['nsd']:.6f}"
)
print(
    "Max NSA sampled-expected diff.:  "
    f"{maximum_sampled_expected_difference['nsa']:.6f}"
)
print(f"Max Cell 8 control difference:     {maximum_control_difference:.3e}")
print(
    "Critical validation checks:      "
    f"{int(validation['severity'].eq('critical').sum()):,}"
)
print(f"Critical failures:                  {len(critical_failures):,}")
print(f"Warnings requiring review:          {len(warning_failures):,}")
print()
print("Full nonstructural damage states:")
print(f"  {final_damage_path}")
print("Occurrence-level damage summary:")
print(f"  {final_event_summary_path}")
print("Validation:")
print(f"  {validation_path}")
print("Summary:")
print(f"  {summary_path}")
print()
print("Next: calculate sampled and analytical expected nonstructural ground-up loss.")

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 9 failed one or more critical checks. Review: "
        f"{validation_path}"
    )


FULL ANNUAL-CATALOG NONSTRUCTURAL DAMAGE-STATE SAMPLING
Ground-motion rows:       4,996,100
Catalog occurrences:      10,630
Portfolio buildings:      470
Occurrences per chunk:    250
Production chunks:        43
Damage IM:                exact SA0P4
Drift damage namespace:   notebook5_nonstructural_drift_damage_v1
Acceleration namespace:   notebook5_nonstructural_acceleration_damage_v1

------------------------------------------------------------------------------
NONSTRUCTURAL DAMAGE CHUNK 1 OF 43 [ID 0000]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
NONSTRUCTURAL DAMAGE CHUNK 2 OF 43 [ID 0001]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
NONSTRUCTURAL DAMAGE CHUNK 3 OF 43 [ID 0002]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
NONSTRUCTURAL DAMAG

In [21]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

DAMAGE_STATES = ["None", "Slight", "Moderate", "Extensive", "Complete"]
COMPONENTS = {
    "nsd": {
        "label": "nonstructural_drift_sensitive",
        "repair_role": "nsd_repair_ratios",
        "sample_column": "sampled_nsd_damage_state",
        "probability_columns": [
            "nsd_p_state_none",
            "nsd_p_state_slight",
            "nsd_p_state_moderate",
            "nsd_p_state_extensive",
            "nsd_p_state_complete",
        ],
    },
    "nsa": {
        "label": "nonstructural_acceleration_sensitive",
        "repair_role": "nsa_repair_ratios",
        "sample_column": "sampled_nsa_damage_state",
        "probability_columns": [
            "nsa_p_state_none",
            "nsa_p_state_slight",
            "nsa_p_state_moderate",
            "nsa_p_state_extensive",
            "nsa_p_state_complete",
        ],
    },
}
EXPECTED_SITES = 470
DOLLAR_YEAR = 2022
PIPELINE_VERSION = "notebook5_cell10_nonstructural_ground_up_loss_v1"
NUMERIC_TOLERANCE = 1e-10
LOSS_RECONCILIATION_WARNING_TOLERANCE = 0.05


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_9_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_occupancy(value: Any) -> str:
    text = normalize_text(value).upper().replace(" ", "")
    if not text:
        return ""
    if text.startswith("RES1-"):
        return "RES1"
    return text.split("-", 1)[0] if text.startswith("RES1") else text


def clean_excel_table(path: Path, sheet_name: str | int = 0) -> pd.DataFrame:
    table = pd.read_excel(path, sheet_name=sheet_name)
    table = table.dropna(axis=1, how="all").dropna(axis=0, how="all")
    table.columns = [normalize_text(column) for column in table.columns]
    return table.reset_index(drop=True)


def ratio_columns(prefix: str) -> list[str]:
    return [f"{prefix}_ratio_{state.lower()}" for state in DAMAGE_STATES]


def expand_repair_ratios(path: Path, prefix: str) -> pd.DataFrame:
    raw = clean_excel_table(path)
    required = {"occ_type", "DS_0", "DS_1", "DS_2", "DS_3"}
    missing = sorted(required.difference(raw.columns))
    if missing:
        raise ValueError(f"{prefix.upper()} repair-cost table is missing columns: {missing}")

    source_columns = ["DS_0", "DS_1", "DS_2", "DS_3"]
    for column in source_columns:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")
    if raw[source_columns].isna().any().any():
        raise ValueError(f"{prefix.upper()} repair-cost table contains nonnumeric ratios.")
    if (raw[source_columns] < 0).any().any():
        raise ValueError(f"{prefix.upper()} repair-cost table contains negative ratios.")

    rows: list[dict[str, Any]] = []
    for _, row in raw.iterrows():
        original = normalize_text(row["occ_type"]).upper().replace(" ", "")
        normalized = normalize_occupancy(original)
        keys = [normalized]
        match = re.fullmatch(r"([A-Z]+\d+)([A-Z])-([A-Z])", original)
        if match:
            base, start, end = match.groups()
            keys = [base + chr(code) for code in range(ord(start), ord(end) + 1)]

        for occupancy in keys:
            rows.append(
                {
                    "occ_type": occupancy,
                    f"{prefix}_source_occ_type": original,
                    f"{prefix}_ratio_none": 0.0,
                    f"{prefix}_ratio_slight": float(row["DS_0"]) / 100.0,
                    f"{prefix}_ratio_moderate": float(row["DS_1"]) / 100.0,
                    f"{prefix}_ratio_extensive": float(row["DS_2"]) / 100.0,
                    f"{prefix}_ratio_complete": float(row["DS_3"]) / 100.0,
                }
            )

    expanded = pd.DataFrame(rows)
    keep_rows: list[pd.Series] = []
    conflicts: list[str] = []
    columns = ratio_columns(prefix)
    for occupancy, group in expanded.groupby("occ_type", sort=True):
        if len(group[columns].drop_duplicates()) > 1:
            conflicts.append(str(occupancy))
        keep_rows.append(group.iloc[-1])
    if conflicts:
        raise ValueError(
            f"Conflicting {prefix.upper()} repair ratios for occupancy classes: "
            f"{sorted(conflicts)}"
        )
    return pd.DataFrame(keep_rows).sort_values("occ_type").reset_index(drop=True)


def reference_record(summary: dict[str, Any], role: str) -> dict[str, Any]:
    matches = [
        row
        for row in summary.get("hazus_reference_files", [])
        if str(row.get("role")) == role
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one HAZUS reference record for role {role!r}; "
            f"found {len(matches)}."
        )
    return matches[0]


def stable_frame_hash(frame: pd.DataFrame, columns: list[str]) -> str:
    digest = hashlib.sha256()
    subset = frame[columns].copy()
    for column in columns:
        values = subset[column]
        if pd.api.types.is_float_dtype(values):
            encoded = values.map(lambda value: format(float(value), ".17g"))
        else:
            encoded = values.astype(str)
        for value in encoded:
            digest.update(value.encode("utf-8"))
            digest.update(b"\x1f")
        digest.update(b"\x1e")
    return digest.hexdigest()


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    buffer = io.StringIO()
    frame.to_csv(buffer, index=False, lineterminator="\n")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as compressed:
            compressed.write(buffer.getvalue().encode("utf-8"))
    temporary.replace(path)


def concatenate_gzip_csv_files(chunk_paths: list[Path], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    expected_header: bytes | None = None
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw_output, mtime=0) as target:
            for chunk_path in chunk_paths:
                with gzip.open(chunk_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        target.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            f"Chunk header mismatch while concatenating {chunk_path}."
                        )
                    while block := source.read(8 * 1024 * 1024):
                        target.write(block)
    temporary.replace(output_path)


def summarize_occurrences(frame: pd.DataFrame) -> pd.DataFrame:
    aggregations: dict[str, tuple[str, str]] = {
        "buildings": ("site_id", "size"),
        "sampled_nsd_ground_up_loss_2022_usd": (
            "sampled_nsd_ground_up_loss_2022_usd",
            "sum",
        ),
        "analytical_expected_nsd_ground_up_loss_2022_usd": (
            "analytical_expected_nsd_ground_up_loss_2022_usd",
            "sum",
        ),
        "sampled_nsa_ground_up_loss_2022_usd": (
            "sampled_nsa_ground_up_loss_2022_usd",
            "sum",
        ),
        "analytical_expected_nsa_ground_up_loss_2022_usd": (
            "analytical_expected_nsa_ground_up_loss_2022_usd",
            "sum",
        ),
        "sampled_total_nonstructural_ground_up_loss_2022_usd": (
            "sampled_total_nonstructural_ground_up_loss_2022_usd",
            "sum",
        ),
        "analytical_expected_total_nonstructural_ground_up_loss_2022_usd": (
            "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
            "sum",
        ),
        "buildings_with_sampled_nsd_damage": ("sampled_nsd_damage_state", lambda x: int((x > 0).sum())),
        "buildings_with_sampled_nsa_damage": ("sampled_nsa_damage_state", lambda x: int((x > 0).sum())),
        "maximum_sampled_nsd_damage_state": ("sampled_nsd_damage_state", "max"),
        "maximum_sampled_nsa_damage_state": ("sampled_nsa_damage_state", "max"),
    }
    event_fields = [
        "catalog_year",
        "catalog_event_id",
        "rupture_ordinal",
        "rupture_template_event_id",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
    ]
    group_columns = [column for column in event_fields if column in frame.columns]
    return (
        frame.groupby(group_columns, sort=False, as_index=False)
        .agg(**aggregations)
        .reset_index(drop=True)
    )


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_nonstructural_loss"
CHUNK_DIR = OUTPUT_DIR / "chunks"
WORK_DIR = OUTPUT_DIR / "work"
MARKER_DIR = WORK_DIR / "markers"
CHUNK_VALIDATION_DIR = WORK_DIR / "chunk_validations"

for directory in [METADATA_DIR, PARAMETER_DIR, OUTPUT_DIR, CHUNK_DIR, MARKER_DIR, CHUNK_VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CELL1_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_1_summary.json"
CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_1_validation.csv"
CELL5_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_5_summary.json"
CELL5_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_5_validation.csv"
CELL9_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_9_summary.json"
CELL9_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_9_validation.csv"
CELL9_CHUNK_MANIFEST_PATH = METADATA_DIR / "notebook_5_cell_9_chunk_manifest.csv"
BUILDING_PARAMETER_PATH = (
    PARAMETER_DIR / "seaside_w2_replacement_values_and_nonstructural_ratios.csv"
)
RATIO_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_10_nonstructural_ratio_summary.csv"
LOSS_RECONCILIATION_PATH = METADATA_DIR / "notebook_5_cell_10_loss_reconciliation.csv"
CELL10_CHUNK_MANIFEST_PATH = METADATA_DIR / "notebook_5_cell_10_chunk_manifest.csv"
CELL10_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_10_validation.csv"
CELL10_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_10_summary.json"
FINAL_BUILDING_LOSS_PATH = OUTPUT_DIR / "full_nonstructural_ground_up_loss.csv.gz"
FINAL_EVENT_LOSS_PATH = OUTPUT_DIR / "nonstructural_ground_up_loss_event_summary.csv.gz"

required_paths = [
    CELL1_SUMMARY_PATH,
    CELL1_VALIDATION_PATH,
    CELL5_SUMMARY_PATH,
    CELL5_VALIDATION_PATH,
    CELL9_SUMMARY_PATH,
    CELL9_VALIDATION_PATH,
    CELL9_CHUNK_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required prior-cell outputs are missing: {missing_paths}")

validation_rows: list[dict[str, Any]] = []
cell1_summary = load_json(CELL1_SUMMARY_PATH)
cell5_summary = load_json(CELL5_SUMMARY_PATH)
cell9_summary = load_json(CELL9_SUMMARY_PATH)
cell1_validation = pd.read_csv(CELL1_VALIDATION_PATH)
cell5_validation = pd.read_csv(CELL5_VALIDATION_PATH)
cell9_validation = pd.read_csv(CELL9_VALIDATION_PATH)
cell9_manifest = pd.read_csv(CELL9_CHUNK_MANIFEST_PATH)

for cell_number, summary, validation in [
    (1, cell1_summary, cell1_validation),
    (5, cell5_summary, cell5_validation),
    (9, cell9_summary, cell9_validation),
]:
    append_check(
        validation_rows,
        f"cell{cell_number}_critical_checks_passed",
        summary.get("all_critical_checks_passed") is True,
        f"all_critical_checks_passed={summary.get('all_critical_checks_passed')}",
    )
    parsed = parse_bool_series(validation["passed"])
    unresolved_critical = validation.loc[
        validation["severity"].astype(str).str.lower().eq("critical") & ~parsed
    ]
    append_check(
        validation_rows,
        f"cell{cell_number}_has_no_unresolved_critical_checks",
        unresolved_critical.empty,
        f"unresolved_critical={len(unresolved_critical)}",
    )

expected_rows = int(cell9_summary["production"]["rows"])
expected_occurrences = int(cell9_summary["production"]["occurrences"])
expected_sites = int(cell9_summary["production"]["sites"])
expected_chunks = int(cell9_summary["production"]["chunks"])
declared_catalog_years = int(cell1_summary["annual_catalog"]["declared_duration_years"])

append_check(
    validation_rows,
    "portfolio_has_470_buildings",
    expected_sites == EXPECTED_SITES,
    f"sites={expected_sites}",
)
append_check(
    validation_rows,
    "rows_match_occurrences_times_sites",
    expected_rows == expected_occurrences * expected_sites,
    f"rows={expected_rows}; occurrences={expected_occurrences}; sites={expected_sites}",
)
append_check(
    validation_rows,
    "catalog_duration_positive",
    declared_catalog_years > 0,
    f"years={declared_catalog_years}",
)

cell9_damage_info = cell9_summary["outputs"]["full_nonstructural_damage_states"]
cell9_damage_path = resolve_recorded_path(PROJECT_ROOT, str(cell9_damage_info["path"]))
append_check(
    validation_rows,
    "cell9_final_damage_file_exists",
    cell9_damage_path.exists(),
    str(cell9_damage_path),
)
append_check(
    validation_rows,
    "cell9_final_damage_hash_matches_summary",
    sha256_file(cell9_damage_path) == str(cell9_damage_info["sha256"]),
    "Final Cell 9 nonstructural damage file hash must match its accepted summary.",
)

cell5_building_info = cell5_summary["outputs"]["building_value_and_ratio_table"]
cell5_building_path = resolve_recorded_path(PROJECT_ROOT, str(cell5_building_info["path"]))
append_check(
    validation_rows,
    "cell5_building_value_table_exists",
    cell5_building_path.exists(),
    str(cell5_building_path),
)
append_check(
    validation_rows,
    "cell5_building_value_hash_matches_summary",
    sha256_file(cell5_building_path) == str(cell5_building_info["sha256"]),
    "Cell 5 building replacement-value table hash must match its accepted summary.",
)

building_values = pd.read_csv(cell5_building_path)
required_building_columns = {
    "site_id",
    "occ_type",
    "building_replacement_value_2022_usd",
    "structural_ratio_complete",
}
missing_building_columns = sorted(required_building_columns.difference(building_values.columns))
append_check(
    validation_rows,
    "cell5_building_value_columns_present",
    not missing_building_columns,
    f"missing={missing_building_columns}",
)
if missing_building_columns:
    raise KeyError(f"Cell 5 building-value table is missing columns: {missing_building_columns}")

building_values["site_id"] = building_values["site_id"].astype(str)
building_values["occ_type"] = building_values["occ_type"].map(normalize_occupancy)
building_values["building_replacement_value_2022_usd"] = pd.to_numeric(
    building_values["building_replacement_value_2022_usd"], errors="raise"
)
append_check(
    validation_rows,
    "cell5_building_value_sites_unique_complete",
    len(building_values) == expected_sites
    and building_values["site_id"].is_unique
    and building_values["site_id"].notna().all(),
    f"rows={len(building_values)}; duplicates={int(building_values['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "replacement_values_positive_finite",
    np.isfinite(building_values["building_replacement_value_2022_usd"]).all()
    and (building_values["building_replacement_value_2022_usd"] > 0.0).all(),
    (
        f"minimum={building_values['building_replacement_value_2022_usd'].min():.2f}; "
        f"maximum={building_values['building_replacement_value_2022_usd'].max():.2f}"
    ),
)

reference_paths: dict[str, Path] = {}
reference_hashes: dict[str, str] = {}
for prefix, specification in COMPONENTS.items():
    role = specification["repair_role"]
    record = reference_record(cell1_summary, role)
    path = resolve_recorded_path(PROJECT_ROOT, str(record["path"]))
    actual_hash = sha256_file(path)
    reference_paths[prefix] = path
    reference_hashes[prefix] = actual_hash
    append_check(
        validation_rows,
        f"{prefix}_repair_ratio_hash_matches_cell1",
        actual_hash == str(record["sha256"]),
        f"calculated={actual_hash}; recorded={record['sha256']}",
    )

nsd_ratios = expand_repair_ratios(reference_paths["nsd"], "nsd")
nsa_ratios = expand_repair_ratios(reference_paths["nsa"], "nsa")

building_parameters = building_values.merge(
    nsd_ratios, on="occ_type", how="left", validate="many_to_one"
).merge(
    nsa_ratios, on="occ_type", how="left", validate="many_to_one"
)

for prefix in COMPONENTS:
    columns = ratio_columns(prefix)
    missing_sites = building_parameters.loc[
        building_parameters[columns].isna().any(axis=1), "site_id"
    ].tolist()
    append_check(
        validation_rows,
        f"{prefix}_repair_ratios_assigned_to_every_building",
        not missing_sites,
        f"missing_sites={missing_sites[:20]}",
    )
    matrix = building_parameters[columns].to_numpy(dtype=np.float64)
    append_check(
        validation_rows,
        f"{prefix}_repair_ratios_finite_and_bounded",
        np.isfinite(matrix).all()
        and np.all((matrix >= 0.0) & (matrix <= 1.0)),
        f"minimum={np.nanmin(matrix):.6f}; maximum={np.nanmax(matrix):.6f}",
    )
    append_check(
        validation_rows,
        f"{prefix}_repair_ratios_nondecreasing_by_state",
        np.all(np.diff(matrix, axis=1) >= -NUMERIC_TOLERANCE),
        "None <= Slight <= Moderate <= Extensive <= Complete for every building.",
    )

complete_component_total = (
    pd.to_numeric(building_parameters["structural_ratio_complete"], errors="raise")
    + building_parameters["nsd_ratio_complete"]
    + building_parameters["nsa_ratio_complete"]
)
append_check(
    validation_rows,
    "complete_component_ratios_sum_to_one",
    np.allclose(complete_component_total.to_numpy(), 1.0, atol=1e-12, rtol=0.0),
    (
        f"minimum={complete_component_total.min():.12f}; "
        f"maximum={complete_component_total.max():.12f}"
    ),
)

building_parameters["nsd_repair_ratio_source_sha256"] = reference_hashes["nsd"]
building_parameters["nsa_repair_ratio_source_sha256"] = reference_hashes["nsa"]
building_parameters.to_csv(BUILDING_PARAMETER_PATH, index=False)

ratio_summary_rows: list[dict[str, Any]] = []
for occupancy, group in building_parameters.groupby("occ_type", sort=True):
    row: dict[str, Any] = {
        "occ_type": occupancy,
        "buildings": int(len(group)),
        "total_replacement_value_2022_usd": float(
            group["building_replacement_value_2022_usd"].sum()
        ),
    }
    for prefix in COMPONENTS:
        for state in DAMAGE_STATES:
            column = f"{prefix}_ratio_{state.lower()}"
            values = group[column].drop_duplicates()
            if len(values) != 1:
                raise RuntimeError(
                    f"Occupancy {occupancy} has multiple {prefix.upper()} ratios in {column}."
                )
            row[column] = float(values.iloc[0])
    ratio_summary_rows.append(row)
pd.DataFrame(ratio_summary_rows).to_csv(RATIO_SUMMARY_PATH, index=False)

required_manifest_columns = {
    "chunk_id",
    "rows",
    "occurrences",
    "output_path",
    "output_sha256",
}
missing_manifest_columns = sorted(required_manifest_columns.difference(cell9_manifest.columns))
append_check(
    validation_rows,
    "cell9_chunk_manifest_columns_present",
    not missing_manifest_columns,
    f"missing={missing_manifest_columns}",
)
if missing_manifest_columns:
    raise KeyError(f"Cell 9 chunk manifest is missing columns: {missing_manifest_columns}")

cell9_manifest["chunk_id"] = cell9_manifest["chunk_id"].astype(str).str.zfill(4)
cell9_manifest = cell9_manifest.sort_values("chunk_id").reset_index(drop=True)
append_check(
    validation_rows,
    "cell9_chunk_manifest_rows_match_summary",
    len(cell9_manifest) == expected_chunks,
    f"manifest={len(cell9_manifest)}; summary={expected_chunks}",
)

basis_payload = {
    "pipeline_version": PIPELINE_VERSION,
    "cell1_summary_sha256": sha256_file(CELL1_SUMMARY_PATH),
    "cell5_summary_sha256": sha256_file(CELL5_SUMMARY_PATH),
    "cell9_summary_sha256": sha256_file(CELL9_SUMMARY_PATH),
    "cell9_chunk_manifest_sha256": sha256_file(CELL9_CHUNK_MANIFEST_PATH),
    "cell5_building_value_sha256": sha256_file(cell5_building_path),
    "nsd_repair_ratio_sha256": reference_hashes["nsd"],
    "nsa_repair_ratio_sha256": reference_hashes["nsa"],
    "dollar_year": DOLLAR_YEAR,
}
basis_hash = hashlib.sha256(
    json.dumps(basis_payload, sort_keys=True).encode("utf-8")
).hexdigest()

lookup_columns = [
    "site_id",
    "occ_type",
    "building_replacement_value_2022_usd",
    "structural_ratio_complete",
    *ratio_columns("nsd"),
    *ratio_columns("nsa"),
]
parameter_lookup = building_parameters[lookup_columns].copy()

chunk_paths: list[Path] = []
chunk_manifest_rows: list[dict[str, Any]] = []
event_summary_frames: list[pd.DataFrame] = []
total_rows = 0
seen_occurrences: set[str] = set()
source_sampled_totals = Counter()
component_sampled_total = {prefix: 0.0 for prefix in COMPONENTS}
component_expected_total = {prefix: 0.0 for prefix in COMPONENTS}
global_max_sampled_equation_error = 0.0
global_max_expected_equation_error = 0.0
global_max_total_equation_error = 0.0
global_max_sampled_total_ratio = 0.0
global_max_expected_total_ratio = 0.0

print("=" * 78)
print("FULL ANNUAL-CATALOG NONSTRUCTURAL GROUND-UP LOSS")
print("=" * 78)
print(f"Nonstructural damage rows:     {expected_rows:,}")
print(f"Catalog occurrences:           {expected_occurrences:,}")
print(f"Portfolio buildings:           {expected_sites:,}")
print(f"Cell 9 production chunks:      {len(cell9_manifest):,}")
print(
    "Portfolio replacement value:   "
    f"${building_parameters['building_replacement_value_2022_usd'].sum():,.0f} "
    f"({DOLLAR_YEAR} USD)"
)
print("Loss scope:                     drift- and acceleration-sensitive nonstructural repair")

for manifest_index, manifest_row in cell9_manifest.iterrows():
    chunk_id = str(manifest_row["chunk_id"]).zfill(4)
    print()
    print("-" * 78)
    print(
        f"NONSTRUCTURAL LOSS CHUNK {manifest_index + 1} OF {len(cell9_manifest)} "
        f"[ID {chunk_id}]"
    )

    damage_chunk_path = resolve_recorded_path(
        PROJECT_ROOT, str(manifest_row["output_path"])
    )
    actual_input_hash = sha256_file(damage_chunk_path)
    if actual_input_hash != str(manifest_row["output_sha256"]):
        raise RuntimeError(
            f"Cell 9 chunk {chunk_id} hash mismatch: calculated={actual_input_hash}; "
            f"recorded={manifest_row['output_sha256']}"
        )

    damage = pd.read_csv(damage_chunk_path)
    required_damage_columns = {
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
        "site_ordinal",
        "site_id",
        "sa0p4_simulated_g",
        "occ_type",
        "sampled_nsd_damage_state",
        "sampled_nsa_damage_state",
        *COMPONENTS["nsd"]["probability_columns"],
        *COMPONENTS["nsa"]["probability_columns"],
    }
    missing_damage_columns = sorted(required_damage_columns.difference(damage.columns))
    if missing_damage_columns:
        raise KeyError(f"Cell 9 chunk {chunk_id} is missing columns: {missing_damage_columns}")

    damage["site_id"] = damage["site_id"].astype(str)
    damage["occurrence_id"] = damage["occurrence_id"].astype(str)
    damage["occ_type"] = damage["occ_type"].map(normalize_occupancy)
    damage["source_type"] = damage["source_type"].astype(str).str.strip().str.upper()
    for prefix, specification in COMPONENTS.items():
        damage[specification["sample_column"]] = pd.to_numeric(
            damage[specification["sample_column"]], errors="raise"
        ).astype(np.int8)
        for column in specification["probability_columns"]:
            damage[column] = pd.to_numeric(damage[column], errors="raise")

    input_hash_columns = [
        "occurrence_ordinal",
        "occurrence_id",
        "site_ordinal",
        "site_id",
        "sampled_nsd_damage_state",
        "sampled_nsa_damage_state",
        *COMPONENTS["nsd"]["probability_columns"],
        *COMPONENTS["nsa"]["probability_columns"],
    ]
    input_frame_hash = stable_frame_hash(damage, input_hash_columns)
    chunk_output_path = CHUNK_DIR / f"nonstructural_loss_chunk_{chunk_id}.csv.gz"
    marker_path = MARKER_DIR / f"nonstructural_loss_chunk_{chunk_id}.json"
    chunk_validation_path = (
        CHUNK_VALIDATION_DIR / f"nonstructural_loss_chunk_{chunk_id}_validation.csv"
    )

    marker_valid = False
    marker: dict[str, Any] = {}
    if marker_path.exists() and chunk_output_path.exists() and chunk_validation_path.exists():
        try:
            marker = load_json(marker_path)
            marker_valid = (
                marker.get("basis_hash") == basis_hash
                and marker.get("cell9_chunk_sha256") == actual_input_hash
                and marker.get("input_frame_hash") == input_frame_hash
                and int(marker.get("rows", -1)) == len(damage)
                and marker.get("output_sha256") == sha256_file(chunk_output_path)
            )
        except Exception:
            marker_valid = False

    if marker_valid:
        output = pd.read_csv(chunk_output_path)
        print(f"Reused validated loss chunk with {len(output):,} rows.")
    else:
        chunk_validation_rows: list[dict[str, Any]] = []
        merged = damage.merge(
            parameter_lookup,
            on="site_id",
            how="left",
            suffixes=("", "_parameter"),
            validate="many_to_one",
        )
        missing_parameters = int(
            merged["building_replacement_value_2022_usd"].isna().sum()
        )
        append_check(
            chunk_validation_rows,
            "all_sites_have_nonstructural_loss_parameters",
            missing_parameters == 0,
            f"missing_rows={missing_parameters}",
        )
        if missing_parameters:
            raise RuntimeError(
                f"Chunk {chunk_id} has {missing_parameters} rows without loss parameters."
            )

        occupancy_match = merged["occ_type"].eq(merged["occ_type_parameter"])
        append_check(
            chunk_validation_rows,
            "damage_and_parameter_occupancies_match",
            occupancy_match.all(),
            f"mismatches={int((~occupancy_match).sum())}",
        )
        if not occupancy_match.all():
            raise RuntimeError(f"Chunk {chunk_id} has occupancy mismatches after site join.")
        merged = merged.drop(columns=["occ_type_parameter"])

        replacement_values = merged[
            "building_replacement_value_2022_usd"
        ].to_numpy(dtype=np.float64)

        for prefix, specification in COMPONENTS.items():
            states = merged[specification["sample_column"]].to_numpy(dtype=np.int8)
            state_valid = np.all((states >= 0) & (states <= 4))
            append_check(
                chunk_validation_rows,
                f"{prefix}_sampled_damage_states_in_range",
                state_valid,
                f"minimum={states.min()}; maximum={states.max()}",
            )
            if not state_valid:
                raise RuntimeError(f"Chunk {chunk_id} has invalid {prefix.upper()} states.")

            probability_matrix = merged[
                specification["probability_columns"]
            ].to_numpy(dtype=np.float64)
            ratio_matrix = merged[ratio_columns(prefix)].to_numpy(dtype=np.float64)
            probability_error = float(
                np.max(np.abs(probability_matrix.sum(axis=1) - 1.0))
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_probabilities_finite_nonnegative",
                np.isfinite(probability_matrix).all()
                and probability_matrix.min() >= -1e-12,
                f"minimum={probability_matrix.min():.3e}",
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_probabilities_sum_to_one",
                probability_error <= 2e-12,
                f"maximum_error={probability_error:.3e}",
            )

            sampled_ratios = ratio_matrix[np.arange(len(merged)), states]
            expected_ratios = np.sum(probability_matrix * ratio_matrix, axis=1)
            merged[f"sampled_{prefix}_repair_ratio"] = sampled_ratios
            merged[f"analytical_expected_{prefix}_repair_ratio"] = expected_ratios
            merged[f"sampled_{prefix}_ground_up_loss_2022_usd"] = (
                replacement_values * sampled_ratios
            )
            merged[f"analytical_expected_{prefix}_ground_up_loss_2022_usd"] = (
                replacement_values * expected_ratios
            )

            sampled_ratio_check = ratio_matrix[np.arange(len(merged)), states]
            expected_ratio_check = np.sum(probability_matrix * ratio_matrix, axis=1)
            sampled_error = float(np.max(np.abs(sampled_ratios - sampled_ratio_check)))
            expected_error = float(np.max(np.abs(expected_ratios - expected_ratio_check)))
            append_check(
                chunk_validation_rows,
                f"{prefix}_sampled_ratio_equation_exact",
                sampled_error <= NUMERIC_TOLERANCE,
                f"maximum_error={sampled_error:.3e}",
            )
            append_check(
                chunk_validation_rows,
                f"{prefix}_expected_ratio_equation_exact",
                expected_error <= NUMERIC_TOLERANCE,
                f"maximum_error={expected_error:.3e}",
            )

        merged["sampled_total_nonstructural_repair_ratio"] = (
            merged["sampled_nsd_repair_ratio"] + merged["sampled_nsa_repair_ratio"]
        )
        merged["analytical_expected_total_nonstructural_repair_ratio"] = (
            merged["analytical_expected_nsd_repair_ratio"]
            + merged["analytical_expected_nsa_repair_ratio"]
        )
        merged["sampled_total_nonstructural_ground_up_loss_2022_usd"] = (
            merged["sampled_nsd_ground_up_loss_2022_usd"]
            + merged["sampled_nsa_ground_up_loss_2022_usd"]
        )
        merged[
            "analytical_expected_total_nonstructural_ground_up_loss_2022_usd"
        ] = (
            merged["analytical_expected_nsd_ground_up_loss_2022_usd"]
            + merged["analytical_expected_nsa_ground_up_loss_2022_usd"]
        )

        sampled_total_equation_error = float(
            np.max(
                np.abs(
                    merged["sampled_total_nonstructural_ground_up_loss_2022_usd"]
                    - replacement_values
                    * merged["sampled_total_nonstructural_repair_ratio"]
                )
            )
        )
        expected_total_equation_error = float(
            np.max(
                np.abs(
                    merged[
                        "analytical_expected_total_nonstructural_ground_up_loss_2022_usd"
                    ]
                    - replacement_values
                    * merged["analytical_expected_total_nonstructural_repair_ratio"]
                )
            )
        )
        append_check(
            chunk_validation_rows,
            "sampled_total_nonstructural_loss_equation_exact",
            sampled_total_equation_error <= 1e-6,
            f"maximum_error={sampled_total_equation_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "expected_total_nonstructural_loss_equation_exact",
            expected_total_equation_error <= 1e-6,
            f"maximum_error={expected_total_equation_error:.3e}",
        )

        sampled_total_ratios = merged[
            "sampled_total_nonstructural_repair_ratio"
        ].to_numpy(dtype=np.float64)
        expected_total_ratios = merged[
            "analytical_expected_total_nonstructural_repair_ratio"
        ].to_numpy(dtype=np.float64)
        append_check(
            chunk_validation_rows,
            "sampled_total_nonstructural_ratios_bounded",
            np.isfinite(sampled_total_ratios).all()
            and np.all(sampled_total_ratios >= -NUMERIC_TOLERANCE)
            and np.all(sampled_total_ratios <= 1.0 + NUMERIC_TOLERANCE),
            f"minimum={sampled_total_ratios.min():.6f}; maximum={sampled_total_ratios.max():.6f}",
        )
        append_check(
            chunk_validation_rows,
            "expected_total_nonstructural_ratios_bounded",
            np.isfinite(expected_total_ratios).all()
            and np.all(expected_total_ratios >= -NUMERIC_TOLERANCE)
            and np.all(expected_total_ratios <= 1.0 + NUMERIC_TOLERANCE),
            f"minimum={expected_total_ratios.min():.6f}; maximum={expected_total_ratios.max():.6f}",
        )

        id_columns = [
            column
            for column in [
                "catalog_year",
                "catalog_event_id",
                "rupture_ordinal",
                "rupture_template_event_id",
                "occurrence_ordinal",
                "occurrence_id",
                "rupture_id",
                "source_type",
                "magnitude",
                "site_ordinal",
                "site_id",
                "sa0p4_simulated_g",
                "structural_type",
                "design_level",
                "occ_type",
                "year_built",
            ]
            if column in merged.columns
        ]
        output_columns = [
            *id_columns,
            "building_replacement_value_2022_usd",
            "sampled_nsd_damage_state",
            "sampled_nsa_damage_state",
            "sampled_nsd_repair_ratio",
            "analytical_expected_nsd_repair_ratio",
            "sampled_nsd_ground_up_loss_2022_usd",
            "analytical_expected_nsd_ground_up_loss_2022_usd",
            "sampled_nsa_repair_ratio",
            "analytical_expected_nsa_repair_ratio",
            "sampled_nsa_ground_up_loss_2022_usd",
            "analytical_expected_nsa_ground_up_loss_2022_usd",
            "sampled_total_nonstructural_repair_ratio",
            "analytical_expected_total_nonstructural_repair_ratio",
            "sampled_total_nonstructural_ground_up_loss_2022_usd",
            "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
        ]
        output = merged[output_columns].copy()

        append_check(
            chunk_validation_rows,
            "output_rows_match_input",
            len(output) == len(damage),
            f"output={len(output)}; input={len(damage)}",
        )
        append_check(
            chunk_validation_rows,
            "occurrence_site_pairs_unique",
            not output.duplicated(["occurrence_id", "site_id"]).any(),
            f"duplicates={int(output.duplicated(['occurrence_id', 'site_id']).sum())}",
        )
        occurrence_sizes = output.groupby("occurrence_id", sort=False).size()
        append_check(
            chunk_validation_rows,
            "occurrences_have_complete_portfolios",
            occurrence_sizes.eq(expected_sites).all(),
            (
                f"occurrences={len(occurrence_sizes)}; minimum={int(occurrence_sizes.min())}; "
                f"maximum={int(occurrence_sizes.max())}; expected={expected_sites}"
            ),
        )

        chunk_validation = pd.DataFrame(chunk_validation_rows)
        chunk_failures = chunk_validation.loc[
            chunk_validation["severity"].eq("critical") & ~chunk_validation["passed"]
        ]
        chunk_validation.to_csv(chunk_validation_path, index=False)
        if not chunk_failures.empty:
            raise RuntimeError(
                f"Nonstructural loss chunk {chunk_id} failed validation. "
                f"Review {chunk_validation_path}."
            )

        write_gzip_csv_deterministic(output, chunk_output_path)
        marker = {
            "pipeline_version": PIPELINE_VERSION,
            "basis_hash": basis_hash,
            "cell9_chunk_sha256": actual_input_hash,
            "input_frame_hash": input_frame_hash,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "output_path": str(chunk_output_path),
            "output_sha256": sha256_file(chunk_output_path),
            "validation_path": str(chunk_validation_path),
            "validation_sha256": sha256_file(chunk_validation_path),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        write_json(marker_path, marker)
        print(
            f"Calculated {len(output):,} rows for "
            f"{output['occurrence_id'].nunique():,} occurrences."
        )

    required_output_columns = {
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "source_type",
        "site_ordinal",
        "site_id",
        "building_replacement_value_2022_usd",
        "sampled_nsd_damage_state",
        "sampled_nsa_damage_state",
        "sampled_nsd_repair_ratio",
        "analytical_expected_nsd_repair_ratio",
        "sampled_nsd_ground_up_loss_2022_usd",
        "analytical_expected_nsd_ground_up_loss_2022_usd",
        "sampled_nsa_repair_ratio",
        "analytical_expected_nsa_repair_ratio",
        "sampled_nsa_ground_up_loss_2022_usd",
        "analytical_expected_nsa_ground_up_loss_2022_usd",
        "sampled_total_nonstructural_repair_ratio",
        "analytical_expected_total_nonstructural_repair_ratio",
        "sampled_total_nonstructural_ground_up_loss_2022_usd",
        "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
    }
    missing_output_columns = sorted(required_output_columns.difference(output.columns))
    if missing_output_columns:
        raise KeyError(f"Chunk {chunk_id} output is missing columns: {missing_output_columns}")

    numeric_columns = [
        "building_replacement_value_2022_usd",
        "sampled_nsd_repair_ratio",
        "analytical_expected_nsd_repair_ratio",
        "sampled_nsd_ground_up_loss_2022_usd",
        "analytical_expected_nsd_ground_up_loss_2022_usd",
        "sampled_nsa_repair_ratio",
        "analytical_expected_nsa_repair_ratio",
        "sampled_nsa_ground_up_loss_2022_usd",
        "analytical_expected_nsa_ground_up_loss_2022_usd",
        "sampled_total_nonstructural_repair_ratio",
        "analytical_expected_total_nonstructural_repair_ratio",
        "sampled_total_nonstructural_ground_up_loss_2022_usd",
        "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
    ]
    for column in numeric_columns:
        output[column] = pd.to_numeric(output[column], errors="raise")
    for column in ["sampled_nsd_damage_state", "sampled_nsa_damage_state"]:
        output[column] = pd.to_numeric(output[column], errors="raise").astype(np.int8)

    occurrence_ids = set(output["occurrence_id"].astype(str).unique().tolist())
    repeated_occurrences = occurrence_ids.intersection(seen_occurrences)
    if repeated_occurrences:
        raise RuntimeError(
            f"Chunk {chunk_id} repeats occurrences from prior chunks: "
            f"{sorted(repeated_occurrences)[:5]}"
        )
    seen_occurrences.update(occurrence_ids)

    replacement_values = output[
        "building_replacement_value_2022_usd"
    ].to_numpy(dtype=np.float64)
    for prefix in COMPONENTS:
        sampled_ratios = output[f"sampled_{prefix}_repair_ratio"].to_numpy(dtype=np.float64)
        expected_ratios = output[
            f"analytical_expected_{prefix}_repair_ratio"
        ].to_numpy(dtype=np.float64)
        sampled_losses = output[
            f"sampled_{prefix}_ground_up_loss_2022_usd"
        ].to_numpy(dtype=np.float64)
        expected_losses = output[
            f"analytical_expected_{prefix}_ground_up_loss_2022_usd"
        ].to_numpy(dtype=np.float64)
        global_max_sampled_equation_error = max(
            global_max_sampled_equation_error,
            float(np.max(np.abs(sampled_losses - replacement_values * sampled_ratios))),
        )
        global_max_expected_equation_error = max(
            global_max_expected_equation_error,
            float(np.max(np.abs(expected_losses - replacement_values * expected_ratios))),
        )
        component_sampled_total[prefix] += float(sampled_losses.sum())
        component_expected_total[prefix] += float(expected_losses.sum())

    sampled_total_losses = output[
        "sampled_total_nonstructural_ground_up_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    expected_total_losses = output[
        "analytical_expected_total_nonstructural_ground_up_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    sampled_total_ratios = output[
        "sampled_total_nonstructural_repair_ratio"
    ].to_numpy(dtype=np.float64)
    expected_total_ratios = output[
        "analytical_expected_total_nonstructural_repair_ratio"
    ].to_numpy(dtype=np.float64)
    global_max_total_equation_error = max(
        global_max_total_equation_error,
        float(np.max(np.abs(sampled_total_losses - replacement_values * sampled_total_ratios))),
        float(np.max(np.abs(expected_total_losses - replacement_values * expected_total_ratios))),
    )
    global_max_sampled_total_ratio = max(
        global_max_sampled_total_ratio, float(sampled_total_ratios.max())
    )
    global_max_expected_total_ratio = max(
        global_max_expected_total_ratio, float(expected_total_ratios.max())
    )

    source_totals = output.groupby("source_type", sort=False)[
        "sampled_total_nonstructural_ground_up_loss_2022_usd"
    ].sum()
    for source_type, value in source_totals.items():
        source_sampled_totals[str(source_type)] += float(value)

    event_summary_frames.append(summarize_occurrences(output))
    chunk_paths.append(chunk_output_path)
    total_rows += len(output)
    chunk_manifest_rows.append(
        {
            "chunk_id": chunk_id,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_occurrence_ordinal": int(output["occurrence_ordinal"].min()),
            "maximum_occurrence_ordinal": int(output["occurrence_ordinal"].max()),
            "output_path": str(chunk_output_path),
            "output_size_bytes": int(chunk_output_path.stat().st_size),
            "output_sha256": sha256_file(chunk_output_path),
            "marker_path": str(marker_path),
            "validation_path": str(chunk_validation_path),
            "reused": bool(marker_valid),
        }
    )

if len(chunk_paths) != expected_chunks:
    raise RuntimeError(f"Processed {len(chunk_paths)} chunks but expected {expected_chunks}.")

concatenate_gzip_csv_files(chunk_paths, FINAL_BUILDING_LOSS_PATH)
event_summary = pd.concat(event_summary_frames, ignore_index=True)
event_summary = event_summary.sort_values("occurrence_ordinal").reset_index(drop=True)
write_gzip_csv_deterministic(event_summary, FINAL_EVENT_LOSS_PATH)

chunk_manifest = pd.DataFrame(chunk_manifest_rows)
chunk_manifest.to_csv(CELL10_CHUNK_MANIFEST_PATH, index=False)

sampled_total = float(sum(component_sampled_total.values()))
expected_total = float(sum(component_expected_total.values()))
relative_difference = (
    abs(sampled_total - expected_total) / expected_total if expected_total > 0.0 else 0.0
)
sampled_preliminary_aal = sampled_total / declared_catalog_years
expected_preliminary_aal = expected_total / declared_catalog_years

append_check(
    validation_rows,
    "production_rows_match_expected",
    total_rows == expected_rows,
    f"processed={total_rows}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "production_occurrences_match_expected",
    len(seen_occurrences) == expected_occurrences,
    f"processed={len(seen_occurrences)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "final_building_loss_file_exists",
    FINAL_BUILDING_LOSS_PATH.exists(),
    str(FINAL_BUILDING_LOSS_PATH),
)
append_check(
    validation_rows,
    "event_summary_rows_match_occurrences",
    len(event_summary) == expected_occurrences
    and event_summary["occurrence_id"].astype(str).is_unique,
    f"rows={len(event_summary)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "event_summary_buildings_complete",
    event_summary["buildings"].eq(expected_sites).all(),
    (
        f"minimum={int(event_summary['buildings'].min())}; "
        f"maximum={int(event_summary['buildings'].max())}; expected={expected_sites}"
    ),
)
append_check(
    validation_rows,
    "sampled_loss_equations_exact",
    global_max_sampled_equation_error <= 1e-6,
    f"maximum_error={global_max_sampled_equation_error:.3e}",
)
append_check(
    validation_rows,
    "expected_loss_equations_exact",
    global_max_expected_equation_error <= 1e-6,
    f"maximum_error={global_max_expected_equation_error:.3e}",
)
append_check(
    validation_rows,
    "total_nonstructural_loss_equations_exact",
    global_max_total_equation_error <= 1e-6,
    f"maximum_error={global_max_total_equation_error:.3e}",
)
append_check(
    validation_rows,
    "sampled_total_nonstructural_ratio_does_not_exceed_one",
    global_max_sampled_total_ratio <= 1.0 + NUMERIC_TOLERANCE,
    f"maximum={global_max_sampled_total_ratio:.6f}",
)
append_check(
    validation_rows,
    "expected_total_nonstructural_ratio_does_not_exceed_one",
    global_max_expected_total_ratio <= 1.0 + NUMERIC_TOLERANCE,
    f"maximum={global_max_expected_total_ratio:.6f}",
)
append_check(
    validation_rows,
    "component_totals_sum_to_total_sampled_loss",
    abs(sampled_total - (component_sampled_total["nsd"] + component_sampled_total["nsa"])) <= 1e-6,
    f"total={sampled_total:.6f}",
)
append_check(
    validation_rows,
    "component_totals_sum_to_total_expected_loss",
    abs(expected_total - (component_expected_total["nsd"] + component_expected_total["nsa"])) <= 1e-6,
    f"total={expected_total:.6f}",
)
append_check(
    validation_rows,
    "sampled_total_reasonably_close_to_analytical_expected_total",
    relative_difference <= LOSS_RECONCILIATION_WARNING_TOLERANCE,
    (
        f"relative_difference={relative_difference:.6%}; "
        f"warning_tolerance={LOSS_RECONCILIATION_WARNING_TOLERANCE:.2%}"
    ),
    severity="warning",
)

loss_reconciliation = pd.DataFrame(
    [
        {
            "component": "nonstructural_drift_sensitive",
            "sampled_loss_total_2022_usd": component_sampled_total["nsd"],
            "analytical_expected_loss_total_2022_usd": component_expected_total["nsd"],
            "sampled_preliminary_aal_2022_usd": component_sampled_total["nsd"] / declared_catalog_years,
            "analytical_expected_preliminary_aal_2022_usd": component_expected_total["nsd"] / declared_catalog_years,
        },
        {
            "component": "nonstructural_acceleration_sensitive",
            "sampled_loss_total_2022_usd": component_sampled_total["nsa"],
            "analytical_expected_loss_total_2022_usd": component_expected_total["nsa"],
            "sampled_preliminary_aal_2022_usd": component_sampled_total["nsa"] / declared_catalog_years,
            "analytical_expected_preliminary_aal_2022_usd": component_expected_total["nsa"] / declared_catalog_years,
        },
        {
            "component": "total_nonstructural",
            "sampled_loss_total_2022_usd": sampled_total,
            "analytical_expected_loss_total_2022_usd": expected_total,
            "sampled_preliminary_aal_2022_usd": sampled_preliminary_aal,
            "analytical_expected_preliminary_aal_2022_usd": expected_preliminary_aal,
        },
    ]
)
loss_reconciliation["absolute_sampled_expected_difference_2022_usd"] = abs(
    loss_reconciliation["sampled_loss_total_2022_usd"]
    - loss_reconciliation["analytical_expected_loss_total_2022_usd"]
)
loss_reconciliation["relative_sampled_expected_difference"] = (
    loss_reconciliation["absolute_sampled_expected_difference_2022_usd"]
    / loss_reconciliation["analytical_expected_loss_total_2022_usd"].replace(0.0, np.nan)
).fillna(0.0)
loss_reconciliation.to_csv(LOSS_RECONCILIATION_PATH, index=False)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
]
warnings = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
]
validation.to_csv(CELL10_VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warnings.to_dict(orient="records"),
    "scope": {
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
        "loss_components": [
            "drift-sensitive nonstructural repair",
            "acceleration-sensitive nonstructural repair",
        ],
        "not_included": [
            "structural repair loss, calculated separately in Cell 5",
            "contents loss",
            "business interruption",
            "insurance policy terms",
            "reinsurance terms",
        ],
    },
    "annual_catalog": {
        "declared_duration_years": declared_catalog_years,
        "occurrences": expected_occurrences,
        "rows": expected_rows,
        "sites": expected_sites,
        "zero_event_years_rule": (
            "Preliminary AAL uses all declared catalog years. Full annual aggregation "
            "is performed after structural and nonstructural losses are combined."
        ),
    },
    "portfolio_valuation": {
        "buildings": int(len(building_parameters)),
        "occupancy_types": sorted(building_parameters["occ_type"].unique().tolist()),
        "total_replacement_value_2022_usd": float(
            building_parameters["building_replacement_value_2022_usd"].sum()
        ),
        "cell5_building_value_path": str(cell5_building_path),
        "cell5_building_value_sha256": sha256_file(cell5_building_path),
    },
    "repair_ratios": {
        "mapping": {
            "None": 0.0,
            "Slight": "DS_0 / 100",
            "Moderate": "DS_1 / 100",
            "Extensive": "DS_2 / 100",
            "Complete": "DS_3 / 100",
        },
        "nsd": {
            "source_path": str(reference_paths["nsd"]),
            "source_sha256": reference_hashes["nsd"],
        },
        "nsa": {
            "source_path": str(reference_paths["nsa"]),
            "source_sha256": reference_hashes["nsa"],
        },
        "complete_component_reconciliation": (
            "structural_complete + nsd_complete + nsa_complete = 1.0 for every building"
        ),
    },
    "loss_equations": {
        "sampled_component": (
            "sampled component loss = building replacement value * component repair "
            "ratio for sampled component damage state"
        ),
        "analytical_expected_component": (
            "expected component loss = building replacement value * "
            "sum_d[P(component DS=d|SA0P4) * component repair ratio_d]"
        ),
        "total_nonstructural": "total nonstructural loss = NSD loss + NSA loss",
    },
    "production": {
        "rows": total_rows,
        "occurrences": len(seen_occurrences),
        "chunks": len(chunk_paths),
        "sampled_nsd_loss_total_2022_usd": component_sampled_total["nsd"],
        "analytical_expected_nsd_loss_total_2022_usd": component_expected_total["nsd"],
        "sampled_nsa_loss_total_2022_usd": component_sampled_total["nsa"],
        "analytical_expected_nsa_loss_total_2022_usd": component_expected_total["nsa"],
        "sampled_total_nonstructural_loss_2022_usd": sampled_total,
        "analytical_expected_total_nonstructural_loss_2022_usd": expected_total,
        "sampled_preliminary_aal_2022_usd": sampled_preliminary_aal,
        "analytical_expected_preliminary_aal_2022_usd": expected_preliminary_aal,
        "relative_sampled_expected_difference": relative_difference,
        "maximum_sampled_total_nonstructural_repair_ratio": global_max_sampled_total_ratio,
        "maximum_expected_total_nonstructural_repair_ratio": global_max_expected_total_ratio,
        "source_sampled_loss_totals_2022_usd": dict(source_sampled_totals),
    },
    "outputs": {
        "building_value_and_nonstructural_ratio_table": {
            "path": str(BUILDING_PARAMETER_PATH),
            "rows": int(len(building_parameters)),
            "sha256": sha256_file(BUILDING_PARAMETER_PATH),
        },
        "full_nonstructural_ground_up_loss": {
            "path": str(FINAL_BUILDING_LOSS_PATH),
            "rows": total_rows,
            "size_bytes": int(FINAL_BUILDING_LOSS_PATH.stat().st_size),
            "sha256": sha256_file(FINAL_BUILDING_LOSS_PATH),
            "row_granularity": "one annual-catalog occurrence and one portfolio building",
        },
        "event_nonstructural_ground_up_loss": {
            "path": str(FINAL_EVENT_LOSS_PATH),
            "rows": int(len(event_summary)),
            "size_bytes": int(FINAL_EVENT_LOSS_PATH.stat().st_size),
            "sha256": sha256_file(FINAL_EVENT_LOSS_PATH),
        },
        "ratio_summary": str(RATIO_SUMMARY_PATH),
        "loss_reconciliation": str(LOSS_RECONCILIATION_PATH),
        "chunk_manifest": str(CELL10_CHUNK_MANIFEST_PATH),
        "validation": str(CELL10_VALIDATION_PATH),
        "summary": str(CELL10_SUMMARY_PATH),
    },
    "next_cell": (
        "Cell 11: combine Cell 5 structural and Cell 10 nonstructural losses at the "
        "occurrence-building level, validate component reconciliation, and create "
        "total building ground-up loss and occurrence-level portfolio loss tables."
    ),
}
write_json(CELL10_SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 10 failed critical validation. Review "
        f"{CELL10_VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 5 CELL 10 NONSTRUCTURAL GROUND-UP LOSS COMPLETE")
print("=" * 78)
print(f"Production loss rows:                  {total_rows:,}")
print(f"Catalog occurrences:                   {len(seen_occurrences):,}")
print(
    "Portfolio replacement value:           "
    f"${building_parameters['building_replacement_value_2022_usd'].sum():,.0f}"
)
print(f"Sampled NSD loss total:                ${component_sampled_total['nsd']:,.0f}")
print(f"Expected NSD loss total:               ${component_expected_total['nsd']:,.0f}")
print(f"Sampled NSA loss total:                ${component_sampled_total['nsa']:,.0f}")
print(f"Expected NSA loss total:               ${component_expected_total['nsa']:,.0f}")
print(f"Sampled nonstructural loss total:      ${sampled_total:,.0f}")
print(f"Expected nonstructural loss total:     ${expected_total:,.0f}")
print(f"Sampled preliminary AAL:               ${sampled_preliminary_aal:,.2f}")
print(f"Expected preliminary AAL:              ${expected_preliminary_aal:,.2f}")
print(f"Sampled-expected relative difference:  {relative_difference:.4%}")
print(f"Maximum sampled equation error:        {global_max_sampled_equation_error:.3e}")
print(f"Maximum expected equation error:       {global_max_expected_equation_error:.3e}")
print(f"Critical validation checks:            {int(validation['severity'].eq('critical').sum())}")
print(f"Critical failures:                     {len(critical_failures)}")
print(f"Warnings requiring review:             {len(warnings)}")
print()
print("Full nonstructural ground-up loss:")
print(f"  {FINAL_BUILDING_LOSS_PATH}")
print("Occurrence-level nonstructural loss:")
print(f"  {FINAL_EVENT_LOSS_PATH}")
print("Building nonstructural repair parameters:")
print(f"  {BUILDING_PARAMETER_PATH}")
print("Validation:")
print(f"  {CELL10_VALIDATION_PATH}")
print("Summary:")
print(f"  {CELL10_SUMMARY_PATH}")
print()
print("Next: combine structural and nonstructural losses into total ground-up loss.")


FULL ANNUAL-CATALOG NONSTRUCTURAL GROUND-UP LOSS
Nonstructural damage rows:     4,996,100
Catalog occurrences:           10,630
Portfolio buildings:           470
Cell 9 production chunks:      43
Portfolio replacement value:   $384,236,605 (2022 USD)
Loss scope:                     drift- and acceleration-sensitive nonstructural repair

------------------------------------------------------------------------------
NONSTRUCTURAL LOSS CHUNK 1 OF 43 [ID 0000]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
NONSTRUCTURAL LOSS CHUNK 2 OF 43 [ID 0001]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
NONSTRUCTURAL LOSS CHUNK 3 OF 43 [ID 0002]
Calculated 117,500 rows for 250 occurrences.

------------------------------------------------------------------------------
NONSTRUCTURAL LOSS CHUNK 4 OF 43 [ID 0003]
Calculated 117,500 rows for 250 oc

In [24]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

DAMAGE_STATES = ["None", "Slight", "Moderate", "Extensive", "Complete"]
DOLLAR_YEAR = 2022
PIPELINE_VERSION = "notebook5_cell11_total_ground_up_loss_v1"
NUMERIC_TOLERANCE = 1e-10
LOSS_EQUATION_TOLERANCE_USD = 1e-6
AGGREGATE_RECONCILIATION_TOLERANCE_USD = 0.01
EXPECTED_SAMPLED_WARNING_TOLERANCE = 0.05


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_10_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def prior_validation_passed(path: Path) -> tuple[bool, int]:
    table = pd.read_csv(path)
    required = {"severity", "passed"}
    missing = sorted(required.difference(table.columns))
    if missing:
        raise KeyError(f"Validation file {path} is missing columns: {missing}")
    table["passed"] = parse_bool_series(table["passed"])
    failures = table.loc[
        table["severity"].astype(str).str.lower().eq("critical") & ~table["passed"]
    ]
    return failures.empty, int(len(failures))


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    buffer = io.StringIO()
    frame.to_csv(buffer, index=False, lineterminator="\n")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as compressed:
            compressed.write(buffer.getvalue().encode("utf-8"))
    temporary.replace(path)


def concatenate_gzip_csv_files(chunk_paths: list[Path], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    expected_header: bytes | None = None
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(fileobj=raw_output, mode="wb", mtime=0) as compressed_output:
            for chunk_path in chunk_paths:
                with gzip.open(chunk_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        compressed_output.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            f"Chunk header mismatch while concatenating {chunk_path}."
                        )
                    shutil.copyfileobj(source, compressed_output, length=8 * 1024 * 1024)
    temporary.replace(output_path)


def normalized_text(values: pd.Series) -> pd.Series:
    return values.fillna("").astype(str).str.strip()


def compare_text_columns(
    left: pd.DataFrame,
    right: pd.DataFrame,
    columns: list[str],
) -> tuple[bool, dict[str, int]]:
    mismatches: dict[str, int] = {}
    for column in columns:
        if column not in left.columns or column not in right.columns:
            continue
        mismatch_count = int(
            (~normalized_text(left[column]).eq(normalized_text(right[column]))).sum()
        )
        mismatches[column] = mismatch_count
    return all(value == 0 for value in mismatches.values()), mismatches


def compare_numeric_columns(
    left: pd.DataFrame,
    right: pd.DataFrame,
    columns: list[str],
    absolute_tolerance: float,
) -> tuple[bool, dict[str, float]]:
    errors: dict[str, float] = {}
    for column in columns:
        if column not in left.columns or column not in right.columns:
            continue
        left_values = pd.to_numeric(left[column], errors="raise").to_numpy(
            dtype=np.float64
        )
        right_values = pd.to_numeric(right[column], errors="raise").to_numpy(
            dtype=np.float64
        )
        error = float(np.max(np.abs(left_values - right_values))) if len(left) else 0.0
        errors[column] = error
    return all(value <= absolute_tolerance for value in errors.values()), errors


def summarize_occurrences(frame: pd.DataFrame) -> pd.DataFrame:
    event_fields = [
        "catalog_year",
        "catalog_event_id",
        "rupture_ordinal",
        "rupture_template_event_id",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
    ]
    group_columns = [column for column in event_fields if column in frame.columns]

    output = (
        frame.groupby(group_columns, sort=False)
        .agg(
            buildings=("site_id", "size"),
            portfolio_replacement_value_2022_usd=(
                "building_replacement_value_2022_usd",
                "sum",
            ),
            sampled_structural_ground_up_loss_2022_usd=(
                "sampled_structural_ground_up_loss_2022_usd",
                "sum",
            ),
            analytical_expected_structural_ground_up_loss_2022_usd=(
                "analytical_expected_structural_ground_up_loss_2022_usd",
                "sum",
            ),
            sampled_nsd_ground_up_loss_2022_usd=(
                "sampled_nsd_ground_up_loss_2022_usd",
                "sum",
            ),
            analytical_expected_nsd_ground_up_loss_2022_usd=(
                "analytical_expected_nsd_ground_up_loss_2022_usd",
                "sum",
            ),
            sampled_nsa_ground_up_loss_2022_usd=(
                "sampled_nsa_ground_up_loss_2022_usd",
                "sum",
            ),
            analytical_expected_nsa_ground_up_loss_2022_usd=(
                "analytical_expected_nsa_ground_up_loss_2022_usd",
                "sum",
            ),
            sampled_total_nonstructural_ground_up_loss_2022_usd=(
                "sampled_total_nonstructural_ground_up_loss_2022_usd",
                "sum",
            ),
            analytical_expected_total_nonstructural_ground_up_loss_2022_usd=(
                "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
                "sum",
            ),
            sampled_total_ground_up_loss_2022_usd=(
                "sampled_total_ground_up_loss_2022_usd",
                "sum",
            ),
            analytical_expected_total_ground_up_loss_2022_usd=(
                "analytical_expected_total_ground_up_loss_2022_usd",
                "sum",
            ),
            sampled_structurally_damaged_buildings=(
                "sampled_structurally_damaged_indicator",
                "sum",
            ),
            sampled_nsd_damaged_buildings=("sampled_nsd_damaged_indicator", "sum"),
            sampled_nsa_damaged_buildings=("sampled_nsa_damaged_indicator", "sum"),
            sampled_any_component_damaged_buildings=(
                "sampled_any_component_damaged_indicator",
                "sum",
            ),
            maximum_building_sampled_total_ground_up_loss_2022_usd=(
                "sampled_total_ground_up_loss_2022_usd",
                "max",
            ),
            mean_sa0p4_g=("sa0p4_simulated_g", "mean"),
            maximum_sa0p4_g=("sa0p4_simulated_g", "max"),
        )
        .reset_index()
    )

    replacement = output["portfolio_replacement_value_2022_usd"].to_numpy(
        dtype=np.float64
    )
    output["sampled_total_ground_up_loss_ratio"] = (
        output["sampled_total_ground_up_loss_2022_usd"] / replacement
    )
    output["analytical_expected_total_ground_up_loss_ratio"] = (
        output["analytical_expected_total_ground_up_loss_2022_usd"] / replacement
    )
    sampled_total = output["sampled_total_ground_up_loss_2022_usd"].to_numpy(
        dtype=np.float64
    )
    output["sampled_structural_loss_share"] = np.divide(
        output["sampled_structural_ground_up_loss_2022_usd"],
        sampled_total,
        out=np.zeros(len(output), dtype=np.float64),
        where=sampled_total > 0.0,
    )
    output["sampled_nsd_loss_share"] = np.divide(
        output["sampled_nsd_ground_up_loss_2022_usd"],
        sampled_total,
        out=np.zeros(len(output), dtype=np.float64),
        where=sampled_total > 0.0,
    )
    output["sampled_nsa_loss_share"] = np.divide(
        output["sampled_nsa_ground_up_loss_2022_usd"],
        sampled_total,
        out=np.zeros(len(output), dtype=np.float64),
        where=sampled_total > 0.0,
    )
    return output.sort_values("occurrence_ordinal").reset_index(drop=True)


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
STRUCTURAL_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_structural_loss"
NONSTRUCTURAL_DIR = (
    PROJECT_ROOT / "data" / "processed" / "notebook_5_nonstructural_loss"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_total_ground_up_loss"
CHUNK_DIR = OUTPUT_DIR / "chunks"
WORK_DIR = OUTPUT_DIR / "work"
MARKER_DIR = WORK_DIR / "markers"
CHUNK_VALIDATION_DIR = WORK_DIR / "chunk_validations"

for directory in [
    METADATA_DIR,
    OUTPUT_DIR,
    CHUNK_DIR,
    WORK_DIR,
    MARKER_DIR,
    CHUNK_VALIDATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CELL5_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_5_summary.json"
CELL5_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_5_validation.csv"
CELL5_MANIFEST_PATH = METADATA_DIR / "notebook_5_cell_5_chunk_manifest.csv"
CELL10_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_10_summary.json"
CELL10_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_10_validation.csv"
CELL10_MANIFEST_PATH = METADATA_DIR / "notebook_5_cell_10_chunk_manifest.csv"

FINAL_BUILDING_LOSS_PATH = OUTPUT_DIR / "full_total_ground_up_loss.csv.gz"
FINAL_EVENT_LOSS_PATH = OUTPUT_DIR / "total_ground_up_loss_event_summary.csv.gz"
COMPONENT_RECONCILIATION_PATH = (
    METADATA_DIR / "notebook_5_cell_11_component_reconciliation.csv"
)
CHUNK_MANIFEST_PATH = METADATA_DIR / "notebook_5_cell_11_chunk_manifest.csv"
CELL11_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_11_validation.csv"
CELL11_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_11_summary.json"

required_paths = [
    CELL5_SUMMARY_PATH,
    CELL5_VALIDATION_PATH,
    CELL5_MANIFEST_PATH,
    CELL10_SUMMARY_PATH,
    CELL10_VALIDATION_PATH,
    CELL10_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required prior-cell outputs are missing: {missing_paths}")

validation_rows: list[dict[str, Any]] = []
cell5_summary = load_json(CELL5_SUMMARY_PATH)
cell10_summary = load_json(CELL10_SUMMARY_PATH)
cell5_manifest = pd.read_csv(CELL5_MANIFEST_PATH)
cell10_manifest = pd.read_csv(CELL10_MANIFEST_PATH)

for cell_number, summary, validation_path in [
    (5, cell5_summary, CELL5_VALIDATION_PATH),
    (10, cell10_summary, CELL10_VALIDATION_PATH),
]:
    validation_passed, failure_count = prior_validation_passed(validation_path)
    append_check(
        validation_rows,
        f"cell_{cell_number}_critical_validation_passed",
        validation_passed and bool(summary.get("all_critical_checks_passed", False)),
        f"critical_failures={failure_count}",
    )

cell5_catalog = cell5_summary["annual_catalog"]
cell10_catalog = cell10_summary["annual_catalog"]
declared_catalog_years = int(cell5_catalog["declared_duration_years"])
expected_occurrences = int(cell5_catalog["occurrences"])
expected_sites = int(cell5_catalog["sites"])
expected_rows = int(cell5_catalog["rows"])
expected_chunks = int(cell5_summary["production"]["chunks"])

catalog_fields = [
    "declared_duration_years",
    "occurrences",
    "sites",
    "rows",
]
catalog_mismatches = {
    field: (cell5_catalog.get(field), cell10_catalog.get(field))
    for field in catalog_fields
    if int(cell5_catalog.get(field)) != int(cell10_catalog.get(field))
}
append_check(
    validation_rows,
    "cell_5_and_cell_10_catalog_dimensions_match",
    not catalog_mismatches,
    f"mismatches={catalog_mismatches}",
)
append_check(
    validation_rows,
    "declared_catalog_years_positive",
    declared_catalog_years > 0,
    f"years={declared_catalog_years}",
)
append_check(
    validation_rows,
    "expected_row_identity",
    expected_rows == expected_occurrences * expected_sites,
    (
        f"rows={expected_rows}; occurrences={expected_occurrences}; "
        f"sites={expected_sites}"
    ),
)

required_manifest_columns = {
    "chunk_id",
    "rows",
    "occurrences",
    "minimum_occurrence_ordinal",
    "maximum_occurrence_ordinal",
    "output_path",
    "output_sha256",
}
for label, manifest in [("cell_5", cell5_manifest), ("cell_10", cell10_manifest)]:
    missing = sorted(required_manifest_columns.difference(manifest.columns))
    append_check(
        validation_rows,
        f"{label}_manifest_schema_complete",
        not missing,
        f"missing={missing}",
    )
    if missing:
        raise KeyError(f"{label} manifest is missing columns: {missing}")

for manifest in [cell5_manifest, cell10_manifest]:
    manifest["chunk_id"] = manifest["chunk_id"].astype(str).str.zfill(4)
    for column in [
        "rows",
        "occurrences",
        "minimum_occurrence_ordinal",
        "maximum_occurrence_ordinal",
    ]:
        manifest[column] = pd.to_numeric(manifest[column], errors="raise").astype(
            np.int64
        )
    manifest.sort_values("minimum_occurrence_ordinal", inplace=True)
    manifest.reset_index(drop=True, inplace=True)

append_check(
    validation_rows,
    "cell_5_manifest_chunk_count_matches_expected",
    len(cell5_manifest) == expected_chunks,
    f"manifest={len(cell5_manifest)}; expected={expected_chunks}",
)
append_check(
    validation_rows,
    "cell_10_manifest_chunk_count_matches_expected",
    len(cell10_manifest) == expected_chunks,
    f"manifest={len(cell10_manifest)}; expected={expected_chunks}",
)

manifest_compare_columns = [
    "chunk_id",
    "rows",
    "occurrences",
    "minimum_occurrence_ordinal",
    "maximum_occurrence_ordinal",
]
manifest_alignment = cell5_manifest[manifest_compare_columns].equals(
    cell10_manifest[manifest_compare_columns]
)
append_check(
    validation_rows,
    "cell_5_and_cell_10_chunk_boundaries_match",
    manifest_alignment,
    f"chunks={len(cell5_manifest)}",
)
if not manifest_alignment:
    raise RuntimeError("Cell 5 and Cell 10 chunk boundaries do not align.")

portfolio_value_cell5 = float(
    cell5_summary["portfolio_valuation"]["total_replacement_value_2022_usd"]
)
portfolio_value_cell10 = float(
    cell10_summary["portfolio_valuation"]["total_replacement_value_2022_usd"]
)
append_check(
    validation_rows,
    "cell_5_and_cell_10_portfolio_values_match",
    abs(portfolio_value_cell5 - portfolio_value_cell10) <= LOSS_EQUATION_TOLERANCE_USD,
    f"cell5={portfolio_value_cell5:.6f}; cell10={portfolio_value_cell10:.6f}",
)

basis_payload = {
    "pipeline_version": PIPELINE_VERSION,
    "cell5_summary_sha256": sha256_file(CELL5_SUMMARY_PATH),
    "cell5_manifest_sha256": sha256_file(CELL5_MANIFEST_PATH),
    "cell10_summary_sha256": sha256_file(CELL10_SUMMARY_PATH),
    "cell10_manifest_sha256": sha256_file(CELL10_MANIFEST_PATH),
    "declared_catalog_years": declared_catalog_years,
    "expected_occurrences": expected_occurrences,
    "expected_sites": expected_sites,
    "expected_rows": expected_rows,
    "dollar_year": DOLLAR_YEAR,
}
basis_hash = hashlib.sha256(
    json.dumps(basis_payload, sort_keys=True).encode("utf-8")
).hexdigest()

print("=" * 78)
print("FULL ANNUAL-CATALOG TOTAL GROUND-UP LOSS")
print("=" * 78)
print(f"Structural loss rows:          {expected_rows:,}")
print(f"Nonstructural loss rows:       {expected_rows:,}")
print(f"Catalog occurrences:           {expected_occurrences:,}")
print(f"Portfolio buildings:           {expected_sites:,}")
print(f"Production chunks:             {expected_chunks:,}")
print(f"Portfolio replacement value:   ${portfolio_value_cell5:,.0f} ({DOLLAR_YEAR} USD)")
print("Loss scope:                     structural + NSD + NSA repair")

chunk_paths: list[Path] = []
chunk_manifest_rows: list[dict[str, Any]] = []
event_summary_frames: list[pd.DataFrame] = []
seen_pairs = np.zeros(expected_rows, dtype=bool)
occurrence_counts = np.zeros(expected_occurrences, dtype=np.int32)
site_counts = np.zeros(expected_sites, dtype=np.int32)
seen_occurrences: set[str] = set()
source_sampled_totals: Counter[str] = Counter()
component_sampled_totals = {"structural": 0.0, "nsd": 0.0, "nsa": 0.0}
component_expected_totals = {"structural": 0.0, "nsd": 0.0, "nsa": 0.0}
total_rows = 0
maximum_sampled_equation_error = 0.0
maximum_expected_equation_error = 0.0
maximum_component_sum_error = 0.0
maximum_sampled_total_ratio = 0.0
maximum_expected_total_ratio = 0.0

key_columns = ["occurrence_ordinal", "occurrence_id", "site_ordinal", "site_id"]
text_compare_columns = [
    "catalog_event_id",
    "rupture_template_event_id",
    "rupture_id",
    "source_type",
    "structural_type",
    "design_level",
    "occ_type",
]
numeric_compare_columns = [
    "catalog_year",
    "rupture_ordinal",
    "occurrence_ordinal",
    "site_ordinal",
    "magnitude",
    "sa0p4_simulated_g",
    "building_replacement_value_2022_usd",
]

for manifest_index in range(expected_chunks):
    structural_record = cell5_manifest.iloc[manifest_index]
    nonstructural_record = cell10_manifest.iloc[manifest_index]
    chunk_id = str(structural_record["chunk_id"]).zfill(4)

    print()
    print("-" * 78)
    print(
        f"TOTAL GROUND-UP LOSS CHUNK {manifest_index + 1} OF {expected_chunks} "
        f"[ID {chunk_id}]"
    )

    structural_path = resolve_recorded_path(
        PROJECT_ROOT, str(structural_record["output_path"])
    )
    nonstructural_path = resolve_recorded_path(
        PROJECT_ROOT, str(nonstructural_record["output_path"])
    )
    structural_hash = sha256_file(structural_path)
    nonstructural_hash = sha256_file(nonstructural_path)
    if structural_hash != str(structural_record["output_sha256"]):
        raise RuntimeError(f"Cell 5 chunk {chunk_id} hash mismatch.")
    if nonstructural_hash != str(nonstructural_record["output_sha256"]):
        raise RuntimeError(f"Cell 10 chunk {chunk_id} hash mismatch.")

    chunk_output_path = CHUNK_DIR / f"total_ground_up_loss_chunk_{chunk_id}.csv.gz"
    marker_path = MARKER_DIR / f"total_ground_up_loss_chunk_{chunk_id}.json"
    chunk_validation_path = (
        CHUNK_VALIDATION_DIR / f"total_ground_up_loss_chunk_{chunk_id}_validation.csv"
    )

    marker_valid = False
    marker: dict[str, Any] = {}
    if marker_path.exists() and chunk_output_path.exists() and chunk_validation_path.exists():
        try:
            marker = load_json(marker_path)
            marker_valid = (
                marker.get("pipeline_version") == PIPELINE_VERSION
                and marker.get("basis_hash") == basis_hash
                and marker.get("cell5_chunk_sha256") == structural_hash
                and marker.get("cell10_chunk_sha256") == nonstructural_hash
                and int(marker.get("rows", -1)) == int(structural_record["rows"])
                and marker.get("output_sha256") == sha256_file(chunk_output_path)
                and marker.get("validation_sha256")
                == sha256_file(chunk_validation_path)
            )
        except Exception:
            marker_valid = False

    if marker_valid:
        output = pd.read_csv(chunk_output_path)
        print(f"Reused validated total-loss chunk with {len(output):,} rows.")
    else:
        structural = pd.read_csv(structural_path)
        nonstructural = pd.read_csv(nonstructural_path)
        chunk_validation_rows: list[dict[str, Any]] = []

        required_structural_columns = {
            *key_columns,
            "catalog_year",
            "rupture_id",
            "source_type",
            "magnitude",
            "sa0p4_simulated_g",
            "occ_type",
            "building_replacement_value_2022_usd",
            "sampled_structural_damage_state",
            "sampled_structural_repair_ratio",
            "analytical_expected_structural_repair_ratio",
            "sampled_structural_ground_up_loss_2022_usd",
            "analytical_expected_structural_ground_up_loss_2022_usd",
            "sampled_structurally_damaged_indicator",
        }
        required_nonstructural_columns = {
            *key_columns,
            "catalog_year",
            "rupture_id",
            "source_type",
            "magnitude",
            "sa0p4_simulated_g",
            "occ_type",
            "building_replacement_value_2022_usd",
            "sampled_nsd_damage_state",
            "sampled_nsa_damage_state",
            "sampled_nsd_repair_ratio",
            "analytical_expected_nsd_repair_ratio",
            "sampled_nsd_ground_up_loss_2022_usd",
            "analytical_expected_nsd_ground_up_loss_2022_usd",
            "sampled_nsa_repair_ratio",
            "analytical_expected_nsa_repair_ratio",
            "sampled_nsa_ground_up_loss_2022_usd",
            "analytical_expected_nsa_ground_up_loss_2022_usd",
            "sampled_total_nonstructural_repair_ratio",
            "analytical_expected_total_nonstructural_repair_ratio",
            "sampled_total_nonstructural_ground_up_loss_2022_usd",
            "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
        }
        missing_structural = sorted(required_structural_columns.difference(structural.columns))
        missing_nonstructural = sorted(
            required_nonstructural_columns.difference(nonstructural.columns)
        )
        append_check(
            chunk_validation_rows,
            "structural_chunk_schema_complete",
            not missing_structural,
            f"missing={missing_structural}",
        )
        append_check(
            chunk_validation_rows,
            "nonstructural_chunk_schema_complete",
            not missing_nonstructural,
            f"missing={missing_nonstructural}",
        )
        if missing_structural or missing_nonstructural:
            raise KeyError(
                f"Chunk {chunk_id} missing structural={missing_structural}; "
                f"nonstructural={missing_nonstructural}"
            )

        for frame in [structural, nonstructural]:
            frame["occurrence_id"] = frame["occurrence_id"].astype(str)
            frame["site_id"] = frame["site_id"].astype(str)
            frame["occurrence_ordinal"] = pd.to_numeric(
                frame["occurrence_ordinal"], errors="raise"
            ).astype(np.int64)
            frame["site_ordinal"] = pd.to_numeric(
                frame["site_ordinal"], errors="raise"
            ).astype(np.int64)
            frame.sort_values(key_columns, inplace=True)
            frame.reset_index(drop=True, inplace=True)

        append_check(
            chunk_validation_rows,
            "structural_and_nonstructural_row_counts_match",
            len(structural) == len(nonstructural),
            f"structural={len(structural)}; nonstructural={len(nonstructural)}",
        )
        append_check(
            chunk_validation_rows,
            "structural_occurrence_site_pairs_unique",
            not structural.duplicated(key_columns).any(),
            f"duplicates={int(structural.duplicated(key_columns).sum())}",
        )
        append_check(
            chunk_validation_rows,
            "nonstructural_occurrence_site_pairs_unique",
            not nonstructural.duplicated(key_columns).any(),
            f"duplicates={int(nonstructural.duplicated(key_columns).sum())}",
        )

        keys_match = structural[key_columns].equals(nonstructural[key_columns])
        append_check(
            chunk_validation_rows,
            "structural_and_nonstructural_keys_match_exactly",
            keys_match,
            f"rows={len(structural)}",
        )
        if not keys_match:
            raise RuntimeError(f"Chunk {chunk_id} occurrence-site keys do not align.")

        text_match, text_mismatches = compare_text_columns(
            structural, nonstructural, text_compare_columns
        )
        numeric_match, numeric_errors = compare_numeric_columns(
            structural,
            nonstructural,
            numeric_compare_columns,
            absolute_tolerance=LOSS_EQUATION_TOLERANCE_USD,
        )
        append_check(
            chunk_validation_rows,
            "shared_text_metadata_match",
            text_match,
            f"mismatches={text_mismatches}",
        )
        append_check(
            chunk_validation_rows,
            "shared_numeric_metadata_match",
            numeric_match,
            f"maximum_errors={numeric_errors}",
        )
        if not text_match or not numeric_match:
            raise RuntimeError(f"Chunk {chunk_id} metadata differs between Cells 5 and 10.")

        replacement_values = pd.to_numeric(
            structural["building_replacement_value_2022_usd"], errors="raise"
        ).to_numpy(dtype=np.float64)

        sampled_structural_ratios = pd.to_numeric(
            structural["sampled_structural_repair_ratio"], errors="raise"
        ).to_numpy(dtype=np.float64)
        expected_structural_ratios = pd.to_numeric(
            structural["analytical_expected_structural_repair_ratio"], errors="raise"
        ).to_numpy(dtype=np.float64)
        sampled_nsd_ratios = pd.to_numeric(
            nonstructural["sampled_nsd_repair_ratio"], errors="raise"
        ).to_numpy(dtype=np.float64)
        expected_nsd_ratios = pd.to_numeric(
            nonstructural["analytical_expected_nsd_repair_ratio"], errors="raise"
        ).to_numpy(dtype=np.float64)
        sampled_nsa_ratios = pd.to_numeric(
            nonstructural["sampled_nsa_repair_ratio"], errors="raise"
        ).to_numpy(dtype=np.float64)
        expected_nsa_ratios = pd.to_numeric(
            nonstructural["analytical_expected_nsa_repair_ratio"], errors="raise"
        ).to_numpy(dtype=np.float64)

        sampled_total_ratios = (
            sampled_structural_ratios + sampled_nsd_ratios + sampled_nsa_ratios
        )
        expected_total_ratios = (
            expected_structural_ratios + expected_nsd_ratios + expected_nsa_ratios
        )

        sampled_structural_losses = pd.to_numeric(
            structural["sampled_structural_ground_up_loss_2022_usd"], errors="raise"
        ).to_numpy(dtype=np.float64)
        expected_structural_losses = pd.to_numeric(
            structural[
                "analytical_expected_structural_ground_up_loss_2022_usd"
            ],
            errors="raise",
        ).to_numpy(dtype=np.float64)
        sampled_nsd_losses = pd.to_numeric(
            nonstructural["sampled_nsd_ground_up_loss_2022_usd"], errors="raise"
        ).to_numpy(dtype=np.float64)
        expected_nsd_losses = pd.to_numeric(
            nonstructural["analytical_expected_nsd_ground_up_loss_2022_usd"],
            errors="raise",
        ).to_numpy(dtype=np.float64)
        sampled_nsa_losses = pd.to_numeric(
            nonstructural["sampled_nsa_ground_up_loss_2022_usd"], errors="raise"
        ).to_numpy(dtype=np.float64)
        expected_nsa_losses = pd.to_numeric(
            nonstructural["analytical_expected_nsa_ground_up_loss_2022_usd"],
            errors="raise",
        ).to_numpy(dtype=np.float64)

        sampled_total_losses = (
            sampled_structural_losses + sampled_nsd_losses + sampled_nsa_losses
        )
        expected_total_losses = (
            expected_structural_losses + expected_nsd_losses + expected_nsa_losses
        )

        sampled_equation_error = float(
            np.max(np.abs(sampled_total_losses - replacement_values * sampled_total_ratios))
        )
        expected_equation_error = float(
            np.max(np.abs(expected_total_losses - replacement_values * expected_total_ratios))
        )
        sampled_component_sum_error = float(
            np.max(
                np.abs(
                    sampled_total_losses
                    - (
                        sampled_structural_losses
                        + sampled_nsd_losses
                        + sampled_nsa_losses
                    )
                )
            )
        )
        expected_component_sum_error = float(
            np.max(
                np.abs(
                    expected_total_losses
                    - (
                        expected_structural_losses
                        + expected_nsd_losses
                        + expected_nsa_losses
                    )
                )
            )
        )

        append_check(
            chunk_validation_rows,
            "sampled_total_loss_equation_exact",
            sampled_equation_error <= LOSS_EQUATION_TOLERANCE_USD,
            f"maximum_error={sampled_equation_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "expected_total_loss_equation_exact",
            expected_equation_error <= LOSS_EQUATION_TOLERANCE_USD,
            f"maximum_error={expected_equation_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "sampled_component_losses_sum_exactly",
            sampled_component_sum_error <= LOSS_EQUATION_TOLERANCE_USD,
            f"maximum_error={sampled_component_sum_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "expected_component_losses_sum_exactly",
            expected_component_sum_error <= LOSS_EQUATION_TOLERANCE_USD,
            f"maximum_error={expected_component_sum_error:.3e}",
        )
        append_check(
            chunk_validation_rows,
            "sampled_total_repair_ratios_bounded",
            np.isfinite(sampled_total_ratios).all()
            and np.all(sampled_total_ratios >= -NUMERIC_TOLERANCE)
            and np.all(sampled_total_ratios <= 1.0 + NUMERIC_TOLERANCE),
            (
                f"minimum={sampled_total_ratios.min():.6f}; "
                f"maximum={sampled_total_ratios.max():.6f}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "expected_total_repair_ratios_bounded",
            np.isfinite(expected_total_ratios).all()
            and np.all(expected_total_ratios >= -NUMERIC_TOLERANCE)
            and np.all(expected_total_ratios <= 1.0 + NUMERIC_TOLERANCE),
            (
                f"minimum={expected_total_ratios.min():.6f}; "
                f"maximum={expected_total_ratios.max():.6f}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "total_ground_up_losses_do_not_exceed_replacement_value",
            np.all(sampled_total_losses <= replacement_values + LOSS_EQUATION_TOLERANCE_USD)
            and np.all(
                expected_total_losses <= replacement_values + LOSS_EQUATION_TOLERANCE_USD
            ),
            (
                f"sampled_max_ratio={sampled_total_ratios.max():.6f}; "
                f"expected_max_ratio={expected_total_ratios.max():.6f}"
            ),
        )

        identifier_columns = [
            column
            for column in [
                "catalog_year",
                "catalog_event_id",
                "rupture_ordinal",
                "rupture_template_event_id",
                "occurrence_ordinal",
                "occurrence_id",
                "rupture_id",
                "source_type",
                "magnitude",
                "site_ordinal",
                "site_id",
                "sa0p4_simulated_g",
                "structural_type",
                "design_level",
                "occ_type",
                "year_built",
                "floor_area_sqft",
                "replacement_cost_per_sqft_2022_usd",
                "building_replacement_value_2022_usd",
            ]
            if column in structural.columns
        ]
        output = structural[identifier_columns].copy()

        structural_columns = [
            column
            for column in [
                "sampled_structural_damage_state",
                "sampled_structural_damage_state_name",
                "sampled_structural_repair_ratio",
                "analytical_expected_structural_repair_ratio",
                "sampled_structural_ground_up_loss_2022_usd",
                "analytical_expected_structural_ground_up_loss_2022_usd",
                "sampled_structurally_damaged_indicator",
                "analytical_expected_structurally_damaged_indicator",
            ]
            if column in structural.columns
        ]
        for column in structural_columns:
            output[column] = structural[column].to_numpy()

        nonstructural_columns = [
            "sampled_nsd_damage_state",
            "sampled_nsa_damage_state",
            "sampled_nsd_repair_ratio",
            "analytical_expected_nsd_repair_ratio",
            "sampled_nsd_ground_up_loss_2022_usd",
            "analytical_expected_nsd_ground_up_loss_2022_usd",
            "sampled_nsa_repair_ratio",
            "analytical_expected_nsa_repair_ratio",
            "sampled_nsa_ground_up_loss_2022_usd",
            "analytical_expected_nsa_ground_up_loss_2022_usd",
            "sampled_total_nonstructural_repair_ratio",
            "analytical_expected_total_nonstructural_repair_ratio",
            "sampled_total_nonstructural_ground_up_loss_2022_usd",
            "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
        ]
        for column in nonstructural_columns:
            output[column] = nonstructural[column].to_numpy()

        output["sampled_nsd_damage_state_name"] = pd.Categorical.from_codes(
            pd.to_numeric(output["sampled_nsd_damage_state"], errors="raise").astype(int),
            DAMAGE_STATES,
        ).astype(str)
        output["sampled_nsa_damage_state_name"] = pd.Categorical.from_codes(
            pd.to_numeric(output["sampled_nsa_damage_state"], errors="raise").astype(int),
            DAMAGE_STATES,
        ).astype(str)
        output["sampled_nsd_damaged_indicator"] = (
            pd.to_numeric(output["sampled_nsd_damage_state"], errors="raise") > 0
        ).astype(np.int8)
        output["sampled_nsa_damaged_indicator"] = (
            pd.to_numeric(output["sampled_nsa_damage_state"], errors="raise") > 0
        ).astype(np.int8)
        output["sampled_any_component_damaged_indicator"] = (
            (
                pd.to_numeric(
                    output["sampled_structural_damage_state"], errors="raise"
                )
                > 0
            )
            | (pd.to_numeric(output["sampled_nsd_damage_state"], errors="raise") > 0)
            | (pd.to_numeric(output["sampled_nsa_damage_state"], errors="raise") > 0)
        ).astype(np.int8)
        output["sampled_total_ground_up_repair_ratio"] = sampled_total_ratios
        output[
            "analytical_expected_total_ground_up_repair_ratio"
        ] = expected_total_ratios
        output["sampled_total_ground_up_loss_2022_usd"] = sampled_total_losses
        output[
            "analytical_expected_total_ground_up_loss_2022_usd"
        ] = expected_total_losses

        append_check(
            chunk_validation_rows,
            "output_rows_match_inputs",
            len(output) == len(structural) == len(nonstructural),
            (
                f"output={len(output)}; structural={len(structural)}; "
                f"nonstructural={len(nonstructural)}"
            ),
        )
        append_check(
            chunk_validation_rows,
            "output_occurrence_site_pairs_unique",
            not output.duplicated(key_columns).any(),
            f"duplicates={int(output.duplicated(key_columns).sum())}",
        )
        occurrence_sizes = output.groupby("occurrence_id", sort=False).size()
        append_check(
            chunk_validation_rows,
            "occurrences_have_complete_portfolios",
            occurrence_sizes.eq(expected_sites).all(),
            (
                f"occurrences={len(occurrence_sizes)}; "
                f"minimum={int(occurrence_sizes.min())}; "
                f"maximum={int(occurrence_sizes.max())}; expected={expected_sites}"
            ),
        )

        chunk_validation = pd.DataFrame(chunk_validation_rows)
        chunk_failures = chunk_validation.loc[
            chunk_validation["severity"].eq("critical") & ~chunk_validation["passed"]
        ]
        chunk_validation.to_csv(chunk_validation_path, index=False)
        if not chunk_failures.empty:
            raise RuntimeError(
                f"Total ground-up loss chunk {chunk_id} failed validation. "
                f"Review {chunk_validation_path}."
            )

        write_gzip_csv_deterministic(output, chunk_output_path)
        marker = {
            "pipeline_version": PIPELINE_VERSION,
            "basis_hash": basis_hash,
            "cell5_chunk_sha256": structural_hash,
            "cell10_chunk_sha256": nonstructural_hash,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "output_path": str(chunk_output_path),
            "output_sha256": sha256_file(chunk_output_path),
            "validation_path": str(chunk_validation_path),
            "validation_sha256": sha256_file(chunk_validation_path),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        write_json(marker_path, marker)
        print(
            f"Calculated {len(output):,} rows for "
            f"{output['occurrence_id'].nunique():,} occurrences."
        )

    required_output_columns = {
        *key_columns,
        "catalog_year",
        "rupture_id",
        "source_type",
        "magnitude",
        "sa0p4_simulated_g",
        "occ_type",
        "building_replacement_value_2022_usd",
        "sampled_structural_damage_state",
        "sampled_nsd_damage_state",
        "sampled_nsa_damage_state",
        "sampled_structural_repair_ratio",
        "analytical_expected_structural_repair_ratio",
        "sampled_nsd_repair_ratio",
        "analytical_expected_nsd_repair_ratio",
        "sampled_nsa_repair_ratio",
        "analytical_expected_nsa_repair_ratio",
        "sampled_structural_ground_up_loss_2022_usd",
        "analytical_expected_structural_ground_up_loss_2022_usd",
        "sampled_nsd_ground_up_loss_2022_usd",
        "analytical_expected_nsd_ground_up_loss_2022_usd",
        "sampled_nsa_ground_up_loss_2022_usd",
        "analytical_expected_nsa_ground_up_loss_2022_usd",
        "sampled_total_nonstructural_ground_up_loss_2022_usd",
        "analytical_expected_total_nonstructural_ground_up_loss_2022_usd",
        "sampled_total_ground_up_repair_ratio",
        "analytical_expected_total_ground_up_repair_ratio",
        "sampled_total_ground_up_loss_2022_usd",
        "analytical_expected_total_ground_up_loss_2022_usd",
        "sampled_structurally_damaged_indicator",
        "sampled_nsd_damaged_indicator",
        "sampled_nsa_damaged_indicator",
        "sampled_any_component_damaged_indicator",
    }
    missing_output = sorted(required_output_columns.difference(output.columns))
    if missing_output:
        raise KeyError(f"Chunk {chunk_id} output is missing columns: {missing_output}")

    output["occurrence_ordinal"] = pd.to_numeric(
        output["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    output["site_ordinal"] = pd.to_numeric(
        output["site_ordinal"], errors="raise"
    ).astype(np.int64)
    occurrence_ordinals = output["occurrence_ordinal"].to_numpy(dtype=np.int64)
    site_ordinals = output["site_ordinal"].to_numpy(dtype=np.int64)
    if np.any(occurrence_ordinals < 0) or np.any(
        occurrence_ordinals >= expected_occurrences
    ):
        raise RuntimeError(f"Chunk {chunk_id} has occurrence ordinals outside range.")
    if np.any(site_ordinals < 0) or np.any(site_ordinals >= expected_sites):
        raise RuntimeError(f"Chunk {chunk_id} has site ordinals outside range.")

    pair_indices = occurrence_ordinals * expected_sites + site_ordinals
    if len(np.unique(pair_indices)) != len(pair_indices):
        raise RuntimeError(f"Chunk {chunk_id} contains duplicate ordinal pairs.")
    if seen_pairs[pair_indices].any():
        raise RuntimeError(f"Chunk {chunk_id} repeats previously processed pairs.")
    seen_pairs[pair_indices] = True
    np.add.at(occurrence_counts, occurrence_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    sampled_structural = pd.to_numeric(
        output["sampled_structural_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    expected_structural = pd.to_numeric(
        output["analytical_expected_structural_ground_up_loss_2022_usd"],
        errors="raise",
    ).to_numpy(dtype=np.float64)
    sampled_nsd = pd.to_numeric(
        output["sampled_nsd_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    expected_nsd = pd.to_numeric(
        output["analytical_expected_nsd_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    sampled_nsa = pd.to_numeric(
        output["sampled_nsa_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    expected_nsa = pd.to_numeric(
        output["analytical_expected_nsa_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    sampled_total = pd.to_numeric(
        output["sampled_total_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    expected_total = pd.to_numeric(
        output["analytical_expected_total_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    sampled_total_ratios = pd.to_numeric(
        output["sampled_total_ground_up_repair_ratio"], errors="raise"
    ).to_numpy(dtype=np.float64)
    expected_total_ratios = pd.to_numeric(
        output["analytical_expected_total_ground_up_repair_ratio"], errors="raise"
    ).to_numpy(dtype=np.float64)
    replacement_values = pd.to_numeric(
        output["building_replacement_value_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)

    component_sampled_totals["structural"] += float(sampled_structural.sum())
    component_sampled_totals["nsd"] += float(sampled_nsd.sum())
    component_sampled_totals["nsa"] += float(sampled_nsa.sum())
    component_expected_totals["structural"] += float(expected_structural.sum())
    component_expected_totals["nsd"] += float(expected_nsd.sum())
    component_expected_totals["nsa"] += float(expected_nsa.sum())

    maximum_sampled_equation_error = max(
        maximum_sampled_equation_error,
        float(np.max(np.abs(sampled_total - replacement_values * sampled_total_ratios))),
    )
    maximum_expected_equation_error = max(
        maximum_expected_equation_error,
        float(np.max(np.abs(expected_total - replacement_values * expected_total_ratios))),
    )
    maximum_component_sum_error = max(
        maximum_component_sum_error,
        float(np.max(np.abs(sampled_total - sampled_structural - sampled_nsd - sampled_nsa))),
        float(np.max(np.abs(expected_total - expected_structural - expected_nsd - expected_nsa))),
    )
    maximum_sampled_total_ratio = max(
        maximum_sampled_total_ratio, float(sampled_total_ratios.max())
    )
    maximum_expected_total_ratio = max(
        maximum_expected_total_ratio, float(expected_total_ratios.max())
    )

    source_totals = output.groupby("source_type", sort=False)[
        "sampled_total_ground_up_loss_2022_usd"
    ].sum()
    for source_type, value in source_totals.items():
        source_sampled_totals[str(source_type)] += float(value)

    event_summary_frames.append(summarize_occurrences(output))
    chunk_paths.append(chunk_output_path)
    total_rows += len(output)
    seen_occurrences.update(output["occurrence_id"].astype(str).unique().tolist())
    chunk_manifest_rows.append(
        {
            "chunk_id": chunk_id,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_occurrence_ordinal": int(output["occurrence_ordinal"].min()),
            "maximum_occurrence_ordinal": int(output["occurrence_ordinal"].max()),
            "sampled_total_loss_sum_2022_usd": float(sampled_total.sum()),
            "analytical_expected_total_loss_sum_2022_usd": float(expected_total.sum()),
            "input_cell5_path": str(structural_path),
            "input_cell5_sha256": structural_hash,
            "input_cell10_path": str(nonstructural_path),
            "input_cell10_sha256": nonstructural_hash,
            "output_path": str(chunk_output_path),
            "output_size_bytes": int(chunk_output_path.stat().st_size),
            "output_sha256": sha256_file(chunk_output_path),
            "marker_path": str(marker_path),
            "validation_path": str(chunk_validation_path),
            "reused": bool(marker_valid),
        }
    )

if len(chunk_paths) != expected_chunks:
    raise RuntimeError(f"Processed {len(chunk_paths)} chunks but expected {expected_chunks}.")

concatenate_gzip_csv_files(chunk_paths, FINAL_BUILDING_LOSS_PATH)
event_summary = pd.concat(event_summary_frames, ignore_index=True)
event_summary = event_summary.sort_values("occurrence_ordinal").reset_index(drop=True)
write_gzip_csv_deterministic(event_summary, FINAL_EVENT_LOSS_PATH)

chunk_manifest = pd.DataFrame(chunk_manifest_rows)
chunk_manifest.to_csv(CHUNK_MANIFEST_PATH, index=False)

sampled_total_loss = float(sum(component_sampled_totals.values()))
expected_total_loss = float(sum(component_expected_totals.values()))
sampled_preliminary_aal = sampled_total_loss / declared_catalog_years
expected_preliminary_aal = expected_total_loss / declared_catalog_years
relative_sampled_expected_difference = (
    abs(sampled_total_loss - expected_total_loss) / expected_total_loss
    if expected_total_loss > 0.0
    else 0.0
)

cell5_sampled = float(cell5_summary["production"]["sampled_loss_total_2022_usd"])
cell5_expected = float(
    cell5_summary["production"]["analytical_expected_loss_total_2022_usd"]
)
cell10_sampled_nsd = float(
    cell10_summary["production"]["sampled_nsd_loss_total_2022_usd"]
)
cell10_expected_nsd = float(
    cell10_summary["production"]["analytical_expected_nsd_loss_total_2022_usd"]
)
cell10_sampled_nsa = float(
    cell10_summary["production"]["sampled_nsa_loss_total_2022_usd"]
)
cell10_expected_nsa = float(
    cell10_summary["production"]["analytical_expected_nsa_loss_total_2022_usd"]
)

component_reconciliation = pd.DataFrame(
    [
        {
            "component": "structural",
            "sampled_loss_total_2022_usd": component_sampled_totals["structural"],
            "prior_cell_sampled_total_2022_usd": cell5_sampled,
            "sampled_difference_2022_usd": component_sampled_totals["structural"]
            - cell5_sampled,
            "analytical_expected_loss_total_2022_usd": component_expected_totals[
                "structural"
            ],
            "prior_cell_expected_total_2022_usd": cell5_expected,
            "expected_difference_2022_usd": component_expected_totals["structural"]
            - cell5_expected,
        },
        {
            "component": "nonstructural_drift_sensitive",
            "sampled_loss_total_2022_usd": component_sampled_totals["nsd"],
            "prior_cell_sampled_total_2022_usd": cell10_sampled_nsd,
            "sampled_difference_2022_usd": component_sampled_totals["nsd"]
            - cell10_sampled_nsd,
            "analytical_expected_loss_total_2022_usd": component_expected_totals[
                "nsd"
            ],
            "prior_cell_expected_total_2022_usd": cell10_expected_nsd,
            "expected_difference_2022_usd": component_expected_totals["nsd"]
            - cell10_expected_nsd,
        },
        {
            "component": "nonstructural_acceleration_sensitive",
            "sampled_loss_total_2022_usd": component_sampled_totals["nsa"],
            "prior_cell_sampled_total_2022_usd": cell10_sampled_nsa,
            "sampled_difference_2022_usd": component_sampled_totals["nsa"]
            - cell10_sampled_nsa,
            "analytical_expected_loss_total_2022_usd": component_expected_totals[
                "nsa"
            ],
            "prior_cell_expected_total_2022_usd": cell10_expected_nsa,
            "expected_difference_2022_usd": component_expected_totals["nsa"]
            - cell10_expected_nsa,
        },
    ]
)
component_reconciliation["sampled_share_of_total"] = (
    component_reconciliation["sampled_loss_total_2022_usd"] / sampled_total_loss
)
component_reconciliation["analytical_expected_share_of_total"] = (
    component_reconciliation["analytical_expected_loss_total_2022_usd"]
    / expected_total_loss
)
component_reconciliation["sampled_aal_2022_usd"] = (
    component_reconciliation["sampled_loss_total_2022_usd"]
    / declared_catalog_years
)
component_reconciliation["analytical_expected_aal_2022_usd"] = (
    component_reconciliation["analytical_expected_loss_total_2022_usd"]
    / declared_catalog_years
)
component_reconciliation.to_csv(COMPONENT_RECONCILIATION_PATH, index=False)

sampled_prior_reconciliation_error = float(
    component_reconciliation["sampled_difference_2022_usd"].abs().max()
)
expected_prior_reconciliation_error = float(
    component_reconciliation["expected_difference_2022_usd"].abs().max()
)
event_sampled_total = float(
    event_summary["sampled_total_ground_up_loss_2022_usd"].sum()
)
event_expected_total = float(
    event_summary["analytical_expected_total_ground_up_loss_2022_usd"].sum()
)

append_check(
    validation_rows,
    "all_production_rows_processed",
    total_rows == expected_rows and seen_pairs.all(),
    f"processed={total_rows}; expected={expected_rows}; unseen={int((~seen_pairs).sum())}",
)
append_check(
    validation_rows,
    "all_catalog_occurrences_processed",
    len(seen_occurrences) == expected_occurrences,
    f"processed={len(seen_occurrences)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "every_occurrence_has_complete_portfolio",
    np.all(occurrence_counts == expected_sites),
    f"minimum={occurrence_counts.min()}; maximum={occurrence_counts.max()}",
)
append_check(
    validation_rows,
    "every_building_has_all_occurrences",
    np.all(site_counts == expected_occurrences),
    f"minimum={site_counts.min()}; maximum={site_counts.max()}",
)
append_check(
    validation_rows,
    "event_summary_has_one_row_per_occurrence",
    len(event_summary) == expected_occurrences
    and event_summary["occurrence_id"].astype(str).nunique() == expected_occurrences,
    f"rows={len(event_summary)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "event_summary_portfolios_are_complete",
    event_summary["buildings"].eq(expected_sites).all(),
    (
        f"minimum={int(event_summary['buildings'].min())}; "
        f"maximum={int(event_summary['buildings'].max())}"
    ),
)
append_check(
    validation_rows,
    "component_totals_reconcile_with_prior_cells",
    sampled_prior_reconciliation_error <= AGGREGATE_RECONCILIATION_TOLERANCE_USD
    and expected_prior_reconciliation_error <= AGGREGATE_RECONCILIATION_TOLERANCE_USD,
    (
        f"sampled_max_error={sampled_prior_reconciliation_error:.3e}; "
        f"expected_max_error={expected_prior_reconciliation_error:.3e}; "
        f"tolerance={AGGREGATE_RECONCILIATION_TOLERANCE_USD:.2f} USD"
    ),
)
append_check(
    validation_rows,
    "event_and_building_sampled_totals_reconcile",
    abs(event_sampled_total - sampled_total_loss)
    <= AGGREGATE_RECONCILIATION_TOLERANCE_USD,
    (
        f"difference={event_sampled_total - sampled_total_loss:.3e}; "
        f"tolerance={AGGREGATE_RECONCILIATION_TOLERANCE_USD:.2f} USD"
    ),
)
append_check(
    validation_rows,
    "event_and_building_expected_totals_reconcile",
    abs(event_expected_total - expected_total_loss)
    <= AGGREGATE_RECONCILIATION_TOLERANCE_USD,
    (
        f"difference={event_expected_total - expected_total_loss:.3e}; "
        f"tolerance={AGGREGATE_RECONCILIATION_TOLERANCE_USD:.2f} USD"
    ),
)
append_check(
    validation_rows,
    "maximum_sampled_total_loss_equation_error_small",
    maximum_sampled_equation_error <= LOSS_EQUATION_TOLERANCE_USD,
    f"maximum_error={maximum_sampled_equation_error:.3e}",
)
append_check(
    validation_rows,
    "maximum_expected_total_loss_equation_error_small",
    maximum_expected_equation_error <= LOSS_EQUATION_TOLERANCE_USD,
    f"maximum_error={maximum_expected_equation_error:.3e}",
)
append_check(
    validation_rows,
    "maximum_component_sum_error_small",
    maximum_component_sum_error <= LOSS_EQUATION_TOLERANCE_USD,
    f"maximum_error={maximum_component_sum_error:.3e}",
)
append_check(
    validation_rows,
    "maximum_total_repair_ratios_bounded",
    maximum_sampled_total_ratio <= 1.0 + NUMERIC_TOLERANCE
    and maximum_expected_total_ratio <= 1.0 + NUMERIC_TOLERANCE,
    (
        f"sampled={maximum_sampled_total_ratio:.6f}; "
        f"expected={maximum_expected_total_ratio:.6f}"
    ),
)
append_check(
    validation_rows,
    "sampled_and_expected_total_losses_reconcile_statistically",
    relative_sampled_expected_difference <= EXPECTED_SAMPLED_WARNING_TOLERANCE,
    (
        f"relative_difference={relative_sampled_expected_difference:.6%}; "
        f"warning_threshold={EXPECTED_SAMPLED_WARNING_TOLERANCE:.2%}"
    ),
    severity="warning",
)
append_check(
    validation_rows,
    "full_total_ground_up_loss_file_exists",
    FINAL_BUILDING_LOSS_PATH.exists() and FINAL_BUILDING_LOSS_PATH.stat().st_size > 0,
    f"path={FINAL_BUILDING_LOSS_PATH}",
)
append_check(
    validation_rows,
    "event_total_ground_up_loss_file_exists",
    FINAL_EVENT_LOSS_PATH.exists() and FINAL_EVENT_LOSS_PATH.stat().st_size > 0,
    f"path={FINAL_EVENT_LOSS_PATH}",
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
]
warnings = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
]
validation.to_csv(CELL11_VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warnings.to_dict(orient="records"),
    "scope": {
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
        "included_loss_components": [
            "structural repair",
            "drift-sensitive nonstructural repair",
            "acceleration-sensitive nonstructural repair",
        ],
        "not_included": [
            "contents loss",
            "business interruption",
            "insurance policy terms",
            "reinsurance terms",
        ],
    },
    "annual_catalog": {
        "declared_duration_years": declared_catalog_years,
        "occurrences": expected_occurrences,
        "sites": expected_sites,
        "rows": expected_rows,
        "zero_event_years_rule": (
            "Preliminary AAL uses all declared catalog years. Cell 12 creates the "
            "complete annual loss series and explicitly inserts zero-event years."
        ),
    },
    "portfolio_valuation": {
        "total_replacement_value_2022_usd": portfolio_value_cell5,
        "buildings": expected_sites,
    },
    "validation_tolerances": {
        "row_equation_usd": LOSS_EQUATION_TOLERANCE_USD,
        "aggregate_reconciliation_usd": AGGREGATE_RECONCILIATION_TOLERANCE_USD,
        "rationale": (
            "Row-level equations retain a micro-dollar tolerance. Aggregate "
            "reconciliations allow one cent for floating-point summation-order "
            "differences across nearly five million rows."
        ),
    },
    "loss_equations": {
        "sampled_total": (
            "sampled total ground-up loss = sampled structural loss + sampled NSD "
            "loss + sampled NSA loss"
        ),
        "analytical_expected_total": (
            "expected total ground-up loss = expected structural loss + expected NSD "
            "loss + expected NSA loss"
        ),
        "repair_ratio_bound": (
            "structural + NSD + NSA repair ratios remain between 0 and 1 because "
            "the three Complete-state component ratios reconcile to total replacement"
        ),
    },
    "production": {
        "rows": total_rows,
        "occurrences": len(seen_occurrences),
        "chunks": len(chunk_paths),
        "sampled_structural_loss_total_2022_usd": component_sampled_totals[
            "structural"
        ],
        "analytical_expected_structural_loss_total_2022_usd": component_expected_totals[
            "structural"
        ],
        "sampled_nsd_loss_total_2022_usd": component_sampled_totals["nsd"],
        "analytical_expected_nsd_loss_total_2022_usd": component_expected_totals[
            "nsd"
        ],
        "sampled_nsa_loss_total_2022_usd": component_sampled_totals["nsa"],
        "analytical_expected_nsa_loss_total_2022_usd": component_expected_totals[
            "nsa"
        ],
        "sampled_total_ground_up_loss_2022_usd": sampled_total_loss,
        "analytical_expected_total_ground_up_loss_2022_usd": expected_total_loss,
        "sampled_preliminary_aal_2022_usd": sampled_preliminary_aal,
        "analytical_expected_preliminary_aal_2022_usd": expected_preliminary_aal,
        "relative_sampled_expected_difference": relative_sampled_expected_difference,
        "maximum_sampled_total_ground_up_repair_ratio": maximum_sampled_total_ratio,
        "maximum_expected_total_ground_up_repair_ratio": maximum_expected_total_ratio,
        "source_sampled_loss_totals_2022_usd": dict(source_sampled_totals),
    },
    "component_shares": {
        row["component"]: {
            "sampled_share": float(row["sampled_share_of_total"]),
            "analytical_expected_share": float(
                row["analytical_expected_share_of_total"]
            ),
        }
        for _, row in component_reconciliation.iterrows()
    },
    "outputs": {
        "full_total_ground_up_loss": {
            "path": str(FINAL_BUILDING_LOSS_PATH),
            "rows": total_rows,
            "size_bytes": int(FINAL_BUILDING_LOSS_PATH.stat().st_size),
            "sha256": sha256_file(FINAL_BUILDING_LOSS_PATH),
            "row_granularity": "one annual-catalog occurrence and one portfolio building",
        },
        "event_total_ground_up_loss": {
            "path": str(FINAL_EVENT_LOSS_PATH),
            "rows": int(len(event_summary)),
            "size_bytes": int(FINAL_EVENT_LOSS_PATH.stat().st_size),
            "sha256": sha256_file(FINAL_EVENT_LOSS_PATH),
        },
        "component_reconciliation": str(COMPONENT_RECONCILIATION_PATH),
        "chunk_manifest": str(CHUNK_MANIFEST_PATH),
        "validation": str(CELL11_VALIDATION_PATH),
        "summary": str(CELL11_SUMMARY_PATH),
    },
    "next_cell": (
        "Cell 12: construct the complete annual total ground-up loss series, insert "
        "all zero-event years, and calculate total ground-up AAL, OEP, AEP, and PML."
    ),
}
write_json(CELL11_SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 11 failed critical validation. Review "
        f"{CELL11_VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 5 CELL 11 TOTAL GROUND-UP LOSS COMPLETE")
print("=" * 78)
print(f"Production loss rows:                 {total_rows:,}")
print(f"Catalog occurrences:                  {len(seen_occurrences):,}")
print(f"Portfolio replacement value:          ${portfolio_value_cell5:,.0f}")
print(f"Sampled structural loss total:        ${component_sampled_totals['structural']:,.0f}")
print(f"Sampled NSD loss total:               ${component_sampled_totals['nsd']:,.0f}")
print(f"Sampled NSA loss total:               ${component_sampled_totals['nsa']:,.0f}")
print(f"Sampled total ground-up loss:         ${sampled_total_loss:,.0f}")
print(f"Expected total ground-up loss:        ${expected_total_loss:,.0f}")
print(f"Sampled preliminary AAL:              ${sampled_preliminary_aal:,.2f}")
print(f"Expected preliminary AAL:             ${expected_preliminary_aal:,.2f}")
print(
    "Sampled-expected relative diff.:    "
    f"{relative_sampled_expected_difference:.4%}"
)
print(f"Maximum sampled total repair ratio:   {maximum_sampled_total_ratio:.6f}")
print(f"Maximum expected total repair ratio:  {maximum_expected_total_ratio:.6f}")
print(f"Critical validation checks:           {len(validation.loc[validation['severity'].eq('critical')]):,}")
print(f"Critical failures:                    {len(critical_failures):,}")
print(f"Warnings requiring review:            {len(warnings):,}")
print()
print("Full total ground-up loss:")
print(f"  {FINAL_BUILDING_LOSS_PATH}")
print("Occurrence-level total ground-up loss:")
print(f"  {FINAL_EVENT_LOSS_PATH}")
print("Component reconciliation:")
print(f"  {COMPONENT_RECONCILIATION_PATH}")
print("Validation:")
print(f"  {CELL11_VALIDATION_PATH}")
print("Summary:")
print(f"  {CELL11_SUMMARY_PATH}")
print()
print("Next: calculate total ground-up annual loss and risk metrics.")


FULL ANNUAL-CATALOG TOTAL GROUND-UP LOSS
Structural loss rows:          4,996,100
Nonstructural loss rows:       4,996,100
Catalog occurrences:           10,630
Portfolio buildings:           470
Production chunks:             43
Portfolio replacement value:   $384,236,605 (2022 USD)
Loss scope:                     structural + NSD + NSA repair

------------------------------------------------------------------------------
TOTAL GROUND-UP LOSS CHUNK 1 OF 43 [ID 0000]
Reused validated total-loss chunk with 117,500 rows.

------------------------------------------------------------------------------
TOTAL GROUND-UP LOSS CHUNK 2 OF 43 [ID 0001]
Reused validated total-loss chunk with 117,500 rows.

------------------------------------------------------------------------------
TOTAL GROUND-UP LOSS CHUNK 3 OF 43 [ID 0021]
Reused validated total-loss chunk with 117,500 rows.

------------------------------------------------------------------------------
TOTAL GROUND-UP LOSS CHUNK 4 OF 43 [ID 

In [25]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook5_cell12_total_ground_up_risk_metrics_v1"
DOLLAR_YEAR = 2022
RETURN_PERIODS_YEARS = [
    50,
    100,
    200,
    250,
    500,
    1_000,
    2_000,
    2_500,
    5_000,
    10_000,
    20_000,
    50_000,
    100_000,
    200_000,
    500_000,
    1_000_000,
    2_000_000,
]
NUMERIC_ATOL_USD = 1e-4
TAIL_SUPPORT_WARNING_RANK = 20


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "05_calculate_ground_up_losses.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_11_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 5 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_output,
            compresslevel=6,
            mtime=0,
        ) as compressed_output:
            with io.TextIOWrapper(
                compressed_output,
                encoding="utf-8",
                newline="",
            ) as text_output:
                frame.to_csv(text_output, index=False, lineterminator="\n")
    temporary.replace(path)


def empirical_pml(
    losses: np.ndarray,
    return_period_years: int,
) -> tuple[float, int, float]:
    n_years = int(losses.size)
    if return_period_years <= 0 or return_period_years > n_years:
        raise ValueError(
            f"Return period must be in [1, {n_years}], "
            f"received {return_period_years}."
        )
    descending_rank = max(1, int(math.ceil(n_years / return_period_years)))
    sorted_losses = np.sort(losses)[::-1]
    loss = float(sorted_losses[descending_rank - 1])
    empirical_aep = descending_rank / n_years
    return loss, descending_rank, empirical_aep


def build_ep_curve(
    losses: np.ndarray,
    curve_type: str,
    loss_basis: str,
    n_years: int,
) -> pd.DataFrame:
    positive = losses[losses > 0.0]
    if positive.size == 0:
        return pd.DataFrame(
            columns=[
                "curve_type",
                "loss_basis",
                "descending_rank",
                "annual_exceedance_probability",
                "return_period_years",
                "loss_2022_usd",
            ]
        )

    ordered = np.sort(positive)[::-1]
    ranks = np.arange(1, ordered.size + 1, dtype=np.int64)
    return pd.DataFrame(
        {
            "curve_type": curve_type,
            "loss_basis": loss_basis,
            "descending_rank": ranks,
            "annual_exceedance_probability": ranks / n_years,
            "return_period_years": n_years / ranks,
            "loss_2022_usd": ordered,
        }
    )


def annual_statistics(losses: np.ndarray) -> dict[str, float | int]:
    mean_loss = float(np.mean(losses))
    standard_deviation = float(np.std(losses, ddof=0))
    positive = losses[losses > 0.0]
    return {
        "years": int(losses.size),
        "positive_loss_years": int(positive.size),
        "zero_loss_years": int(losses.size - positive.size),
        "annual_probability_of_positive_loss": float(positive.size / losses.size),
        "mean_annual_loss_2022_usd": mean_loss,
        "annual_loss_standard_deviation_2022_usd": standard_deviation,
        "annual_loss_cov": (
            float(standard_deviation / mean_loss) if mean_loss > 0.0 else 0.0
        ),
        "mean_positive_year_loss_2022_usd": (
            float(np.mean(positive)) if positive.size else 0.0
        ),
        "median_positive_year_loss_2022_usd": (
            float(np.median(positive)) if positive.size else 0.0
        ),
        "maximum_annual_loss_2022_usd": float(np.max(losses)),
    }


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_total_ground_up_risk"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CELL11_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_11_summary.json"
CELL11_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_11_validation.csv"
HANDOFF_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_4_final_handoff"
    / "notebook_5_input_handoff.json"
)
ANNUAL_SERIES_PATH = OUTPUT_DIR / "total_ground_up_annual_loss_series.csv.gz"
EP_CURVE_PATH = OUTPUT_DIR / "total_ground_up_exceedance_probability_curve.csv.gz"
PML_TABLE_PATH = OUTPUT_DIR / "total_ground_up_pml_table.csv"
RISK_METRICS_PATH = OUTPUT_DIR / "total_ground_up_risk_metrics.csv"
SOURCE_AAL_PATH = OUTPUT_DIR / "total_ground_up_source_aal_summary.csv"
COMPONENT_AAL_PATH = OUTPUT_DIR / "total_ground_up_component_aal_summary.csv"
PLOT_PNG_PATH = PLOT_DIR / "total_ground_up_aep_oep_curve.png"
PLOT_PDF_PATH = PLOT_DIR / "total_ground_up_aep_oep_curve.pdf"
CELL12_VALIDATION_PATH = METADATA_DIR / "notebook_5_cell_12_validation.csv"
CELL12_SUMMARY_PATH = METADATA_DIR / "notebook_5_cell_12_summary.json"

validation_rows: list[dict[str, Any]] = []
append_check(
    validation_rows,
    "cell11_summary_exists",
    CELL11_SUMMARY_PATH.exists(),
    str(CELL11_SUMMARY_PATH),
)
append_check(
    validation_rows,
    "cell11_validation_exists",
    CELL11_VALIDATION_PATH.exists(),
    str(CELL11_VALIDATION_PATH),
)
append_check(
    validation_rows,
    "notebook4_handoff_exists",
    HANDOFF_PATH.exists(),
    str(HANDOFF_PATH),
)

if not all(
    path.exists()
    for path in [CELL11_SUMMARY_PATH, CELL11_VALIDATION_PATH, HANDOFF_PATH]
):
    missing = [
        str(path)
        for path in [CELL11_SUMMARY_PATH, CELL11_VALIDATION_PATH, HANDOFF_PATH]
        if not path.exists()
    ]
    raise FileNotFoundError(f"Required inputs are missing: {missing}")

cell11_summary = load_json(CELL11_SUMMARY_PATH)
handoff = load_json(HANDOFF_PATH)
cell11_validation = pd.read_csv(CELL11_VALIDATION_PATH)
cell11_validation["passed"] = parse_bool_series(cell11_validation["passed"])
cell11_critical_failures = cell11_validation.loc[
    cell11_validation["severity"].eq("critical")
    & ~cell11_validation["passed"]
]
append_check(
    validation_rows,
    "cell11_critical_checks_passed",
    cell11_critical_failures.empty
    and bool(cell11_summary.get("all_critical_checks_passed")),
    f"critical_failures={len(cell11_critical_failures)}",
)

annual_catalog = handoff["annual_catalog"]
declared_catalog_years = int(annual_catalog["declared_duration_years"])
expected_occurrences = int(annual_catalog["occurrences"])
expected_occupied_years = int(annual_catalog["occupied_years"])
expected_zero_event_years = int(annual_catalog["zero_event_years"])
expected_multiple_event_years = int(annual_catalog["multiple_event_years"])
expected_maximum_events = int(annual_catalog["maximum_events_in_one_year"])

append_check(
    validation_rows,
    "declared_catalog_duration_positive",
    declared_catalog_years > 0,
    f"years={declared_catalog_years}",
)
append_check(
    validation_rows,
    "return_periods_supported_by_catalog",
    max(RETURN_PERIODS_YEARS) <= declared_catalog_years,
    (
        f"maximum_return_period={max(RETURN_PERIODS_YEARS)}; "
        f"catalog_years={declared_catalog_years}"
    ),
)

record = cell11_summary["outputs"]["event_total_ground_up_loss"]
event_loss_path = resolve_recorded_path(PROJECT_ROOT, str(record["path"]))
actual_event_hash = sha256_file(event_loss_path)
append_check(
    validation_rows,
    "cell11_event_loss_hash_matches",
    actual_event_hash == str(record["sha256"]),
    f"expected={record['sha256']}; actual={actual_event_hash}",
)

events = pd.read_csv(event_loss_path, compression="gzip")
required_event_columns = {
    "catalog_year",
    "occurrence_id",
    "source_type",
    "portfolio_replacement_value_2022_usd",
    "sampled_structural_ground_up_loss_2022_usd",
    "analytical_expected_structural_ground_up_loss_2022_usd",
    "sampled_nsd_ground_up_loss_2022_usd",
    "analytical_expected_nsd_ground_up_loss_2022_usd",
    "sampled_nsa_ground_up_loss_2022_usd",
    "analytical_expected_nsa_ground_up_loss_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "analytical_expected_total_ground_up_loss_2022_usd",
}
missing_event_columns = sorted(required_event_columns.difference(events.columns))
append_check(
    validation_rows,
    "event_loss_schema_complete",
    not missing_event_columns,
    f"missing={missing_event_columns}",
)
if missing_event_columns:
    raise RuntimeError(
        f"Cell 11 event-loss file is missing columns: {missing_event_columns}"
    )

numeric_event_columns = [
    "catalog_year",
    "portfolio_replacement_value_2022_usd",
    "sampled_structural_ground_up_loss_2022_usd",
    "analytical_expected_structural_ground_up_loss_2022_usd",
    "sampled_nsd_ground_up_loss_2022_usd",
    "analytical_expected_nsd_ground_up_loss_2022_usd",
    "sampled_nsa_ground_up_loss_2022_usd",
    "analytical_expected_nsa_ground_up_loss_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "analytical_expected_total_ground_up_loss_2022_usd",
]
for column in numeric_event_columns:
    events[column] = pd.to_numeric(events[column], errors="raise")
events["catalog_year"] = events["catalog_year"].astype(np.int64)

append_check(
    validation_rows,
    "event_summary_occurrence_count_matches",
    len(events) == expected_occurrences and events["occurrence_id"].is_unique,
    (
        f"rows={len(events)}; expected={expected_occurrences}; "
        f"unique={events['occurrence_id'].nunique()}"
    ),
)
append_check(
    validation_rows,
    "catalog_year_labels_within_declared_duration",
    events["catalog_year"].between(1, declared_catalog_years).all(),
    (
        f"minimum={events['catalog_year'].min()}; "
        f"maximum={events['catalog_year'].max()}"
    ),
)
loss_columns = [
    "sampled_structural_ground_up_loss_2022_usd",
    "analytical_expected_structural_ground_up_loss_2022_usd",
    "sampled_nsd_ground_up_loss_2022_usd",
    "analytical_expected_nsd_ground_up_loss_2022_usd",
    "sampled_nsa_ground_up_loss_2022_usd",
    "analytical_expected_nsa_ground_up_loss_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "analytical_expected_total_ground_up_loss_2022_usd",
]
loss_matrix = events[loss_columns].to_numpy(dtype=np.float64)
append_check(
    validation_rows,
    "event_losses_finite_and_nonnegative",
    np.isfinite(loss_matrix).all() and np.all(loss_matrix >= 0.0),
    "all occurrence-level component and total losses checked",
)

sampled_component_sum = (
    events["sampled_structural_ground_up_loss_2022_usd"]
    + events["sampled_nsd_ground_up_loss_2022_usd"]
    + events["sampled_nsa_ground_up_loss_2022_usd"]
)
expected_component_sum = (
    events["analytical_expected_structural_ground_up_loss_2022_usd"]
    + events["analytical_expected_nsd_ground_up_loss_2022_usd"]
    + events["analytical_expected_nsa_ground_up_loss_2022_usd"]
)
append_check(
    validation_rows,
    "event_sampled_components_sum_to_total",
    np.allclose(
        sampled_component_sum,
        events["sampled_total_ground_up_loss_2022_usd"],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    (
        "maximum_error="
        f"{np.max(np.abs(sampled_component_sum - events['sampled_total_ground_up_loss_2022_usd'])):.3e}"
    ),
)
append_check(
    validation_rows,
    "event_expected_components_sum_to_total",
    np.allclose(
        expected_component_sum,
        events["analytical_expected_total_ground_up_loss_2022_usd"],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    (
        "maximum_error="
        f"{np.max(np.abs(expected_component_sum - events['analytical_expected_total_ground_up_loss_2022_usd'])):.3e}"
    ),
)

annual_occupied = (
    events.groupby("catalog_year", sort=True)
    .agg(
        event_count=("occurrence_id", "size"),
        sampled_aep_total_ground_up_loss_2022_usd=(
            "sampled_total_ground_up_loss_2022_usd",
            "sum",
        ),
        sampled_oep_total_ground_up_loss_2022_usd=(
            "sampled_total_ground_up_loss_2022_usd",
            "max",
        ),
        analytical_expected_aep_total_ground_up_loss_2022_usd=(
            "analytical_expected_total_ground_up_loss_2022_usd",
            "sum",
        ),
        analytical_expected_oep_total_ground_up_loss_2022_usd=(
            "analytical_expected_total_ground_up_loss_2022_usd",
            "max",
        ),
    )
    .reset_index()
)

catalog_year = np.arange(1, declared_catalog_years + 1, dtype=np.int64)
event_count = np.zeros(declared_catalog_years, dtype=np.int16)
sampled_aep = np.zeros(declared_catalog_years, dtype=np.float64)
sampled_oep = np.zeros(declared_catalog_years, dtype=np.float64)
expected_aep = np.zeros(declared_catalog_years, dtype=np.float64)
expected_oep = np.zeros(declared_catalog_years, dtype=np.float64)

positions = annual_occupied["catalog_year"].to_numpy(dtype=np.int64) - 1
event_count[positions] = annual_occupied["event_count"].to_numpy(dtype=np.int16)
sampled_aep[positions] = annual_occupied[
    "sampled_aep_total_ground_up_loss_2022_usd"
].to_numpy(dtype=np.float64)
sampled_oep[positions] = annual_occupied[
    "sampled_oep_total_ground_up_loss_2022_usd"
].to_numpy(dtype=np.float64)
expected_aep[positions] = annual_occupied[
    "analytical_expected_aep_total_ground_up_loss_2022_usd"
].to_numpy(dtype=np.float64)
expected_oep[positions] = annual_occupied[
    "analytical_expected_oep_total_ground_up_loss_2022_usd"
].to_numpy(dtype=np.float64)

annual_series = pd.DataFrame(
    {
        "catalog_year": catalog_year,
        "event_count": event_count,
        "zero_event_year": event_count == 0,
        "sampled_aep_total_ground_up_loss_2022_usd": sampled_aep,
        "sampled_oep_total_ground_up_loss_2022_usd": sampled_oep,
        "analytical_expected_aep_total_ground_up_loss_2022_usd": expected_aep,
        "analytical_expected_oep_total_ground_up_loss_2022_usd": expected_oep,
    }
)
write_gzip_csv_deterministic(annual_series, ANNUAL_SERIES_PATH)

occupied_years = int(np.count_nonzero(event_count))
zero_event_years = int(np.count_nonzero(event_count == 0))
multiple_event_years = int(np.count_nonzero(event_count > 1))
maximum_events_in_year = int(event_count.max())

append_check(
    validation_rows,
    "annual_series_contains_all_catalog_years",
    len(annual_series) == declared_catalog_years
    and annual_series["catalog_year"].iloc[0] == 1
    and annual_series["catalog_year"].iloc[-1] == declared_catalog_years,
    (
        f"rows={len(annual_series)}; first={annual_series['catalog_year'].iloc[0]}; "
        f"last={annual_series['catalog_year'].iloc[-1]}"
    ),
)
append_check(
    validation_rows,
    "event_counts_reconcile",
    int(event_count.sum()) == expected_occurrences,
    f"annual_sum={int(event_count.sum())}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "occupied_year_count_matches_handoff",
    occupied_years == expected_occupied_years,
    f"actual={occupied_years}; expected={expected_occupied_years}",
)
append_check(
    validation_rows,
    "zero_event_year_count_matches_handoff",
    zero_event_years == expected_zero_event_years,
    f"actual={zero_event_years}; expected={expected_zero_event_years}",
)
append_check(
    validation_rows,
    "multiple_event_year_count_matches_handoff",
    multiple_event_years == expected_multiple_event_years,
    f"actual={multiple_event_years}; expected={expected_multiple_event_years}",
)
append_check(
    validation_rows,
    "maximum_events_per_year_matches_handoff",
    maximum_events_in_year == expected_maximum_events,
    f"actual={maximum_events_in_year}; expected={expected_maximum_events}",
)
append_check(
    validation_rows,
    "aep_not_less_than_oep",
    np.all(sampled_aep + NUMERIC_ATOL_USD >= sampled_oep)
    and np.all(expected_aep + NUMERIC_ATOL_USD >= expected_oep),
    "checked sampled and analytical expected annual losses",
)
append_check(
    validation_rows,
    "single_and_zero_event_years_have_equal_aep_oep",
    np.allclose(
        sampled_aep[event_count <= 1],
        sampled_oep[event_count <= 1],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    )
    and np.allclose(
        expected_aep[event_count <= 1],
        expected_oep[event_count <= 1],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    f"years_checked={int(np.count_nonzero(event_count <= 1))}",
)
append_check(
    validation_rows,
    "sampled_annual_total_matches_event_total",
    math.isclose(
        float(sampled_aep.sum()),
        float(events["sampled_total_ground_up_loss_2022_usd"].sum()),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"annual={sampled_aep.sum():.6f}; "
        f"event={events['sampled_total_ground_up_loss_2022_usd'].sum():.6f}"
    ),
)
append_check(
    validation_rows,
    "expected_annual_total_matches_event_total",
    math.isclose(
        float(expected_aep.sum()),
        float(events["analytical_expected_total_ground_up_loss_2022_usd"].sum()),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"annual={expected_aep.sum():.6f}; "
        f"event={events['analytical_expected_total_ground_up_loss_2022_usd'].sum():.6f}"
    ),
)

sampled_aal = float(sampled_aep.mean())
expected_aal = float(expected_aep.mean())
cell11_sampled_aal = float(
    cell11_summary["production"]["sampled_preliminary_aal_2022_usd"]
)
cell11_expected_aal = float(
    cell11_summary["production"]["analytical_expected_preliminary_aal_2022_usd"]
)
append_check(
    validation_rows,
    "sampled_aal_matches_cell11",
    math.isclose(
        sampled_aal,
        cell11_sampled_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    f"cell12={sampled_aal:.9f}; cell11={cell11_sampled_aal:.9f}",
)
append_check(
    validation_rows,
    "expected_aal_matches_cell11",
    math.isclose(
        expected_aal,
        cell11_expected_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    f"cell12={expected_aal:.9f}; cell11={cell11_expected_aal:.9f}",
)

curve_frames = [
    build_ep_curve(sampled_aep, "AEP", "sampled", declared_catalog_years),
    build_ep_curve(sampled_oep, "OEP", "sampled", declared_catalog_years),
    build_ep_curve(
        expected_aep,
        "AEP",
        "analytical_expected",
        declared_catalog_years,
    ),
    build_ep_curve(
        expected_oep,
        "OEP",
        "analytical_expected",
        declared_catalog_years,
    ),
]
ep_curve = pd.concat(curve_frames, ignore_index=True)
write_gzip_csv_deterministic(ep_curve, EP_CURVE_PATH)

pml_rows: list[dict[str, Any]] = []
loss_vectors = {
    ("sampled", "AEP"): sampled_aep,
    ("sampled", "OEP"): sampled_oep,
    ("analytical_expected", "AEP"): expected_aep,
    ("analytical_expected", "OEP"): expected_oep,
}
for return_period in RETURN_PERIODS_YEARS:
    row: dict[str, Any] = {
        "return_period_years": int(return_period),
        "target_annual_exceedance_probability": 1.0 / return_period,
    }
    supporting_ranks: list[int] = []
    for (loss_basis, curve_type), values in loss_vectors.items():
        loss, rank, empirical_aep = empirical_pml(values, return_period)
        prefix = f"{loss_basis}_{curve_type.lower()}"
        row[f"{prefix}_pml_2022_usd"] = loss
        row[f"{prefix}_supporting_descending_rank"] = rank
        row[f"{prefix}_empirical_aep"] = empirical_aep
        supporting_ranks.append(rank)
    row["minimum_supporting_descending_rank"] = min(supporting_ranks)
    row["tail_support_flag"] = (
        "thin_tail_support"
        if min(supporting_ranks) < TAIL_SUPPORT_WARNING_RANK
        else "adequate_order_statistic_support"
    )
    pml_rows.append(row)
pml_table = pd.DataFrame(pml_rows)
pml_table.to_csv(PML_TABLE_PATH, index=False)

for loss_basis in ["sampled", "analytical_expected"]:
    for curve_type in ["aep", "oep"]:
        column = f"{loss_basis}_{curve_type}_pml_2022_usd"
        differences = np.diff(pml_table[column].to_numpy(dtype=float))
        minimum_difference = float(differences.min()) if differences.size else 0.0
        append_check(
            validation_rows,
            f"{loss_basis}_{curve_type}_pml_nondecreasing_with_return_period",
            np.all(differences >= -NUMERIC_ATOL_USD),
            f"minimum_difference={minimum_difference:.6f}",
        )
append_check(
    validation_rows,
    "sampled_aep_pml_not_less_than_sampled_oep_pml",
    np.all(
        pml_table["sampled_aep_pml_2022_usd"].to_numpy(dtype=float)
        + NUMERIC_ATOL_USD
        >= pml_table["sampled_oep_pml_2022_usd"].to_numpy(dtype=float)
    ),
    "checked all requested return periods",
)
append_check(
    validation_rows,
    "expected_aep_pml_not_less_than_expected_oep_pml",
    np.all(
        pml_table["analytical_expected_aep_pml_2022_usd"].to_numpy(dtype=float)
        + NUMERIC_ATOL_USD
        >= pml_table["analytical_expected_oep_pml_2022_usd"].to_numpy(dtype=float)
    ),
    "checked all requested return periods",
)

risk_rows: list[dict[str, Any]] = []
for loss_basis, curve_type, values in [
    ("sampled", "AEP", sampled_aep),
    ("sampled", "OEP", sampled_oep),
    ("analytical_expected", "AEP", expected_aep),
    ("analytical_expected", "OEP", expected_oep),
]:
    risk_rows.append(
        {
            "loss_basis": loss_basis,
            "curve_type": curve_type,
            **annual_statistics(values),
        }
    )
risk_metrics = pd.DataFrame(risk_rows)
risk_metrics.to_csv(RISK_METRICS_PATH, index=False)

source_aal = (
    events.groupby("source_type", sort=True)
    .agg(
        occurrences=("occurrence_id", "size"),
        sampled_total_ground_up_loss_2022_usd=(
            "sampled_total_ground_up_loss_2022_usd",
            "sum",
        ),
        analytical_expected_total_ground_up_loss_2022_usd=(
            "analytical_expected_total_ground_up_loss_2022_usd",
            "sum",
        ),
        sampled_structural_ground_up_loss_2022_usd=(
            "sampled_structural_ground_up_loss_2022_usd",
            "sum",
        ),
        sampled_nsd_ground_up_loss_2022_usd=(
            "sampled_nsd_ground_up_loss_2022_usd",
            "sum",
        ),
        sampled_nsa_ground_up_loss_2022_usd=(
            "sampled_nsa_ground_up_loss_2022_usd",
            "sum",
        ),
    )
    .reset_index()
)
source_aal["sampled_total_ground_up_aal_2022_usd"] = (
    source_aal["sampled_total_ground_up_loss_2022_usd"]
    / declared_catalog_years
)
source_aal["analytical_expected_total_ground_up_aal_2022_usd"] = (
    source_aal["analytical_expected_total_ground_up_loss_2022_usd"]
    / declared_catalog_years
)
source_aal["sampled_aal_share"] = np.where(
    sampled_aal > 0.0,
    source_aal["sampled_total_ground_up_aal_2022_usd"] / sampled_aal,
    0.0,
)
source_aal["analytical_expected_aal_share"] = np.where(
    expected_aal > 0.0,
    source_aal["analytical_expected_total_ground_up_aal_2022_usd"]
    / expected_aal,
    0.0,
)
source_aal.to_csv(SOURCE_AAL_PATH, index=False)

append_check(
    validation_rows,
    "source_sampled_aal_reconciles",
    math.isclose(
        float(source_aal["sampled_total_ground_up_aal_2022_usd"].sum()),
        sampled_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        "source_sum="
        f"{source_aal['sampled_total_ground_up_aal_2022_usd'].sum():.9f}; "
        f"portfolio={sampled_aal:.9f}"
    ),
)
append_check(
    validation_rows,
    "source_expected_aal_reconciles",
    math.isclose(
        float(
            source_aal[
                "analytical_expected_total_ground_up_aal_2022_usd"
            ].sum()
        ),
        expected_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        "source_sum="
        f"{source_aal['analytical_expected_total_ground_up_aal_2022_usd'].sum():.9f}; "
        f"portfolio={expected_aal:.9f}"
    ),
)

component_rows = []
for component, sampled_column, expected_column in [
    (
        "structural",
        "sampled_structural_ground_up_loss_2022_usd",
        "analytical_expected_structural_ground_up_loss_2022_usd",
    ),
    (
        "nonstructural_drift_sensitive",
        "sampled_nsd_ground_up_loss_2022_usd",
        "analytical_expected_nsd_ground_up_loss_2022_usd",
    ),
    (
        "nonstructural_acceleration_sensitive",
        "sampled_nsa_ground_up_loss_2022_usd",
        "analytical_expected_nsa_ground_up_loss_2022_usd",
    ),
]:
    sampled_total = float(events[sampled_column].sum())
    expected_total = float(events[expected_column].sum())
    component_rows.append(
        {
            "component": component,
            "sampled_loss_total_2022_usd": sampled_total,
            "analytical_expected_loss_total_2022_usd": expected_total,
            "sampled_aal_2022_usd": sampled_total / declared_catalog_years,
            "analytical_expected_aal_2022_usd": (
                expected_total / declared_catalog_years
            ),
            "sampled_share_of_total_aal": (
                sampled_total / float(events["sampled_total_ground_up_loss_2022_usd"].sum())
            ),
            "analytical_expected_share_of_total_aal": (
                expected_total
                / float(
                    events[
                        "analytical_expected_total_ground_up_loss_2022_usd"
                    ].sum()
                )
            ),
        }
    )
component_aal = pd.DataFrame(component_rows)
component_aal.to_csv(COMPONENT_AAL_PATH, index=False)
append_check(
    validation_rows,
    "component_sampled_aal_reconciles",
    math.isclose(
        float(component_aal["sampled_aal_2022_usd"].sum()),
        sampled_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"component_sum={component_aal['sampled_aal_2022_usd'].sum():.9f}; "
        f"portfolio={sampled_aal:.9f}"
    ),
)
append_check(
    validation_rows,
    "component_expected_aal_reconciles",
    math.isclose(
        float(component_aal["analytical_expected_aal_2022_usd"].sum()),
        expected_aal,
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        "component_sum="
        f"{component_aal['analytical_expected_aal_2022_usd'].sum():.9f}; "
        f"portfolio={expected_aal:.9f}"
    ),
)

plot_sampled = ep_curve.loc[ep_curve["loss_basis"].eq("sampled")].copy()
fig, ax = plt.subplots(figsize=(8.0, 5.0))
for curve_type in ["OEP", "AEP"]:
    subset = plot_sampled.loc[plot_sampled["curve_type"].eq(curve_type)]
    ax.plot(
        subset["return_period_years"],
        subset["loss_2022_usd"] / 1_000_000.0,
        linewidth=1.8,
        label=f"Sampled {curve_type}",
    )
ax.set_xscale("log")
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Total ground-up loss (million 2022 USD)")
ax.set_title("Total ground-up annual exceedance curves")
ax.grid(True, which="both", linewidth=0.4, alpha=0.35)
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_PNG_PATH, dpi=300)
fig.savefig(PLOT_PDF_PATH)
plt.close(fig)

for output_path, check_id in [
    (ANNUAL_SERIES_PATH, "annual_series_file_exists"),
    (EP_CURVE_PATH, "ep_curve_file_exists"),
    (PML_TABLE_PATH, "pml_table_file_exists"),
    (RISK_METRICS_PATH, "risk_metrics_file_exists"),
    (SOURCE_AAL_PATH, "source_aal_file_exists"),
    (COMPONENT_AAL_PATH, "component_aal_file_exists"),
    (PLOT_PNG_PATH, "ep_plot_png_exists"),
    (PLOT_PDF_PATH, "ep_plot_pdf_exists"),
]:
    append_check(
        validation_rows,
        check_id,
        output_path.exists() and output_path.stat().st_size > 0,
        str(output_path),
    )

thin_tail_return_periods = pml_table.loc[
    pml_table["tail_support_flag"].eq("thin_tail_support"),
    "return_period_years",
].astype(int).tolist()
append_check(
    validation_rows,
    "extreme_return_period_tail_support_documented",
    len(thin_tail_return_periods) == 0,
    (
        f"thin_tail_return_periods={thin_tail_return_periods}; "
        f"threshold_rank={TAIL_SUPPORT_WARNING_RANK}"
    ),
    severity="warning",
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].eq("critical") & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].eq("warning") & ~validation["passed"]
].copy()
validation.to_csv(CELL12_VALIDATION_PATH, index=False)

portfolio_replacement_value = float(
    events["portfolio_replacement_value_2022_usd"].iloc[0]
)
summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warning_failures.to_dict(orient="records"),
    "scope": {
        "loss_components": [
            "structural repair",
            "drift-sensitive nonstructural repair",
            "acceleration-sensitive nonstructural repair",
        ],
        "loss_basis": "total ground-up building repair loss",
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
        "official_ep_basis": (
            "sampled structural, NSD, and NSA damage states and sampled total "
            "ground-up losses"
        ),
        "analytical_expected_curves_use": (
            "reconciliation diagnostic based on conditional expected component "
            "losses; not a replacement for the sampled annual loss distribution"
        ),
    },
    "annual_catalog": {
        "declared_duration_years": declared_catalog_years,
        "occurrences": expected_occurrences,
        "occupied_years": occupied_years,
        "multiple_event_years": multiple_event_years,
        "zero_event_years": zero_event_years,
        "maximum_events_in_one_year": maximum_events_in_year,
    },
    "portfolio": {
        "replacement_value_2022_usd": portfolio_replacement_value,
    },
    "risk_metrics": {
        "sampled_total_ground_up_aal_2022_usd": sampled_aal,
        "analytical_expected_total_ground_up_aal_2022_usd": expected_aal,
        "sampled_positive_aep_years": int(np.count_nonzero(sampled_aep > 0.0)),
        "sampled_positive_oep_years": int(np.count_nonzero(sampled_oep > 0.0)),
        "maximum_sampled_annual_aggregate_loss_2022_usd": float(
            sampled_aep.max()
        ),
        "maximum_sampled_annual_occurrence_loss_2022_usd": float(
            sampled_oep.max()
        ),
        "maximum_sampled_aep_loss_ratio": float(
            sampled_aep.max() / portfolio_replacement_value
        ),
        "maximum_sampled_oep_loss_ratio": float(
            sampled_oep.max() / portfolio_replacement_value
        ),
    },
    "pml_method": {
        "definition": (
            "For return period R, sort all declared annual losses in descending "
            "order and select rank ceil(N/R), where N is the declared catalog "
            "duration."
        ),
        "return_periods_years": RETURN_PERIODS_YEARS,
        "tail_support_warning_rank": TAIL_SUPPORT_WARNING_RANK,
        "thin_tail_return_periods": thin_tail_return_periods,
    },
    "source_files": {
        "cell11_event_loss_path": str(event_loss_path),
        "cell11_event_loss_sha256": actual_event_hash,
        "cell11_summary_path": str(CELL11_SUMMARY_PATH),
        "cell11_summary_sha256": sha256_file(CELL11_SUMMARY_PATH),
        "notebook4_handoff_path": str(HANDOFF_PATH),
        "notebook4_handoff_sha256": sha256_file(HANDOFF_PATH),
    },
    "outputs": {
        "annual_loss_series": {
            "path": str(ANNUAL_SERIES_PATH),
            "sha256": sha256_file(ANNUAL_SERIES_PATH),
            "rows": declared_catalog_years,
        },
        "exceedance_probability_curve": {
            "path": str(EP_CURVE_PATH),
            "sha256": sha256_file(EP_CURVE_PATH),
            "rows": int(len(ep_curve)),
        },
        "pml_table": {
            "path": str(PML_TABLE_PATH),
            "sha256": sha256_file(PML_TABLE_PATH),
            "rows": int(len(pml_table)),
        },
        "risk_metrics": {
            "path": str(RISK_METRICS_PATH),
            "sha256": sha256_file(RISK_METRICS_PATH),
            "rows": int(len(risk_metrics)),
        },
        "source_aal_summary": {
            "path": str(SOURCE_AAL_PATH),
            "sha256": sha256_file(SOURCE_AAL_PATH),
            "rows": int(len(source_aal)),
        },
        "component_aal_summary": {
            "path": str(COMPONENT_AAL_PATH),
            "sha256": sha256_file(COMPONENT_AAL_PATH),
            "rows": int(len(component_aal)),
        },
        "plot_png": str(PLOT_PNG_PATH),
        "plot_pdf": str(PLOT_PDF_PATH),
        "validation": str(CELL12_VALIDATION_PATH),
        "summary": str(CELL12_SUMMARY_PATH),
    },
    "next_notebook": (
        "Notebook 6: assign insurance policy terms and reinsurance layers, then "
        "calculate gross insured, retained, and ceded annual loss metrics."
    ),
}
write_json(CELL12_SUMMARY_PATH, summary)

print()
print("=" * 78)
print("NOTEBOOK 5 CELL 12 TOTAL GROUND-UP RISK METRICS COMPLETE")
print("=" * 78)
print(f"Declared catalog years:            {declared_catalog_years:,}")
print(f"Catalog occurrences:               {expected_occurrences:,}")
print(f"Occupied years:                    {occupied_years:,}")
print(f"Multiple-event years:              {multiple_event_years:,}")
print(f"Zero-event years:                  {zero_event_years:,}")
print(f"Sampled total ground-up AAL:       ${sampled_aal:,.2f}")
print(f"Analytical expected AAL:           ${expected_aal:,.2f}")
print(f"Maximum sampled AEP loss:          ${sampled_aep.max():,.0f}")
print(f"Maximum sampled OEP loss:          ${sampled_oep.max():,.0f}")
print(
    "Maximum sampled AEP loss ratio:    "
    f"{sampled_aep.max() / portfolio_replacement_value:.4%}"
)
print(f"Critical validation checks:        {int(validation['severity'].eq('critical').sum())}")
print(f"Critical failures:                 {len(critical_failures)}")
print(f"Warnings requiring review:         {len(warning_failures)}")
print()
print("Annual loss series:")
print(f"  {ANNUAL_SERIES_PATH}")
print("PML table:")
print(f"  {PML_TABLE_PATH}")
print("Exceedance curve:")
print(f"  {EP_CURVE_PATH}")
print("Component AAL summary:")
print(f"  {COMPONENT_AAL_PATH}")
print("Validation:")
print(f"  {CELL12_VALIDATION_PATH}")
print("Summary:")
print(f"  {CELL12_SUMMARY_PATH}")
print()
print(
    "Next: begin Notebook 6 and apply insurance policy and reinsurance terms "
    "to the completed total ground-up loss results."
)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 5 Cell 12 failed one or more critical checks. Review: "
        f"{CELL12_VALIDATION_PATH}"
    )



NOTEBOOK 5 CELL 12 TOTAL GROUND-UP RISK METRICS COMPLETE
Declared catalog years:            2,000,000
Catalog occurrences:               10,630
Occupied years:                    10,593
Multiple-event years:              36
Zero-event years:                  1,989,407
Sampled total ground-up AAL:       $195,922.45
Analytical expected AAL:           $196,250.58
Maximum sampled AEP loss:          $282,749,468
Maximum sampled OEP loss:          $282,749,468
Maximum sampled AEP loss ratio:    73.5873%
Critical validation checks:        43
Critical failures:                 0
Warnings requiring review:         1

Annual loss series:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_total_ground_up_risk\total_ground_up_annual_loss_series.csv.gz
PML table:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_5_total_ground_up_risk\total_ground_up_pml_table.csv
Exceedance curve:
  c:\Users\USER\Documents\GitHub

In [26]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", None)

PROJECT_ROOT = Path(
    r"."
)

validation_path = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_5_damage_loss"
    / "notebook_5_cell_12_validation.csv"
)

validation = pd.read_csv(validation_path)

warnings = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"].astype(bool)
]

display(warnings[["check_id", "severity", "passed", "detail"]])

,check_id,severity,passed,detail
43,extreme_return_period_tail_support_documented,warning,False,"thin_tail_return_periods=[200000, 500000, 1000000, 2000000]; threshold_rank=20"
